In [ ]:
# ============ Plan A · Paso 1 · Inventario de lo que tenemos (solo lectura) ============
# 1. Fuentes: existen, cuántas filas y columnas.
# 2. Columnas de la base final: familia, uso en el plan A, tipo, cobertura total y fuera del catálogo.
# 3. Columnas de fecha: rango y valores fuera de 2015-2026.
# 4. Guarda todo en 03_resultados/c15_inventario_columnas.csv
import os, glob
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
R   = '03_resultados'
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
pd.set_option('display.width', 220); pd.set_option('display.max_rows', 300)

# ---------- 1. Fuentes ----------
def buscar(patron):
    r = sorted(glob.glob(patron, recursive=True))
    return r[0] if r else None
casa = os.path.expanduser('~')
FUENTES = {
    'base final de procesos':        f'{R}/final_procesos.parquet',
    'catastro RUC del SRI':          f'{R}/sri_ruc_catastro.parquet',
    'directorio Superintendencia':  buscar(f'{R}/**/*supercias*.parquet'),
    'procesos solo planeación':     buscar('**/procesos_solo_planeacion*.parquet'),
    'procesos sin monto':           buscar('**/procesos_sin_monto*.parquet'),
    'consolidado OCDS':             '01_parquet/sercop_consolidado.parquet',
    'catastro fantasmas SRI (csv)': buscar(os.path.join(casa, 'Downloads', 'SRI_Catastro_Empresas_Fantasmas*.csv')),
    'directorio compañías (xlsx)':  buscar(os.path.join(casa, 'Downloads', 'directorio_companias*.xlsx')),
}
print('='*90 + '\n1. FUENTES\n' + '='*90)
filas_f = []
for nombre, ruta in FUENTES.items():
    if not ruta or not os.path.exists(ruta):
        filas_f.append((nombre, ruta or '(no encontrado)', False, None, None, None)); continue
    n = c = None
    try:
        if ruta.endswith('.parquet'):
            n = con.sql(f"SELECT count(*) FROM read_parquet('{ruta}')").fetchone()[0]
            c = len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{ruta}')").fetchall())
        elif ruta.endswith('.csv'):
            d = con.sql(f"SELECT * FROM read_csv_auto('{ruta}', ignore_errors=true)").df()
            n, c = len(d), d.shape[1]
    except Exception as e:
        try: con.rollback()
        except Exception: pass
        print(f'  aviso {nombre}: {type(e).__name__} {str(e)[:80]}')
    filas_f.append((nombre, ruta, True, n, c, round(os.path.getsize(ruta)/1e6, 1)))
print(pd.DataFrame(filas_f, columns=['fuente', 'ruta', 'existe', 'filas', 'columnas', 'mb']).to_string(index=False))

# ---------- 2. Columnas de la base final ----------
BASE = FUENTES['base final de procesos']
con.sql(f"CREATE OR REPLACE VIEW fp AS SELECT * FROM read_parquet('{BASE}')")
tipos = {r[0]: r[1] for r in con.sql("DESCRIBE fp").fetchall()}
N = con.sql("SELECT count(*) FROM fp").fetchone()[0]
NOCAT = ("NOT coalesce(CAST(es_catalogo AS BOOLEAN), false)" if 'es_catalogo' in tipos
         else "familia_procedimiento <> 'Catalogo electronico'")
ocid_unico = con.sql("SELECT count(DISTINCT ocid) FROM fp").fetchone()[0] == N
print('\n' + '='*90 + f'\n2. BASE FINAL: {N:,} procesos x {len(tipos)} columnas | ocid único: {ocid_unico}\n' + '='*90)

def familia_y_uso(c):
    l = c.lower()
    if l in ('ocid', 'proveedor_ruc', 'entidad_ruc', 'anio_proceso'):          return 'llave', 'llave'
    if l.startswith(('y_', 'f_marca', 'f_reactivacion')) or 'fantasma' in l or l == 'en_catastro':
        return 'etiqueta SRI', 'solo validación: nunca como variable'
    if l.startswith(('red_', 'cob_', 'com_', 'idx_')):                          return 'red', 'fuera de alcance (redes)'
    if l.startswith(('c_', 'k_')) or l.startswith('cri'):                        return 'CRI', 'comparación con la línea base'
    if l == 'monto_tender':                                                      return 'monto', 'solo fuera de subasta (presupuesto)'
    if 'monto' in l:                                                             return 'monto', 'candidata: precio'
    if 'fecha' in l or l.startswith('dt_'):                                      return 'fecha', 'candidata: plazos'
    if 'oferente' in l or 'unico' in l:                                          return 'competencia', 'candidata: concurrencia'
    if l.startswith(('sup_', 'capital', 'anio_constitucion', 'fecha_constitucion')) or 'supercia' in l:
        return 'proveedor (Supercias)', 'candidata: perfil del proveedor'
    if l.startswith('proveedor_'):                                               return 'proveedor', 'candidata: perfil del proveedor'
    if l.startswith(('ent_', 'entidad_')):                                       return 'entidad', 'candidata: contexto de la entidad'
    if l in ('familia_procedimiento', 'clase', 'es_catalogo', 'universo_competencia', 'tipo_contratacion'):
        return 'procedimiento', 'define grupos comparables'
    if 'cpc' in l:                                                               return 'objeto (CPC)', 'define grupos de precio'
    return 'otra', 'revisar'

cols = list(tipos)
partes = []
for i in range(0, len(cols), 60):                       # en bloques para no hacer una consulta gigante
    bloque = cols[i:i+60]
    sel = ', '.join(f'count("{c}") AS "t_{j}", count("{c}") FILTER (WHERE {NOCAT}) AS "n_{j}"'
                    for j, c in enumerate(bloque))
    fila = con.sql(f"SELECT count(*) FILTER (WHERE {NOCAT}) AS nn, {sel} FROM fp").df().iloc[0]
    for j, c in enumerate(bloque):
        partes.append({'columna': c, 'tipo': tipos[c],
                       'cobertura_total_%': round(100 * fila[f't_{j}'] / N, 1),
                       'cobertura_sin_catalogo_%': round(100 * fila[f'n_{j}'] / max(fila['nn'], 1), 1)})
INV = pd.DataFrame(partes)
INV[['familia', 'uso_plan_A']] = INV.columna.apply(lambda c: pd.Series(familia_y_uso(c)))
INV = INV[['familia', 'columna', 'tipo', 'uso_plan_A', 'cobertura_total_%', 'cobertura_sin_catalogo_%']]

print('\nResumen por uso en el plan A:')
print(INV.groupby('uso_plan_A').agg(columnas=('columna', 'size'),
      cobertura_media_sin_cat=('cobertura_sin_catalogo_%', 'mean')).round(1)
      .sort_values('columnas', ascending=False).to_string())
for uso in ['candidata: precio', 'candidata: plazos', 'candidata: concurrencia',
            'candidata: perfil del proveedor', 'candidata: contexto de la entidad',
            'define grupos comparables', 'define grupos de precio', 'revisar']:
    t = INV[INV.uso_plan_A == uso]
    if len(t):
        print(f'\n--- {uso} ({len(t)}) ---')
        print(t[['columna', 'tipo', 'cobertura_total_%', 'cobertura_sin_catalogo_%']].to_string(index=False))

# ---------- 3. Fechas ----------
fechas = [c for c, t in tipos.items() if ('DATE' in t.upper() or 'TIMESTAMP' in t.upper())]
if fechas:
    print('\n' + '='*90 + '\n3. COLUMNAS DE FECHA\n' + '='*90)
    filas_d = []
    for c in fechas:
        r = con.sql(f"""SELECT min("{c}"), max("{c}"),
                        count(*) FILTER (WHERE "{c}" IS NOT NULL AND (year("{c}") < 2015 OR year("{c}") > 2026))
                        FROM fp""").fetchone()
        filas_d.append((c, str(r[0])[:19], str(r[1])[:19], r[2]))
    print(pd.DataFrame(filas_d, columns=['columna', 'mínimo', 'máximo', 'fuera_2015_2026']).to_string(index=False))

INV.to_csv(f'{R}/c15_inventario_columnas.csv', index=False)
print(f'\n  -> {R}/c15_inventario_columnas.csv')
print('Pega aquí toda la salida.')

In [ ]:
# ============ Plan A · Paso 2 · Extracción complementaria desde el JSON (unos 10 a 12 minutos) ============
# Trae SOLO lo que la base final no tiene y la parte C mostró que existe:
#   fechas de adjudicación y de firma, período del contrato y de adjudicación, fin de preguntas,
#   precio unitario y cantidad del ítem principal, y las ofertas (bids) de menor cuantía.
# Guarda 03_resultados/c15_complemento_json.parquet (una fila por ocid) y lo comprueba contra la base.
import os, glob, time
import pandas as pd, duckdb
try:
    import ijson
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ijson']); import ijson

os.chdir(r'C:\SERCOP_FINAL')
R = '03_resultados'
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")

def prefijo_de(ruta):
    with open(ruta, 'rb') as fh:
        while True:
            c = fh.read(1)
            if not c: return 'releases.item'
            if not c.isspace():
                return 'item.releases.item' if c == b'[' else 'releases.item'

def lista(v):
    if isinstance(v, list): return [x for x in v if isinstance(x, dict)]
    if isinstance(v, dict) and v and all(str(k).isdigit() for k in v):
        return [x for x in v.values() if isinstance(x, dict)]
    return []

def g(d, *ruta):
    for k in ruta:
        if not isinstance(d, dict): return None
        d = d.get(k)
    return d

def num(v):
    try: return float(v)
    except (TypeError, ValueError): return None

def fila(rel):
    t   = rel.get('tender') if isinstance(rel.get('tender'), dict) else {}
    aws = lista(rel.get('awards')); cts = lista(rel.get('contracts'))
    # ítem principal = el de mayor valor (precio unitario x cantidad), como en el cuaderno 6
    items = lista(t.get('items')) or [i for a in aws for i in lista(a.get('items'))]
    mejor, mejor_v = None, -1.0
    for it in items:
        pu, q = num(g(it, 'unit', 'value', 'amount')), num(it.get('quantity'))
        v = (pu or 0) * (q if q is not None else 1)
        if v > mejor_v: mejor, mejor_v = it, v
    bids = rel.get('bids')
    det  = lista(g(bids, 'details')) if isinstance(bids, dict) else lista(bids)
    vals = [x for x in (num(g(b, 'value', 'amount')) for b in det) if x is not None]
    fechas_adj   = [a.get('date') for a in aws if a.get('date')]
    fechas_firma = [c.get('dateSigned') for c in cts if c.get('dateSigned')]
    return {
        'ocid': rel.get('ocid'),
        'j_fecha_adjudicacion': min(fechas_adj) if fechas_adj else None,
        'j_fecha_firma':        min(fechas_firma) if fechas_firma else None,
        'j_contrato_inicio':    g(cts[0], 'period', 'startDate') if cts else None,
        'j_contrato_fin':       g(cts[0], 'period', 'endDate') if cts else None,
        'j_periodo_adj_inicio': g(t, 'awardPeriod', 'startDate'),
        'j_periodo_adj_fin':    g(t, 'awardPeriod', 'endDate'),
        'j_fin_preguntas':      g(t, 'enquiryPeriod', 'endDate'),
        'j_item_cpc':           str(g(mejor, 'classification', 'id')) if g(mejor, 'classification', 'id') else None,
        'j_item_cantidad':      num(mejor.get('quantity')) if mejor else None,
        'j_item_precio_unit':   num(g(mejor, 'unit', 'value', 'amount')) if mejor else None,
        'j_item_unidad':        g(mejor, 'unit', 'name') if mejor else None,
        'j_n_items':            len(items),
        'j_n_ofertas':          len(det) if det else None,
        'j_oferta_min':         min(vals) if vals else None,
        'j_oferta_max':         max(vals) if vals else None,
    }

partes = []
for ruta in sorted(glob.glob('00_json_crudo/releases_*.json')):
    if ruta.endswith('releases_2024.json') and os.path.exists(ruta.replace('.json', '_reparado.json')):
        continue
    t0, filas = time.time(), []
    with open(ruta, 'rb') as fh:
        for rel in ijson.items(fh, prefijo_de(ruta)):
            if isinstance(rel, dict): filas.append(fila(rel))
    partes.append(pd.DataFrame(filas)); del filas
    print(f'  {os.path.basename(ruta)}: {len(partes[-1]):,} procesos ({time.time()-t0:.0f} s)')

J = pd.concat(partes, ignore_index=True); del partes
for c in [c for c in J.columns if c.startswith(('j_fecha', 'j_contrato', 'j_periodo', 'j_fin'))]:
    J[c] = pd.to_datetime(J[c], errors='coerce', utc=True).dt.tz_localize(None)
dup = J.ocid.duplicated().sum()
print(f'\nextraídos: {len(J):,} filas | ocid repetidos: {dup}')
if dup:
    J = J.drop_duplicates('ocid', keep='first'); print('  se dejó la primera aparición de cada ocid')
J.to_parquet(f'{R}/c15_complemento_json.parquet', index=False)
print(f'  -> {R}/c15_complemento_json.parquet')

# ---------- Comprobación contra la base final ----------
con.sql(f"CREATE OR REPLACE VIEW fp AS SELECT * FROM read_parquet('{R}/final_procesos.parquet')")
con.sql(f"CREATE OR REPLACE VIEW j AS SELECT * FROM read_parquet('{R}/c15_complemento_json.parquet')")
n_fp, n_union = con.sql("""SELECT (SELECT count(*) FROM fp),
                                  (SELECT count(*) FROM fp LEFT JOIN j USING (ocid))""").fetchone()
sin_par = con.sql("SELECT count(*) FROM fp ANTI JOIN j USING (ocid)").fetchone()[0]
print(f'\nbase: {n_fp:,} | tras la unión: {n_union:,} ({"no duplica" if n_fp == n_union else "DUPLICA, revisar"}) '
      f'| procesos de la base sin fila en el JSON: {sin_par:,}')

campos = ['j_fecha_adjudicacion', 'j_fecha_firma', 'j_contrato_fin', 'j_periodo_adj_fin', 'j_fin_preguntas',
          'j_item_cpc', 'j_item_precio_unit', 'j_item_cantidad', 'j_n_ofertas']
sel = ', '.join(f'round(100.0*count({c})/count(*), 1) AS {c}' for c in campos)
print('\n% con dato, por familia (sin catálogo):')
print(con.sql(f"""SELECT familia_procedimiento AS familia, count(*) AS procesos, {sel}
                  FROM fp LEFT JOIN j USING (ocid)
                  WHERE NOT coalesce(CAST(es_catalogo AS BOOLEAN), false)
                  GROUP BY 1 ORDER BY procesos DESC""").df().to_string(index=False))

print('\nPlazos en días, mediana por familia (sin catálogo), y casos con fechas al revés:')
print(con.sql("""
  SELECT familia_procedimiento AS familia,
         median(date_diff('day', fecha_inicio_tender, j_fecha_adjudicacion)) AS inicio_a_adjudicacion,
         median(date_diff('day', j_fecha_adjudicacion, j_fecha_firma))       AS adjudicacion_a_firma,
         median(date_diff('day', j_fecha_firma, j_contrato_fin))             AS duracion_contrato,
         count(*) FILTER (WHERE j_fecha_adjudicacion < fecha_inicio_tender)  AS adj_antes_de_inicio,
         count(*) FILTER (WHERE j_fecha_firma < j_fecha_adjudicacion)        AS firma_antes_de_adj
  FROM fp LEFT JOIN j USING (ocid)
  WHERE NOT coalesce(CAST(es_catalogo AS BOOLEAN), false)
  GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('\nPega aquí toda la salida.')

In [ ]:
# %%
# ===== CELDA 1 · Carpeta plan_A, ubicar archivos y construir el universo con segmentos =====
import duckdb, hashlib, re, time
import pandas as pd
from pathlib import Path

RAIZ   = Path(r"C:\SERCOP_FINAL")
PLAN_A = RAIZ / "plan_A"          # carpeta de trabajo del plan A (todo lo nuevo va aquí)
TABLAS = PLAN_A / "tablas"        # resultados en CSV
TABLAS.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
tmp = (PLAN_A / "tmp_duckdb"); tmp.mkdir(exist_ok=True)
con.execute(f"SET temp_directory='{tmp.as_posix()}'")

# ---- ayudas pequeñas (no hace falta entenderlas, solo ejecutarlas) ----
def buscar(*nombres):
    """Busca el archivo por nombre dentro de C:\\SERCOP_FINAL y usa el más reciente."""
    for nom in nombres:
        h = [p for p in RAIZ.rglob(nom) if "plan_A" not in p.parts]
        if h:
            h.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            if len(h) > 1:
                print(f"  ojo: {nom} aparece {len(h)} veces; uso el más reciente: {h[0]}")
            return h[0]
    raise FileNotFoundError(f"No encontré {nombres} dentro de {RAIZ}")

def q(sql):      return con.execute(sql).df()
def sp(p):       return "'" + Path(p).as_posix().replace("'", "''") + "'"
def qc(c):       return '"' + c.replace('"', '""') + '"'
def columnas(p): return q(f"DESCRIBE SELECT * FROM read_parquet({sp(p)})")["column_name"].tolist()
def elegir(cands, cols):
    bajo = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in bajo: return bajo[c.lower()]
    return None

CIFRAS = {}   # cada cifra queda anotada con su universo para el registro de cifras
def cifra(nombre, valor, universo, celda):
    CIFRAS[nombre] = (valor, universo, celda)
    pd.DataFrame([(k,) + v for k, v in CIFRAS.items()],
                 columns=["cifra", "valor", "universo", "celda"]
                 ).to_csv(TABLAS / "planA_cifras_para_registro.csv", index=False, encoding="utf-8-sig")

def informe_nulos(ruta, etiqueta, donde="TRUE"):
    """Por columna: cuántos nulos y cuántos 'vacíos' ('' , [] , {}). Una consulta por columna."""
    desc = q(f"DESCRIBE SELECT * FROM read_parquet({sp(ruta)})")
    filas = []
    for c, t in zip(desc["column_name"], desc["column_type"]):
        r = q(f"""SELECT count(*) n,
                  count(*) FILTER (WHERE {qc(c)} IS NULL) nulos,
                  count(*) FILTER (WHERE {qc(c)} IS NOT NULL
                       AND trim(CAST({qc(c)} AS VARCHAR)) IN ('','[]','{{}}')) vacios
                  FROM read_parquet({sp(ruta)}) WHERE {donde}""").iloc[0]
        filas.append((c, t, int(r.n), int(r.nulos), int(r.vacios)))
    d = pd.DataFrame(filas, columns=["columna", "tipo", "filas", "nulos", "vacios"])
    d["pct_nulo"]  = (100 * d.nulos  / d.filas).round(2)
    d["pct_vacio"] = (100 * d.vacios / d.filas).round(2)
    d["pct_sin_dato"] = (d.pct_nulo + d.pct_vacio).round(2)
    d = d.sort_values("pct_sin_dato", ascending=False)
    d.to_csv(TABLAS / f"planA_nulos_{etiqueta}.csv", index=False, encoding="utf-8-sig")
    return d

# ---- ubicar los archivos que ya tienes ----
F  = buscar("final_procesos.parquet", "procesos.parquet")
SP = buscar("procesos_solo_planeacion.parquet")
SM = buscar("procesos_sin_monto.parquet")
print("base de procesos     :", F)
print("solo planeación      :", SP)
print("sin monto            :", SM)
cF, cSP, cSM = columnas(F), columnas(SP), columnas(SM)

# ---- detectar nombres de columnas (si alguno sale None, avísame y lo ajusto) ----
c_ocid  = elegir(["ocid"], cF)
c_cat   = elegir(["es_catalogo"], cF)
c_fam   = elegir(["familia_procedimiento", "familia", "procedimiento_familia", "familia_proc"], cF)
c_monto = elegir(["monto_final"], cF)
c_ruc   = elegir(["proveedor_ruc"], cF)
c_anio  = elegir(["anio_proceso"], cF)
c_f1    = elegir(["fecha_inicio_tender"], cF)
c_f2    = elegir(["fecha_release"], cF)
c_etp   = elegir(["etapas"], cF)
c_ent   = elegir(["entidad_ruc", "comprador_ruc", "buyer_ruc"], cF)
c_entn  = elegir(["entidad_nombre", "comprador_nombre", "buyer_name", "entidad"], cF)
mapa = dict(ocid=c_ocid, es_catalogo=c_cat, familia=c_fam, monto=c_monto, proveedor_ruc=c_ruc,
            anio=c_anio, fecha_tender=c_f1, fecha_release=c_f2, etapas=c_etp,
            entidad_ruc=c_ent, entidad_nombre=c_entn)
print("\ncolumnas detectadas:"); [print(f"  {k:15s} -> {v}") for k, v in mapa.items()]
if None in (c_ocid, c_cat, c_monto, c_ruc):
    print("\nFALTA una columna clave. Columnas parecidas:")
    print([c for c in cF if re.search("cat|famil|metodo|monto|ruc", c, re.I)])
    raise SystemExit("Pégame esa lista y ajusto la celda.")

def col(c, tipo="VARCHAR"):  return f"f.{qc(c)}" if c else f"CAST(NULL AS {tipo})"

fecha_ref = "COALESCE(" + ", ".join(f"TRY_CAST(f.{qc(c)} AS DATE)" for c in (c_f1, c_f2) if c) + ")"
anio_exp  = (f"CAST(f.{qc(c_anio)} AS INTEGER)" if c_anio else f"CAST(year({fecha_ref}) AS INTEGER)")
periodo   = (f"CASE WHEN {fecha_ref} IS NULL THEN 'sin_fecha' "
             f"WHEN {fecha_ref} >= DATE '2025-10-07' THEN 'post_reforma_2025-10-07' "
             f"ELSE 'pre_reforma' END")

# ---- parte 1: procesos de la base (2.736.387) ----
base_sql = f"""
SELECT f.*,
  CASE WHEN COALESCE(CAST({col(c_cat)} AS BOOLEAN), false) THEN 'S2_catalogo'
       WHEN {col(c_ruc)} IS NOT NULL AND trim(CAST({col(c_ruc)} AS VARCHAR)) <> ''
            AND COALESCE({col(c_monto,'DOUBLE')}, 0) > 0 THEN 'S3_modelo_sin_catalogo'
       ELSE 'S4_sin_catalogo_sin_proveedor_o_monto' END AS planA_segmento,
  {col(c_fam)}  AS planA_familia,
  {anio_exp}    AS planA_anio,
  {periodo}     AS planA_periodo_normativo,
  'base'        AS planA_origen
FROM read_parquet({sp(F)}) f"""

# ---- parte 2: los que quedaron fuera de la base (solo planeación, sin monto) ----
def excl_sql(ruta, cols, seg, origen):
    o  = elegir(["ocid"], cols)
    an = elegir(["anio_proceso"], cols)
    fe = elegir(["fecha_inicio_tender", "fecha_release"], cols)
    fa = elegir(["familia_procedimiento", "familia", "procedimiento_familia", "familia_proc"], cols)
    if an:   a = f"CAST({qc(an)} AS INTEGER)"
    elif fe: a = f"CAST(year(TRY_CAST({qc(fe)} AS DATE)) AS INTEGER)"
    else:    a = "TRY_CAST(regexp_extract(CAST(" + qc(o) + " AS VARCHAR), '(20[0-3][0-9])', 1) AS INTEGER)"
    fam = qc(fa) if fa else "CAST(NULL AS VARCHAR)"
    return (f"SELECT CAST({qc(o)} AS VARCHAR) AS ocid, '{seg}' AS planA_segmento, "
            f"CAST({fam} AS VARCHAR) AS planA_familia, {a} AS planA_anio, "
            f"'sin_fecha' AS planA_periodo_normativo, '{origen}' AS planA_origen "
            f"FROM read_parquet({sp(ruta)})"), (an or fe or "ocid (extraído del código; menos fiable)")

s0, f0 = excl_sql(SP, cSP, "S0_solo_planeacion", "procesos_solo_planeacion")
s1, f1 = excl_sql(SM, cSM, "S1_sin_monto", "procesos_sin_monto")
print(f"\naño de los solo-planeación sale de: {f0}\naño de los sin-monto sale de: {f1}")

U = PLAN_A / "planA_universo_total.parquet"
t0 = time.time()
con.execute(f"""COPY (
  SELECT *, (planA_anio = 2026) AS planA_piloto_2026
  FROM ( {base_sql} UNION ALL BY NAME {s0} UNION ALL BY NAME {s1} )
) TO {sp(U)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f"\nUniverso escrito en {U.name} ({time.time()-t0:.0f} s)")

seg = q(f"""SELECT planA_segmento segmento, count(*) procesos,
            round(100.0*count(*)/sum(count(*)) OVER (),2) pct
            FROM read_parquet({sp(U)}) GROUP BY 1 ORDER BY 1""")
print(seg.to_string(index=False))
tot = int(seg.procesos.sum())
print(f"\nTOTAL = {tot:,}  (esperado 2.785.355) ->", "COINCIDE" if tot == 2785355 else "NO COINCIDE: revisar")
dup = q(f"SELECT count(*)-count(DISTINCT ocid) d FROM read_parquet({sp(U)})").d[0]
print(f"ocid repetidos en el universo: {dup}  (debe ser 0)")
for _, r in seg.iterrows():
    cifra(f"procesos_{r.segmento}", int(r.procesos), f"universo total {tot:,}", "celda1")
cifra("universo_total", tot, "12 JSON, 2015-2026", "celda1")

# ---- archivos por subconjunto (se COPIAN; los originales no se tocan) ----
def exportar(nombre, donde):
    ruta = PLAN_A / nombre
    con.execute(f"COPY (SELECT * FROM read_parquet({sp(U)}) WHERE {donde}) TO {sp(ruta)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    n = q(f"SELECT count(*) n FROM read_parquet({sp(ruta)})").n[0]
    print(f"  {nombre:46s} {n:>10,} filas")
exportar("planA_P1_modelo_sin_catalogo.parquet",   "planA_segmento='S3_modelo_sin_catalogo'")
exportar("planA_P0_catalogo.parquet",              "planA_segmento='S2_catalogo'")
exportar("planA_excluidos_solo_planeacion.parquet","planA_segmento='S0_solo_planeacion'")
exportar("planA_excluidos_sin_monto.parquet",      "planA_segmento='S1_sin_monto'")
exportar("planA_sin_catalogo_sin_proveedor_o_monto.parquet", "planA_segmento='S4_sin_catalogo_sin_proveedor_o_monto'")
P1 = PLAN_A / "planA_P1_modelo_sin_catalogo.parquet"

# %%
# ===== CELDA 2 · Revisión de nulos =====
# (a) todas las columnas del universo de modelado P1; (b) columnas clave por segmento
nul = informe_nulos(P1, "P1_modelo")
print("Columnas de P1 con algún dato faltante (nulo o vacío), de mayor a menor:\n")
con_falta = nul[nul.pct_sin_dato > 0]
print(con_falta[["columna", "tipo", "nulos", "vacios", "pct_sin_dato"]].head(60).to_string(index=False))
print(f"\nColumnas de P1: {len(nul)} | con algún faltante: {len(con_falta)} | "
      f"con más de 50 % faltante: {(nul.pct_sin_dato > 50).sum()} | 100 % vacías: {(nul.pct_sin_dato >= 100).sum()}")
cifra("P1_columnas_total", len(nul), "P1 sin catálogo", "celda2")
cifra("P1_columnas_con_faltantes", len(con_falta), "P1 sin catálogo", "celda2")
cifra("P1_columnas_mas_50pct_faltante", int((nul.pct_sin_dato > 50).sum()), "P1 sin catálogo", "celda2")

claves = [c for c in [c_ocid, c_ruc, c_monto, c_fam, c_anio, c_f1, c_f2, c_etp, c_ent, c_entn,
                      "planA_familia", "planA_anio"] if c]
claves = list(dict.fromkeys(claves))
sel = ", ".join(f"round(100.0*count(*) FILTER (WHERE {qc(c)} IS NULL OR trim(CAST({qc(c)} AS VARCHAR)) IN ('','[]'))/count(*),1) AS {qc(c)}"
                for c in claves)
por_seg = q(f"SELECT planA_segmento AS segmento, count(*) AS filas, {sel} FROM read_parquet({sp(U)}) GROUP BY 1 ORDER BY 1")
por_seg.to_csv(TABLAS / "planA_nulos_por_segmento.csv", index=False, encoding="utf-8-sig")
print("\n% sin dato por segmento (columnas clave). Un segmento sin esa columna sale 100 %: es esperable\n"
      "(p. ej. los solo-planeación no traen proveedor ni monto).\n")
print(por_seg.T.to_string(header=False))

# %%
# ===== CELDA 3 · Auditoría de unión por RUC con SRI (todas las provincias) y Superintendencia =====
CAT = buscar("sri_ruc_catastro.parquet")
SC  = buscar("c8_supercias.parquet")
cCAT, cSC = columnas(CAT), columnas(SC)
print("Catastro SRI :", len(cCAT), "columnas ->", cCAT)
print("Supercias    :", len(cSC),  "columnas ->", cSC)
kc = elegir(["ruc", "numero_ruc", "num_ruc"], cCAT) or next((c for c in cCAT if "ruc" in c.lower()), None)
ks = elegir(["ruc", "numero_ruc", "num_ruc"], cSC)  or next((c for c in cSC  if "ruc" in c.lower()), None)
print(f"\nclave SRI = {kc} | clave Supercias = {ks}")
if kc is None or ks is None: raise SystemExit("No encontré la columna RUC; pégame las columnas.")

def norm(c): return f"trim(CAST({qc(c)} AS VARCHAR))"

# 1) calidad de la clave en cada fuente
for nom, ruta, k in (("SRI", CAT, kc), ("Supercias", SC, ks)):
    r = q(f"""SELECT count(*) filas, count(DISTINCT {norm(k)}) distintos,
              count(*) FILTER (WHERE {qc(k)} IS NULL) nulos,
              count(*) FILTER (WHERE length({norm(k)}) <> 13) largo_no_13
              FROM read_parquet({sp(ruta)})""").iloc[0]
    print(f"\n{nom}: {int(r.filas):,} filas | {int(r.distintos):,} RUC distintos | "
          f"RUC repetidos (filas extra): {int(r.filas-r.distintos):,} | nulos: {int(r.nulos)} | largo distinto de 13: {int(r.largo_no_13):,}")
    cifra(f"{nom}_filas", int(r.filas), f"parquet {ruta.name}", "celda3")
    cifra(f"{nom}_ruc_distintos", int(r.distintos), f"parquet {ruta.name}", "celda3")

# 2) provincias: ¿están todas?
cprov = next((c for c in cCAT if "provincia" in c.lower()), None)
if cprov:
    pv = q(f"SELECT {qc(cprov)} provincia, count(*) filas FROM read_parquet({sp(CAT)}) GROUP BY 1 ORDER BY 2 DESC")
    print(f"\nProvincias en el parquet del SRI: {len(pv)}"); print(pv.to_string(index=False))
    cifra("SRI_provincias_en_parquet", len(pv), "sri_ruc_catastro.parquet", "celda3")
else:
    print("\n(el parquet del SRI no tiene columna de provincia; reviso los CSV)")
csvs = sorted(RAIZ.rglob("SRI_RUC_*.csv"))
print(f"\nCSV de SRI encontrados: {len(csvs)}")
CONTAR_CSV = True    # ponlo en False si tarda demasiado
if CONTAR_CSV and csvs:
    filas_csv = []
    for f in csvs:
        try:
            n = con.execute(f"SELECT count(*) FROM read_csv({sp(f)}, header=true, all_varchar=true, "
                            f"encoding='latin-1', ignore_errors=true)").fetchone()[0]
        except Exception as e:
            n = None; print("  no pude contar", f.name, "->", str(e)[:80])
        filas_csv.append((f.name, n))
    dc = pd.DataFrame(filas_csv, columns=["archivo", "filas_csv"])
    print(dc.to_string(index=False))
    print(f"\nSuma de filas en los CSV: {int(dc.filas_csv.fillna(0).sum()):,} (orientativo) | "
          f"filas en el parquet: {int(q(f'SELECT count(*) n FROM read_parquet({sp(CAT)})').n[0]):,}")
    dc.to_csv(TABLAS / "planA_sri_csv_por_provincia.csv", index=False, encoding="utf-8-sig")
    cifra("SRI_csv_archivos", len(csvs), "carpeta SRI", "celda3")
    cifra("SRI_csv_filas_suma", int(dc.filas_csv.fillna(0).sum()), "CSV SRI (conteo orientativo)", "celda3")

# 3) ¿cuántos RUC de proveedores y entidades de SERCOP aparecen en cada fuente?
prov = f"""SELECT DISTINCT planA_segmento seg, {norm(c_ruc)} ruc FROM read_parquet({sp(U)})
           WHERE {qc(c_ruc)} IS NOT NULL AND {norm(c_ruc)} <> ''"""
r = q(f"""WITH p AS ({prov}),
  c AS (SELECT DISTINCT {norm(kc)} k FROM read_parquet({sp(CAT)})),
  s AS (SELECT DISTINCT {norm(ks)} k FROM read_parquet({sp(SC)}))
  SELECT p.seg segmento, count(*) proveedores,
         count(c.k) en_sri, round(100.0*count(c.k)/count(*),2) pct_sri,
         count(s.k) en_supercias, round(100.0*count(s.k)/count(*),2) pct_supercias
  FROM p LEFT JOIN c ON c.k=p.ruc LEFT JOIN s ON s.k=p.ruc GROUP BY 1 ORDER BY 1""")
print("\nProveedores (RUC distintos) de SERCOP encontrados en cada fuente, por segmento:\n")
print(r.to_string(index=False))
r.to_csv(TABLAS / "planA_cobertura_ruc_por_segmento.csv", index=False, encoding="utf-8-sig")
for _, x in r.iterrows():
    cifra(f"prov_{x.segmento}_en_SRI_pct", float(x.pct_sri), f"{int(x.proveedores):,} RUC distintos del segmento", "celda3")
    cifra(f"prov_{x.segmento}_en_supercias_pct", float(x.pct_supercias), f"{int(x.proveedores):,} RUC distintos del segmento", "celda3")

if c_ent:
    e = q(f"""WITH p AS (SELECT DISTINCT {norm(c_ent)} ruc FROM read_parquet({sp(U)}) WHERE {qc(c_ent)} IS NOT NULL AND {norm(c_ent)}<>''),
      c AS (SELECT DISTINCT {norm(kc)} k FROM read_parquet({sp(CAT)}))
      SELECT count(*) entidades, count(c.k) en_sri, round(100.0*count(c.k)/count(*),2) pct FROM p LEFT JOIN c ON c.k=p.ruc""").iloc[0]
    print(f"\nEntidades compradoras (RUC distintos): {int(e.entidades):,} | en SRI: {int(e.en_sri):,} ({e.pct} %)")
    cifra("entidades_en_SRI_pct", float(e.pct), f"{int(e.entidades):,} RUC de entidad distintos", "celda3")

# 4) ¿la base de procesos YA venía unida? Comparo contra una unión nueva
c_fin_f = elegir(["proveedor_fecha_inicio_actividades"], cF)
c_fin_c = next((c for c in cCAT if "inicio" in c.lower()), None)
if c_fin_f and c_fin_c:
    z = q(f"""WITH a AS (SELECT {norm(c_ruc)} ruc, max({qc(c_fin_f)}) fin FROM read_parquet({sp(U)})
                        WHERE {qc(c_ruc)} IS NOT NULL AND {norm(c_ruc)}<>'' GROUP BY 1),
      c AS (SELECT {norm(kc)} k, max({qc(c_fin_c)}) fin_cat FROM read_parquet({sp(CAT)}) GROUP BY 1)
      SELECT count(*) proveedores,
        count(*) FILTER (WHERE c.k IS NOT NULL) en_catastro,
        count(*) FILTER (WHERE c.k IS NOT NULL AND a.fin IS NULL) en_catastro_pero_sin_fecha_en_base,
        count(*) FILTER (WHERE c.k IS NOT NULL AND c.fin_cat IS NULL) en_catastro_con_fecha_nula_en_origen
      FROM a LEFT JOIN c ON c.k=a.ruc""").iloc[0]
    print("\nRevisión de la unión que ya traía la base (fecha de inicio de actividades):")
    print(z.to_string())
    cifra("prov_en_catastro_pero_sin_fecha_inicio_en_base", int(z.en_catastro_pero_sin_fecha_en_base),
          f"{int(z.proveedores):,} proveedores", "celda3")
else:
    print("\n(no pude comparar fecha de inicio: falta la columna en la base o en el catastro)")

# 5) estados y fechas del catastro (RUC cerrado, suspensión...)
for c in [c for c in cCAT if re.search("estado|condicion|contribuyente|obligado", c, re.I)][:5]:
    print(f"\nValores de '{c}' (SRI):")
    print(q(f"SELECT {qc(c)} valor, count(*) n FROM read_parquet({sp(CAT)}) GROUP BY 1 ORDER BY 2 DESC LIMIT 8").to_string(index=False))
filas = []
for nom, ruta, cols in (("SRI", CAT, cCAT), ("Supercias", SC, cSC)):
    for c in [c for c in cols if "fecha" in c.lower()]:
        r = q(f"""SELECT count(*) n, count(*) FILTER (WHERE TRY_CAST({qc(c)} AS DATE) IS NULL) nulos,
                  min(TRY_CAST({qc(c)} AS DATE)) minimo, max(TRY_CAST({qc(c)} AS DATE)) maximo,
                  count(*) FILTER (WHERE TRY_CAST({qc(c)} AS DATE) < DATE '1900-01-01') antes_1900,
                  count(*) FILTER (WHERE TRY_CAST({qc(c)} AS DATE) > DATE '2026-09-05') despues_del_corte
                  FROM read_parquet({sp(ruta)})""").iloc[0]
        filas.append((nom, c, int(r.n), int(r.nulos), str(r.minimo), str(r.maximo), int(r.antes_1900), int(r.despues_del_corte)))
fe = pd.DataFrame(filas, columns=["fuente", "columna", "filas", "nulos_o_ilegibles", "minimo", "maximo", "antes_1900", "despues_5sep2026"])
fe.to_csv(TABLAS / "planA_fechas_calidad_sri_supercias.csv", index=False, encoding="utf-8-sig")
print("\nFechas en SRI / Supercias (mínimos absurdos = hallazgo de calidad, no error tuyo):\n")
print(fe.to_string(index=False))
for _, x in fe.iterrows():
    cifra(f"{x.fuente}_{x.columna}_antes_1900", int(x.antes_1900), f"{x.filas:,} filas", "celda3")

# 6) base consolidada: UNA fila por RUC de proveedor, con SRI y Supercias unidos por RUC
sel_c = ", ".join(f"c.{qc(c)} AS {qc('sri_'+c)}" for c in cCAT if c != kc)
sel_s = ", ".join(f"s.{qc(c)} AS {qc('sc_'+c)}" for c in cSC if c != ks)
PROV = PLAN_A / "planA_proveedores_ruc.parquet"
con.execute(f"""COPY (
 WITH p AS (
   SELECT {norm(c_ruc)} AS ruc, count(*) AS n_procesos,
          count(*) FILTER (WHERE planA_segmento='S3_modelo_sin_catalogo') AS n_procesos_P1,
          sum(COALESCE({qc(c_monto)},0)) AS monto_total,
          min(planA_anio) AS primer_anio, max(planA_anio) AS ultimo_anio
   FROM read_parquet({sp(U)}) WHERE {qc(c_ruc)} IS NOT NULL AND {norm(c_ruc)}<>'' GROUP BY 1),
 c AS (SELECT *, count(*) OVER (PARTITION BY {norm(kc)}) AS n_filas_ruc FROM read_parquet({sp(CAT)})
       QUALIFY row_number() OVER (PARTITION BY {norm(kc)} ORDER BY {norm(kc)}) = 1),
 s AS (SELECT * FROM read_parquet({sp(SC)}) QUALIFY row_number() OVER (PARTITION BY {norm(ks)} ORDER BY {norm(ks)}) = 1)
 SELECT p.*, (c.{qc(kc)} IS NOT NULL) AS en_sri, (s.{qc(ks)} IS NOT NULL) AS en_supercias,
        c.n_filas_ruc AS sri_n_filas_ruc, {sel_c}, {sel_s}
 FROM p LEFT JOIN c ON trim(CAST(c.{qc(kc)} AS VARCHAR)) = p.ruc
        LEFT JOIN s ON trim(CAST(s.{qc(ks)} AS VARCHAR)) = p.ruc
) TO {sp(PROV)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
nprov = q(f"SELECT count(*) n, count(*) FILTER (WHERE en_sri) a, count(*) FILTER (WHERE en_supercias) b FROM read_parquet({sp(PROV)})").iloc[0]
print(f"\n{PROV.name}: {int(nprov.n):,} proveedores | con SRI {int(nprov.a):,} | con Supercias {int(nprov.b):,}")
cifra("proveedores_consolidados", int(nprov.n), "RUC de proveedor distintos en el universo", "celda3")
nn = informe_nulos(PROV, "proveedores_ruc")
print("\nColumnas de la base consolidada con más datos faltantes:")
print(nn[["columna", "pct_sin_dato"]].head(25).to_string(index=False))
print("\nAviso de fuga: columnas sri_* de ESTADO ACTUAL describen hoy, no la fecha del proceso;")
print("no deben ser variables del modelo (solo las fechas históricas sirven para reglas temporales).")

# %%
# ===== CELDA 4 · Embudo por etapas, justificación estadística de descartar solo-planeación, muestra manual, manifiesto =====
tot = int(q(f"SELECT count(*) n FROM read_parquet({sp(U)})").n[0])

# 1) qué etapas existen y cuántos procesos las tienen (sin suponer nombres)
if c_etp:
    et = q(f"""SELECT s etapa, count(*) procesos FROM (SELECT unnest({qc(c_etp)}) s FROM read_parquet({sp(U)})
              WHERE {qc(c_etp)} IS NOT NULL) GROUP BY 1 ORDER BY 2 DESC""")
    print("Etapas registradas en la columna 'etapas' (procesos que la contienen):\n"); print(et.to_string(index=False))
    emb = q(f"""SELECT planA_segmento segmento,
         COALESCE(array_to_string(list_sort({qc(c_etp)}), ' + '), '(sin etapas)') combinacion, count(*) procesos
         FROM read_parquet({sp(U)}) GROUP BY 1,2 ORDER BY 1, 3 DESC""")
    emb["pct_del_segmento"] = (100 * emb.procesos / emb.groupby("segmento").procesos.transform("sum")).round(2)
    emb.to_csv(TABLAS / "planA_embudo_etapas.csv", index=False, encoding="utf-8-sig")
    print("\nCombinaciones de etapas por segmento (las 4 más frecuentes de cada uno):\n")
    print(emb.groupby("segmento").head(4).to_string(index=False))
    for _, x in et.iterrows():
        cifra(f"procesos_con_etapa_{x.etapa}", int(x.procesos), "procesos de la base (con columna etapas)", "celda4")

# 2) solo-planeación por año: ¿es un fenómeno reciente (procesos en curso) o estable (abandonados)?
ya = q(f"""SELECT planA_anio anio, count(*) procesos,
   count(*) FILTER (WHERE planA_segmento='S0_solo_planeacion') solo_planeacion,
   round(100.0*count(*) FILTER (WHERE planA_segmento='S0_solo_planeacion')/count(*),2) pct_solo_planeacion,
   count(*) FILTER (WHERE planA_segmento='S1_sin_monto') sin_monto
   FROM read_parquet({sp(U)}) GROUP BY 1 ORDER BY 1""")
ya.to_csv(TABLAS / "planA_solo_planeacion_por_anio.csv", index=False, encoding="utf-8-sig")
print("\nSolo planeación por año:\n"); print(ya.to_string(index=False))
n0 = int(ya.solo_planeacion.sum()); n1 = int(ya.sin_monto.sum())
p0 = 100 * n0 / tot
ep = (1.96 * (p0/100 * (1 - p0/100) / tot) ** 0.5) * 100
frase = (f"De {tot:,} procesos publicados entre 2015 y junio de 2026, {n0:,} ({p0:.2f} %, IC95 % ±{ep:.3f} puntos) "
         f"quedaron solo en planeación (sin llamado, adjudicación ni contrato) y {n1:,} ({100*n1/tot:.2f} %) "
         f"no tienen monto en ninguna fuente; ambos grupos se excluyen del modelo y se conservan como estadística descriptiva.")
print("\nFRASE PARA LA TESIS (borrador, depende de lo que digan los años):\n" + frase)
cifra("solo_planeacion_n", n0, f"{tot:,} procesos", "celda4")
cifra("solo_planeacion_pct", round(p0, 2), f"{tot:,} procesos", "celda4")
ant = ya[(ya.anio >= 2015) & (ya.anio <= 2025)]
if len(ant):
    print(f"\nTasa 2015-2025: mínimo {ant.pct_solo_planeacion.min()} % | máximo {ant.pct_solo_planeacion.max()} %. "
          f"Si es estable en los años cerrados, no son procesos 'en curso' sino que no avanzaron.")
    cifra("solo_planeacion_tasa_min_2015_2025", float(ant.pct_solo_planeacion.min()), "por año", "celda4")
    cifra("solo_planeacion_tasa_max_2015_2025", float(ant.pct_solo_planeacion.max()), "por año", "celda4")

# 3) ¿cuáles procesos quedan para modelar, por año y por familia?
fa = q(f"""SELECT planA_familia familia, planA_periodo_normativo periodo, count(*) procesos
           FROM read_parquet({sp(P1)}) GROUP BY 1,2 ORDER BY 1,2""")
fa.to_csv(TABLAS / "planA_P1_por_familia_y_periodo.csv", index=False, encoding="utf-8-sig")
print("\nUniverso de modelado P1 por familia y periodo normativo:\n"); print(fa.to_string(index=False))
peq = fa[fa.procesos < 30]
print(f"\nGrupos con menos de 30 procesos (usarán el grupo más amplio): {len(peq)}")

# 4) muestra para revisar a mano en el portal (reproducible)
cols_m = [c for c in ["ocid", "planA_segmento", "planA_anio", "planA_familia", c_entn, c_etp, c_monto] if c]
cols_m = list(dict.fromkeys(cols_m))
def sel_cols(c):
    return f"array_to_string({qc(c)}, ' + ') AS {qc(c)}" if c == c_etp else qc(c)
d = q(f"SELECT {', '.join(sel_cols(c) for c in cols_m)} FROM read_parquet({sp(U)})")
partes = []
s0 = d[d.planA_segmento == "S0_solo_planeacion"]
for a in sorted(s0.planA_anio.dropna().unique()):      # 2 por año para ver si el patrón cambia con el tiempo
    g = s0[s0.planA_anio == a]
    partes.append(g.sample(min(len(g), 2), random_state=42))
for sg, n in (("S3_modelo_sin_catalogo", 12), ("S2_catalogo", 5), ("S1_sin_monto", 5), ("S4_sin_catalogo_sin_proveedor_o_monto", 5)):
    x = d[d.planA_segmento == sg]
    if len(x): partes.append(x.sample(min(n, len(x)), random_state=42))
m = pd.concat(partes, ignore_index=True)
m.insert(1, "codigo_candidato", m.ocid.str.replace(r"^ocds-[a-z0-9]+-", "", regex=True))
for c in ("portal_etapa_observada", "portal_estado_observado", "coincide_con_nuestros_datos", "notas"):
    m[c] = ""
RUTA_M = TABLAS / "planA_muestra_revision_manual.csv"
m.to_csv(RUTA_M, index=False, encoding="utf-8-sig")
print(f"\nMuestra para revisar en el portal: {len(m)} procesos -> {RUTA_M}")
print(m[["ocid", "codigo_candidato", "planA_segmento", "planA_anio"]].head(15).to_string(index=False))
print("\nCómo revisarla: abre el CSV, copia 'codigo_candidato' en el buscador de procesos del portal de Compras")
print("Públicas, mira en qué etapa está (como en tu captura: Planeación activa y el resto en gris) y anota")
print("'portal_etapa_observada' y 'portal_estado_observado' (¿cancelado, desierto, en curso, nada?).")
print("Si el código no aparece, prueba con el ocid completo y dime cuál formato funcionó.")
cifra("muestra_revision_manual_n", len(m), "muestra aleatoria reproducible (semilla 42)", "celda4")

# 5) manifiesto de la carpeta plan_A
filas = []
for f in sorted(PLAN_A.glob("*.parquet")):
    n = q(f"SELECT count(*) n FROM read_parquet({sp(f)})").n[0]
    k = len(columnas(f))
    h = hashlib.sha256(f.read_bytes()).hexdigest()[:16] if f.stat().st_size < 3e9 else "(omitido)"
    filas.append((f.name, int(n), k, round(f.stat().st_size/1e6, 1), time.strftime("%Y-%m-%d %H:%M", time.localtime(f.stat().st_mtime)), h))
man = pd.DataFrame(filas, columns=["archivo", "filas", "columnas", "MB", "creado", "sha256_16"])
man.to_csv(PLAN_A / "planA_manifiesto.csv", index=False, encoding="utf-8-sig")
print("\nMANIFIESTO plan_A:\n"); print(man.to_string(index=False))
print(f"\nCifras listas para el registro: {TABLAS/'planA_cifras_para_registro.csv'}")

In [ ]:
# %%
# ===== CELDA 1 · Carpeta plan_A, ubicar archivos y construir el universo con segmentos =====
import duckdb, hashlib, re, time
import pandas as pd
from pathlib import Path

RAIZ   = Path(r"C:\SERCOP_FINAL")
PLAN_A = RAIZ / "plan_A"          # carpeta de trabajo del plan A (todo lo nuevo va aquí)
TABLAS = PLAN_A / "tablas"        # resultados en CSV
TABLAS.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
tmp = (PLAN_A / "tmp_duckdb"); tmp.mkdir(exist_ok=True)
con.execute(f"SET temp_directory='{tmp.as_posix()}'")

# ---- ayudas pequeñas (no hace falta entenderlas, solo ejecutarlas) ----
def buscar(*nombres):
    """Busca el archivo por nombre dentro de C:\\SERCOP_FINAL y usa el más reciente."""
    for nom in nombres:
        h = [p for p in RAIZ.rglob(nom) if "plan_A" not in p.parts]
        if h:
            h.sort(key=lambda p: p.stat().st_mtime, reverse=True)
            if len(h) > 1:
                print(f"  ojo: {nom} aparece {len(h)} veces; uso el más reciente: {h[0]}")
            return h[0]
    raise FileNotFoundError(f"No encontré {nombres} dentro de {RAIZ}")

def q(sql):      return con.execute(sql).df()
def sp(p):       return "'" + Path(p).as_posix().replace("'", "''") + "'"
def qc(c):       return '"' + c.replace('"', '""') + '"'
def columnas(p): return q(f"DESCRIBE SELECT * FROM read_parquet({sp(p)})")["column_name"].tolist()
def elegir(cands, cols):
    bajo = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in bajo: return bajo[c.lower()]
    return None

CIFRAS = {}   # cada cifra queda anotada con su universo para el registro de cifras
def cifra(nombre, valor, universo, celda):
    CIFRAS[nombre] = (valor, universo, celda)
    pd.DataFrame([(k,) + v for k, v in CIFRAS.items()],
                 columns=["cifra", "valor", "universo", "celda"]
                 ).to_csv(TABLAS / "planA_cifras_para_registro.csv", index=False, encoding="utf-8-sig")

def informe_nulos(ruta, etiqueta, donde="TRUE"):
    """Por columna: cuántos nulos y cuántos 'vacíos' ('' , [] , {}). Una consulta por columna."""
    desc = q(f"DESCRIBE SELECT * FROM read_parquet({sp(ruta)})")
    filas = []
    for c, t in zip(desc["column_name"], desc["column_type"]):
        r = q(f"""SELECT count(*) n,
                  count(*) FILTER (WHERE {qc(c)} IS NULL) nulos,
                  count(*) FILTER (WHERE {qc(c)} IS NOT NULL
                       AND trim(CAST({qc(c)} AS VARCHAR)) IN ('','[]','{{}}')) vacios
                  FROM read_parquet({sp(ruta)}) WHERE {donde}""").iloc[0]
        filas.append((c, t, int(r.n), int(r.nulos), int(r.vacios)))
    d = pd.DataFrame(filas, columns=["columna", "tipo", "filas", "nulos", "vacios"])
    d["pct_nulo"]  = (100 * d.nulos  / d.filas).round(2)
    d["pct_vacio"] = (100 * d.vacios / d.filas).round(2)
    d["pct_sin_dato"] = (d.pct_nulo + d.pct_vacio).round(2)
    d = d.sort_values("pct_sin_dato", ascending=False)
    d.to_csv(TABLAS / f"planA_nulos_{etiqueta}.csv", index=False, encoding="utf-8-sig")
    return d

# ---- ubicar los archivos que ya tienes ----
F  = buscar("final_procesos.parquet", "procesos.parquet")
SP = buscar("procesos_solo_planeacion.parquet")
SM = buscar("procesos_sin_monto.parquet")
print("base de procesos     :", F)
print("solo planeación      :", SP)
print("sin monto            :", SM)
cF, cSP, cSM = columnas(F), columnas(SP), columnas(SM)

# ---- detectar nombres de columnas (si alguno sale None, avísame y lo ajusto) ----
c_ocid  = elegir(["ocid"], cF)
c_cat   = elegir(["es_catalogo"], cF)
c_fam   = elegir(["familia_procedimiento", "familia", "procedimiento_familia", "familia_proc"], cF)
c_monto = elegir(["monto_final"], cF)
c_ruc   = elegir(["proveedor_ruc"], cF)
c_anio  = elegir(["anio_proceso"], cF)
c_f1    = elegir(["fecha_inicio_tender"], cF)
c_f2    = elegir(["fecha_release"], cF)
c_etp   = elegir(["etapas"], cF)
c_ent   = elegir(["entidad_ruc", "comprador_ruc", "buyer_ruc"], cF)
c_entn  = elegir(["entidad_nombre", "comprador_nombre", "buyer_name", "entidad"], cF)
mapa = dict(ocid=c_ocid, es_catalogo=c_cat, familia=c_fam, monto=c_monto, proveedor_ruc=c_ruc,
            anio=c_anio, fecha_tender=c_f1, fecha_release=c_f2, etapas=c_etp,
            entidad_ruc=c_ent, entidad_nombre=c_entn)
print("\ncolumnas detectadas:"); [print(f"  {k:15s} -> {v}") for k, v in mapa.items()]
if None in (c_ocid, c_cat, c_monto, c_ruc):
    print("\nFALTA una columna clave. Columnas parecidas:")
    print([c for c in cF if re.search("cat|famil|metodo|monto|ruc", c, re.I)])
    raise SystemExit("Pégame esa lista y ajusto la celda.")

def col(c, tipo="VARCHAR"):  return f"f.{qc(c)}" if c else f"CAST(NULL AS {tipo})"

fecha_ref = "COALESCE(" + ", ".join(f"TRY_CAST(f.{qc(c)} AS DATE)" for c in (c_f1, c_f2) if c) + ")"
anio_exp  = (f"CAST(f.{qc(c_anio)} AS INTEGER)" if c_anio else f"CAST(year({fecha_ref}) AS INTEGER)")
periodo   = (f"CASE WHEN {fecha_ref} IS NULL THEN 'sin_fecha' "
             f"WHEN {fecha_ref} >= DATE '2025-10-07' THEN 'post_reforma_2025-10-07' "
             f"ELSE 'pre_reforma' END")

def anio_codigo(expr):
    """Año que aparece dentro del código del proceso (p. ej. SIE-GADICC-028-2017-94877 -> 2017)."""
    return (f"TRY_CAST(NULLIF(regexp_extract(CAST({expr} AS VARCHAR), '-(20[0-3][0-9])(-|[0-9])', 1), '') AS INTEGER)")

# ---- parte 1: procesos de la base (2.736.387) ----
base_sql = f"""
SELECT f.*,
  CASE WHEN COALESCE(CAST({col(c_cat)} AS BOOLEAN), false) THEN 'S2_catalogo'
       WHEN {col(c_ruc)} IS NOT NULL AND trim(CAST({col(c_ruc)} AS VARCHAR)) <> ''
            AND COALESCE({col(c_monto,'DOUBLE')}, 0) > 0 THEN 'S3_modelo_sin_catalogo'
       ELSE 'S4_sin_catalogo_sin_proveedor_o_monto' END AS planA_segmento,
  {col(c_fam)}  AS planA_familia,
  {anio_exp}    AS planA_anio,
  'columna base (anio_proceso / fecha)' AS planA_anio_fuente,
  {anio_codigo('f.' + qc(c_ocid))}  AS planA_anio_codigo,
  {periodo}     AS planA_periodo_normativo,
  'base'        AS planA_origen
FROM read_parquet({sp(F)}) f"""

# ---- parte 2: los que quedaron fuera de la base (solo planeación, sin monto) ----
def excl_sql(ruta, cols, seg, origen):
    o  = elegir(["ocid"], cols)
    an = elegir(["anio_proceso"], cols)
    f1 = elegir(["fecha_inicio_tender"], cols)
    f2 = elegir(["fecha_release"], cols)
    fa = elegir(["familia_procedimiento", "familia", "procedimiento_familia", "familia_proc"], cols)
    cand = []                                   # (nombre de la fuente, expresión SQL del año)
    if an: cand.append((an, f"CAST({qc(an)} AS INTEGER)"))
    if f1: cand.append((f1, f"CAST(year(TRY_CAST({qc(f1)} AS DATE)) AS INTEGER)"))
    if f2: cand.append((f2, f"CAST(year(TRY_CAST({qc(f2)} AS DATE)) AS INTEGER)"))
    cand.append(("codigo_del_proceso", anio_codigo(qc(o))))
    a   = "COALESCE(" + ", ".join(e for _, e in cand) + ")"
    fte = "CASE " + " ".join(f"WHEN {e} IS NOT NULL THEN '{n}'" for n, e in cand) + " ELSE 'sin_año' END"
    fam = qc(fa) if fa else "CAST(NULL AS VARCHAR)"
    return (f"SELECT CAST({qc(o)} AS VARCHAR) AS ocid, '{seg}' AS planA_segmento, "
            f"CAST({fam} AS VARCHAR) AS planA_familia, {a} AS planA_anio, {fte} AS planA_anio_fuente, "
            f"{anio_codigo(qc(o))} AS planA_anio_codigo, "
            f"'sin_fecha' AS planA_periodo_normativo, '{origen}' AS planA_origen "
            f"FROM read_parquet({sp(ruta)})"), ", ".join(n for n, _ in cand)

s0, f0 = excl_sql(SP, cSP, "S0_solo_planeacion", "procesos_solo_planeacion")
s1, f1 = excl_sql(SM, cSM, "S1_sin_monto", "procesos_sin_monto")
print(f"\naño de los solo-planeación: se prueba en este orden -> {f0}\naño de los sin-monto: se prueba en este orden -> {f1}")

U = PLAN_A / "planA_universo_total.parquet"
t0 = time.time()
con.execute(f"""COPY (
  SELECT *, (planA_anio = 2026) AS planA_piloto_2026
  FROM ( {base_sql} UNION ALL BY NAME {s0} UNION ALL BY NAME {s1} )
) TO {sp(U)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f"\nUniverso escrito en {U.name} ({time.time()-t0:.0f} s)")

seg = q(f"""SELECT planA_segmento segmento, count(*) procesos,
            round(100.0*count(*)/sum(count(*)) OVER (),2) pct
            FROM read_parquet({sp(U)}) GROUP BY 1 ORDER BY 1""")
print(seg.to_string(index=False))
tot = int(seg.procesos.sum())
print(f"\nTOTAL = {tot:,}  (esperado 2.785.355) ->", "COINCIDE" if tot == 2785355 else "NO COINCIDE: revisar")
dup = q(f"SELECT count(*)-count(DISTINCT ocid) d FROM read_parquet({sp(U)})").d[0]
print(f"ocid repetidos en el universo: {dup}  (debe ser 0)")
for _, r in seg.iterrows():
    cifra(f"procesos_{r.segmento}", int(r.procesos), f"universo total {tot:,}", "celda1")
cifra("universo_total", tot, "12 JSON, 2015-2026", "celda1")

# ---- archivos por subconjunto (se COPIAN; los originales no se tocan) ----
def exportar(nombre, donde):
    ruta = PLAN_A / nombre
    con.execute(f"COPY (SELECT * FROM read_parquet({sp(U)}) WHERE {donde}) TO {sp(ruta)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    n = q(f"SELECT count(*) n FROM read_parquet({sp(ruta)})").n[0]
    print(f"  {nombre:46s} {n:>10,} filas")
exportar("planA_P1_modelo_sin_catalogo.parquet",   "planA_segmento='S3_modelo_sin_catalogo'")
exportar("planA_P0_catalogo.parquet",              "planA_segmento='S2_catalogo'")
exportar("planA_excluidos_solo_planeacion.parquet","planA_segmento='S0_solo_planeacion'")
exportar("planA_excluidos_sin_monto.parquet",      "planA_segmento='S1_sin_monto'")
exportar("planA_sin_catalogo_sin_proveedor_o_monto.parquet", "planA_segmento='S4_sin_catalogo_sin_proveedor_o_monto'")
P1 = PLAN_A / "planA_P1_modelo_sin_catalogo.parquet"

# %%
# ===== CELDA 2 · Revisión de nulos =====
# (a) todas las columnas del universo de modelado P1; (b) columnas clave por segmento
nul = informe_nulos(P1, "P1_modelo")
print("Columnas de P1 con algún dato faltante (nulo o vacío), de mayor a menor:\n")
con_falta = nul[nul.pct_sin_dato > 0]
print(con_falta[["columna", "tipo", "nulos", "vacios", "pct_sin_dato"]].head(60).to_string(index=False))
print(f"\nColumnas de P1: {len(nul)} | con algún faltante: {len(con_falta)} | "
      f"con más de 50 % faltante: {(nul.pct_sin_dato > 50).sum()} | 100 % vacías: {(nul.pct_sin_dato >= 100).sum()}")
cifra("P1_columnas_total", len(nul), "P1 sin catálogo", "celda2")
cifra("P1_columnas_con_faltantes", len(con_falta), "P1 sin catálogo", "celda2")
cifra("P1_columnas_mas_50pct_faltante", int((nul.pct_sin_dato > 50).sum()), "P1 sin catálogo", "celda2")

claves = [c for c in [c_ocid, c_ruc, c_monto, c_fam, c_anio, c_f1, c_f2, c_etp, c_ent, c_entn,
                      "planA_familia", "planA_anio"] if c]
claves = list(dict.fromkeys(claves))
sel = ", ".join(f"round(100.0*count(*) FILTER (WHERE {qc(c)} IS NULL OR trim(CAST({qc(c)} AS VARCHAR)) IN ('','[]'))/count(*),1) AS {qc(c)}"
                for c in claves)
por_seg = q(f"SELECT planA_segmento AS segmento, count(*) AS filas, {sel} FROM read_parquet({sp(U)}) GROUP BY 1 ORDER BY 1")
por_seg.to_csv(TABLAS / "planA_nulos_por_segmento.csv", index=False, encoding="utf-8-sig")
print("\n% sin dato por segmento (columnas clave). Un segmento sin esa columna sale 100 %: es esperable\n"
      "(p. ej. los solo-planeación no traen proveedor ni monto).\n")
print(por_seg.T.to_string(header=False))

# %%
# ===== CELDA 3 · Auditoría de unión por RUC con SRI (todas las provincias) y Superintendencia =====
CAT = buscar("sri_ruc_catastro.parquet")
SC  = buscar("c8_supercias.parquet")
cCAT, cSC = columnas(CAT), columnas(SC)
print("Catastro SRI :", len(cCAT), "columnas ->", cCAT)
print("Supercias    :", len(cSC),  "columnas ->", cSC)
kc = elegir(["ruc", "numero_ruc", "num_ruc"], cCAT) or next((c for c in cCAT if "ruc" in c.lower()), None)
ks = elegir(["ruc", "numero_ruc", "num_ruc"], cSC)  or next((c for c in cSC  if "ruc" in c.lower()), None)
print(f"\nclave SRI = {kc} | clave Supercias = {ks}")
if kc is None or ks is None: raise SystemExit("No encontré la columna RUC; pégame las columnas.")

def norm(c): return f"trim(CAST({qc(c)} AS VARCHAR))"

# 1) calidad de la clave en cada fuente
for nom, ruta, k in (("SRI", CAT, kc), ("Supercias", SC, ks)):
    r = q(f"""SELECT count(*) filas, count(DISTINCT {norm(k)}) distintos,
              count(*) FILTER (WHERE {qc(k)} IS NULL) nulos,
              count(*) FILTER (WHERE length({norm(k)}) <> 13) largo_no_13
              FROM read_parquet({sp(ruta)})""").iloc[0]
    print(f"\n{nom}: {int(r.filas):,} filas | {int(r.distintos):,} RUC distintos | "
          f"RUC repetidos (filas extra): {int(r.filas-r.distintos):,} | nulos: {int(r.nulos)} | largo distinto de 13: {int(r.largo_no_13):,}")
    cifra(f"{nom}_filas", int(r.filas), f"parquet {ruta.name}", "celda3")
    cifra(f"{nom}_ruc_distintos", int(r.distintos), f"parquet {ruta.name}", "celda3")

# 2) provincias: ¿están todas?
cprov = next((c for c in cCAT if "provincia" in c.lower()), None)
if cprov:
    pv = q(f"SELECT {qc(cprov)} provincia, count(*) filas FROM read_parquet({sp(CAT)}) GROUP BY 1 ORDER BY 2 DESC")
    print(f"\nProvincias en el parquet del SRI: {len(pv)}"); print(pv.to_string(index=False))
    cifra("SRI_provincias_en_parquet", len(pv), "sri_ruc_catastro.parquet", "celda3")
else:
    print("\n(el parquet del SRI no tiene columna de provincia; reviso los CSV)")
csvs = sorted(RAIZ.rglob("SRI_RUC_*.csv"))
print(f"\nCSV de SRI encontrados: {len(csvs)}")
CONTAR_CSV = True    # ponlo en False si tarda demasiado
if CONTAR_CSV and csvs:
    filas_csv = []
    for f in csvs:
        try:
            n = con.execute(f"SELECT count(*) FROM read_csv({sp(f)}, header=true, all_varchar=true, "
                            f"encoding='latin-1', ignore_errors=true)").fetchone()[0]
        except Exception as e:
            n = None; print("  no pude contar", f.name, "->", str(e)[:80])
        filas_csv.append((f.name, n))
    dc = pd.DataFrame(filas_csv, columns=["archivo", "filas_csv"])
    print(dc.to_string(index=False))
    print(f"\nSuma de filas en los CSV: {int(dc.filas_csv.fillna(0).sum()):,} (orientativo) | "
          f"filas en el parquet: {int(q(f'SELECT count(*) n FROM read_parquet({sp(CAT)})').n[0]):,}")
    dc.to_csv(TABLAS / "planA_sri_csv_por_provincia.csv", index=False, encoding="utf-8-sig")
    cifra("SRI_csv_archivos", len(csvs), "carpeta SRI", "celda3")
    cifra("SRI_csv_filas_suma", int(dc.filas_csv.fillna(0).sum()), "CSV SRI (conteo orientativo)", "celda3")

# 3) ¿cuántos RUC de proveedores y entidades de SERCOP aparecen en cada fuente?
prov = f"""SELECT DISTINCT planA_segmento seg, {norm(c_ruc)} ruc FROM read_parquet({sp(U)})
           WHERE {qc(c_ruc)} IS NOT NULL AND {norm(c_ruc)} <> ''"""
r = q(f"""WITH p AS ({prov}),
  c AS (SELECT DISTINCT {norm(kc)} k FROM read_parquet({sp(CAT)})),
  s AS (SELECT DISTINCT {norm(ks)} k FROM read_parquet({sp(SC)}))
  SELECT p.seg segmento, count(*) proveedores,
         count(c.k) en_sri, round(100.0*count(c.k)/count(*),2) pct_sri,
         count(s.k) en_supercias, round(100.0*count(s.k)/count(*),2) pct_supercias
  FROM p LEFT JOIN c ON c.k=p.ruc LEFT JOIN s ON s.k=p.ruc GROUP BY 1 ORDER BY 1""")
print("\nProveedores (RUC distintos) de SERCOP encontrados en cada fuente, por segmento:\n")
print(r.to_string(index=False))
r.to_csv(TABLAS / "planA_cobertura_ruc_por_segmento.csv", index=False, encoding="utf-8-sig")
for _, x in r.iterrows():
    cifra(f"prov_{x.segmento}_en_SRI_pct", float(x.pct_sri), f"{int(x.proveedores):,} RUC distintos del segmento", "celda3")
    cifra(f"prov_{x.segmento}_en_supercias_pct", float(x.pct_supercias), f"{int(x.proveedores):,} RUC distintos del segmento", "celda3")

if c_ent:
    e = q(f"""WITH p AS (SELECT DISTINCT {norm(c_ent)} ruc FROM read_parquet({sp(U)}) WHERE {qc(c_ent)} IS NOT NULL AND {norm(c_ent)}<>''),
      c AS (SELECT DISTINCT {norm(kc)} k FROM read_parquet({sp(CAT)}))
      SELECT count(*) entidades, count(c.k) en_sri, round(100.0*count(c.k)/count(*),2) pct FROM p LEFT JOIN c ON c.k=p.ruc""").iloc[0]
    print(f"\nEntidades compradoras (RUC distintos): {int(e.entidades):,} | en SRI: {int(e.en_sri):,} ({e.pct} %)")
    cifra("entidades_en_SRI_pct", float(e.pct), f"{int(e.entidades):,} RUC de entidad distintos", "celda3")

# 4) ¿la base de procesos YA venía unida? Comparo contra una unión nueva
c_fin_f = elegir(["proveedor_fecha_inicio_actividades"], cF)
c_fin_c = next((c for c in cCAT if "inicio" in c.lower()), None)
if c_fin_f and c_fin_c:
    z = q(f"""WITH a AS (SELECT {norm(c_ruc)} ruc, max({qc(c_fin_f)}) fin FROM read_parquet({sp(U)})
                        WHERE {qc(c_ruc)} IS NOT NULL AND {norm(c_ruc)}<>'' GROUP BY 1),
      c AS (SELECT {norm(kc)} k, max({qc(c_fin_c)}) fin_cat FROM read_parquet({sp(CAT)}) GROUP BY 1)
      SELECT count(*) proveedores,
        count(*) FILTER (WHERE c.k IS NOT NULL) en_catastro,
        count(*) FILTER (WHERE c.k IS NOT NULL AND a.fin IS NULL) en_catastro_pero_sin_fecha_en_base,
        count(*) FILTER (WHERE c.k IS NOT NULL AND c.fin_cat IS NULL) en_catastro_con_fecha_nula_en_origen
      FROM a LEFT JOIN c ON c.k=a.ruc""").iloc[0]
    print("\nRevisión de la unión que ya traía la base (fecha de inicio de actividades):")
    print(z.to_string())
    cifra("prov_en_catastro_pero_sin_fecha_inicio_en_base", int(z.en_catastro_pero_sin_fecha_en_base),
          f"{int(z.proveedores):,} proveedores", "celda3")
else:
    print("\n(no pude comparar fecha de inicio: falta la columna en la base o en el catastro)")

# 5) estados y fechas del catastro (RUC cerrado, suspensión...)
for c in [c for c in cCAT if re.search("estado|condicion|contribuyente|obligado", c, re.I)][:5]:
    print(f"\nValores de '{c}' (SRI):")
    print(q(f"SELECT {qc(c)} valor, count(*) n FROM read_parquet({sp(CAT)}) GROUP BY 1 ORDER BY 2 DESC LIMIT 8").to_string(index=False))
filas = []
for nom, ruta, cols in (("SRI", CAT, cCAT), ("Supercias", SC, cSC)):
    for c in [c for c in cols if "fecha" in c.lower()]:
        r = q(f"""SELECT count(*) n, count(*) FILTER (WHERE TRY_CAST({qc(c)} AS DATE) IS NULL) nulos,
                  min(TRY_CAST({qc(c)} AS DATE)) minimo, max(TRY_CAST({qc(c)} AS DATE)) maximo,
                  count(*) FILTER (WHERE TRY_CAST({qc(c)} AS DATE) < DATE '1900-01-01') antes_1900,
                  count(*) FILTER (WHERE TRY_CAST({qc(c)} AS DATE) > DATE '2026-09-05') despues_del_corte
                  FROM read_parquet({sp(ruta)})""").iloc[0]
        filas.append((nom, c, int(r.n), int(r.nulos), str(r.minimo), str(r.maximo), int(r.antes_1900), int(r.despues_del_corte)))
fe = pd.DataFrame(filas, columns=["fuente", "columna", "filas", "nulos_o_ilegibles", "minimo", "maximo", "antes_1900", "despues_5sep2026"])
fe.to_csv(TABLAS / "planA_fechas_calidad_sri_supercias.csv", index=False, encoding="utf-8-sig")
print("\nFechas en SRI / Supercias (mínimos absurdos = hallazgo de calidad, no error tuyo):\n")
print(fe.to_string(index=False))
for _, x in fe.iterrows():
    cifra(f"{x.fuente}_{x.columna}_antes_1900", int(x.antes_1900), f"{x.filas:,} filas", "celda3")

# 6) base consolidada: UNA fila por RUC de proveedor, con SRI y Supercias unidos por RUC
sel_c = ", ".join(f"c.{qc(c)} AS {qc('sri_'+c)}" for c in cCAT if c != kc)
sel_s = ", ".join(f"s.{qc(c)} AS {qc('sc_'+c)}" for c in cSC if c != ks)
PROV = PLAN_A / "planA_proveedores_ruc.parquet"
con.execute(f"""COPY (
 WITH p AS (
   SELECT {norm(c_ruc)} AS ruc, count(*) AS n_procesos,
          count(*) FILTER (WHERE planA_segmento='S3_modelo_sin_catalogo') AS n_procesos_P1,
          sum(COALESCE({qc(c_monto)},0)) AS monto_total,
          min(planA_anio) AS primer_anio, max(planA_anio) AS ultimo_anio
   FROM read_parquet({sp(U)}) WHERE {qc(c_ruc)} IS NOT NULL AND {norm(c_ruc)}<>'' GROUP BY 1),
 c AS (SELECT *, count(*) OVER (PARTITION BY {norm(kc)}) AS n_filas_ruc FROM read_parquet({sp(CAT)})
       QUALIFY row_number() OVER (PARTITION BY {norm(kc)} ORDER BY {norm(kc)}) = 1),
 s AS (SELECT * FROM read_parquet({sp(SC)}) QUALIFY row_number() OVER (PARTITION BY {norm(ks)} ORDER BY {norm(ks)}) = 1)
 SELECT p.*, (c.{qc(kc)} IS NOT NULL) AS en_sri, (s.{qc(ks)} IS NOT NULL) AS en_supercias,
        c.n_filas_ruc AS sri_n_filas_ruc, {sel_c}, {sel_s}
 FROM p LEFT JOIN c ON trim(CAST(c.{qc(kc)} AS VARCHAR)) = p.ruc
        LEFT JOIN s ON trim(CAST(s.{qc(ks)} AS VARCHAR)) = p.ruc
) TO {sp(PROV)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
nprov = q(f"SELECT count(*) n, count(*) FILTER (WHERE en_sri) a, count(*) FILTER (WHERE en_supercias) b FROM read_parquet({sp(PROV)})").iloc[0]
print(f"\n{PROV.name}: {int(nprov.n):,} proveedores | con SRI {int(nprov.a):,} | con Supercias {int(nprov.b):,}")
cifra("proveedores_consolidados", int(nprov.n), "RUC de proveedor distintos en el universo", "celda3")
nn = informe_nulos(PROV, "proveedores_ruc")
print("\nColumnas de la base consolidada con más datos faltantes:")
print(nn[["columna", "pct_sin_dato"]].head(25).to_string(index=False))
print("\nAviso de fuga: columnas sri_* de ESTADO ACTUAL describen hoy, no la fecha del proceso;")
print("no deben ser variables del modelo (solo las fechas históricas sirven para reglas temporales).")

# %%
# ===== CELDA 4 · Embudo por etapas, justificación estadística de descartar solo-planeación, muestra manual, manifiesto =====
tot = int(q(f"SELECT count(*) n FROM read_parquet({sp(U)})").n[0])

# 1) qué etapas existen y cuántos procesos las tienen (sin suponer nombres)
if c_etp:
    et = q(f"""SELECT s etapa, count(*) procesos FROM (SELECT unnest({qc(c_etp)}) s FROM read_parquet({sp(U)})
              WHERE {qc(c_etp)} IS NOT NULL) GROUP BY 1 ORDER BY 2 DESC""")
    print("Etapas registradas en la columna 'etapas' (procesos que la contienen):\n"); print(et.to_string(index=False))
    emb = q(f"""SELECT planA_segmento segmento,
         COALESCE(array_to_string(list_sort({qc(c_etp)}), ' + '), '(sin etapas)') combinacion, count(*) procesos
         FROM read_parquet({sp(U)}) GROUP BY 1,2 ORDER BY 1, 3 DESC""")
    emb["pct_del_segmento"] = (100 * emb.procesos / emb.groupby("segmento").procesos.transform("sum")).round(2)
    emb.to_csv(TABLAS / "planA_embudo_etapas.csv", index=False, encoding="utf-8-sig")
    print("\nCombinaciones de etapas por segmento (las 4 más frecuentes de cada uno):\n")
    print(emb.groupby("segmento").head(4).to_string(index=False))
    for _, x in et.iterrows():
        cifra(f"procesos_con_etapa_{x.etapa}", int(x.procesos), "procesos de la base (con columna etapas)", "celda4")

# 2) solo-planeación por año. Se usa el AÑO DEL CÓDIGO del proceso porque existe para todos los
#    segmentos con la misma definición (anio_proceso no existe para los solo-planeación).
ya = q(f"""SELECT planA_anio_codigo anio_codigo, count(*) procesos,
   count(*) FILTER (WHERE planA_segmento='S0_solo_planeacion') solo_planeacion,
   round(100.0*count(*) FILTER (WHERE planA_segmento='S0_solo_planeacion')/count(*),2) pct_solo_planeacion,
   count(*) FILTER (WHERE planA_segmento='S1_sin_monto') sin_monto,
   round(100.0*count(*) FILTER (WHERE planA_segmento='S1_sin_monto')/count(*),2) pct_sin_monto
   FROM read_parquet({sp(U)}) GROUP BY 1 ORDER BY 1""")
ya.to_csv(TABLAS / "planA_solo_planeacion_por_anio_codigo.csv", index=False, encoding="utf-8-sig")
print("\nSolo planeación y sin monto por AÑO DEL CÓDIGO del proceso (<NA> = el código no trae año):\n")
print(ya.to_string(index=False))
n0 = int(ya.solo_planeacion.sum()); n1 = int(ya.sin_monto.sum())
p0 = 100 * n0 / tot
frase = (f"De {tot:,} procesos publicados en el portal entre 2015 y el corte del 5 de septiembre de 2026, "
         f"{n0:,} ({p0:.2f} %) quedaron solo en planeación (sin llamado, adjudicación ni contrato) y {n1:,} ({100*n1/tot:.2f} %) "
         f"no tienen monto en ninguna fuente; ambos grupos se excluyen del modelo y se conservan como estadística descriptiva.")
print("\nFRASE PARA LA TESIS (borrador; es un censo, no una muestra, por eso no lleva intervalo de confianza):\n" + frase)
cifra("solo_planeacion_n", n0, f"{tot:,} procesos", "celda4")
cifra("solo_planeacion_pct", round(p0, 2), f"{tot:,} procesos", "celda4")
ant = ya[(ya.anio_codigo >= 2015) & (ya.anio_codigo <= 2025)]
if len(ant):
    print(f"\nTasa 2015-2025 por año del código: mínimo {ant.pct_solo_planeacion.min()} % | máximo {ant.pct_solo_planeacion.max()} %. "
          f"Si es estable en los años cerrados, no son procesos 'en curso' sino que no avanzaron.")
    cifra("solo_planeacion_tasa_min_2015_2025", float(ant.pct_solo_planeacion.min()), "por año del código", "celda4")
    cifra("solo_planeacion_tasa_max_2015_2025", float(ant.pct_solo_planeacion.max()), "por año del código", "celda4")
sin_anio = int(ya[ya.anio_codigo.isna()].procesos.sum()) if ya.anio_codigo.isna().any() else 0
print(f"Procesos cuyo código no trae año: {sin_anio:,} (no entran en la tasa por año).")

# 3) ¿cuáles procesos quedan para modelar, por año y por familia?
fa = q(f"""SELECT planA_familia familia, planA_periodo_normativo periodo, count(*) procesos
           FROM read_parquet({sp(P1)}) GROUP BY 1,2 ORDER BY 1,2""")
fa.to_csv(TABLAS / "planA_P1_por_familia_y_periodo.csv", index=False, encoding="utf-8-sig")
print("\nUniverso de modelado P1 por familia y periodo normativo:\n"); print(fa.to_string(index=False))
peq = fa[fa.procesos < 30]
print(f"\nGrupos con menos de 30 procesos (usarán el grupo más amplio): {len(peq)}")

# 4) muestra para revisar a mano en el portal (reproducible)
cols_m = [c for c in ["ocid", "planA_segmento", "planA_anio", "planA_anio_codigo", "planA_familia", c_entn, c_etp, c_monto] if c]
cols_m = list(dict.fromkeys(cols_m))
def sel_cols(c):
    return f"array_to_string({qc(c)}, ' + ') AS {qc(c)}" if c == c_etp else qc(c)
d = q(f"SELECT {', '.join(sel_cols(c) for c in cols_m)} FROM read_parquet({sp(U)})")
partes = []
s0 = d[d.planA_segmento == "S0_solo_planeacion"]
anios_s0 = sorted(s0.planA_anio_codigo.dropna().unique())
for a in anios_s0:                                  # 2 por año para ver si el patrón cambia con el tiempo
    g = s0[s0.planA_anio_codigo == a]
    partes.append(g.sample(min(len(g), 2), random_state=42))
if len(s0) and not anios_s0:                        # respaldo: si ningún código trae año, 20 al azar
    partes.append(s0.sample(min(20, len(s0)), random_state=42))
for sg, n in (("S3_modelo_sin_catalogo", 12), ("S2_catalogo", 5), ("S1_sin_monto", 5), ("S4_sin_catalogo_sin_proveedor_o_monto", 5)):
    x = d[d.planA_segmento == sg]
    if len(x): partes.append(x.sample(min(n, len(x)), random_state=42))
m = pd.concat(partes, ignore_index=True)
m.insert(1, "codigo_candidato", m.ocid.str.replace(r"^ocds-[a-z0-9]+-", "", regex=True))
# hipótesis: el número final (tras el último guion) sería un identificador interno y no parte del código visible
m.insert(2, "codigo_sin_sufijo", m.codigo_candidato.str.replace(r"-[0-9]+$", "", regex=True))
for c in ("portal_etapa_observada", "portal_estado_observado", "coincide_con_nuestros_datos", "notas"):
    m[c] = ""
RUTA_M = TABLAS / "planA_muestra_revision_manual.csv"
m.to_csv(RUTA_M, index=False, encoding="utf-8-sig")
print(f"\nMuestra para revisar en el portal: {len(m)} procesos -> {RUTA_M}")
print(m[["codigo_candidato", "codigo_sin_sufijo", "planA_segmento", "planA_anio_codigo"]].head(20).to_string(index=False))
print("\nCómo revisarla: abre el CSV, prueba primero 'codigo_sin_sufijo' y luego 'codigo_candidato' en el buscador de procesos del portal de Compras")
print("Públicas, mira en qué etapa está (como en tu captura: Planeación activa y el resto en gris) y anota")
print("'portal_etapa_observada' y 'portal_estado_observado' (¿cancelado, desierto, en curso, nada?).")
print("Si el código no aparece, prueba con el ocid completo y dime cuál formato funcionó.")
cifra("muestra_revision_manual_n", len(m), "muestra aleatoria reproducible (semilla 42)", "celda4")

# 5) manifiesto de la carpeta plan_A
filas = []
for f in sorted(PLAN_A.glob("*.parquet")):
    n = q(f"SELECT count(*) n FROM read_parquet({sp(f)})").n[0]
    k = len(columnas(f))
    h = hashlib.sha256(f.read_bytes()).hexdigest()[:16] if f.stat().st_size < 3e9 else "(omitido)"
    filas.append((f.name, int(n), k, round(f.stat().st_size/1e6, 1), time.strftime("%Y-%m-%d %H:%M", time.localtime(f.stat().st_mtime)), h))
man = pd.DataFrame(filas, columns=["archivo", "filas", "columnas", "MB", "creado", "sha256_16"])
man.to_csv(PLAN_A / "planA_manifiesto.csv", index=False, encoding="utf-8-sig")
print("\nMANIFIESTO plan_A:\n"); print(man.to_string(index=False))
print(f"\nCifras listas para el registro: {TABLAS/'planA_cifras_para_registro.csv'}")

# %%
# ===== CELDA 5 · Diagnósticos que quedaron abiertos: definición de año, SRI en CSV, Supercias por tipo, procesos sin proveedor =====
# (a) ¿De dónde sale 'anio_proceso'? Se compara con el año de las fechas y con el año del código, solo en P1.
r = q(f"""SELECT count(*) filas,
   round(100.0*count(*) FILTER (WHERE planA_anio = year(TRY_CAST(fecha_inicio_tender AS DATE)))/count(*),2) pct_igual_a_fecha_inicio_tender,
   round(100.0*count(*) FILTER (WHERE planA_anio = year(TRY_CAST(fecha_release AS DATE)))/count(*),2) pct_igual_a_fecha_release,
   round(100.0*count(*) FILTER (WHERE planA_anio = planA_anio_codigo)/count(*),2) pct_igual_al_anio_del_codigo,
   round(100.0*count(*) FILTER (WHERE planA_anio_codigo IS NULL)/count(*),2) pct_codigo_sin_anio
   FROM read_parquet({sp(P1)})""")
print("(a) Coincidencia de anio_proceso en P1 con:\n"); print(r.T.to_string(header=False))
d = q(f"""SELECT planA_anio - planA_anio_codigo AS diferencia_anio, count(*) procesos,
          round(100.0*count(*)/sum(count(*)) OVER (),2) pct
          FROM read_parquet({sp(P1)}) WHERE planA_anio_codigo IS NOT NULL GROUP BY 1 ORDER BY 1""")
print("\nanio_proceso menos año del código (0 = coinciden):\n"); print(d.to_string(index=False))
d.to_csv(TABLAS / "planA_diagnostico_anio.csv", index=False, encoding="utf-8-sig")

# (b) Qué trae el archivo de solo-planeación (columnas con datos)
print("\n(b) Solo planeación: columnas del archivo original y % sin dato en cada una\n")
cs = columnas(SP)
filas = []
for c in cs:
    x = q(f"SELECT round(100.0*count(*) FILTER (WHERE {qc(c)} IS NULL)/count(*),1) p FROM read_parquet({sp(SP)})").p[0]
    filas.append((c, x))
print(pd.DataFrame(filas, columns=["columna", "pct_nulo"]).to_string(index=False))

# (c) Los CSV del SRI. Los archivos son UTF-8 (el intento anterior con latin-1 fue un error mío).
print("\n(c) CSV del SRI leídos como UTF-8; se compara filas leídas contra líneas del archivo\n")
filas, todos = [], []
for f in csvs:
    try:
        n_lineas = sum(1 for _ in open(f, "rb")) - 1
        dd = con.execute(f"DESCRIBE SELECT * FROM read_csv({sp(f)}, header=true, all_varchar=true, ignore_errors=true)").df()
        kk = next(c for c in dd.column_name if "ruc" in c.lower())
        n, nd = con.execute(f"SELECT count(*), count(DISTINCT {qc(kk)}) FROM read_csv({sp(f)}, header=true, all_varchar=true, ignore_errors=true)").fetchone()
        filas.append((f.name, n_lineas, n, nd)); todos.append(f)
    except Exception as e:
        filas.append((f.name, None, None, None)); print("  fallo", f.name, str(e)[:100])
dc = pd.DataFrame(filas, columns=["archivo", "lineas_archivo", "filas_leidas", "ruc_distintos_en_archivo"])
print(dc.to_string(index=False))
dc.to_csv(TABLAS / "planA_sri_csv_por_provincia.csv", index=False, encoding="utf-8-sig")
tl, tr = int(dc.lineas_archivo.fillna(0).sum()), int(dc.filas_leidas.fillna(0).sum())
print(f"\nLíneas totales {tl:,} | filas leídas {tr:,} | diferencia {tl-tr:,} (si no es ~0, el lector descartó filas)")
if todos:
    lista_csv = ", ".join(sp(f) for f in todos)
    kk = next(c for c in con.execute(f"DESCRIBE SELECT * FROM read_csv({sp(todos[0])}, header=true, all_varchar=true, ignore_errors=true)").df().column_name if "ruc" in c.lower())
    nd = con.execute(f"SELECT count(DISTINCT {qc(kk)}) FROM read_csv([{lista_csv}], header=true, all_varchar=true, ignore_errors=true, union_by_name=true)").fetchone()[0]
    print(f"RUC distintos entre todos los CSV: {nd:,} | RUC en el parquet: 7,124,906")
    cifra("SRI_csv_lineas_total", tl, "24 CSV del SRI", "celda5")
    cifra("SRI_csv_filas_leidas", tr, "24 CSV del SRI (lector UTF-8)", "celda5")
    cifra("SRI_csv_ruc_distintos", int(nd), "24 CSV del SRI", "celda5")

# (d) Supercias solo cubre sociedades: la cobertura justa se mide dentro de las sociedades
tp = next((c for c in cCAT if "tipo_contrib" in c.lower()), None)
if tp:
    z = q(f"""WITH p AS (SELECT ruc, en_sri, en_supercias, sri_{tp} AS tipo FROM read_parquet({sp(PROV)}))
      SELECT COALESCE(tipo,'(sin SRI)') tipo_contribuyente, count(*) proveedores,
             count(*) FILTER (WHERE en_supercias) en_supercias,
             round(100.0*count(*) FILTER (WHERE en_supercias)/count(*),2) pct_en_supercias
      FROM p GROUP BY 1 ORDER BY 2 DESC""")
    print("\n(d) Cobertura de Supercias por tipo de contribuyente (proveedores de SERCOP):\n"); print(z.to_string(index=False))
    z.to_csv(TABLAS / "planA_supercias_por_tipo.csv", index=False, encoding="utf-8-sig")
    for _, x in z.iterrows():
        cifra(f"supercias_cobertura_{x.tipo_contribuyente}_pct", float(x.pct_en_supercias), f"{int(x.proveedores):,} proveedores", "celda5")

# (e) S4: procesos sin catálogo y sin proveedor, pero con etapa de adjudicación
if c_etp:
    w = q(f"""SELECT count(*) procesos_S4,
       count(*) FILTER (WHERE list_contains({qc(c_etp)}, 'award')) con_etapa_adjudicacion,
       count(*) FILTER (WHERE list_contains({qc(c_etp)}, 'contract')) con_etapa_contrato,
       round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_millones
       FROM read_parquet({sp(PLAN_A / 'planA_sin_catalogo_sin_proveedor_o_monto.parquet')})""")
    print("\n(e) S4 (sin catálogo, sin RUC de proveedor):\n"); print(w.T.to_string(header=False))
    for k, v in w.iloc[0].items():
        cifra(f"S4_{k}", float(v), "8.637 procesos S4", "celda5")

In [ ]:
# %%
# ===== CELDA 8 · Reconciliar los fantasmas, revisar fugas por anacronismo y entender S4 =====
# Requiere celdas 1, 3 y 6 (usa U, P1, PLAN_A, TABLAS, VAL, DIMP, c_ruc, c_monto, c_etp, cF).
import re
 
# ---------- (1) 170 contra 163: ¿y_fantasma y proveedor_fantasma_estado cuentan lo mismo? ----------
yf = elegir(["y_fantasma"], cF); fe = elegir(["proveedor_fantasma_estado"], cF)
print("="*95, "\n(1) RECONCILIACIÓN DE LOS FANTASMAS\n", "="*95, sep="")
if yf and fe:
    cr = q(f"""SELECT COALESCE(CAST({qc(yf)} AS INTEGER), -1) AS y_fantasma,
          ({qc(fe)} IS NOT NULL) AS tiene_estado, count(*) procesos,
          count(DISTINCT {qc(c_ruc)}) ruc
          FROM read_parquet({sp(U)}) GROUP BY 1,2 ORDER BY 1,2""")
    print("\nTabla cruzada en el universo completo (y_fantasma frente a 'tiene estado del SRI'):\n")
    print(cr.to_string(index=False))
    disc = q(f"""SELECT planA_segmento segmento, planA_familia familia, count(*) procesos,
          count(DISTINCT {qc(c_ruc)}) ruc, round(sum(COALESCE({qc(c_monto)},0))/1e6,3) monto_millones
          FROM read_parquet({sp(U)})
          WHERE {qc(fe)} IS NOT NULL AND COALESCE(CAST({qc(yf)} AS INTEGER),0) = 0
          GROUP BY 1,2 ORDER BY 3 DESC""")
    print("\nLos procesos que tienen estado del SRI pero y_fantasma = 0 (la diferencia):\n")
    print(disc.to_string(index=False) if len(disc) else "  ninguno: las dos columnas coinciden")
    if len(disc):
        d2 = q(f"""SELECT DISTINCT {norm(c_ruc)} ruc, {qc(fe)} estado,
              CAST(min(TRY_CAST(f_marca AS DATE)) OVER (PARTITION BY {norm(c_ruc)}) AS VARCHAR) fecha_marca
              FROM read_parquet({sp(U)})
              WHERE {qc(fe)} IS NOT NULL AND COALESCE(CAST({qc(yf)} AS INTEGER),0) = 0""")
        print(f"\nRUC involucrados en la diferencia: {len(d2)}\n"); print(d2.to_string(index=False))
    tot = q(f"""SELECT
          CAST(sum(CASE WHEN COALESCE(CAST({qc(yf)} AS INTEGER),0)=1 THEN 1 ELSE 0 END) AS BIGINT) por_y_fantasma,
          CAST(sum(CASE WHEN {qc(fe)} IS NOT NULL THEN 1 ELSE 0 END) AS BIGINT) por_estado_sri,
          count(DISTINCT CASE WHEN COALESCE(CAST({qc(yf)} AS INTEGER),0)=1 THEN {qc(c_ruc)} END) ruc_y,
          count(DISTINCT CASE WHEN {qc(fe)} IS NOT NULL THEN {qc(c_ruc)} END) ruc_estado
          FROM read_parquet({sp(U)})""").iloc[0]
    print(f"\nTotales en el universo: por y_fantasma {int(tot.por_y_fantasma)} procesos / {int(tot.ruc_y)} RUC | "
          f"por estado del SRI {int(tot.por_estado_sri)} procesos / {int(tot.ruc_estado)} RUC")
    p1c = q(f"""SELECT CAST(sum(CASE WHEN COALESCE(CAST({qc(yf)} AS INTEGER),0)=1 THEN 1 ELSE 0 END) AS BIGINT) a,
          CAST(sum(CASE WHEN {qc(fe)} IS NOT NULL THEN 1 ELSE 0 END) AS BIGINT) b FROM read_parquet({sp(P1)})""").iloc[0]
    print(f"Dentro de P1: por y_fantasma {int(p1c.a)} | por estado del SRI {int(p1c.b)}")
    cifra("fantasma_procesos_por_y_fantasma_universo", int(tot.por_y_fantasma), "universo total", "celda8")
    cifra("fantasma_procesos_por_estado_sri_universo", int(tot.por_estado_sri), "universo total", "celda8")
    cifra("fantasma_procesos_por_y_fantasma_P1", int(p1c.a), "442.295 procesos de P1", "celda8")
    cifra("fantasma_procesos_por_estado_sri_P1", int(p1c.b), "442.295 procesos de P1", "celda8")
else:
    print("falta y_fantasma o proveedor_fantasma_estado en la base")
 
# ---------- (2) El catastro de inexistentes del SRI, leído de verdad ----------
print("\n" + "="*95 + "\n(2) CATASTRO DE EMPRESAS INEXISTENTES DEL SRI (denominador de la validación)\n" + "="*95)
cands = [p for p in RAIZ.rglob("*") if p.is_file() and p.suffix.lower() in (".csv", ".txt", ".xlsx")
         and re.search("fantasma|inexistent|fictici", p.name, re.I) and "plan_A" not in p.parts]
print(f"Archivos candidatos: {[p.name for p in cands]}")
def leer_catastro(archivos):
    """Prueba codificaciones y separadores hasta que el archivo se lea con sentido."""
    for p in archivos:
        for enc in ("utf-8", "latin-1", "cp1252"):
            for sep in (";", ",", "|", "\t"):
                try:
                    lect = (f"read_csv({sp(p)}, header=true, all_varchar=true, "
                            f"encoding='{enc}', delim='{sep}', ignore_errors=true)")
                    d = con.execute(f"SELECT * FROM {lect} LIMIT 5").df()
                    if d.shape[1] >= 2 and len(d):
                        n = con.execute(f"SELECT count(*) FROM {lect}").fetchone()[0]
                        print(f"\n  {p.name}: encoding={enc} delim='{sep}' -> {n:,} filas, {d.shape[1]} columnas")
                        print("  columnas:", list(d.columns))
                        return p, enc, sep, n, lect
                except Exception:
                    continue
    return None
 
cat_f = leer_catastro(cands)
if cat_f:
    p, enc, sep, n, lect = cat_f
    cols = con.execute(f"DESCRIBE SELECT * FROM {lect}").df().column_name.tolist()
    kf = next((c for c in cols if re.search("ruc", c, re.I)), None)
    if kf:
        r = q(f"""WITH c AS (SELECT DISTINCT trim(CAST({qc(kf)} AS VARCHAR)) AS ruc FROM {lect}),
              p AS (SELECT DISTINCT {norm(c_ruc)} AS ruc FROM read_parquet({sp(U)}) WHERE {norm(c_ruc)} <> '')
              SELECT (SELECT count(*) FROM c) ruc_en_el_catastro,
                     (SELECT count(*) FROM c JOIN p USING (ruc)) con_contratos_sercop""").iloc[0]
        a, b = int(r.ruc_en_el_catastro), int(r.con_contratos_sercop)
        print(f"\n  RUC en el catastro de inexistentes: {a:,}")
        print(f"  De esos, con al menos un proceso en SERCOP: {b:,} ({100*b/a:.3f} % del catastro)")
        print(f"  Y son el {100*b/73953:.4f} % de los 73.953 proveedores del portal.")
        cifra("SRI_catastro_inexistentes_ruc", a, "catastro de inexistentes del SRI", "celda8")
        cifra("SRI_inexistentes_con_contratos", b, f"{a:,} RUC del catastro", "celda8")
        cifra("SRI_inexistentes_pct_de_proveedores", round(100*b/73953, 4), "73.953 proveedores", "celda8")
    else:
        print("  no encontré columna de RUC; columnas:", cols)
else:
    print("  No pude leer ningún archivo del catastro de fantasmas. Si existe, pásame su nombre exacto.")
 
# ---------- (3) Contratar después de la marca, con fechas de día, no de año ----------
print("\n" + "="*95 + "\n(3) PROCESOS POSTERIORES A LA MARCA DEL SRI (regla verificable, no variable)\n" + "="*95)
if VAL.exists():
    fproc = "COALESCE(TRY_CAST(u.fecha_inicio_tender AS DATE), TRY_CAST(u.fecha_release AS DATE))"
    t = q(f"""WITH v AS (SELECT * FROM read_parquet({sp(VAL)}) WHERE fantasma_fecha_marca IS NOT NULL)
      SELECT CASE WHEN {fproc} > v.fantasma_fecha_marca THEN 'despues de la marca'
                  WHEN {fproc} IS NULL THEN 'sin fecha de proceso' ELSE 'antes o el mismo dia' END momento,
             count(*) procesos, count(DISTINCT u.{qc(c_ruc)}) ruc,
             round(sum(COALESCE(u.{qc(c_monto)},0))/1e6,3) monto_millones,
             CAST(min({fproc}) AS VARCHAR) desde, CAST(max({fproc}) AS VARCHAR) hasta
      FROM read_parquet({sp(U)}) u JOIN v ON v.ruc = {norm(c_ruc)}
      GROUP BY 1 ORDER BY 2 DESC""")
    print("\nComparando la fecha del proceso con la fecha de la marca, día a día:\n"); print(t.to_string(index=False))
    t.to_csv(TABLAS / "planA_fantasmas_antes_despues_marca.csv", index=False, encoding="utf-8-sig")
    det = q(f"""WITH v AS (SELECT * FROM read_parquet({sp(VAL)}) WHERE fantasma_fecha_marca IS NOT NULL)
      SELECT u.planA_segmento segmento, u.planA_familia familia, count(*) procesos,
             count(DISTINCT u.{qc(c_ruc)}) ruc, round(sum(COALESCE(u.{qc(c_monto)},0))/1e6,3) monto_millones,
             round(avg(date_diff('day', v.fantasma_fecha_marca, {fproc}))) dias_promedio_despues
      FROM read_parquet({sp(U)}) u JOIN v ON v.ruc = {norm(c_ruc)}
      WHERE {fproc} > v.fantasma_fecha_marca GROUP BY 1,2 ORDER BY 5 DESC""")
    print("\nSolo los posteriores, por segmento y familia:\n"); print(det.to_string(index=False))
    det.to_csv(TABLAS / "planA_fantasmas_posteriores_detalle.csv", index=False, encoding="utf-8-sig")
    for _, x in t.iterrows():
        cifra(f"fantasma_{x.momento.replace(' ','_')}_procesos", int(x.procesos), "procesos de los 15 RUC con fecha de marca", "celda8")
        cifra(f"fantasma_{x.momento.replace(' ','_')}_monto_MM", float(x.monto_millones), "ídem", "celda8")
 
# ---------- (4) Fuga por anacronismo: variables de "alguna vez" que miran el futuro ----------
print("\n" + "="*95 + "\n(4) FUGA POR ANACRONISMO: variables que usan hechos posteriores al proceso\n" + "="*95)
sosp = [c for c in cF if re.search("alguna_vez|tras_suspension|es_sociedad_registrada|antiguedad", c, re.I)]
print(f"Columnas sospechosas encontradas: {sosp}\n")
fsus = elegir(["proveedor_fecha_suspension_definitiva"], cF)
if fsus:
    fproc2 = "COALESCE(TRY_CAST(fecha_inicio_tender AS DATE), TRY_CAST(fecha_release AS DATE))"
    r = q(f"""SELECT count(*) procesos_con_suspension,
      count(*) FILTER (WHERE TRY_CAST({qc(fsus)} AS DATE) > {fproc2}) suspension_posterior_al_proceso,
      round(100.0*count(*) FILTER (WHERE TRY_CAST({qc(fsus)} AS DATE) > {fproc2})/count(*),2) pct_posterior
      FROM read_parquet({sp(P1)}) WHERE TRY_CAST({qc(fsus)} AS DATE) IS NOT NULL""").iloc[0]
    print("De los procesos de P1 cuyo proveedor tiene fecha de suspensión definitiva:")
    print(r.to_string())
    print("\nSi 'suspension_posterior_al_proceso' es alto, cualquier variable del tipo 'fue suspendido alguna")
    print("vez' le está contando al modelo algo que ocurrió DESPUÉS del proceso: eso es fuga temporal.")
    print("La versión correcta es 'estaba suspendido a la fecha del proceso'.")
    cifra("P1_suspension_posterior_al_proceso_pct", float(r.pct_posterior), f"{int(r.procesos_con_suspension):,} procesos de P1 con fecha de suspensión", "celda8")
    for c in [c for c in sosp if re.search("alguna_vez|tras_suspension", c, re.I)]:
        z = q(f"""SELECT {qc(c)} valor, count(*) procesos,
              count(*) FILTER (WHERE TRY_CAST({qc(fsus)} AS DATE) > {fproc2}) con_suspension_posterior
              FROM read_parquet({sp(P1)}) GROUP BY 1 ORDER BY 2 DESC LIMIT 5""")
        print(f"\n{c}:"); print(z.to_string(index=False))
 
# ---------- (5) S4: adjudicaciones sin RUC. El caso de la subasta inversa ----------
print("\n" + "="*95 + "\n(5) S4: 2.684 subastas con adjudicación y sin RUC de proveedor\n" + "="*95)
S4 = PLAN_A / "planA_sin_catalogo_sin_proveedor_o_monto.parquet"
pn = elegir(["proveedor_nombre"], cF); pid = elegir(["proveedor_id"], cF)
piezas = ["planA_familia familia", "count(*) procesos"]
for c, al in ((pn, "con_nombre_de_proveedor"), (pid, "con_id_de_proveedor")):
    if c: piezas.append(f"count(*) FILTER (WHERE {qc(c)} IS NOT NULL AND trim(CAST({qc(c)} AS VARCHAR)) <> '') {al}")
for c in ("n_awards", "n_parties", "n_contracts"):
    if elegir([c], cF): piezas.append(f"round(avg({qc(c)}),2) {c}_promedio")
if elegir(["fuente_monto"], cF): piezas.append("any_value(fuente_monto) fuente_monto")
s4 = q(f"SELECT {', '.join(piezas)} FROM read_parquet({sp(S4)}) GROUP BY 1 ORDER BY 2 DESC")
print("\n¿Tienen nombre de proveedor aunque les falte el RUC?\n"); print(s4.to_string(index=False))
s4.to_csv(TABLAS / "planA_S4_diagnostico.csv", index=False, encoding="utf-8-sig")
if pn:
    ej = q(f"""SELECT planA_familia familia, ocid, {qc(pn)} proveedor_nombre,
          round(COALESCE({qc(c_monto)},0)/1e6,3) monto_millones FROM read_parquet({sp(S4)})
          WHERE {qc(pn)} IS NOT NULL AND trim(CAST({qc(pn)} AS VARCHAR)) <> ''
          ORDER BY {qc(c_monto)} DESC NULLS LAST LIMIT 10""")
    print("\nLos diez de mayor monto con nombre pero sin RUC (si hay nombre, el RUC se puede recuperar):\n")
    print(ej.to_string(index=False))
fm = elegir(["fuente_monto"], cF)
if fm:
    print("\n¿De dónde sale el monto en cada segmento?\n")
    print(q(f"""SELECT planA_segmento segmento, {qc(fm)} fuente_monto, count(*) procesos
          FROM read_parquet({sp(U)}) GROUP BY 1,2 ORDER BY 1,3 DESC""").to_string(index=False))
 
# ---------- (6) ¿Qué etapas traen de verdad los 32.161 "solo planeación"? ----------
# En el universo salen "(sin etapas)" porque la celda 1 no arrastró esa columna desde su archivo.
# El archivo original sí la tiene, así que la leo ahí. Esta es la prueba directa del descarte.
print("\n" + "="*95 + "\n(6) ETAPAS REALES DE LOS 32.161 SOLO-PLANEACIÓN (desde su archivo original)\n" + "="*95)
cSP2 = columnas(SP)
e_sp = elegir(["etapas"], cSP2)
if e_sp:
    tp = q(f"DESCRIBE SELECT * FROM read_parquet({sp(SP)})")
    tipo = dict(zip(tp.column_name, tp.column_type))[e_sp]
    print(f"Tipo de la columna '{e_sp}' en el archivo: {tipo}")
    if tipo.endswith("[]"):
        a = q(f"""SELECT COALESCE(array_to_string(list_sort({qc(e_sp)}), ' + '), '(nulo)') combinacion,
              count(*) procesos, round(100.0*count(*)/sum(count(*)) OVER (),2) pct
              FROM read_parquet({sp(SP)}) GROUP BY 1 ORDER BY 2 DESC LIMIT 15""")
        b = q(f"""SELECT s etapa, count(*) procesos FROM (SELECT unnest({qc(e_sp)}) s FROM read_parquet({sp(SP)})
              WHERE {qc(e_sp)} IS NOT NULL) GROUP BY 1 ORDER BY 2 DESC""")
    else:
        a = q(f"""SELECT COALESCE(CAST({qc(e_sp)} AS VARCHAR),'(nulo)') combinacion, count(*) procesos,
              round(100.0*count(*)/sum(count(*)) OVER (),2) pct FROM read_parquet({sp(SP)})
              GROUP BY 1 ORDER BY 2 DESC LIMIT 15""")
        b = None
    print("\nCombinaciones de etapas en los 32.161:\n"); print(a.to_string(index=False))
    if b is not None: print("\nEtapas individuales:\n"); print(b.to_string(index=False))
    a.to_csv(TABLAS / "planA_S0_etapas_reales.csv", index=False, encoding="utf-8-sig")
    print("\nCómo se lee: si el 100 % trae solo 'planning', el descarte queda probado con el dato:")
    print("son procesos sin llamado, sin adjudicación y sin contrato, así que no hay precio ni")
    print("adjudicatario que puntuar. Si aparecen otras etapas, el filtro del cuaderno 9 hay que revisarlo.")
    for _, x in a.head(5).iterrows():
        cifra(f"S0_etapas_{x.combinacion.replace(' ','')}", int(x.procesos), "32.161 solo-planeación", "celda8")
    ts = elegir(["tender_status"], cSP2)
    if ts:
        print(f"\n'{ts}' en los 32.161:")
        print(q(f"SELECT COALESCE(CAST({qc(ts)} AS VARCHAR),'(nulo)') estado, count(*) n FROM read_parquet({sp(SP)}) GROUP BY 1 ORDER BY 2 DESC").to_string(index=False))
else:
    print("el archivo de solo-planeación no tiene columna de etapas; columnas:", cSP2)
 
 

In [ ]:
# ===== CELDA 9 · ¿De dónde sale el monto?, las columnas que no varían y la tasa base =====
# Requiere las celdas 1 y 3. Responde a una duda que puede invalidar una cifra ya escrita.
print("="*95 + "\n(1) ¿monto_final es siempre el MONTO ADJUDICADO?\n" + "="*95)
fm = elegir(["fuente_monto"], cF); ma = elegir(["monto_award"], cF); mt = elegir(["monto_tender"], cF)
tiene = lambda e: f"list_contains({qc(c_etp)}, '{e}')"
sel = ["planA_segmento segmento",
       f"CASE WHEN {qc(c_etp)} IS NULL THEN '(sin etapas)' "
       f"WHEN {tiene('award')} AND {tiene('contract')} THEN 'adjudicacion y contrato' "
       f"WHEN {tiene('award')} THEN 'solo adjudicacion' "
       f"WHEN {tiene('contract')} THEN 'solo contrato' ELSE 'ni adjudicacion ni contrato' END etapas_de_dinero",
       "count(*) procesos", f"round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_millones"]
if fm: sel.append(f"string_agg(DISTINCT CAST({qc(fm)} AS VARCHAR), ', ') fuentes_declaradas")
if ma: sel.append(f"round(100.0*count({qc(ma)})/count(*),1) pct_con_monto_award")
if mt: sel.append(f"round(100.0*count({qc(mt)})/count(*),1) pct_con_monto_tender")
t = q(f"SELECT {', '.join(sel)} FROM read_parquet({sp(U)}) GROUP BY 1,2 ORDER BY 1,3 DESC")
print(t.to_string(index=False))
t.to_csv(TABLAS / "planA_origen_del_monto.csv", index=False, encoding="utf-8-sig")
print("\nLa fila que importa: procesos CON monto pero SIN adjudicación ni contrato. Si existen y su")
print("monto sale del pliego, entonces 'monto_final' NO siempre es el adjudicado y hay que decirlo.")
if ma:
    r = q(f"""SELECT count(*) procesos_con_monto,
      count(*) FILTER (WHERE {qc(ma)} IS NULL) sin_monto_adjudicado,
      round(sum(COALESCE({qc(c_monto)},0)) FILTER (WHERE {qc(ma)} IS NULL)/1e6,1) monto_millones_sin_adjudicado,
      round(100.0*count(*) FILTER (WHERE abs(COALESCE({qc(c_monto)},0)-COALESCE({qc(ma)},0))<0.01)/count(*),2) pct_identico_al_adjudicado
      FROM read_parquet({sp(U)}) WHERE COALESCE({qc(c_monto)},0) > 0""").iloc[0]
    print("\nEn todo el universo con monto positivo:\n" + r.to_string())
    cifra("monto_final_sin_monto_award_procesos", int(r.sin_monto_adjudicado), "universo con monto positivo", "celda9")
    cifra("monto_final_sin_monto_award_MM", float(r.monto_millones_sin_adjudicado), "ídem", "celda9")

print("\n" + "="*95 + "\n(2) LAS COLUMNAS QUE NO VARÍAN EN P1: ¿control que pasó o indicador muerto?\n" + "="*95)
for c in ["fuente_monto", "tiene_periodo_competitivo", "monto_cero", "entidad_es_fantasma", "es_catalogo"]:
    cc = elegir([c], cF)
    if not cc: continue
    a = q(f"SELECT COALESCE(CAST({qc(cc)} AS VARCHAR),'(nulo)') valor, count(*) n FROM read_parquet({sp(P1)}) GROUP BY 1 ORDER BY 2 DESC")
    b = q(f"SELECT COALESCE(CAST({qc(cc)} AS VARCHAR),'(nulo)') valor, count(*) n FROM read_parquet({sp(U)}) GROUP BY 1 ORDER BY 2 DESC")
    print(f"\n{cc}  ->  en P1: {dict(zip(a.valor, a.n))}")
    print(f"{' '*len(cc)}  ->  en el universo: {dict(zip(b.valor, b.n))}")
print("\nSi una columna es constante en P1 pero varía en el universo, es un control que pasó (sirve para")
print("documentar, no como variable). Si es constante en los dos, el indicador nunca se activó y hay que")
print("revisar cómo se construyó o retirarlo.")

print("\n" + "="*95 + "\n(3) TASA BASE Y PESO EN DINERO DE LA VALIDACIÓN\n" + "="*95)
fe = elegir(["proveedor_fantasma_estado"], cF)
if fe:
    r = q(f"""SELECT count(*) procesos_P1,
      CAST(sum(CASE WHEN {qc(fe)} IS NOT NULL THEN 1 ELSE 0 END) AS BIGINT) marcados,
      round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_P1_MM,
      round(sum(CASE WHEN {qc(fe)} IS NOT NULL THEN COALESCE({qc(c_monto)},0) ELSE 0 END)/1e6,2) monto_marcados_MM
      FROM read_parquet({sp(P1)})""").iloc[0]
    n, m = int(r.procesos_P1), int(r.marcados)
    print(f"Procesos de P1: {n:,} | con proveedor marcado: {m}")
    print(f"Tasa base: 1 proceso marcado por cada {n/m:,.0f} procesos ({100*m/n:.4f} %)")
    print(f"Monto: {r.monto_marcados_MM:,.2f} de {r.monto_P1_MM:,.1f} millones ({100*r.monto_marcados_MM/r.monto_P1_MM:.4f} %)")
    print("\nPor qué importa: con una clase positiva de 1 en varios miles, cualquier 'precisión' alta es")
    print("engañosa. Por eso el trabajo reporta lift y percentil, no precisión, y declara la potencia.")
    cifra("P1_tasa_base_1_cada", round(n/m), "442.295 procesos de P1", "celda9")
    cifra("P1_monto_marcados_pct", round(100*r.monto_marcados_MM/r.monto_P1_MM, 4), "monto de P1", "celda9")

print("\n" + "="*95 + "\n(4) SUBASTA INVERSA: ¿qué tan raro es que falte el RUC?\n" + "="*95)
z = q(f"""SELECT planA_familia familia,
  count(*) procesos_sin_catalogo,
  count(*) FILTER (WHERE planA_segmento='S4_sin_catalogo_sin_proveedor_o_monto') sin_ruc,
  round(100.0*count(*) FILTER (WHERE planA_segmento='S4_sin_catalogo_sin_proveedor_o_monto')/count(*),2) pct_sin_ruc
  FROM read_parquet({sp(U)})
  WHERE planA_segmento IN ('S3_modelo_sin_catalogo','S4_sin_catalogo_sin_proveedor_o_monto')
  GROUP BY 1 ORDER BY 4 DESC""")
print(z.to_string(index=False))
print("\nEsta es la lectura correcta: el 100 % de las subastas de S4 tienen adjudicación, pero las")
print("subastas de S4 son solo una fracción de todas las subastas. El denominador correcto es este.")
z.to_csv(TABLAS / "planA_sin_ruc_por_familia.csv", index=False, encoding="utf-8-sig")
for _, x in z.iterrows():
    cifra(f"sin_ruc_pct_{x.familia.replace(' ','_')}", float(x.pct_sin_ruc), f"{int(x.procesos_sin_catalogo):,} procesos sin catálogo de esa familia", "celda9")

In [ ]:
# ===== CELDA 10 · Muestras para revisión manual (portal SERCOP y página del SRI) =====
# Junta en un solo CSV las dudas que el dato no resuelve y que hay que mirar a mano.
filas = []
def agregar(bloque, identificador, contexto, pregunta):
    filas.append({"bloque": bloque, "identificador": identificador, "contexto": contexto,
                  "pregunta": pregunta, "respuesta_observada": "", "notas": ""})

# A. Los 4.031 procesos con presupuesto del pliego y sin adjudicación: ¿desiertos, cancelados o en curso?
S4 = PLAN_A / "planA_sin_catalogo_sin_proveedor_o_monto.parquet"
sin_adj = (f"NOT list_contains({qc(c_etp)}, 'award') AND NOT list_contains({qc(c_etp)}, 'contract')")
a = q(f"""SELECT ocid, planA_familia familia, planA_anio anio, {qc(c_entn)} entidad,
      round(COALESCE({qc(c_monto)},0)/1e6,3) monto_millones
      FROM read_parquet({sp(S4)}) WHERE {sin_adj}
      ORDER BY {qc(c_monto)} DESC NULLS LAST LIMIT 10""")
print("A. Procesos con monto del pliego y SIN adjudicación ni contrato (los 10 de mayor monto):\n")
print(a.to_string(index=False))
for _, x in a.iterrows():
    agregar("A. sin adjudicacion", x.ocid,
            f"{x.familia} {x.anio} · {x.entidad} · {x.monto_millones} MM",
            "¿El portal lo muestra como desierto, cancelado, en curso o adjudicado?")

# B. Los 4.031 más antiguos, para ver si son procesos todavía en curso
b = q(f"""SELECT ocid, planA_familia familia, planA_anio anio, {qc(c_entn)} entidad
      FROM read_parquet({sp(S4)}) WHERE {sin_adj} AND planA_anio <= 2023
      ORDER BY random() LIMIT 5""")
for _, x in b.iterrows():
    agregar("B. sin adjudicacion antiguo", x.ocid, f"{x.familia} {x.anio} · {x.entidad}",
            "Es de 2023 o antes: ¿sigue 'en curso' o quedó desierto/cancelado?")

# C. Procesos que solo llegaron a planeación: ¿qué muestra el portal?
c = q(f"""SELECT ocid, planA_anio anio FROM read_parquet({sp(PLAN_A / 'planA_excluidos_solo_planeacion.parquet')})
      WHERE planA_anio IN (2021, 2022, 2026) ORDER BY random() LIMIT 6""")
for _, x in c.iterrows():
    agregar("C. solo planeacion", x.ocid, f"publicado en {x.anio}",
            "¿El portal lo muestra solo en Planeación? ¿Dice PAC, en curso, cancelado?")

# D. Las empresas marcadas como inexistentes que HOY figuran activas en el SRI
if VAL.exists():
    d = q(f"""SELECT v.ruc, v.fantasma_estado, CAST(v.fantasma_fecha_marca AS VARCHAR) fecha_marca,
          v.n_procesos_marcados procesos FROM read_parquet({sp(VAL)}) v
          WHERE upper(CAST(v.fantasma_estado AS VARCHAR)) <> 'SUSPENDIDO' ORDER BY 4 DESC""")
    print("\nD. Marcadas como inexistentes y hoy NO suspendidas ante el SRI:\n"); print(d.to_string(index=False))
    for _, x in d.iterrows():
        agregar("D. fantasma no suspendida", x.ruc,
                f"estado hoy: {x.fantasma_estado} · marca del {x.fecha_marca} · {x.procesos} procesos",
                "En consultas.sri.gob.ec: ¿qué estado y qué razón social tiene hoy este RUC?")

# E. Sociedades que el SRI reconoce y la Superintendencia no registra
e = q(f"""SELECT ruc, sri_RAZON_SOCIAL razon_social, n_procesos, round(monto_total/1e6,3) monto_millones
      FROM read_parquet({sp(DIMP)}) WHERE sri_es_sociedad AND NOT en_supercias
      ORDER BY monto_total DESC NULLS LAST LIMIT 6""")
print("\nE. Sociedades segun el SRI que NO estan en el directorio de companias (mayor monto):\n")
print(e.to_string(index=False))
for _, x in e.iterrows():
    agregar("E. sociedad sin Supercias", x.ruc, f"{x.razon_social} · {x.n_procesos} procesos · {x.monto_millones} MM",
            "¿Aparece en supercias.gob.ec? Si no, ¿qué tipo de entidad es (pública, consorcio, extranjera)?")

# F. El único proceso fuera del catálogo que no tiene etapa de planeación
f = q(f"""SELECT ocid, planA_segmento segmento, planA_familia familia, planA_anio anio
      FROM read_parquet({sp(U)}) WHERE planA_segmento IN ('S3_modelo_sin_catalogo','S4_sin_catalogo_sin_proveedor_o_monto')
      AND ({qc(c_etp)} IS NULL OR NOT list_contains({qc(c_etp)}, 'planning')) LIMIT 3""")
print("\nF. Procesos fuera del catálogo sin etapa de planeación (deberían ser 1):\n")
print(f.to_string(index=False) if len(f) else "  ninguno")
for _, x in f.iterrows():
    agregar("F. sin planeacion", x.ocid, f"{x.segmento} · {x.familia} {x.anio}", "¿Qué muestra el portal para este proceso?")

# G. Concurso público: la familia con mayor pérdida de RUC
g = q(f"""SELECT ocid, planA_anio anio, {qc(c_entn)} entidad, round(COALESCE({qc(c_monto)},0)/1e6,3) monto_millones
      FROM read_parquet({sp(S4)}) WHERE planA_familia ILIKE '%oncurso%'
      ORDER BY {qc(c_monto)} DESC NULLS LAST LIMIT 4""")
for _, x in g.iterrows():
    agregar("G. concurso sin RUC", x.ocid, f"{x.anio} · {x.entidad} · {x.monto_millones} MM",
            "¿El portal muestra adjudicatario? Si lo muestra, el RUC se perdió al procesar.")

M = TABLAS / "planA_revision_manual_dudas.csv"
pd.DataFrame(filas).to_csv(M, index=False, encoding="utf-8-sig")
print(f"\n{len(filas)} casos para revisar -> {M}")
print("Llena 'respuesta_observada' y 'notas'. Con eso cierro cada duda con evidencia, no con suposición.")

In [ ]:
import pandas as pd
m = pd.read_csv(r"C:\SERCOP_FINAL\plan_A\tablas\planA_revision_manual_dudas.csv", encoding="utf-8-sig")
m["codigo"] = m.identificador.str.replace(r"^ocds-[a-z0-9]+-", "", regex=True).str.replace(r"-[0-9]+$", "", regex=True)
for b in ["C. solo planeacion", "B. sin adjudicacion antiguo", "G. concurso sin RUC"]:
    d = m[m.bloque == b]
    if len(d):
        print(f"\n{'='*80}\n{b}  ({len(d)} casos)\n{'='*80}")
        print(d[["codigo", "contexto", "pregunta"]].to_string(index=False))

In [ ]:
# ===== CELDA 12 · Desiertos, efecto de la fecha de corte, RUC recuperable y oferentes =====
# Requiere celdas 1 y 3. Nace de la revisión manual del 9 de octubre en el portal.
CORTE = "DATE '2026-09-05'"
S4 = PLAN_A / "planA_sin_catalogo_sin_proveedor_o_monto.parquet"
sin_adj = f"NOT list_contains({qc(c_etp)}, 'award') AND NOT list_contains({qc(c_etp)}, 'contract')"
fp = "COALESCE(TRY_CAST(fecha_inicio_tender AS DATE), TRY_CAST(fecha_release AS DATE))"

print("="*95 + "\n(1) LOS 4.031 SIN ADJUDICACIÓN: ¿desiertos o todavía en curso al 5 de septiembre de 2026?\n" + "="*95)
t = q(f"""SELECT year({fp}) anio_licitacion, count(*) procesos,
   round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_millones,
   round(avg(date_diff('day', {fp}, {CORTE}))) dias_promedio_hasta_el_corte,
   count(*) FILTER (WHERE date_diff('day', {fp}, {CORTE}) <= 180) iniciados_en_los_ultimos_180_dias
   FROM read_parquet({sp(S4)}) WHERE {sin_adj} GROUP BY 1 ORDER BY 1""")
print(t.to_string(index=False))
t.to_csv(TABLAS / "planA_sin_adjudicacion_por_anio.csv", index=False, encoding="utf-8-sig")
r = q(f"""SELECT count(*) total,
   count(*) FILTER (WHERE date_diff('day', {fp}, {CORTE}) <= 180) en_curso_plausible,
   count(*) FILTER (WHERE date_diff('day', {fp}, {CORTE}) > 180) cerrados_sin_adjudicacion,
   round(sum(COALESCE({qc(c_monto)},0)) FILTER (WHERE date_diff('day', {fp}, {CORTE}) > 180)/1e6,1) monto_cerrados_MM
   FROM read_parquet({sp(S4)}) WHERE {sin_adj}""").iloc[0]
print(f"\nDe los {int(r.total):,} procesos sin adjudicación:")
print(f"  iniciados dentro de los 180 días previos al corte (pueden seguir en curso): {int(r.en_curso_plausible):,}")
print(f"  iniciados antes (ya debieron resolverse; candidatos a desierto o cancelado): {int(r.cerrados_sin_adjudicacion):,}"
      f"  ->  {r.monto_cerrados_MM:,.1f} millones")
print("\nVerificado a mano: LCS-EPCONST-002-2021 y LICS-EPP-2023-041 figuran como DESIERTA en el portal,")
print("y LICO-GADMCA-2026-001 como SUSPENDIDO. El estado existe en el portal pero NO en los datos OCDS.")
for k in ("total","en_curso_plausible","cerrados_sin_adjudicacion"):
    cifra(f"sin_adjudicacion_{k}", int(r[k]), "procesos sin catálogo, sin RUC y sin adjudicación", "celda12")
cifra("sin_adjudicacion_monto_cerrados_MM", float(r.monto_cerrados_MM), "ídem, iniciados a más de 180 días del corte", "celda12")

print("\n" + "="*95 + "\n(2) EL RUC ESTÁ DENTRO DE proveedor_id: formato EC-RUC-<ruc>-<id de entidad>\n" + "="*95)
pid = elegir(["proveedor_id"], cF)
if pid:
    pidv = f"CAST({qc(pid)} AS VARCHAR)"
    print(q(f"""SELECT CASE WHEN {pidv} LIKE 'EC-RUC-%' THEN 'EC-RUC- (contribuyente ecuatoriano)'
             WHEN {pidv} LIKE 'ID-%' THEN 'ID- (identificador extranjero u otro)'
             WHEN {pidv} IS NULL THEN '(nulo)' ELSE 'otro formato' END prefijo,
           count(*) procesos, count(DISTINCT {pidv}) distintos
           FROM read_parquet({sp(S4)}) GROUP BY 1 ORDER BY 2 DESC""").to_string(index=False))
    ruc = f"regexp_extract({pidv}, '^EC-RUC-([0-9]+)', 1)"
    z = q(f"""WITH s AS (SELECT planA_familia familia, {ruc} AS cand FROM read_parquet({sp(S4)}))
      SELECT familia, count(*) procesos,
        count(*) FILTER (WHERE length(cand)=13) ruc_de_13,
        count(*) FILTER (WHERE length(cand)>13) mas_de_13_digitos,
        count(*) FILTER (WHERE cand='' OR cand IS NULL) sin_ruc_en_el_id
      FROM s GROUP BY 1 ORDER BY 2 DESC""")
    print("\n¿Cuántos traen un RUC de 13 dígitos dentro del identificador?\n"); print(z.to_string(index=False))
    v = q(f"""WITH s AS (SELECT {ruc} AS cand FROM read_parquet({sp(S4)}) WHERE {ruc} <> ''),
      s2 AS (SELECT CASE WHEN length(cand)=14 THEN substr(cand,1,13) ELSE cand END AS ruc FROM s),
      c AS (SELECT DISTINCT {norm(kc)} AS ruc FROM read_parquet({sp(CAT)}))
      SELECT count(*) candidatos, count(DISTINCT s2.ruc) distintos,
        count(*) FILTER (WHERE c.ruc IS NOT NULL) existen_en_el_SRI,
        round(100.0*count(*) FILTER (WHERE c.ruc IS NOT NULL)/count(*),1) pct
      FROM s2 LEFT JOIN c ON c.ruc = s2.ruc""").iloc[0]
    print(f"\nRecortando a 13 dígitos los que vienen con 14 (caso CNT: 17681525600011 -> 1768152560001):")
    print(f"  candidatos {int(v.candidatos):,} | distintos {int(v.distintos):,} | "
          f"existen en el catastro del SRI {int(v.existen_en_el_SRI):,} ({v.pct} %)")
    cifra("S4_ruc_recuperados_validados", int(v.existen_en_el_SRI), f"{int(v.candidatos):,} candidatos de proveedor_id", "celda12")
    cifra("S4_ruc_recuperados_pct", float(v.pct), "ídem", "celda12")
    print("\nSi el porcentaje es alto, el RUC NO falta en la fuente: se perdió al extraer, y el segmento S4")
    print("se reduce. Los que queden con prefijo ID- son los proveedores extranjeros, que sí carecen de RUC.")

print("\n" + "="*95 + "\n(3) LOS 15 RUC DE LA VALIDACIÓN, PARA VERIFICAR UNO POR UNO EN EL SRI\n" + "="*95)
if VAL.exists():
    d = q(f"""SELECT v.ruc, CAST(v.fantasma_fecha_marca AS VARCHAR) fecha_marca, v.fantasma_estado estado_en_base,
          v.n_procesos_marcados procesos, round(m.monto/1e6,3) monto_millones
          FROM read_parquet({sp(VAL)}) v
          LEFT JOIN (SELECT {norm(c_ruc)} ruc, sum(COALESCE({qc(c_monto)},0)) monto
                     FROM read_parquet({sp(U)}) GROUP BY 1) m ON m.ruc = v.ruc
          ORDER BY v.n_procesos_marcados DESC""")
    print(d.to_string(index=False))
    d.to_csv(TABLAS / "planA_15_ruc_para_verificar_sri.csv", index=False, encoding="utf-8-sig")
    print("\nEn consultas.sri.gob.ec cada ficha trae DOS marcas distintas que hay que anotar por separado:")
    print("  'Contribuyente fantasma'  y  'Contribuyente con transacciones inexistentes'.")
    print("No son lo mismo, y el catastro que descargamos mezcla las dos. Hasta separarlas, el conjunto")
    print("de validación no está bien definido.")

print("\n" + "="*95 + "\n(4) OFERENTES: ¿en qué familias hay información de participantes?\n" + "="*95)
nof = elegir(["n_oferentes"], cF); npa = elegir(["n_parties"], cF)
if nof:
    o = q(f"""SELECT planA_familia familia, count(*) procesos,
      round(100.0*count({qc(nof)})/count(*),1) pct_con_n_oferentes,
      round(avg({qc(nof)}),2) oferentes_promedio, max({qc(nof)}) maximo
      {", round(avg(" + qc(npa) + "),2) parties_promedio" if npa else ""}
      FROM read_parquet({sp(P1)}) GROUP BY 1 ORDER BY 2 DESC""")
    print(o.to_string(index=False))
    o.to_csv(TABLAS / "planA_oferentes_por_familia.csv", index=False, encoding="utf-8-sig")
    print("\nEl portal muestra la lista de licitadores con nombre e identificador en licitación (ejemplo")
    print("verificado: LICO-CELECEP-2025-07152 con 12 licitadores). Si aquí n_oferentes tiene cobertura")
    print("alta en esas familias, el indicador de concurrencia sirve más allá de la menor cuantía.")

In [ ]:
# ===== CELDA 13 · Conjunto de validación verificado y recuperación del RUC (regex corregida) =====
# Requiere celdas 1 y 3. La celda 12 falló en (2) porque mi patrón solo buscaba 'EC-RUC-';
# los proveedores vienen con prefijo 'ID-' y el RUC está igual adentro.
import pandas as pd
print("="*95 + "\n(1) CONJUNTO DE VALIDACIÓN VERIFICADO A MANO EN EL SRI (9 de octubre de 2026)\n" + "="*95)
VER = [  # ruc, razon_social, tipo, estado_sri, fantasma, transacciones_inexistentes
 ("RUC_OCULTO_01","CONTRIBUYENTE_01","PERSONA NATURAL","SUSPENDIDO",False,True),
 ("RUC_OCULTO_02","CONTRIBUYENTE_02","PERSONA NATURAL","SUSPENDIDO",False,True),
 ("RUC_OCULTO_03","CONTRIBUYENTE_03","SOCIEDAD","SUSPENDIDO",False,True),
 ("RUC_OCULTO_04","CONTRIBUYENTE_04","PERSONA NATURAL","ACTIVO",False,True),
 ("RUC_OCULTO_05","CONTRIBUYENTE_05","PERSONA NATURAL","SUSPENDIDO",False,True),
 ("RUC_OCULTO_06","CONTRIBUYENTE_06","PERSONA NATURAL","SUSPENDIDO",False,True),
 ("RUC_OCULTO_07","CONTRIBUYENTE_07","SOCIEDAD","SUSPENDIDO",True,False),
 ("RUC_OCULTO_08","CONTRIBUYENTE_08","PERSONA NATURAL","SUSPENDIDO",False,True),
 ("RUC_OCULTO_09","CONTRIBUYENTE_09","SOCIEDAD","PASIVO",True,False),
 ("RUC_OCULTO_10","CONTRIBUYENTE_10","SOCIEDAD","SUSPENDIDO",False,True),
 ("RUC_OCULTO_11","CONTRIBUYENTE_11","PERSONA NATURAL","SUSPENDIDO",False,True),
 ("RUC_OCULTO_12","CONTRIBUYENTE_12","SOCIEDAD","ACTIVO",True,False),
 ("RUC_OCULTO_13","CONTRIBUYENTE_13","SOCIEDAD","PASIVO",True,False),
 ("RUC_OCULTO_14","CONTRIBUYENTE_14","SOCIEDAD","PASIVO",False,False),
 ("RUC_OCULTO_15","CONTRIBUYENTE_15","SOCIEDAD","PASIVO",True,False)]
v = pd.DataFrame(VER, columns=["ruc","razon_social","tipo_contribuyente","estado_sri","sri_fantasma","sri_transacciones_inexistentes"])
v["grupo"] = v.apply(lambda r: "fantasma" if r.sri_fantasma else ("transacciones_inexistentes" if r.sri_transacciones_inexistentes else "sin_marca_actual"), axis=1)
v["verificado_el"] = "2026-10-09"; v["fuente"] = "consultas.sri.gob.ec (consulta manual)"
con.register("v_tmp", v)
VER_P = PLAN_A / "planA_validacion_sri_verificada.parquet"
con.execute(f"""COPY (
  SELECT v.*, COALESCE(m.n_procesos,0) AS n_procesos, round(COALESCE(m.monto,0)/1e6,3) AS monto_millones,
         COALESCE(m.n_procesos_P1,0) AS n_procesos_P1
  FROM v_tmp v LEFT JOIN (
    SELECT {norm(c_ruc)} ruc, count(*) n_procesos, sum(COALESCE({qc(c_monto)},0)) monto,
           count(*) FILTER (WHERE planA_segmento='S3_modelo_sin_catalogo') n_procesos_P1
    FROM read_parquet({sp(U)}) GROUP BY 1) m ON m.ruc = v.ruc
) TO {sp(VER_P)} (FORMAT PARQUET)""")
r = q(f"""SELECT grupo, count(*) ruc,
   count(*) FILTER (WHERE tipo_contribuyente='PERSONA NATURAL') personas_naturales,
   CAST(sum(n_procesos) AS BIGINT) procesos, CAST(sum(n_procesos_P1) AS BIGINT) procesos_en_P1,
   round(sum(monto_millones),3) monto_millones
   FROM read_parquet({sp(VER_P)}) GROUP BY 1 ORDER BY 4 DESC""")
print(r.to_string(index=False))
tot = q(f"SELECT count(*) ruc, CAST(sum(n_procesos) AS BIGINT) p, round(sum(monto_millones),3) mm FROM read_parquet({sp(VER_P)})").iloc[0]
print(f"\nTOTAL: {int(tot.ruc)} contribuyentes | {int(tot.p)} procesos | {tot.mm} millones")
print("\nLectura: la etiqueta NO es 'empresas fantasma'. La mayoría del peso viene de contribuyentes con")
print("TRANSACCIONES INEXISTENTES, y siete de los quince son PERSONAS NATURALES, no empresas.")
for _, x in r.iterrows():
    cifra(f"validacion_{x.grupo}_ruc", int(x.ruc), "15 contribuyentes del catastro con contratación", "celda13")
    cifra(f"validacion_{x.grupo}_procesos", int(x.procesos), "170 procesos de los 15", "celda13")
    cifra(f"validacion_{x.grupo}_monto_MM", float(x.monto_millones), "ídem", "celda13")
print(f"\n  -> {VER_P}")
print(q(f"SELECT ruc, razon_social, tipo_contribuyente, estado_sri, grupo, n_procesos, monto_millones FROM read_parquet({sp(VER_P)}) ORDER BY n_procesos DESC").to_string(index=False))

print("\n" + "="*95 + "\n(2) RECUPERAR EL RUC: buscar 13 dígitos en CUALQUIER prefijo del identificador\n" + "="*95)
S4 = PLAN_A / "planA_sin_catalogo_sin_proveedor_o_monto.parquet"
pid = elegir(["proveedor_id"], cF); pidv = f"CAST({qc(pid)} AS VARCHAR)"
print("Diez identificadores de ejemplo (para ver el formato real):\n")
print(q(f"SELECT DISTINCT {pidv} proveedor_id FROM read_parquet({sp(S4)}) WHERE {qc(pid)} IS NOT NULL LIMIT 10").to_string(index=False))
cand = f"regexp_extract({pidv}, '([0-9]{{13}})', 1)"
z = q(f"""SELECT planA_familia familia, count(*) procesos,
   count(*) FILTER (WHERE {cand} <> '') con_ruc_de_13,
   round(100.0*count(*) FILTER (WHERE {cand} <> '')/count(*),1) pct
   FROM read_parquet({sp(S4)}) GROUP BY 1 ORDER BY 2 DESC""")
print("\n¿Hay un número de 13 dígitos dentro del identificador?\n"); print(z.to_string(index=False))
w = q(f"""WITH s AS (SELECT {cand} AS ruc, COALESCE({qc(c_monto)},0) m FROM read_parquet({sp(S4)}) WHERE {cand} <> ''),
  c AS (SELECT DISTINCT {norm(kc)} AS ruc FROM read_parquet({sp(CAT)}))
  SELECT count(*) candidatos, count(DISTINCT s.ruc) distintos,
    count(*) FILTER (WHERE c.ruc IS NOT NULL) existen_en_el_SRI,
    round(100.0*count(*) FILTER (WHERE c.ruc IS NOT NULL)/count(*),1) pct,
    round(sum(s.m) FILTER (WHERE c.ruc IS NOT NULL)/1e6,1) monto_recuperable_MM
  FROM s LEFT JOIN c ON c.ruc = s.ruc""").iloc[0]
print(f"\nCandidatos {int(w.candidatos):,} | distintos {int(w.distintos):,} | validados contra el catastro del SRI "
      f"{int(w.existen_en_el_SRI):,} ({w.pct} %) | {w.monto_recuperable_MM:,.1f} millones")
print(f"\nSi se recuperan, el segmento S4 baja de 8.637 a {8637-int(w.existen_en_el_SRI):,} procesos y el universo")
print("de modelado sube. Hay que rehacer las cifras de S4 antes de escribirlas en la tesis.")
cifra("S4_ruc_recuperables", int(w.existen_en_el_SRI), "8.637 procesos de S4", "celda13")
cifra("S4_monto_recuperable_MM", float(w.monto_recuperable_MM), "ídem", "celda13")

print("\n" + "="*95 + "\n(3) ¿PASA LO MISMO EN LA BASE COMPLETA? (procesos con RUC vacío)\n" + "="*95)
b = q(f"""SELECT planA_segmento segmento, count(*) procesos,
   count(*) FILTER (WHERE {cand} <> '') con_ruc_en_el_identificador
   FROM read_parquet({sp(U)})
   WHERE ({qc(c_ruc)} IS NULL OR {norm(c_ruc)} = '') AND {qc(pid)} IS NOT NULL
   GROUP BY 1 ORDER BY 2 DESC""")
print(b.to_string(index=False) if len(b) else "  ninguno")
print("\nEsto cubre también los 37.733 procesos de catálogo sin RUC. Si ahí también se recupera,")
print("la cifra de 46.370 procesos sin RUC de proveedor cambia y hay que corregirla en todos lados.")

In [ ]:
# ===== CELDA 14 · Reconstruir el universo con el RUC recuperado (versión 2, auditable) =====
# Requiere celdas 1 y 3. Regla: el RUC recuperado NO se mezcla en silencio con el declarado;
# cada fila lleva de dónde salió su RUC, para que cualquiera pueda deshacer la decisión.
import time
pid = elegir(["proveedor_id"], cF); pidv = f"CAST(f.{qc(pid)} AS VARCHAR)"
cand = f"regexp_extract({pidv}, '([0-9]{{13}})', 1)"
decl = f"NULLIF(trim(CAST(f.{qc(c_ruc)} AS VARCHAR)), '')"
ent  = f"NULLIF(trim(CAST(f.{qc(c_ent)} AS VARCHAR)), '')"

print("="*95 + "\n(1) CONTROLES ANTES DE ACEPTAR EL RUC RECUPERADO\n" + "="*95)
ctrl = q(f"""WITH x AS (
  SELECT {decl} AS declarado, NULLIF({cand},'') AS recuperado, {ent} AS entidad
  FROM read_parquet({sp(U)}) f WHERE {decl} IS NULL)
SELECT count(*) sin_ruc_declarado,
  count(recuperado) con_candidato,
  count(*) FILTER (WHERE recuperado = entidad) candidato_igual_al_ruc_de_la_entidad,
  count(*) FILTER (WHERE recuperado IS NOT NULL AND NOT regexp_matches(recuperado,'001$')) no_termina_en_001,
  count(*) FILTER (WHERE recuperado IS NOT NULL AND
       CAST(substr(recuperado,1,2) AS INTEGER) NOT BETWEEN 1 AND 24
       AND CAST(substr(recuperado,1,2) AS INTEGER) <> 30) provincia_fuera_de_rango
FROM x""").iloc[0]
print(ctrl.to_string())
print("\nLos tres últimos controles tienen que dar cero o casi cero. Si el candidato coincide con el RUC")
print("de la entidad, habríamos copiado al comprador como proveedor: ese caso se descarta siempre.")
for k in ctrl.index: cifra(f"recuperacion_{k}", int(ctrl[k]), "procesos sin RUC declarado", "celda14")

ok = (f"NULLIF({cand},'') IS NOT NULL AND NULLIF({cand},'') <> COALESCE({ent},'') "
      f"AND regexp_matches(NULLIF({cand},''),'001$') AND EXISTS "
      f"(SELECT 1 FROM read_parquet({sp(CAT)}) c WHERE trim(CAST(c.{qc(kc)} AS VARCHAR)) = NULLIF({cand},''))")

print("\n" + "="*95 + "\n(2) RECONSTRUCCIÓN DEL UNIVERSO (versión 2)\n" + "="*95)
U2 = PLAN_A / "planA_universo_v2.parquet"
t0 = time.time()
con.execute(f"""COPY (
 SELECT f.* EXCLUDE (planA_segmento),
   f.planA_segmento AS planA_segmento_v1,
   CASE WHEN {decl} IS NOT NULL THEN {decl}
        WHEN {ok} THEN NULLIF({cand},'') END AS proveedor_ruc_final,
   CASE WHEN {decl} IS NOT NULL THEN 'declarado'
        WHEN {ok} THEN 'recuperado_del_identificador'
        WHEN {pidv} IS NOT NULL THEN 'identificador_sin_ruc_ecuatoriano'
        ELSE 'sin_identificador' END AS proveedor_ruc_origen,
   CASE WHEN f.planA_segmento_base = 'S0' THEN 'S0_solo_planeacion'
        WHEN f.planA_segmento_base = 'S1' THEN 'S1_sin_monto'
        WHEN COALESCE(CAST(f.{qc(c_cat)} AS BOOLEAN), false) THEN 'S2_catalogo'
        WHEN (CASE WHEN {decl} IS NOT NULL THEN {decl} WHEN {ok} THEN NULLIF({cand},'') END) IS NOT NULL
             AND COALESCE(f.{qc(c_monto)},0) > 0 THEN 'S3_modelo_sin_catalogo'
        ELSE 'S4_sin_catalogo_sin_proveedor_o_monto' END AS planA_segmento
 FROM (SELECT *, CASE WHEN planA_segmento='S0_solo_planeacion' THEN 'S0'
                      WHEN planA_segmento='S1_sin_monto' THEN 'S1' ELSE 'X' END AS planA_segmento_base
       FROM read_parquet({sp(U)})) f
) TO {sp(U2)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f"Escrito {U2.name} ({time.time()-t0:.0f} s)")

cmp = q(f"""SELECT planA_segmento_v1 antes, planA_segmento despues, count(*) procesos,
   round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_millones
   FROM read_parquet({sp(U2)}) GROUP BY 1,2 HAVING antes <> despues ORDER BY 3 DESC""")
print("\nProcesos que cambian de segmento gracias al RUC recuperado:\n")
print(cmp.to_string(index=False) if len(cmp) else "  ninguno")
seg = q(f"""SELECT planA_segmento segmento, count(*) procesos,
   round(100.0*count(*)/sum(count(*)) OVER (),2) pct,
   round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_millones
   FROM read_parquet({sp(U2)}) GROUP BY 1 ORDER BY 1""")
print("\nSegmentos, versión 2:\n"); print(seg.to_string(index=False))
tot = int(seg.procesos.sum())
print(f"\nTOTAL = {tot:,} -> " + ("COINCIDE con 2.785.355" if tot == 2785355 else "NO COINCIDE: revisar"))
org = q(f"""SELECT proveedor_ruc_origen origen, count(*) procesos,
   round(100.0*count(*)/sum(count(*)) OVER (),2) pct,
   count(DISTINCT proveedor_ruc_final) ruc_distintos
   FROM read_parquet({sp(U2)}) GROUP BY 1 ORDER BY 2 DESC""")
print("\nDe dónde sale el RUC de cada proceso:\n"); print(org.to_string(index=False))
org.to_csv(TABLAS / "planA_origen_del_ruc.csv", index=False, encoding="utf-8-sig")
for _, x in seg.iterrows():
    cifra(f"v2_procesos_{x.segmento}", int(x.procesos), f"universo total {tot:,}", "celda14")
for _, x in org.iterrows():
    cifra(f"v2_ruc_origen_{x.origen}", int(x.procesos), f"universo total {tot:,}", "celda14")
pr = q(f"SELECT count(DISTINCT proveedor_ruc_final) n FROM read_parquet({sp(U2)}) WHERE proveedor_ruc_final IS NOT NULL").n[0]
print(f"\nProveedores distintos: {int(pr):,} (antes 73.953)")
cifra("v2_proveedores_distintos", int(pr), "universo con RUC final", "celda14")

print("\n" + "="*95 + "\n(3) ARCHIVOS DE LA VERSIÓN 2\n" + "="*95)
def exportar2(nombre, donde):
    r = PLAN_A / nombre
    con.execute(f"COPY (SELECT * FROM read_parquet({sp(U2)}) WHERE {donde}) TO {sp(r)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    n = q(f"SELECT count(*) n FROM read_parquet({sp(r)})").n[0]
    print(f"  {nombre:52s} {n:>10,} filas"); return r
P1_2 = exportar2("planA_v2_P1_modelo_sin_catalogo.parquet", "planA_segmento='S3_modelo_sin_catalogo'")
exportar2("planA_v2_P0_catalogo.parquet",                   "planA_segmento='S2_catalogo'")
exportar2("planA_v2_excluidos_solo_planeacion.parquet",     "planA_segmento='S0_solo_planeacion'")
exportar2("planA_v2_excluidos_sin_monto.parquet",           "planA_segmento='S1_sin_monto'")
exportar2("planA_v2_sin_proveedor_identificable.parquet",   "planA_segmento='S4_sin_catalogo_sin_proveedor_o_monto'")

rest = q(f"""SELECT planA_familia familia, count(*) procesos,
   round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_millones,
   count(*) FILTER (WHERE proveedor_ruc_origen='identificador_sin_ruc_ecuatoriano') con_identificador_extranjero
   FROM read_parquet({sp(PLAN_A / 'planA_v2_sin_proveedor_identificable.parquet')})
   GROUP BY 1 ORDER BY 3 DESC""")
print("\nLo que QUEDA sin proveedor identificable, por familia:\n"); print(rest.to_string(index=False))
rest.to_csv(TABLAS / "planA_v2_sin_proveedor_por_familia.csv", index=False, encoding="utf-8-sig")
print("\nEstos son los que de verdad no tienen RUC ecuatoriano: proveedores extranjeros y procesos")
print("sin adjudicación. Ya no es un fallo de extracción, es una característica de la fuente.")

# A partir de aquí las celdas siguientes trabajan con la versión 2.
U, P1 = U2, P1_2
print("\n>>> U y P1 apuntan ahora a la versión 2. Vuelve a correr la CELDA 6 para que las")
print(">>> dimensiones de proveedor y entidad se reconstruyan con el RUC recuperado.")

In [ ]:
# %%
# ===== CELDA 6 · Esquema en estrella: dimensión de proveedor, dimensión de entidad, validación aparte =====
# Requiere las celdas 1 y 3 (usa PLAN_A, TABLAS, U, P1, CAT, SC, kc, ks, cCAT, cSC, c_ruc, c_ent, c_entn).
# Idea: NO duplicar 37 columnas del SRI en 2,78 millones de filas. Se guardan tablas por RUC y se unen
# cuando hace falta. El RUC puede ser llave primaria AQUÍ (una fila por RUC), no en la tabla de procesos.
import re
def norm(c): return f"trim(CAST({qc(c)} AS VARCHAR))"
FUERA = re.compile(r"fantasma|^f_marca$|^f_reactivacion$|estado_actual|riesgoso", re.I)
 
# ---------- A. Dimensión de proveedor (una fila por RUC). Sin ninguna columna de fantasmas. ----------
cols_cat = [c for c in cCAT if c != kc and not FUERA.search(c)]
cols_sc  = [c for c in cSC  if c != ks and not FUERA.search(c)]
sel_c = ", ".join(f"c.{qc(c)} AS {qc('sri_'+c)}" for c in cols_cat)
sel_s = ", ".join(f"s.{qc(c)} AS {qc('sc_'+c)}"  for c in cols_sc)
c_est  = elegir(["ESTADO_CONTRIBUYENTE"], cCAT); c_tip = elegir(["TIPO_CONTRIBUYENTE"], cCAT)
c_fini = elegir(["FECHA_INICIO_ACTIVIDADES"], cCAT); c_fsus = elegir(["FECHA_SUSPENSION_DEFINITIVA"], cCAT)
c_frei = elegir(["FECHA_REINICIO_ACTIVIDADES"], cCAT)
CORTE  = "DATE '2026-09-05'"
 
def der(expr, nombre):  return f"{expr} AS {qc(nombre)}"
derivadas = []
if c_est:  derivadas += [der(f"(upper(trim(CAST(c.{qc(c_est)} AS VARCHAR))) = 'SUSPENDIDO')", "sri_suspendido_hoy"),
                         der(f"(upper(trim(CAST(c.{qc(c_est)} AS VARCHAR))) = 'ACTIVO')", "sri_activo_hoy")]
if c_tip:  derivadas += [der(f"(upper(trim(CAST(c.{qc(c_tip)} AS VARCHAR))) = 'SOCIEDAD')", "sri_es_sociedad")]
if c_fsus: derivadas += [der(f"(TRY_CAST(c.{qc(c_fsus)} AS DATE) IS NOT NULL AND TRY_CAST(c.{qc(c_fsus)} AS DATE) <= {CORTE})", "sri_tiene_suspension_definitiva")]
if c_frei: derivadas += [der(f"(TRY_CAST(c.{qc(c_frei)} AS DATE) IS NOT NULL)", "sri_tiene_reinicio")]
if c_fini: derivadas += [der(f"(TRY_CAST(c.{qc(c_fini)} AS DATE) BETWEEN DATE '1900-01-01' AND {CORTE})", "sri_fecha_inicio_plausible"),
                         der(f"TRY_CAST(c.{qc(c_fini)} AS DATE)", "sri_fecha_inicio_actividades_fecha")]
DIMP = PLAN_A / "planA_dim_proveedor.parquet"
con.execute(f"""COPY (
 WITH p AS (
   SELECT {norm(c_ruc)} AS ruc, count(*) AS n_procesos,
          count(*) FILTER (WHERE planA_segmento='S3_modelo_sin_catalogo') AS n_procesos_P1,
          count(*) FILTER (WHERE planA_segmento='S2_catalogo')            AS n_procesos_catalogo,
          sum(COALESCE({qc(c_monto)},0)) AS monto_total,
          min(planA_anio) AS primer_anio, max(planA_anio) AS ultimo_anio,
          count(DISTINCT {qc(c_ent)}) AS n_entidades
   FROM read_parquet({sp(U)}) WHERE {qc(c_ruc)} IS NOT NULL AND {norm(c_ruc)} <> '' GROUP BY 1),
 c AS (SELECT * FROM read_parquet({sp(CAT)}) QUALIFY row_number() OVER (PARTITION BY {norm(kc)} ORDER BY {norm(kc)}) = 1),
 s AS (SELECT * FROM read_parquet({sp(SC)}) WHERE {qc(ks)} IS NOT NULL
       QUALIFY row_number() OVER (PARTITION BY {norm(ks)} ORDER BY {norm(ks)}) = 1)
 SELECT p.*, (c.{qc(kc)} IS NOT NULL) AS en_sri, (s.{qc(ks)} IS NOT NULL) AS en_supercias,
        {", ".join(derivadas)}, {sel_c}, {sel_s}
 FROM p LEFT JOIN c ON trim(CAST(c.{qc(kc)} AS VARCHAR)) = p.ruc
        LEFT JOIN s ON trim(CAST(s.{qc(ks)} AS VARCHAR)) = p.ruc
) TO {sp(DIMP)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
 
# ---------- B. Dimensión de entidad compradora (una fila por RUC de entidad) ----------
ent_base = [c for c in cF if c.startswith("entidad_") and c != c_ent]
sel_e = ", ".join(f"any_value({qc(c)}) AS {qc(c)}" for c in ent_base) if ent_base else ""
DIME = PLAN_A / "planA_dim_entidad.parquet"
con.execute(f"""COPY (
 WITH e AS (
   SELECT {norm(c_ent)} AS ruc, count(*) AS n_procesos,
          count(*) FILTER (WHERE planA_segmento='S3_modelo_sin_catalogo') AS n_procesos_P1,
          count(*) FILTER (WHERE planA_segmento='S0_solo_planeacion')     AS n_solo_planeacion,
          sum(COALESCE({qc(c_monto)},0)) AS monto_total,
          count(DISTINCT {qc(c_ruc)}) AS n_proveedores{"," if sel_e else ""} {sel_e}
   FROM read_parquet({sp(U)}) WHERE {qc(c_ent)} IS NOT NULL AND {norm(c_ent)} <> '' GROUP BY 1),
 c AS (SELECT * FROM read_parquet({sp(CAT)}) QUALIFY row_number() OVER (PARTITION BY {norm(kc)} ORDER BY {norm(kc)}) = 1)
 SELECT e.*, (c.{qc(kc)} IS NOT NULL) AS en_sri, {sel_c}
 FROM e LEFT JOIN c ON trim(CAST(c.{qc(kc)} AS VARCHAR)) = e.ruc
) TO {sp(DIME)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
 
# ---------- C. Validación aparte: los fantasmas en su propio archivo, nunca en las dimensiones ----------
f_est = elegir(["proveedor_fantasma_estado"], cF); f_det = elegir(["proveedor_fantasma_detalle"], cF)
f_mar = elegir(["f_marca"], cF);                   f_rea = elegir(["f_reactivacion"], cF)
VAL = PLAN_A / "planA_validacion_fantasma.parquet"
if f_est:
    piezas = [f"{norm(c_ruc)} AS ruc", f"any_value({qc(f_est)}) AS fantasma_estado"]
    if f_det: piezas.append(f"any_value({qc(f_det)}) AS fantasma_detalle")
    if f_mar: piezas.append(f"min(TRY_CAST({qc(f_mar)} AS DATE)) AS fantasma_fecha_marca")
    if f_rea: piezas.append(f"min(TRY_CAST({qc(f_rea)} AS DATE)) AS fantasma_fecha_reactivacion")
    piezas += ["count(*) AS n_procesos_marcados",
               "count(*) FILTER (WHERE planA_segmento='S3_modelo_sin_catalogo') AS n_procesos_P1",
               "true AS fantasma_marcado_sri"]
    con.execute(f"""COPY (SELECT {", ".join(piezas)} FROM read_parquet({sp(U)})
        WHERE {qc(f_est)} IS NOT NULL AND {norm(c_ruc)} <> '' GROUP BY 1) TO {sp(VAL)} (FORMAT PARQUET)""")
    v = q(f"SELECT count(*) ruc_marcados, sum(n_procesos_marcados) procesos, sum(n_procesos_P1) procesos_P1 FROM read_parquet({sp(VAL)})").iloc[0]
    print(f"Validación (archivo aparte): {int(v.ruc_marcados)} RUC marcados | {int(v.procesos)} procesos | {int(v.procesos_P1)} en P1")
    print(q(f"SELECT fantasma_estado, count(*) ruc, sum(n_procesos_marcados) procesos FROM read_parquet({sp(VAL)}) GROUP BY 1 ORDER BY 2 DESC").to_string(index=False))
    cifra("fantasma_ruc_marcados", int(v.ruc_marcados), "universo total", "celda6")
    cifra("fantasma_procesos_marcados", int(v.procesos), "universo total", "celda6")
    cifra("fantasma_procesos_en_P1", int(v.procesos_P1), "442.295 procesos de P1", "celda6")
    print("\nOJO: 'fantasma_marcado_sri = true' significa MARCADO POR EL SRI, no 'empresa irregular';")
    print("y la ausencia de marca no es 'empresa limpia', es 'sin marca'. Solo descriptivo y validación.")
else:
    print("No encontré columnas de fantasmas en la base; no se crea el archivo de validación.")
 
# ---------- D. Llaves: comprobar que el RUC sí puede ser llave primaria en las dimensiones ----------
print("\nComprobación de llaves (una fila por RUC = llave primaria válida):")
for nom, ruta in (("dim_proveedor", DIMP), ("dim_entidad", DIME)):
    r = q(f"""SELECT count(*) filas, count(DISTINCT ruc) rucs, count(*) FILTER (WHERE ruc IS NULL) nulos
              FROM read_parquet({sp(ruta)})""").iloc[0]
    ok = "OK" if r.filas == r.rucs and r.nulos == 0 else "PROBLEMA"
    print(f"  {nom:14s} {int(r.filas):>8,} filas | {int(r.rucs):>8,} RUC | nulos {int(r.nulos)} -> {ok}")
    cifra(f"{nom}_filas", int(r.filas), "una fila por RUC", "celda6")
print(f"  procesos (ocid)  {int(q(f'SELECT count(*) n FROM read_parquet({sp(U)})').n[0]):>8,} filas | llave primaria = ocid, NO el RUC")
 
# ---------- E. Diccionario de permisos: qué puede entrar al modelo y qué no ----------
REGLAS = [
    ("llave_o_texto",    r"^ocid$|^planA_|_ruc$|_id$|nombre$|titulo$|^razon|descripcion|^tipo_contratacion$|^metodo_contratacion$|^categoria_principal$"),
    ("prohibido_fuga",   r"fantasma|^f_marca$|^f_reactivacion$|estado_actual|riesgoso|^desviacion_precio_log_z$"),
    ("prohibido_canal_lateral", r"^monto_tender$"),
    ("fuera_de_alcance_redes",  r"^red_|^cob_|^com_|^idx_|coreness|dominancia|rivales|^n_comp_|^rep_|^grupo_|^c_conc_rival$|^c_dominancia$|coincidencias|^n_pares_"),
    ("descriptivo_estado_actual", r"^entidad_superintendencia|^proveedor_superintendencia|^entidad_fecha|_sri$"),
]
# columnas sin varianza en P1 (un solo valor): no pueden ser variables
desc_P1 = q(f"DESCRIBE SELECT * FROM read_parquet({sp(P1)})")
planas = set()
for c, tp in zip(desc_P1.column_name, desc_P1.column_type):
    if tp.endswith("[]") or tp.startswith("STRUCT") or tp.startswith("MAP"): continue
    n = q(f"SELECT count(DISTINCT {qc(c)}) n FROM read_parquet({sp(P1)})").n[0]
    if n <= 1: planas.add(c)
print(f"\nColumnas sin varianza en P1 (un solo valor o todo nulo): {len(planas)}")
if planas: print("  " + ", ".join(sorted(planas)))
filas = []
for c in columnas(P1):
    uso = "modelo_candidato"
    for etq, pat in REGLAS:
        if re.search(pat, c, re.I): uso = etq; break
    if uso == "modelo_candidato" and c in planas: uso = "sin_varianza"
    filas.append((c, uso))
dic = pd.DataFrame(filas, columns=["columna", "uso"])
dic = dic.merge(nul[["columna", "pct_sin_dato"]], on="columna", how="left") if "nul" in dir() else dic
dic.sort_values(["uso", "columna"]).to_csv(TABLAS / "planA_diccionario_permisos.csv", index=False, encoding="utf-8-sig")
print("\nColumnas de P1 por uso permitido:\n")
print(dic.uso.value_counts().rename_axis("uso").reset_index(name="columnas").to_string(index=False))
print(f"\n  -> {TABLAS/'planA_diccionario_permisos.csv'} (revísalo: si algo quedó mal clasificado, dímelo)")
for k, v in dic.uso.value_counts().items():
    cifra(f"P1_columnas_{k}", int(v), "200 columnas de P1", "celda6")
print("\nCandidatas a variable del modelo (las primeras 40):")
print(", ".join(dic.loc[dic.uso == "modelo_candidato", "columna"].head(40)))

In [ ]:
# ===== CELDA 16 · Arreglos finales y limpieza de la carpeta antes de compartirla =====
# Corrige dos fallos de las celdas 14 y 6, y deja la carpeta lista para enviar.
import shutil, hashlib, time, re
U2 = PLAN_A / "planA_universo_v2.parquet"
RUCF = "proveedor_ruc_final"
nf = lambda c: f"trim(CAST({qc(c)} AS VARCHAR))"

print("="*95 + "\n(1) QUITAR LA COLUMNA INTERNA planA_segmento_base\n" + "="*95)
cols_u2 = columnas(U2)
if "planA_segmento_base" in cols_u2:
    tmp = PLAN_A / "_tmp_u2.parquet"
    con.execute(f"COPY (SELECT * EXCLUDE (planA_segmento_base) FROM read_parquet({sp(U2)})) TO {sp(tmp)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    U2.unlink(); tmp.rename(U2)
    print(f"  quitada. El universo queda con {len(columnas(U2))} columnas.")
    for n, d in (("planA_v2_P1_modelo_sin_catalogo.parquet","S3_modelo_sin_catalogo"),
                 ("planA_v2_P0_catalogo.parquet","S2_catalogo"),
                 ("planA_v2_excluidos_solo_planeacion.parquet","S0_solo_planeacion"),
                 ("planA_v2_excluidos_sin_monto.parquet","S1_sin_monto"),
                 ("planA_v2_sin_proveedor_identificable.parquet","S4_sin_catalogo_sin_proveedor_o_monto")):
        con.execute(f"COPY (SELECT * FROM read_parquet({sp(U2)}) WHERE planA_segmento='{d}') TO {sp(PLAN_A/n)} (FORMAT PARQUET, COMPRESSION ZSTD)")
    print("  subconjuntos regenerados sin esa columna.")
else:
    print("  no está: nada que hacer.")
P1_2 = PLAN_A / "planA_v2_P1_modelo_sin_catalogo.parquet"

print("\n" + "="*95 + "\n(2) RECONSTRUIR LAS DIMENSIONES CON EL RUC FINAL (la celda 6 usó el declarado)\n" + "="*95)
cols_cat = [c for c in cCAT if c != kc and not re.search("fantasma|estado_actual|riesgoso", c, re.I)]
cols_sc  = [c for c in cSC  if c != ks and not re.search("fantasma|estado_actual|riesgoso", c, re.I)]
sel_c = ", ".join(f"c.{qc(c)} AS {qc('sri_'+c)}" for c in cols_cat)
sel_s = ", ".join(f"s.{qc(c)} AS {qc('sc_'+c)}"  for c in cols_sc)
c_est = elegir(["ESTADO_CONTRIBUYENTE"], cCAT); c_tip = elegir(["TIPO_CONTRIBUYENTE"], cCAT)
c_fini = elegir(["FECHA_INICIO_ACTIVIDADES"], cCAT); c_fsus = elegir(["FECHA_SUSPENSION_DEFINITIVA"], cCAT)
CORTE = "DATE '2026-09-05'"
der = []
if c_est:  der += [f"(upper(trim(CAST(c.{qc(c_est)} AS VARCHAR)))='SUSPENDIDO') AS sri_suspendido_hoy"]
if c_tip:  der += [f"(upper(trim(CAST(c.{qc(c_tip)} AS VARCHAR)))='SOCIEDAD') AS sri_es_sociedad"]
if c_fsus: der += [f"(TRY_CAST(c.{qc(c_fsus)} AS DATE) IS NOT NULL AND TRY_CAST(c.{qc(c_fsus)} AS DATE) <= {CORTE}) AS sri_tiene_suspension_definitiva"]
if c_fini: der += [f"TRY_CAST(c.{qc(c_fini)} AS DATE) AS sri_fecha_inicio_actividades_fecha",
                   f"(TRY_CAST(c.{qc(c_fini)} AS DATE) BETWEEN DATE '1900-01-01' AND {CORTE}) AS sri_fecha_inicio_plausible"]
DIMP = PLAN_A / "planA_dim_proveedor.parquet"
con.execute(f"""COPY (
 WITH p AS (
   SELECT {nf(RUCF)} AS ruc,
     any_value(proveedor_ruc_origen) AS ruc_origen_mas_comun,
     count(*) AS n_procesos,
     count(*) FILTER (WHERE planA_segmento='S3_modelo_sin_catalogo') AS n_procesos_P1,
     count(*) FILTER (WHERE planA_segmento='S2_catalogo') AS n_procesos_catalogo,
     count(*) FILTER (WHERE proveedor_ruc_origen='recuperado_del_identificador') AS n_procesos_con_ruc_recuperado,
     sum(COALESCE({qc(c_monto)},0)) AS monto_total,
     min(planA_anio) AS primer_anio, max(planA_anio) AS ultimo_anio,
     count(DISTINCT {qc(c_ent)}) AS n_entidades
   FROM read_parquet({sp(U2)}) WHERE {qc(RUCF)} IS NOT NULL AND {nf(RUCF)} <> '' GROUP BY 1),
 c AS (SELECT * FROM read_parquet({sp(CAT)}) QUALIFY row_number() OVER (PARTITION BY {nf(kc)} ORDER BY {nf(kc)})=1),
 s AS (SELECT * FROM read_parquet({sp(SC)}) WHERE {qc(ks)} IS NOT NULL
       QUALIFY row_number() OVER (PARTITION BY {nf(ks)} ORDER BY {nf(ks)})=1)
 SELECT p.*, (c.{qc(kc)} IS NOT NULL) AS en_sri, (s.{qc(ks)} IS NOT NULL) AS en_supercias,
        {", ".join(der)}, {sel_c}, {sel_s}
 FROM p LEFT JOIN c ON trim(CAST(c.{qc(kc)} AS VARCHAR))=p.ruc
        LEFT JOIN s ON trim(CAST(s.{qc(ks)} AS VARCHAR))=p.ruc
) TO {sp(DIMP)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
r = q(f"""SELECT count(*) proveedores, count(DISTINCT ruc) ruc, count(*) FILTER (WHERE en_sri) en_sri,
   count(*) FILTER (WHERE n_procesos_con_ruc_recuperado > 0) con_algun_ruc_recuperado
   FROM read_parquet({sp(DIMP)})""").iloc[0]
print(f"  dim_proveedor: {int(r.proveedores):,} filas | {int(r.ruc):,} RUC | en SRI {int(r.en_sri):,} "
      f"| con RUC recuperado en algún proceso {int(r.con_algun_ruc_recuperado):,}")
print(f"  (antes eran 73.953: entran los proveedores que aparecieron al recuperar el RUC)")
cifra("v2_dim_proveedor_filas", int(r.proveedores), "una fila por RUC final", "celda16")

DIME = PLAN_A / "planA_dim_entidad.parquet"
ent_base = [c for c in cF if c.startswith("entidad_") and c != c_ent]
sel_e = ", ".join(f"any_value({qc(c)}) AS {qc(c)}" for c in ent_base)
con.execute(f"""COPY (
 WITH e AS (SELECT {nf(c_ent)} AS ruc, count(*) AS n_procesos,
     count(*) FILTER (WHERE planA_segmento='S3_modelo_sin_catalogo') AS n_procesos_P1,
     count(*) FILTER (WHERE planA_segmento='S0_solo_planeacion') AS n_solo_planeacion,
     sum(COALESCE({qc(c_monto)},0)) AS monto_total,
     count(DISTINCT {qc(RUCF)}) AS n_proveedores{"," if sel_e else ""} {sel_e}
   FROM read_parquet({sp(U2)}) WHERE {qc(c_ent)} IS NOT NULL AND {nf(c_ent)} <> '' GROUP BY 1),
 c AS (SELECT * FROM read_parquet({sp(CAT)}) QUALIFY row_number() OVER (PARTITION BY {nf(kc)} ORDER BY {nf(kc)})=1)
 SELECT e.*, (c.{qc(kc)} IS NOT NULL) AS en_sri, {sel_c}
 FROM e LEFT JOIN c ON trim(CAST(c.{qc(kc)} AS VARCHAR))=e.ruc
) TO {sp(DIME)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f"  dim_entidad: {int(q(f'SELECT count(*) n FROM read_parquet({sp(DIME)})').n[0]):,} filas")

print("\n" + "="*95 + "\n(3) QUÉ QUEDA REALMENTE SIN PROVEEDOR IDENTIFICABLE\n" + "="*95)
S4_2 = PLAN_A / "planA_v2_sin_proveedor_identificable.parquet"
z = q(f"""SELECT proveedor_ruc_origen origen,
   CASE WHEN list_contains({qc(c_etp)},'award') THEN 'con adjudicacion' ELSE 'sin adjudicacion' END etapa,
   count(*) procesos, round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_millones
   FROM read_parquet({sp(S4_2)}) GROUP BY 1,2 ORDER BY 3 DESC""")
print(z.to_string(index=False))
z.to_csv(TABLAS / "planA_v2_S4_composicion.csv", index=False, encoding="utf-8-sig")
print("\nLos 'sin adjudicacion' no tienen proveedor porque todavía no hay a quién adjudicar.")
print("Los 'identificador_sin_ruc_ecuatoriano' con adjudicación son los proveedores extranjeros.")
for _, x in z.iterrows():
    cifra(f"v2_S4_{x.origen}_{x.etapa.replace(' ','_')}", int(x.procesos), "4.255 procesos de S4 v2", "celda16")

print("\n" + "="*95 + "\n(4) CUADRE DEL MONTO (control que debe dar exacto)\n" + "="*95)
m = q(f"""SELECT planA_segmento segmento, round(sum(COALESCE({qc(c_monto)},0))/1e6,1) mm
   FROM read_parquet({sp(U2)}) GROUP BY 1 ORDER BY 1""")
print(m.to_string(index=False))
print(f"\nSuma: {m.mm.sum():,.1f} millones  |  total canónico declarado: 66.570,9  ->  "
      + ("CUADRA" if abs(m.mm.sum()-66570.9) < 0.5 else "NO CUADRA: revisar"))

print("\n" + "="*95 + "\n(5) ORDENAR LA CARPETA PARA COMPARTIRLA\n" + "="*95)
HIST = PLAN_A / "_version_1_historico"; HIST.mkdir(exist_ok=True)
v1 = ["planA_universo_total.parquet","planA_P1_modelo_sin_catalogo.parquet","planA_P0_catalogo.parquet",
      "planA_excluidos_solo_planeacion.parquet","planA_excluidos_sin_monto.parquet",
      "planA_sin_catalogo_sin_proveedor_o_monto.parquet","planA_proveedores_ruc.parquet",
      "planA_validacion_fantasma.parquet"]
mv = 0
for n in v1:
    f = PLAN_A / n
    if f.exists(): shutil.move(str(f), str(HIST / n)); mv += 1
print(f"  {mv} archivos de la versión 1 movidos a {HIST.name}/ (no se borran, quedan para trazabilidad)")
viejo = TABLAS / "planA_diccionario_permisos.csv"
if viejo.exists():
    viejo.rename(TABLAS / "_obsoleto_planA_diccionario_permisos.csv")
    print("  diccionario de permisos de la celda 6 renombrado a _obsoleto_: el bueno es")
    print("  documentacion/diccionario_columnas.csv, que ya tiene las reglas corregidas.")

man = []
for f in sorted(PLAN_A.glob("*.parquet")):
    n = q(f"SELECT count(*) n FROM read_parquet({sp(f)})").n[0]
    h = hashlib.sha256(f.read_bytes()).hexdigest()[:16] if f.stat().st_size < 3e9 else "(omitido)"
    man.append((f.name, int(n), len(columnas(f)), round(f.stat().st_size/1e6,1),
                time.strftime("%Y-%m-%d %H:%M", time.localtime(f.stat().st_mtime)), h))
M = pd.DataFrame(man, columns=["archivo","filas","columnas","MB","creado","sha256_16"])
M.to_csv(PLAN_A / "documentacion" / "manifiesto.csv", index=False, encoding="utf-8-sig")
print("\nCARPETA LISTA PARA COMPARTIR:\n"); print(M.to_string(index=False))
print(f"\nTamaño a enviar: {M.MB.sum():,.0f} MB (la versión 1 ya no cuenta)")
print("\nVuelve a correr la CELDA 15 para que el README recoja estas correcciones.")

In [ ]:
# ===== CELDA 15 · Documentación de la base para los compañeros =====
# Requiere las celdas 1, 3, 6, 13 y 14. Genera los textos DESDE los datos, no a mano.
import hashlib, time, re
U2 = PLAN_A / "planA_universo_v2.parquet"
P1_2 = PLAN_A / "planA_v2_P1_modelo_sin_catalogo.parquet"
DOC = PLAN_A / "documentacion"; DOC.mkdir(exist_ok=True)

REGLAS = [
    ("llave_o_texto",    r"^ocid$|^planA_|_ruc$|_ruc_final$|_id$|nombre$|titulo$|^razon|descripcion|^tipo_contratacion$|^metodo_contratacion$|^categoria_principal$"),
    ("prohibido_fuga",   r"fantasma|^f_marca$|^f_reactivacion$|estado_actual|riesgoso|^desviacion_precio_log_z$|alguna_vez|tras_suspension"),
    ("prohibido_canal_lateral", r"^monto_tender$|^tiene_periodo_competitivo$|^es_catalogo$"),
    ("fuera_de_alcance_redes",  r"^red_|^cob_|^com_|^idx_|coreness|dominancia|rivales|^n_comp_|^rep_|^grupo_|^c_conc_rival$|^c_dominancia$|coincidencias|^n_pares_"),
    ("descriptivo_estado_actual", r"^entidad_superintendencia|^proveedor_superintendencia|^entidad_fecha|_sri$"),
]
desc = q(f"DESCRIBE SELECT * FROM read_parquet({sp(P1_2)})")
filas = []
for c, tp in zip(desc.column_name, desc.column_type):
    uso = "modelo_candidato"
    for etq, pat in REGLAS:
        if re.search(pat, c, re.I): uso = etq; break
    r = q(f"""SELECT round(100.0*count(*) FILTER (WHERE {qc(c)} IS NULL)/count(*),2) pct_nulo,
          count(DISTINCT {qc(c)}) distintos FROM read_parquet({sp(P1_2)})""").iloc[0]
    if int(r.distintos) <= 1 and uso == "modelo_candidato": uso = "sin_varianza"
    filas.append((c, tp, uso, float(r.pct_nulo), int(r.distintos)))
dic = pd.DataFrame(filas, columns=["columna","tipo","uso_permitido","pct_nulo_en_P1","valores_distintos"])
dic.sort_values(["uso_permitido","columna"]).to_csv(DOC / "diccionario_columnas.csv", index=False, encoding="utf-8-sig")
print("Columnas por uso permitido:\n"); print(dic.uso_permitido.value_counts().rename_axis("uso").reset_index(name="n").to_string(index=False))

seg = q(f"""SELECT planA_segmento segmento, count(*) procesos,
   round(100.0*count(*)/sum(count(*)) OVER (),2) pct, round(sum(COALESCE({qc(c_monto)},0))/1e6,1) mm
   FROM read_parquet({sp(U2)}) GROUP BY 1 ORDER BY 1""")
org = q(f"""SELECT proveedor_ruc_origen origen, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (),2) pct
   FROM read_parquet({sp(U2)}) GROUP BY 1 ORDER BY 2 DESC""")
fam = q(f"""SELECT planA_familia familia, planA_periodo_normativo periodo, count(*) n
   FROM read_parquet({sp(P1_2)}) GROUP BY 1,2 ORDER BY 1,2""")
VER_P = PLAN_A / "planA_validacion_sri_verificada.parquet"
val = q(f"""SELECT grupo, count(*) ruc, CAST(sum(n_procesos) AS BIGINT) procesos, round(sum(monto_millones),3) mm
   FROM read_parquet({sp(VER_P)}) GROUP BY 1 ORDER BY 3 DESC""") if VER_P.exists() else None
nP1 = int(q(f"SELECT count(*) n FROM read_parquet({sp(P1_2)})").n[0])
nTot = int(seg.procesos.sum())

seg2 = seg.rename(columns={"segmento":"segmento","procesos":"procesos","pct":"% del total","mm":"monto (millones USD)"})
org2 = org.rename(columns={"origen":"origen del RUC","n":"procesos","pct":"% del total"})
fam2 = fam.rename(columns={"familia":"familia","periodo":"período normativo","n":"procesos"})
val2 = val.rename(columns={"grupo":"grupo","ruc":"contribuyentes","procesos":"procesos","mm":"monto (millones)"}) if val is not None else None
uso2 = dic.uso_permitido.value_counts().rename_axis("uso permitido").reset_index(name="columnas")

def tabla(df, cols=None):
    d = df if cols is None else df[cols]
    enc = "| " + " | ".join(d.columns) + " |\n|" + "|".join([" --- "]*len(d.columns)) + "|\n"
    return enc + "".join("| " + " | ".join(f"{v:,}" if isinstance(v,(int,)) else str(v) for v in r) + " |\n"
                         for r in d.itertuples(index=False))

R = f"""# Base de datos del plan A · documentación para el equipo

Generado automáticamente el {time.strftime('%d-%m-%Y')} desde los propios archivos.
Fuente: portal de datos abiertos del SERCOP, estándar OCDS, años 2015 a 2026.
**Fecha de corte de la descarga: 5 de septiembre de 2026.**

## 1. Advertencia que hay que leer antes de usar cualquier cifra

Los datos son **una fotografía del 5 de septiembre de 2026**. Un proceso que seguía vivo ese día
aparece incompleto: sin adjudicación ni contrato aunque después los haya tenido. Verificado con el
proceso RE-CEP-MSP-2026-003 del Ministerio de Salud, adjudicado el 8 de septiembre y firmado el 15,
los dos posteriores al corte. Esto afecta sobre todo a 2026.

## 2. Los archivos

| Archivo | Grano | Llave primaria | Filas |
| --- | --- | --- | --- |
| `planA_universo_v2.parquet` | un proceso de contratación | `ocid` | {nTot:,} |
| `planA_v2_P1_modelo_sin_catalogo.parquet` | un proceso | `ocid` | {nP1:,} |
| `planA_v2_P0_catalogo.parquet` | un proceso | `ocid` | {int(seg.loc[seg.segmento=='S2_catalogo','procesos'].iloc[0]):,} |
| `planA_v2_excluidos_solo_planeacion.parquet` | un proceso | `ocid` | {int(seg.loc[seg.segmento=='S0_solo_planeacion','procesos'].iloc[0]):,} |
| `planA_v2_excluidos_sin_monto.parquet` | un proceso | `ocid` | {int(seg.loc[seg.segmento=='S1_sin_monto','procesos'].iloc[0]):,} |
| `planA_v2_sin_proveedor_identificable.parquet` | un proceso | `ocid` | {int(seg.loc[seg.segmento=='S4_sin_catalogo_sin_proveedor_o_monto','procesos'].iloc[0]):,} |
| `planA_dim_proveedor.parquet` | un proveedor | RUC | una fila por RUC |
| `planA_dim_entidad.parquet` | una entidad compradora | RUC | una fila por RUC |
| `planA_validacion_sri_verificada.parquet` | un contribuyente marcado | RUC | 15 |

**El RUC no es llave primaria de la tabla de procesos.** Un mismo proveedor aparece en cientos de
procesos. Las tablas de proveedor y de entidad se unen por RUC cuando hacen falta; así no se repiten
treinta y siete columnas del SRI dos millones de veces.

## 3. Los segmentos

{tabla(seg2)}
Definiciones, aplicadas en este orden:

- **S0 solo planeación**: el registro trae únicamente la etapa `planning`. Verificado: el 100 % de ellos.
- **S1 sin monto**: no tiene monto en ninguna de las tres columnas de monto.
- **S2 catálogo**: compra por catálogo electrónico (convenio marco). Se describe pero no se modela.
- **S3 universo de modelado (P1)**: fuera del catálogo, con RUC de proveedor y monto positivo.
- **S4 sin proveedor identificable**: fuera del catálogo, sin RUC de proveedor aun tras la recuperación.

## 4. El RUC del proveedor: de dónde sale cada uno

{tabla(org2)}
En parte de los registros el campo de RUC venía vacío pero el identificador del proveedor contenía el
número, con el formato `EC-RUC-<ruc>-<id>` o `ID-<ruc>-<id>`. Ese RUC se recupera **solo si** pasa
cuatro controles: tiene trece dígitos, termina en 001, no coincide con el RUC de la entidad compradora
y existe en el catastro del SRI. La columna **`proveedor_ruc_origen`** dice de dónde salió cada uno, de
modo que la decisión es auditable y reversible. La columna **`planA_segmento_v1`** conserva la
clasificación anterior para poder comparar.

**Nunca uses `proveedor_ruc` a secas: usa `proveedor_ruc_final`.**

## 5. Grupos comparables del universo de modelado

Los modelos se ajustan dentro de cada familia de procedimiento y período normativo. El corte de período
es el **7 de octubre de 2025**, fecha de la reforma a la LOSNCP, y se asigna con la fecha de inicio de
la licitación.

{tabla(fam2)}

## 6. El conjunto de validación externa

{tabla(val2) if val2 is not None else '(falta correr la celda 13)'}
Los 15 contribuyentes se verificaron uno por uno en `consultas.sri.gob.ec` el 9 de octubre de 2026.
**No son «empresas fantasma»**: nueve están marcados por transacciones inexistentes, cinco como
empresas fantasma y uno ya no tiene ninguna marca. **Siete de los quince son personas naturales.**

Este conjunto **solo sirve para validar y describir. No es variable del modelo ni objetivo a
predecir**, y por eso vive en un archivo aparte. La tasa base es de un proceso marcado por cada 4.095
del universo de modelado, así que cualquier medida de precisión sería engañosa: se reportan percentil y
lift, y se declara que la potencia es baja.

## 7. Qué columnas NO pueden ser variables del modelo

{tabla(uso2)}
El detalle columna por columna está en `documentacion/diccionario_columnas.csv`. Los motivos:

- **prohibido_fuga**: describen el estado de hoy o vienen de la misma fuente que la etiqueta de
  validación. Las variables del tipo «fue suspendido alguna vez» informan hechos **posteriores** al
  proceso en el 85 % de los casos.
- **prohibido_canal_lateral**: identifican el tipo de procedimiento por la puerta de atrás, que es lo
  que ya usamos para agrupar.
- **fuera_de_alcance_redes**: variables de co-oferta y comunidades, que corresponden al otro tema del
  catálogo del programa y quedan fuera de esta tesis.
- **descriptivo_estado_actual**: sirven para describir, no para predecir.

## 8. Problemas de calidad conocidos de la fuente

- Los datos abiertos **no publican** modificaciones de contrato, ejecución, ni el estado de desierto,
  cancelado o suspendido. El portal transaccional sí los muestra en la ficha del proceso: es una
  brecha entre los dos sistemas del propio SERCOP.
- El identificador de la entidad compradora no coincide con el del bloque de participantes.
- Hay fechas imposibles en el catastro del SRI (104 anteriores a 1900, mínimo en el año 199) y un valor
  centinela 9999-01-01.
- La variable de año de la base es el **año de publicación del registro**, no el año en que se lanzó el
  proceso; coinciden solo en el 78,93 % de los casos. Para análisis económico se usa la fecha de inicio
  de la licitación.
- El número de oferentes no es comparable entre familias: en menor cuantía de obras refleja
  manifestaciones de interés previas al sorteo, no ofertas económicas.

## 9. Reglas de trabajo del equipo

1. Ninguna tasa ni total se escribe sin declarar sobre cuántos registros se calculó.
2. El monto canónico es `monto_final`. En el universo de modelado es siempre el monto adjudicado.
3. Toda cifra que vaya a la tesis se anota en el registro de cifras con su universo y la celda que la
   produjo.
4. Las decisiones que no resuelve el dato se verifican a mano en el portal o en el SRI, y se documenta
   qué se vio y cuándo.
"""
(DOC / "README_base_plan_A.md").write_text(R, encoding="utf-8")
print(f"\n  -> {DOC/'README_base_plan_A.md'}\n  -> {DOC/'diccionario_columnas.csv'}")

man = []
for f in sorted(PLAN_A.glob("*.parquet")):
    n = q(f"SELECT count(*) n FROM read_parquet({sp(f)})").n[0]
    h = hashlib.sha256(f.read_bytes()).hexdigest()[:16] if f.stat().st_size < 3e9 else "(omitido)"
    man.append((f.name, int(n), len(columnas(f)), round(f.stat().st_size/1e6,1),
                time.strftime("%Y-%m-%d %H:%M", time.localtime(f.stat().st_mtime)), h))
M = pd.DataFrame(man, columns=["archivo","filas","columnas","MB","creado","sha256_16"])
M.to_csv(DOC / "manifiesto.csv", index=False, encoding="utf-8-sig")
print("\nMANIFIESTO:\n"); print(M.to_string(index=False))
print(f"\nTamaño total de la carpeta: {M.MB.sum():,.0f} MB")
print("\nPara enviar a los compañeros: comprime la carpeta plan_A completa, o solo documentacion/ y")
print("planA_v2_P1_modelo_sin_catalogo.parquet si solo necesitan el universo de modelado.")

In [ ]:
# ===== CELDA 17 · Dos diagnósticos antes de construir indicadores =====
# (A) ¿Por qué hay procesos adjudicados sin contrato publicado?
# (B) Reconstruir la regla de contratación tras suspensión, esta vez con fechas correctas.
CORTE = "DATE '2026-09-05'"
fp = "COALESCE(TRY_CAST(fecha_inicio_tender AS DATE), TRY_CAST(fecha_release AS DATE))"
con_c = f"list_contains({qc(c_etp)}, 'contract')"

print("="*95 + "\n(A) ADJUDICADOS SIN CONTRATO PUBLICADO\n" + "="*95)
g = q(f"""SELECT CASE WHEN {con_c} THEN 'con contrato' ELSE 'sin contrato' END grupo,
   count(*) procesos, round(100.0*count(*)/sum(count(*)) OVER (),2) pct,
   round(sum(COALESCE({qc(c_monto)},0))/1e6,1) monto_millones,
   round(median(COALESCE({qc(c_monto)},0)),0) monto_mediano,
   round(avg(COALESCE({qc(c_monto)},0)),0) monto_promedio
   FROM read_parquet({sp(P1)}) GROUP BY 1 ORDER BY 2 DESC""")
print(g.to_string(index=False))
n_sin = int(g.loc[g.grupo=='sin contrato','procesos'].iloc[0])
cifra("P1_adjudicados_sin_contrato", n_sin, "446.677 procesos de P1", "celda17")

print("\nPor año de licitación y días hasta el corte (si son recientes, es efecto de la foto):\n")
a = q(f"""SELECT year({fp}) anio, count(*) procesos,
   count(*) FILTER (WHERE NOT {con_c}) sin_contrato,
   round(100.0*count(*) FILTER (WHERE NOT {con_c})/count(*),2) pct_sin_contrato,
   round(avg(date_diff('day', {fp}, {CORTE})) FILTER (WHERE NOT {con_c})) dias_promedio_al_corte
   FROM read_parquet({sp(P1)}) GROUP BY 1 ORDER BY 1""")
print(a.to_string(index=False))
a.to_csv(TABLAS / "planA_sin_contrato_por_anio.csv", index=False, encoding="utf-8-sig")

print("\nPor familia:\n")
f_ = q(f"""SELECT planA_familia familia, count(*) procesos,
   count(*) FILTER (WHERE NOT {con_c}) sin_contrato,
   round(100.0*count(*) FILTER (WHERE NOT {con_c})/count(*),2) pct,
   round(median(COALESCE({qc(c_monto)},0)) FILTER (WHERE NOT {con_c}),0) monto_mediano_sin_contrato,
   round(median(COALESCE({qc(c_monto)},0)) FILTER (WHERE {con_c}),0) monto_mediano_con_contrato
   FROM read_parquet({sp(P1)}) GROUP BY 1 ORDER BY 4 DESC""")
print(f_.to_string(index=False))
f_.to_csv(TABLAS / "planA_sin_contrato_por_familia.csv", index=False, encoding="utf-8-sig")

print("\n¿Se concentra en pocas entidades?\n")
e = q(f"""WITH x AS (SELECT {qc(c_ent)} ruc, any_value({qc(c_entn)}) nombre, count(*) n,
     count(*) FILTER (WHERE NOT {con_c}) sc FROM read_parquet({sp(P1)}) GROUP BY 1 HAVING n >= 100)
   SELECT nombre entidad, n procesos, sc sin_contrato, round(100.0*sc/n,1) pct
   FROM x ORDER BY 4 DESC LIMIT 10""")
print(e.to_string(index=False))
tot_e = q(f"""WITH x AS (SELECT {qc(c_ent)} ruc, count(*) FILTER (WHERE NOT {con_c}) sc
     FROM read_parquet({sp(P1)}) GROUP BY 1)
   SELECT count(*) FILTER (WHERE sc > 0) entidades_con_casos, count(*) entidades_total FROM x""").iloc[0]
print(f"\nEntidades con al menos un caso: {int(tot_e.entidades_con_casos):,} de {int(tot_e.entidades_total):,}")
nc = elegir(["n_contracts"], cF)
if nc:
    print(f"\nValores de {nc} en los que NO tienen etapa de contrato:")
    print(q(f"""SELECT COALESCE(CAST({qc(nc)} AS VARCHAR),'(nulo)') v, count(*) n
          FROM read_parquet({sp(P1)}) WHERE NOT {con_c} GROUP BY 1 ORDER BY 2 DESC LIMIT 5""").to_string(index=False))
print("\nCómo leerlo: si se reparte por años antiguos y por todas las entidades, NO es la fecha de")
print("corte ni un problema de una institución: es que el portal no publica el contrato de una parte")
print("de los procesos, y entra como limitación. Si se concentra en 2026 o en pocas entidades, es otra cosa.")

print("\n" + "="*95 + "\n(B) CONTRATACIÓN CON PROVEEDOR SUSPENDIDO, RECONSTRUIDA\n" + "="*95)
fsus = elegir(["proveedor_fecha_suspension_definitiva"], cF)
frei = elegir(["proveedor_fecha_reinicio_actividades"], cF)
ctr  = elegir(["proveedor_contrato_tras_suspension"], cF)
if fsus:
    S = f"TRY_CAST({qc(fsus)} AS DATE)"
    R = f"TRY_CAST({qc(frei)} AS DATE)" if frei else "CAST(NULL AS DATE)"
    regla = f"({S} IS NOT NULL AND {S} < {fp} AND ({R} IS NULL OR {R} > {fp}))"
    r = q(f"""SELECT count(*) procesos_P1,
      count(*) FILTER (WHERE {S} IS NOT NULL) proveedor_con_fecha_de_suspension,
      count(*) FILTER (WHERE {regla}) suspendido_a_la_fecha_del_proceso,
      round(sum(COALESCE({qc(c_monto)},0)) FILTER (WHERE {regla})/1e6,1) monto_millones,
      count(DISTINCT CASE WHEN {regla} THEN proveedor_ruc_final END) proveedores
      FROM read_parquet({sp(P1)})""").iloc[0]
    print("Regla correcta: la suspensión es ANTERIOR al proceso y todavía no había reinicio.\n")
    print(r.to_string())
    print("\nComparación con la columna existente (que está construida al revés):\n")
    if ctr:
        print(q(f"""SELECT {qc(ctr)} columna_existente, {regla} AS regla_reconstruida, count(*) procesos
              FROM read_parquet({sp(P1)}) GROUP BY 1,2 ORDER BY 3 DESC""").to_string(index=False))
    print("\nLos de la diagonal coinciden; los de fuera son los que la columna vieja marca mal.")
    print("La columna existente se RETIRA y se usa la regla reconstruida, que se reporta aparte de los")
    print("modelos como hallazgo verificable, no como variable.")
    cifra("P1_suspendido_a_la_fecha_del_proceso", int(r.suspendido_a_la_fecha_del_proceso), "446.677 procesos de P1", "celda17")
    cifra("P1_suspendido_monto_MM", float(r.monto_millones), "ídem", "celda17")
    d = q(f"""SELECT planA_familia familia, count(*) procesos,
          count(DISTINCT proveedor_ruc_final) proveedores,
          round(sum(COALESCE({qc(c_monto)},0))/1e6,2) monto_millones,
          round(avg(date_diff('day', {S}, {fp}))) dias_promedio_tras_la_suspension
          FROM read_parquet({sp(P1)}) WHERE {regla} GROUP BY 1 ORDER BY 4 DESC""")
    print("\nPor familia:\n"); print(d.to_string(index=False))
    d.to_csv(TABLAS / "planA_contratacion_tras_suspension.csv", index=False, encoding="utf-8-sig")
    print("\nADVERTENCIA que hay que escribir junto a la cifra: la fecha del proceso aquí es la de inicio")
    print("de la licitación, no la de adjudicación ni la de firma, que todavía no están extraídas del")
    print("JSON. Con la fecha de firma la cifra será mayor, porque la firma es posterior. Es una cota")
    print("INFERIOR, y así hay que declararla hasta correr la extracción complementaria.")
else:
    print("falta la columna de fecha de suspensión")

In [ ]:
# ===== CELDA 18 · Constructor de indicadores por proceso (los que no dependen del JSON) =====
# Produce plan_A/planA_indicadores_v1.parquet, una fila por ocid, SIN tocar el universo.
# Tres reglas que se aplican sin excepción:
#   1) Punto en el tiempo: todo historial mira solo hacia atrás, nunca al futuro del proceso.
#   2) Relativo al grupo comparable: familia de procedimiento x periodo normativo.
#   3) Nada se imputa: cada indicador lleva su indicadora de ausencia.
import time
IND = PLAN_A / "planA_indicadores_v1.parquet"
P1  = PLAN_A / "planA_v2_P1_modelo_sin_catalogo.parquet"
CORTE = "DATE '2026-09-05'"
F = "COALESCE(TRY_CAST(fecha_inicio_tender AS DATE), TRY_CAST(fecha_release AS DATE))"
V12 = "RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND CURRENT ROW"
V30 = "RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND CURRENT ROW"

cP1 = columnas(P1)
c_cpc  = elegir(["cpc","cpc5","codigo_cpc","clasificacion_cpc","item_cpc","cpc_codigo","categoria_cpc"], cP1)
c_fini = elegir(["proveedor_fecha_inicio_actividades"], cP1)
c_cap  = elegir(["proveedor_capital_suscrito"], cP1)
c_unic = elegir(["proveedor_unico"], cP1)
c_nof  = elegir(["n_oferentes"], cP1)
c_pprov= elegir(["provincia_proveedor"], cP1)
c_pent = elegir(["provincia_entidad"], cP1)
print("Columnas detectadas:")
for k, v in dict(cpc=c_cpc, fecha_inicio_prov=c_fini, capital=c_cap, oferente_unico=c_unic,
                 n_oferentes=c_nof, prov_proveedor=c_pprov, prov_entidad=c_pent).items():
    print(f"  {k:20s} -> {v}")
if not c_cpc:
    print("\n  AVISO: no encontré columna de CPC. El fraccionamiento y el precio se calculan sin CPC,")
    print("  que es una versión más gruesa. Si existe con otro nombre, dímelo y la uso.")
CPC5 = f"substr(regexp_replace(CAST({qc(c_cpc)} AS VARCHAR), '[^0-9]', '', 'g'), 1, 5)" if c_cpc else "CAST(NULL AS VARCHAR)"
CPC3 = f"substr(regexp_replace(CAST({qc(c_cpc)} AS VARCHAR), '[^0-9]', '', 'g'), 1, 3)" if c_cpc else "CAST(NULL AS VARCHAR)"

t0 = time.time()
con.execute(f"""CREATE OR REPLACE TEMP VIEW b AS
SELECT ocid, proveedor_ruc_final AS prov, {qc(c_ent)} AS ent, planA_familia AS fam,
  planA_periodo_normativo AS periodo, {F} AS f, year({F}) AS anio,
  COALESCE({qc(c_monto)}, 0) AS monto, {CPC5} AS cpc5, {CPC3} AS cpc3,
  {f'TRY_CAST({qc(c_fini)} AS DATE)' if c_fini else 'CAST(NULL AS DATE)'} AS f_ini_prov,
  {f'{qc(c_cap)}' if c_cap else 'CAST(NULL AS DOUBLE)'} AS capital,
  {f'CAST({qc(c_unic)} AS BOOLEAN)' if c_unic else 'CAST(NULL AS BOOLEAN)'} AS unico,
  {f'{qc(c_nof)}' if c_nof else 'CAST(NULL AS BIGINT)'} AS n_of,
  {f'CAST({qc(c_pprov)} AS VARCHAR)' if c_pprov else 'CAST(NULL AS VARCHAR)'} AS pv,
  {f'CAST({qc(c_pent)} AS VARCHAR)' if c_pent else 'CAST(NULL AS VARCHAR)'} AS pe
FROM read_parquet({sp(P1)})""")

# ---- historial punto en el tiempo (la ventana incluye el día y se resta el propio proceso) ----
con.execute(f"""CREATE OR REPLACE TEMP VIEW h AS
SELECT *,
  sum(monto) OVER (PARTITION BY ent, prov ORDER BY f {V12}) - monto  AS par_monto_12m,
  count(*)   OVER (PARTITION BY ent, prov ORDER BY f {V12}) - 1      AS par_n_12m,
  sum(monto) OVER (PARTITION BY ent       ORDER BY f {V12}) - monto  AS ent_monto_12m,
  count(*)   OVER (PARTITION BY ent       ORDER BY f {V12}) - 1      AS ent_n_12m,
  count(*)   OVER (PARTITION BY prov      ORDER BY f {V12}) - 1      AS prov_n_12m,
  row_number() OVER (PARTITION BY prov ORDER BY f, ocid)             AS orden_prov,
  row_number() OVER (PARTITION BY ent, prov ORDER BY f, ocid)        AS orden_par,
  count(*) OVER (PARTITION BY ent, prov, COALESCE(cpc5,'(sin)') ORDER BY f {V30}) - 1 AS frac_n_30d,
  sum(monto) OVER (PARTITION BY ent, prov, COALESCE(cpc5,'(sin)') ORDER BY f {V30}) - monto AS frac_monto_30d
FROM b""")

# ---- referencia de precio: mediana y MAD del log del monto en el grupo mas fino con 30 o mas ----
con.execute("""CREATE OR REPLACE TEMP VIEW lg AS SELECT *, ln(greatest(monto, 1.0)) AS lm FROM h""")
for n, k in (("g5", "cpc5, fam, anio"), ("g3", "cpc3, fam, anio"), ("gf", "fam, anio")):
    con.execute(f"""CREATE OR REPLACE TEMP VIEW med_{n} AS
      SELECT {k}, count(*) AS n, median(lm) AS md FROM lg GROUP BY {k}""")
    con.execute(f"""CREATE OR REPLACE TEMP VIEW mad_{n} AS
      SELECT l.{k.split(',')[0].strip()} AS k1, l.fam, l.anio, m.n, m.md,
             median(abs(l.lm - m.md)) AS mad
      FROM lg l JOIN med_{n} m USING ({k}) GROUP BY 1,2,3,4,5""")
con.execute(f"""CREATE OR REPLACE TEMP VIEW ref AS
SELECT l.ocid,
  CASE WHEN a5.n >= 30 THEN 'cpc5' WHEN a3.n >= 30 THEN 'cpc3' ELSE 'familia' END AS nivel_referencia,
  CASE WHEN a5.n >= 30 THEN a5.md WHEN a3.n >= 30 THEN a3.md ELSE af.md END AS md,
  CASE WHEN a5.n >= 30 THEN a5.mad WHEN a3.n >= 30 THEN a3.mad ELSE af.mad END AS mad,
  CASE WHEN a5.n >= 30 THEN a5.n WHEN a3.n >= 30 THEN a3.n ELSE af.n END AS n_grupo
FROM lg l
LEFT JOIN mad_g5 a5 ON a5.k1 = l.cpc5 AND a5.fam = l.fam AND a5.anio = l.anio
LEFT JOIN mad_g3 a3 ON a3.k1 = l.cpc3 AND a3.fam = l.fam AND a3.anio = l.anio
LEFT JOIN mad_gf af ON af.k1 = l.fam  AND af.fam = l.fam AND af.anio = l.anio""")

# ---- tabla final de indicadores ----
con.execute(f"""COPY (
WITH x AS (
 SELECT h.*, r.nivel_referencia, r.md, r.mad, r.n_grupo, ln(greatest(h.monto,1.0)) AS lm
 FROM h JOIN ref r USING (ocid)),
y AS (
 SELECT ocid, prov AS proveedor_ruc, ent AS entidad_ruc, fam AS familia, periodo, anio, f AS fecha_proceso,

   -- 1. concurrencia
   unico                                             AS ind_oferente_unico,
   n_of                                              AS ind_n_oferentes,
   CASE WHEN n_of IS NULL THEN NULL ELSE ln(1.0 + n_of) END AS ind_log_oferentes,

   -- 2. concentracion del proveedor en la entidad, 12 meses previos
   par_monto_12m, ent_monto_12m, par_n_12m, ent_n_12m,
   CASE WHEN ent_monto_12m > 0 THEN par_monto_12m / ent_monto_12m END AS ind_cuota_prov_en_entidad_12m,
   CASE WHEN ent_n_12m > 0 THEN CAST(par_n_12m AS DOUBLE) / ent_n_12m END AS ind_cuota_procesos_12m,

   -- 3. recurrencia del par y del proveedor
   par_n_12m                                         AS ind_recurrencia_par_12m,
   prov_n_12m                                        AS ind_procesos_proveedor_12m,
   (orden_par = 1)                                   AS ind_primera_vez_con_la_entidad,
   (orden_prov = 1)                                  AS ind_proveedor_debutante,

   -- 4. fraccionamiento: mismo par y mismo CPC en 30 dias
   frac_n_30d                                        AS ind_fraccionamiento_n_30d,
   frac_monto_30d                                    AS ind_fraccionamiento_monto_30d,
   (frac_n_30d >= 2)                                 AS ind_fraccionamiento_marca,

   -- 5. proveedor: antiguedad y capacidad
   CASE WHEN f_ini_prov BETWEEN DATE '1900-01-01' AND {CORTE} AND f_ini_prov <= f
        THEN date_diff('day', f_ini_prov, f) END     AS ind_antiguedad_dias,
   CASE WHEN capital > 0 THEN monto / capital END    AS ind_monto_sobre_capital,

   -- 6. geografia
   CASE WHEN pv IS NOT NULL AND pe IS NOT NULL THEN (pv <> pe) END AS ind_provincia_distinta,

   -- 7. precio (provisional: monto total; se reemplaza por precio unitario cuando llegue el JSON)
   nivel_referencia                                  AS ind_precio_nivel_referencia,
   n_grupo                                           AS ind_precio_n_grupo,
   CASE WHEN mad > 0 THEN (lm - md) / (1.4826 * mad) END AS ind_desv_monto_log_z,
   monto                                             AS monto_final
 FROM x)
SELECT y.*,
   -- indicadoras de ausencia, una por indicador que puede faltar
   (ind_n_oferentes IS NULL)                 AS falta_n_oferentes,
   (ind_cuota_prov_en_entidad_12m IS NULL)   AS falta_cuota_12m,
   (ind_antiguedad_dias IS NULL)             AS falta_antiguedad,
   (ind_monto_sobre_capital IS NULL)         AS falta_capital,
   (ind_provincia_distinta IS NULL)          AS falta_provincia,
   (ind_desv_monto_log_z IS NULL)            AS falta_desv_precio,
   -- percentil dentro del grupo comparable (familia x periodo), que es como entran al modelo
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_desv_monto_log_z)      AS pct_desv_precio,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_cuota_prov_en_entidad_12m) AS pct_cuota_12m,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_recurrencia_par_12m)   AS pct_recurrencia_par,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_monto_sobre_capital)   AS pct_monto_capital,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_fraccionamiento_n_30d) AS pct_fraccionamiento,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_antiguedad_dias)      AS pct_proveedor_nuevo,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_n_oferentes)          AS pct_poca_concurrencia
FROM y) TO {sp(IND)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f"\nEscrito {IND.name} ({time.time()-t0:.0f} s)")

n = int(q(f"SELECT count(*) n FROM read_parquet({sp(IND)})").n[0])
print(f"Filas: {n:,} | ocid repetidos: {int(q(f'SELECT count(*)-count(DISTINCT ocid) d FROM read_parquet({sp(IND)})').d[0])}")
cifra("indicadores_filas", n, "universo de modelado P1", "celda18")

print("\n" + "="*95 + "\nCOBERTURA DE CADA INDICADOR\n" + "="*95)
inds = [c for c in columnas(IND) if c.startswith("ind_")]
sel = ", ".join(f"round(100.0*count({qc(c)})/count(*),2) AS {qc(c)}" for c in inds)
cob = q(f"SELECT {sel} FROM read_parquet({sp(IND)})").T.rename(columns={0:"pct_con_dato"})
print(cob.to_string())
cob.to_csv(TABLAS / "planA_indicadores_cobertura.csv", encoding="utf-8-sig")

print("\n" + "="*95 + "\nPRUEBA CLAVE: EN EL PRIMER AÑO EL HISTORIAL DEBE ESTAR VACÍO\n" + "="*95)
pr = q(f"""SELECT anio, count(*) procesos,
   round(avg(CAST(ind_recurrencia_par_12m AS DOUBLE)),3) recurrencia_par_promedio,
   round(avg(CAST(ind_procesos_proveedor_12m AS DOUBLE)),3) procesos_prov_promedio,
   round(100.0*avg(CASE WHEN ind_proveedor_debutante THEN 1 ELSE 0 END),1) pct_debutantes,
   round(avg(ind_cuota_prov_en_entidad_12m),4) cuota_promedio
   FROM read_parquet({sp(IND)}) GROUP BY 1 ORDER BY 1""")
print(pr.to_string(index=False))
print("\nEn 2015 la recurrencia y los procesos previos deben ser 0 o casi 0, y el porcentaje de")
print("debutantes cercano al 100 %. Si no, la ventana está mirando al futuro y hay que corregirla.")

print("\n" + "="*95 + "\nDISTRIBUCIÓN DE LOS INDICADORES PRINCIPALES\n" + "="*95)
d = q(f"""SELECT 'desviacion de monto (z)' ind, round(min(ind_desv_monto_log_z),2) minimo,
   round(quantile_cont(ind_desv_monto_log_z,0.25),2) q1, round(median(ind_desv_monto_log_z),2) mediana,
   round(quantile_cont(ind_desv_monto_log_z,0.75),2) q3, round(max(ind_desv_monto_log_z),2) maximo
   FROM read_parquet({sp(IND)})
 UNION ALL SELECT 'cuota del proveedor 12m', round(min(ind_cuota_prov_en_entidad_12m),3),
   round(quantile_cont(ind_cuota_prov_en_entidad_12m,0.25),3), round(median(ind_cuota_prov_en_entidad_12m),3),
   round(quantile_cont(ind_cuota_prov_en_entidad_12m,0.75),3), round(max(ind_cuota_prov_en_entidad_12m),3)
   FROM read_parquet({sp(IND)})
 UNION ALL SELECT 'antiguedad (anios)', round(min(ind_antiguedad_dias)/365.25,1),
   round(quantile_cont(ind_antiguedad_dias,0.25)/365.25,1), round(median(ind_antiguedad_dias)/365.25,1),
   round(quantile_cont(ind_antiguedad_dias,0.75)/365.25,1), round(max(ind_antiguedad_dias)/365.25,1)
   FROM read_parquet({sp(IND)})
 UNION ALL SELECT 'monto sobre capital', round(min(ind_monto_sobre_capital),2),
   round(quantile_cont(ind_monto_sobre_capital,0.25),2), round(median(ind_monto_sobre_capital),2),
   round(quantile_cont(ind_monto_sobre_capital,0.75),2), round(max(ind_monto_sobre_capital),2)
   FROM read_parquet({sp(IND)})""")
print(d.to_string(index=False))

print("\nNivel de referencia usado para el precio (cuanto más fino, mejor):\n")
print(q(f"""SELECT ind_precio_nivel_referencia nivel, count(*) procesos,
   round(100.0*count(*)/sum(count(*)) OVER (),1) pct, round(median(ind_precio_n_grupo)) tamano_mediano_del_grupo
   FROM read_parquet({sp(IND)}) GROUP BY 1 ORDER BY 2 DESC""").to_string(index=False))

print("\nMarcas binarias:\n")
for c in ["ind_oferente_unico","ind_proveedor_debutante","ind_primera_vez_con_la_entidad",
          "ind_fraccionamiento_marca","ind_provincia_distinta"]:
    r = q(f"""SELECT round(100.0*count(*) FILTER (WHERE {qc(c)})/count({qc(c)}),2) pct,
          count({qc(c)}) universo FROM read_parquet({sp(IND)})""").iloc[0]
    print(f"  {c:35s} {r.pct:6.2f} %  sobre {int(r.universo):,} procesos con dato")
    cifra(c, float(r.pct), f"{int(r.universo):,} procesos de P1 con dato", "celda18")
print("\nSi alguna marca sale en 0 % o en 100 %, el indicador no discrimina y hay que revisarlo.")

In [ ]:
# ===== CELDA 19 · Indicadores versión 2: con el CPC y el precio unitario del JSON =====
# Reemplaza a planA_indicadores_v1. Dos cambios de fondo:
#   - el precio se compara dentro de CPC x unidad de medida, no dentro de la familia entera;
#   - se agregan los plazos, que antes no existían.
import time
from pathlib import Path
J   = Path(r"C:\SERCOP_FINAL\03_resultados\c15_complemento_json.parquet")
P1  = PLAN_A / "planA_v2_P1_modelo_sin_catalogo.parquet"
IND = PLAN_A / "planA_indicadores_v2.parquet"
CORTE = "DATE '2026-09-05'"
F = "COALESCE(TRY_CAST(p.fecha_inicio_tender AS DATE), TRY_CAST(p.fecha_release AS DATE))"
V12 = "RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND CURRENT ROW"
V30 = "RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND CURRENT ROW"
cP1 = columnas(P1)
g = lambda n, t="VARCHAR": (f"p.{qc(n)}" if n in cP1 else f"CAST(NULL AS {t})")
c_fini = elegir(["proveedor_fecha_inicio_actividades"], cP1)
c_cap  = elegir(["proveedor_capital_suscrito"], cP1)

t0 = time.time()
con.execute(f"""CREATE OR REPLACE TEMP VIEW b AS
SELECT p.ocid, p.proveedor_ruc_final AS prov, p.{qc(c_ent)} AS ent, p.planA_familia AS fam,
  p.planA_periodo_normativo AS periodo, {F} AS f, year({F}) AS anio,
  COALESCE(p.{qc(c_monto)}, 0) AS monto,
  substr(regexp_replace(CAST(j.j_item_cpc AS VARCHAR),'[^0-9]','','g'),1,5) AS cpc5,
  substr(regexp_replace(CAST(j.j_item_cpc AS VARCHAR),'[^0-9]','','g'),1,3) AS cpc3,
  upper(trim(COALESCE(CAST(j.j_item_unidad AS VARCHAR),'(sin unidad)'))) AS unidad,
  j.j_item_precio_unit AS precio_unit, j.j_item_cantidad AS cantidad,
  TRY_CAST(j.j_fecha_adjudicacion AS DATE) AS f_adj, TRY_CAST(j.j_fecha_firma AS DATE) AS f_firma,
  TRY_CAST(j.j_contrato_inicio AS DATE) AS f_c_ini, TRY_CAST(j.j_contrato_fin AS DATE) AS f_c_fin,
  TRY_CAST(j.j_periodo_tender_ini AS DATE) AS f_t_ini, TRY_CAST(j.j_periodo_tender_fin AS DATE) AS f_t_fin,
  j.j_contrato_monto AS monto_contrato, j.j_n_ofertas AS n_ofertas,
  {f'TRY_CAST(p.{qc(c_fini)} AS DATE)' if c_fini else 'CAST(NULL AS DATE)'} AS f_ini_prov,
  {g(c_cap,'DOUBLE') if c_cap else 'CAST(NULL AS DOUBLE)'} AS capital,
  CAST({g('proveedor_unico','BOOLEAN')} AS BOOLEAN) AS unico, {g('n_oferentes','BIGINT')} AS n_of,
  CAST({g('provincia_proveedor')} AS VARCHAR) AS pv, CAST({g('provincia_entidad')} AS VARCHAR) AS pe
FROM read_parquet({sp(P1)}) p LEFT JOIN read_parquet({sp(J)}) j USING (ocid)""")
print(f"Base unida: {int(q('SELECT count(*) n FROM b').n[0]):,} filas")
print(q("""SELECT round(100.0*count(cpc5)/count(*),2) pct_cpc, round(100.0*count(precio_unit)/count(*),2) pct_precio,
   round(100.0*count(f_adj)/count(*),2) pct_fecha_adj, round(100.0*count(f_firma)/count(*),2) pct_firma
   FROM b""").T.rename(columns={0:"cobertura %"}).to_string())

con.execute(f"""CREATE OR REPLACE TEMP VIEW h AS SELECT *,
  sum(monto) OVER (PARTITION BY ent, prov ORDER BY f {V12}) - monto AS par_monto_12m,
  count(*)   OVER (PARTITION BY ent, prov ORDER BY f {V12}) - 1     AS par_n_12m,
  sum(monto) OVER (PARTITION BY ent       ORDER BY f {V12}) - monto AS ent_monto_12m,
  count(*)   OVER (PARTITION BY ent       ORDER BY f {V12}) - 1     AS ent_n_12m,
  count(*)   OVER (PARTITION BY prov      ORDER BY f {V12}) - 1     AS prov_n_12m,
  row_number() OVER (PARTITION BY prov ORDER BY f, ocid)            AS orden_prov,
  row_number() OVER (PARTITION BY ent, prov ORDER BY f, ocid)       AS orden_par,
  count(*) OVER (PARTITION BY ent, prov, COALESCE(cpc5,'(sin)') ORDER BY f {V30}) - 1 AS frac_n_30d,
  sum(monto) OVER (PARTITION BY ent, prov, COALESCE(cpc5,'(sin)') ORDER BY f {V30}) - monto AS frac_monto_30d,
  ln(greatest(precio_unit, 0.01)) AS lpu
FROM b""")

# ---- referencia de precio UNITARIO en cuatro niveles, del mas fino al mas grueso ----
NIV = [("n1", ["cpc5","unidad","fam","anio"], "cpc5 x unidad"), ("n2", ["cpc5","fam","anio"], "cpc5"),
       ("n3", ["cpc3","fam","anio"], "cpc3"),                  ("n4", ["fam","anio"], "familia")]
for nom, cols, _ in NIV:
    k = ", ".join(cols)
    kh = ", ".join(f"h.{c}" for c in cols)
    con.execute(f"""CREATE OR REPLACE TEMP VIEW m_{nom} AS
      SELECT {k}, count(lpu) n, median(lpu) md FROM h WHERE precio_unit IS NOT NULL GROUP BY {k}""")
    con.execute(f"""CREATE OR REPLACE TEMP VIEW d_{nom} AS
      SELECT {kh}, any_value(x.n) n, any_value(x.md) md, median(abs(h.lpu - x.md)) mad
      FROM h JOIN m_{nom} x USING ({k}) WHERE h.precio_unit IS NOT NULL GROUP BY {kh}""")
con.execute("""CREATE OR REPLACE TEMP VIEW ref AS
SELECT h.ocid,
  CASE WHEN a.n>=30 THEN 'cpc5 x unidad' WHEN b2.n>=30 THEN 'cpc5'
       WHEN c.n>=30 THEN 'cpc3' WHEN d.n>=30 THEN 'familia' END AS nivel,
  CASE WHEN a.n>=30 THEN a.md WHEN b2.n>=30 THEN b2.md WHEN c.n>=30 THEN c.md WHEN d.n>=30 THEN d.md END AS md,
  CASE WHEN a.n>=30 THEN a.mad WHEN b2.n>=30 THEN b2.mad WHEN c.n>=30 THEN c.mad WHEN d.n>=30 THEN d.mad END AS mad,
  CASE WHEN a.n>=30 THEN a.n WHEN b2.n>=30 THEN b2.n WHEN c.n>=30 THEN c.n WHEN d.n>=30 THEN d.n END AS n_grupo
FROM h
LEFT JOIN d_n1 a  ON a.cpc5=h.cpc5 AND a.unidad=h.unidad AND a.fam=h.fam AND a.anio=h.anio
LEFT JOIN d_n2 b2 ON b2.cpc5=h.cpc5 AND b2.fam=h.fam AND b2.anio=h.anio
LEFT JOIN d_n3 c  ON c.cpc3=h.cpc3 AND c.fam=h.fam AND c.anio=h.anio
LEFT JOIN d_n4 d  ON d.fam=h.fam AND d.anio=h.anio""")

con.execute(f"""COPY (
WITH x AS (SELECT h.*, r.nivel, r.md, r.mad, r.n_grupo FROM h LEFT JOIN ref r USING (ocid)),
y AS (SELECT ocid, prov AS proveedor_ruc, ent AS entidad_ruc, fam AS familia, periodo, anio,
   f AS fecha_proceso, monto AS monto_final, cpc5, unidad,
   unico AS ind_oferente_unico, n_of AS ind_n_oferentes,
   CASE WHEN ent_monto_12m > 0 THEN par_monto_12m/ent_monto_12m END AS ind_cuota_prov_en_entidad_12m,
   CASE WHEN ent_n_12m > 0 THEN CAST(par_n_12m AS DOUBLE)/ent_n_12m END AS ind_cuota_procesos_12m,
   par_n_12m AS ind_recurrencia_par_12m, prov_n_12m AS ind_procesos_proveedor_12m,
   (orden_par = 1) AS ind_primera_vez_con_la_entidad, (orden_prov = 1) AS ind_proveedor_debutante,
   frac_n_30d AS ind_fraccionamiento_n_30d, (frac_n_30d >= 2) AS ind_fraccionamiento_marca,
   frac_monto_30d AS ind_fraccionamiento_monto_30d,
   CASE WHEN f_ini_prov BETWEEN DATE '1900-01-01' AND {CORTE} AND f_ini_prov <= f
        THEN date_diff('day', f_ini_prov, f) END AS ind_antiguedad_dias,
   (f_ini_prov < DATE '1950-01-01') AS alerta_fecha_inicio_sospechosa,
   CASE WHEN capital > 0 THEN monto/capital END AS ind_monto_sobre_capital,
   CASE WHEN pv IS NOT NULL AND pe IS NOT NULL THEN (pv <> pe) END AS ind_provincia_distinta,
   -- precio unitario, el bueno
   nivel AS ind_precio_nivel, n_grupo AS ind_precio_n_grupo, precio_unit, cantidad,
   CASE WHEN mad > 0 AND precio_unit IS NOT NULL THEN (lpu - md)/(1.4826*mad) END AS ind_desv_precio_unit_z,
   -- plazos
   CASE WHEN f_t_fin >= f_t_ini THEN date_diff('day', f_t_ini, f_t_fin) END AS ind_plazo_ofertas_dias,
   CASE WHEN f_adj   >= f      THEN date_diff('day', f, f_adj) END          AS ind_plazo_inicio_adj_dias,
   CASE WHEN f_firma >= f_adj  THEN date_diff('day', f_adj, f_firma) END    AS ind_plazo_adj_firma_dias,
   CASE WHEN f_c_fin >= f_c_ini THEN date_diff('day', f_c_ini, f_c_fin) END AS ind_duracion_contrato_dias,
   CASE WHEN monto > 0 AND monto_contrato > 0 THEN monto_contrato/monto END AS ind_contrato_sobre_adjudicado,
   n_ofertas AS ind_n_ofertas_bids
 FROM x)
SELECT y.*,
   (ind_desv_precio_unit_z IS NULL) AS falta_precio, (ind_plazo_inicio_adj_dias IS NULL) AS falta_plazo_adj,
   (ind_plazo_adj_firma_dias IS NULL) AS falta_plazo_firma, (ind_duracion_contrato_dias IS NULL) AS falta_duracion,
   (ind_cuota_prov_en_entidad_12m IS NULL) AS falta_cuota_12m, (ind_antiguedad_dias IS NULL) AS falta_antiguedad,
   (ind_monto_sobre_capital IS NULL) AS falta_capital, (ind_provincia_distinta IS NULL) AS falta_provincia,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_desv_precio_unit_z)        AS pct_precio_alto,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_cuota_prov_en_entidad_12m) AS pct_cuota_12m,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_recurrencia_par_12m)       AS pct_recurrencia_par,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_monto_sobre_capital)       AS pct_monto_capital,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_fraccionamiento_n_30d)     AS pct_fraccionamiento,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_antiguedad_dias)          AS pct_proveedor_nuevo,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_n_oferentes)              AS pct_poca_concurrencia,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_plazo_ofertas_dias)       AS pct_plazo_corto,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_plazo_adj_firma_dias)     AS pct_firma_rapida
FROM y) TO {sp(IND)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f"\nEscrito {IND.name} ({time.time()-t0:.0f} s) | filas: {int(q(f'SELECT count(*) n FROM read_parquet({sp(IND)})').n[0]):,}")

print("\n" + "="*95 + "\n(1) EL PRECIO, AHORA CON CPC REAL\n" + "="*95)
r = q(f"""SELECT COALESCE(ind_precio_nivel,'(sin referencia)') nivel, count(*) procesos,
   round(100.0*count(*)/sum(count(*)) OVER (),1) pct, round(median(ind_precio_n_grupo)) grupo_mediano
   FROM read_parquet({sp(IND)}) GROUP BY 1 ORDER BY 2 DESC""")
print(r.to_string(index=False))
print("\nAntes el 100 % caía en 'familia' con grupos de 9.263 procesos. Cuanto más arriba esté el peso")
print("en esta tabla, más fina es la comparación y más creíble el indicador de precio.")
for _, x in r.iterrows(): cifra(f"precio_nivel_{x.nivel.replace(' ','_')}", int(x.procesos), "446.677 procesos de P1", "celda19")

print("\n" + "="*95 + "\n(2) PRUEBA PUNTO EN EL TIEMPO, BIEN HECHA\n" + "="*95)
t1 = q(f"""SELECT count(*) debutantes,
   count(*) FILTER (WHERE ind_procesos_proveedor_12m <> 0) con_historial_imposible,
   count(*) FILTER (WHERE ind_recurrencia_par_12m <> 0) con_recurrencia_imposible
   FROM read_parquet({sp(IND)}) WHERE ind_proveedor_debutante""").iloc[0]
print("Prueba 1. El PRIMER proceso de cada proveedor no puede tener historial previo:")
print(f"  debutantes: {int(t1.debutantes):,} | con procesos previos: {int(t1.con_historial_imposible)} "
      f"| con recurrencia previa: {int(t1.con_recurrencia_imposible)}  -> "
      + ("CORRECTO" if t1.con_historial_imposible == 0 else "FALLA: la ventana mira al futuro"))
t2 = q(f"""WITH m AS (SELECT min(fecha_proceso) f0 FROM read_parquet({sp(IND)}))
   SELECT count(*) procesos_primeros_30_dias,
     round(avg(CAST(ind_procesos_proveedor_12m AS DOUBLE)),3) historial_promedio,
     max(ind_procesos_proveedor_12m) historial_maximo
   FROM read_parquet({sp(IND)}), m WHERE fecha_proceso < f0 + INTERVAL 30 DAY""").iloc[0]
print("\nPrueba 2. En los primeros 30 días de toda la serie casi nadie puede tener historial de 12 meses:")
print(f"  procesos: {int(t2.procesos_primeros_30_dias):,} | historial promedio: {t2.historial_promedio} "
      f"| máximo: {int(t2.historial_maximo)}")
print("\n(El promedio por año que imprimí en la celda 18 NO era una prueba válida: con ventana móvil de")
print(" 365 días, un proceso de diciembre de 2015 sí tiene historial de enero de 2015. Mi error.)")

print("\n" + "="*95 + "\n(3) PLAZOS: COBERTURA Y MEDIANAS POR FAMILIA\n" + "="*95)
pl = q(f"""SELECT familia, count(*) procesos,
   round(100.0*count(ind_plazo_ofertas_dias)/count(*),1) pct_plazo_ofertas,
   round(median(ind_plazo_ofertas_dias)) mediana_ofertas,
   round(100.0*count(ind_plazo_inicio_adj_dias)/count(*),1) pct_inicio_adj,
   round(median(ind_plazo_inicio_adj_dias)) mediana_inicio_adj,
   round(100.0*count(ind_plazo_adj_firma_dias)/count(*),1) pct_adj_firma,
   round(median(ind_plazo_adj_firma_dias)) mediana_adj_firma,
   round(100.0*count(ind_duracion_contrato_dias)/count(*),1) pct_duracion,
   round(median(ind_duracion_contrato_dias)) mediana_duracion
   FROM read_parquet({sp(IND)}) GROUP BY 1 ORDER BY 2 DESC""")
print(pl.to_string(index=False))
pl.to_csv(TABLAS / "planA_plazos_por_familia.csv", index=False, encoding="utf-8-sig")
print("\nMenor cuantía no tiene fecha de adjudicación en el JSON: ese plazo no existe para sus 68.459")
print("procesos y se declara como ausente, no se imputa.")

print("\n" + "="*95 + "\n(4) CONTROLES DE VALORES EXTREMOS\n" + "="*95)
ex = q(f"""SELECT
   count(*) FILTER (WHERE alerta_fecha_inicio_sospechosa) fecha_inicio_antes_de_1950,
   count(*) FILTER (WHERE ind_antiguedad_dias > 36500) antiguedad_mayor_a_100_anios,
   count(*) FILTER (WHERE ind_monto_sobre_capital > 1000) monto_mas_de_1000_veces_el_capital,
   count(*) FILTER (WHERE ind_duracion_contrato_dias > 3650) contratos_de_mas_de_10_anios,
   count(*) FILTER (WHERE ind_contrato_sobre_adjudicado > 1.1) contrato_supera_lo_adjudicado_10pct,
   count(*) FILTER (WHERE ind_contrato_sobre_adjudicado < 0.9) contrato_menor_a_lo_adjudicado_10pct
   FROM read_parquet({sp(IND)})""").iloc[0]
print(ex.to_string())
print("\nNinguno se recorta ni se imputa: entran al modelo por su percentil, que es insensible a la cola.")
print("Pero las cifras descriptivas de la tesis se reportan con mediana, nunca con promedio.")
for k in ex.index: cifra(f"extremos_{k}", int(ex[k]), "446.677 procesos de P1", "celda19")
print("\nLa última pareja es interesante por sí misma: si el monto del contrato difiere del adjudicado,")
print("es la única forma que tenemos de ver una modificación, porque el portal no publica enmiendas.")

In [ ]:
# ===== CELDA 20 · Indicadores versión 3: ventanas corregidas (DEFINITIVA) =====
# Corrige el fallo que detectó la prueba de la celda 19: la ventana 'CURRENT ROW' incluye los
# procesos del MISMO DIA, de modo que un proveedor con diez procesos el mismo dia veia nueve
# 'previos' aunque fuera su estreno. Ahora la ventana termina el dia ANTERIOR, asi que el
# historial es estrictamente de dias previos. Esta version reemplaza a la v1 y la v2.
import time
from pathlib import Path
J   = Path(r"C:\SERCOP_FINAL\03_resultados\c15_complemento_json.parquet")
P1  = PLAN_A / "planA_v2_P1_modelo_sin_catalogo.parquet"
IND = PLAN_A / "planA_indicadores_v3.parquet"
CORTE = "DATE '2026-09-05'"
F = "COALESCE(TRY_CAST(p.fecha_inicio_tender AS DATE), TRY_CAST(p.fecha_release AS DATE))"
V12 = "RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING"  # solo dias anteriores
V30 = "RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING"
cP1 = columnas(P1)
g = lambda n, t="VARCHAR": (f"p.{qc(n)}" if n in cP1 else f"CAST(NULL AS {t})")
c_fini = elegir(["proveedor_fecha_inicio_actividades"], cP1)
c_cap  = elegir(["proveedor_capital_suscrito"], cP1)

t0 = time.time()
con.execute(f"""CREATE OR REPLACE TEMP VIEW b AS
SELECT p.ocid, p.proveedor_ruc_final AS prov, p.{qc(c_ent)} AS ent, p.planA_familia AS fam,
  p.planA_periodo_normativo AS periodo, {F} AS f, year({F}) AS anio,
  COALESCE(p.{qc(c_monto)}, 0) AS monto,
  substr(regexp_replace(CAST(j.j_item_cpc AS VARCHAR),'[^0-9]','','g'),1,5) AS cpc5,
  substr(regexp_replace(CAST(j.j_item_cpc AS VARCHAR),'[^0-9]','','g'),1,3) AS cpc3,
  upper(trim(COALESCE(CAST(j.j_item_unidad AS VARCHAR),'(sin unidad)'))) AS unidad,
  j.j_item_precio_unit AS precio_unit, j.j_item_cantidad AS cantidad,
  TRY_CAST(j.j_fecha_adjudicacion AS DATE) AS f_adj, TRY_CAST(j.j_fecha_firma AS DATE) AS f_firma,
  TRY_CAST(j.j_contrato_inicio AS DATE) AS f_c_ini, TRY_CAST(j.j_contrato_fin AS DATE) AS f_c_fin,
  TRY_CAST(j.j_periodo_tender_ini AS DATE) AS f_t_ini, TRY_CAST(j.j_periodo_tender_fin AS DATE) AS f_t_fin,
  j.j_contrato_monto AS monto_contrato, j.j_n_ofertas AS n_ofertas,
  {f'TRY_CAST(p.{qc(c_fini)} AS DATE)' if c_fini else 'CAST(NULL AS DATE)'} AS f_ini_prov,
  {g(c_cap,'DOUBLE') if c_cap else 'CAST(NULL AS DOUBLE)'} AS capital,
  CAST({g('proveedor_unico','BOOLEAN')} AS BOOLEAN) AS unico, {g('n_oferentes','BIGINT')} AS n_of,
  CAST({g('provincia_proveedor')} AS VARCHAR) AS pv, CAST({g('provincia_entidad')} AS VARCHAR) AS pe
FROM read_parquet({sp(P1)}) p LEFT JOIN read_parquet({sp(J)}) j USING (ocid)""")
print(f"Base unida: {int(q('SELECT count(*) n FROM b').n[0]):,} filas")
print(q("""SELECT round(100.0*count(cpc5)/count(*),2) pct_cpc, round(100.0*count(precio_unit)/count(*),2) pct_precio,
   round(100.0*count(f_adj)/count(*),2) pct_fecha_adj, round(100.0*count(f_firma)/count(*),2) pct_firma
   FROM b""").T.rename(columns={0:"cobertura %"}).to_string())

con.execute(f"""CREATE OR REPLACE TEMP VIEW h AS SELECT *,
  COALESCE(sum(monto) OVER (PARTITION BY ent, prov ORDER BY f {V12}), 0) AS par_monto_12m,
  count(*)            OVER (PARTITION BY ent, prov ORDER BY f {V12})     AS par_n_12m,
  COALESCE(sum(monto) OVER (PARTITION BY ent       ORDER BY f {V12}), 0) AS ent_monto_12m,
  count(*)            OVER (PARTITION BY ent       ORDER BY f {V12})     AS ent_n_12m,
  count(*)            OVER (PARTITION BY prov      ORDER BY f {V12})     AS prov_n_12m,
  row_number() OVER (PARTITION BY prov ORDER BY f, ocid)            AS orden_prov,
  row_number() OVER (PARTITION BY ent, prov ORDER BY f, ocid)       AS orden_par,
  count(*) OVER (PARTITION BY ent, prov, COALESCE(cpc5,'(sin)') ORDER BY f {V30}) AS frac_n_30d,
  COALESCE(sum(monto) OVER (PARTITION BY ent, prov, COALESCE(cpc5,'(sin)') ORDER BY f {V30}), 0) AS frac_monto_30d,
  ln(greatest(precio_unit, 0.01)) AS lpu
FROM b""")

# ---- referencia de precio UNITARIO en cuatro niveles, del mas fino al mas grueso ----
NIV = [("n1", ["cpc5","unidad","fam","anio"], "cpc5 x unidad"), ("n2", ["cpc5","fam","anio"], "cpc5"),
       ("n3", ["cpc3","fam","anio"], "cpc3"),                  ("n4", ["fam","anio"], "familia")]
for nom, cols, _ in NIV:
    k = ", ".join(cols)
    kh = ", ".join(f"h.{c}" for c in cols)
    con.execute(f"""CREATE OR REPLACE TEMP VIEW m_{nom} AS
      SELECT {k}, count(lpu) n, median(lpu) md FROM h WHERE precio_unit IS NOT NULL GROUP BY {k}""")
    con.execute(f"""CREATE OR REPLACE TEMP VIEW d_{nom} AS
      SELECT {kh}, any_value(x.n) n, any_value(x.md) md, median(abs(h.lpu - x.md)) mad
      FROM h JOIN m_{nom} x USING ({k}) WHERE h.precio_unit IS NOT NULL GROUP BY {kh}""")
con.execute("""CREATE OR REPLACE TEMP VIEW ref AS
SELECT h.ocid,
  CASE WHEN a.n>=30 THEN 'cpc5 x unidad' WHEN b2.n>=30 THEN 'cpc5'
       WHEN c.n>=30 THEN 'cpc3' WHEN d.n>=30 THEN 'familia' END AS nivel,
  CASE WHEN a.n>=30 THEN a.md WHEN b2.n>=30 THEN b2.md WHEN c.n>=30 THEN c.md WHEN d.n>=30 THEN d.md END AS md,
  CASE WHEN a.n>=30 THEN a.mad WHEN b2.n>=30 THEN b2.mad WHEN c.n>=30 THEN c.mad WHEN d.n>=30 THEN d.mad END AS mad,
  CASE WHEN a.n>=30 THEN a.n WHEN b2.n>=30 THEN b2.n WHEN c.n>=30 THEN c.n WHEN d.n>=30 THEN d.n END AS n_grupo
FROM h
LEFT JOIN d_n1 a  ON a.cpc5=h.cpc5 AND a.unidad=h.unidad AND a.fam=h.fam AND a.anio=h.anio
LEFT JOIN d_n2 b2 ON b2.cpc5=h.cpc5 AND b2.fam=h.fam AND b2.anio=h.anio
LEFT JOIN d_n3 c  ON c.cpc3=h.cpc3 AND c.fam=h.fam AND c.anio=h.anio
LEFT JOIN d_n4 d  ON d.fam=h.fam AND d.anio=h.anio""")

con.execute(f"""COPY (
WITH x AS (SELECT h.*, r.nivel, r.md, r.mad, r.n_grupo FROM h LEFT JOIN ref r USING (ocid)),
y AS (SELECT ocid, prov AS proveedor_ruc, ent AS entidad_ruc, fam AS familia, periodo, anio,
   f AS fecha_proceso, monto AS monto_final, cpc5, unidad,
   unico AS ind_oferente_unico, n_of AS ind_n_oferentes,
   CASE WHEN ent_monto_12m > 0 THEN par_monto_12m/ent_monto_12m END AS ind_cuota_prov_en_entidad_12m,
   CASE WHEN ent_n_12m > 0 THEN CAST(par_n_12m AS DOUBLE)/ent_n_12m END AS ind_cuota_procesos_12m,
   par_n_12m AS ind_recurrencia_par_12m, prov_n_12m AS ind_procesos_proveedor_12m,
   (orden_par = 1) AS ind_primera_vez_con_la_entidad, (orden_prov = 1) AS ind_proveedor_debutante,
   frac_n_30d AS ind_fraccionamiento_n_30d, (frac_n_30d >= 1) AS ind_fraccionamiento_marca,
   frac_monto_30d AS ind_fraccionamiento_monto_30d,
   CASE WHEN f_ini_prov BETWEEN DATE '1900-01-01' AND {CORTE} AND f_ini_prov <= f
        THEN date_diff('day', f_ini_prov, f) END AS ind_antiguedad_dias,
   (f_ini_prov < DATE '1950-01-01') AS alerta_fecha_inicio_sospechosa,
   CASE WHEN capital > 0 THEN monto/capital END AS ind_monto_sobre_capital,
   CASE WHEN pv IS NOT NULL AND pe IS NOT NULL THEN (pv <> pe) END AS ind_provincia_distinta,
   nivel AS ind_precio_nivel, n_grupo AS ind_precio_n_grupo, precio_unit, cantidad,
   CASE WHEN mad > 0 AND precio_unit IS NOT NULL THEN (lpu - md)/(1.4826*mad) END AS ind_desv_precio_unit_z,
   CASE WHEN f_t_fin >= f_t_ini THEN date_diff('day', f_t_ini, f_t_fin) END AS ind_plazo_ofertas_dias,
   CASE WHEN f_adj   >= f      THEN date_diff('day', f, f_adj) END          AS ind_plazo_inicio_adj_dias,
   CASE WHEN f_firma >= f_adj  THEN date_diff('day', f_adj, f_firma) END    AS ind_plazo_adj_firma_dias,
   CASE WHEN f_c_fin >= f_c_ini THEN date_diff('day', f_c_ini, f_c_fin) END AS ind_duracion_contrato_dias,
   CASE WHEN monto > 0 AND monto_contrato > 0 THEN monto_contrato/monto END AS ind_contrato_sobre_adjudicado,
   n_ofertas AS ind_n_ofertas_bids
 FROM x)
SELECT y.*,
   (ind_desv_precio_unit_z IS NULL) AS falta_precio, (ind_plazo_inicio_adj_dias IS NULL) AS falta_plazo_adj,
   (ind_plazo_adj_firma_dias IS NULL) AS falta_plazo_firma, (ind_duracion_contrato_dias IS NULL) AS falta_duracion,
   (ind_cuota_prov_en_entidad_12m IS NULL) AS falta_cuota_12m, (ind_antiguedad_dias IS NULL) AS falta_antiguedad,
   (ind_monto_sobre_capital IS NULL) AS falta_capital, (ind_provincia_distinta IS NULL) AS falta_provincia,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_desv_precio_unit_z)        AS pct_precio_alto,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_cuota_prov_en_entidad_12m) AS pct_cuota_12m,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_recurrencia_par_12m)       AS pct_recurrencia_par,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_monto_sobre_capital)       AS pct_monto_capital,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY ind_fraccionamiento_n_30d)     AS pct_fraccionamiento,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_antiguedad_dias)          AS pct_proveedor_nuevo,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_n_oferentes)              AS pct_poca_concurrencia,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_plazo_ofertas_dias)       AS pct_plazo_corto,
   percent_rank() OVER (PARTITION BY familia, periodo ORDER BY -ind_plazo_adj_firma_dias)     AS pct_firma_rapida
FROM y) TO {sp(IND)} (FORMAT PARQUET, COMPRESSION ZSTD)""")
print(f"\nEscrito {IND.name} ({time.time()-t0:.0f} s) | filas: {int(q(f'SELECT count(*) n FROM read_parquet({sp(IND)})').n[0]):,}")

print("\n" + "="*95 + "\n(1) EL PRECIO, AHORA CON CPC REAL\n" + "="*95)
r = q(f"""SELECT COALESCE(ind_precio_nivel,'(sin referencia)') nivel, count(*) procesos,
   round(100.0*count(*)/sum(count(*)) OVER (),1) pct, round(median(ind_precio_n_grupo)) grupo_mediano
   FROM read_parquet({sp(IND)}) GROUP BY 1 ORDER BY 2 DESC""")
print(r.to_string(index=False))
for _, x in r.iterrows(): cifra(f"precio_nivel_{x.nivel.replace(' ','_')}", int(x.procesos), "446.677 procesos de P1", "celda20")

print("\n" + "="*95 + "\n(2) PRUEBA PUNTO EN EL TIEMPO (la que falló en la celda 19)\n" + "="*95)
t1 = q(f"""SELECT count(*) debutantes,
   count(*) FILTER (WHERE ind_procesos_proveedor_12m <> 0) con_historial_imposible,
   count(*) FILTER (WHERE ind_recurrencia_par_12m <> 0) con_recurrencia_imposible
   FROM read_parquet({sp(IND)}) WHERE ind_proveedor_debutante""").iloc[0]
print(f"  debutantes: {int(t1.debutantes):,} | con procesos previos: {int(t1.con_historial_imposible)} "
      f"| con recurrencia previa: {int(t1.con_recurrencia_imposible)}  -> "
      + ("CORRECTO" if t1.con_historial_imposible == 0 else "SIGUE FALLANDO, avísame"))

print("\n" + "="*95 + "\n(3) PLAZOS: COBERTURA Y MEDIANAS POR FAMILIA\n" + "="*95)
pl = q(f"""SELECT familia, count(*) procesos,
   round(100.0*count(ind_plazo_ofertas_dias)/count(*),1) pct_plazo_ofertas,
   round(median(ind_plazo_ofertas_dias)) mediana_ofertas,
   round(100.0*count(ind_plazo_inicio_adj_dias)/count(*),1) pct_inicio_adj,
   round(median(ind_plazo_inicio_adj_dias)) mediana_inicio_adj,
   round(100.0*count(ind_plazo_adj_firma_dias)/count(*),1) pct_adj_firma,
   round(median(ind_plazo_adj_firma_dias)) mediana_adj_firma,
   round(100.0*count(ind_duracion_contrato_dias)/count(*),1) pct_duracion,
   round(median(ind_duracion_contrato_dias)) mediana_duracion
   FROM read_parquet({sp(IND)}) GROUP BY 1 ORDER BY 2 DESC""")
print(pl.to_string(index=False))
pl.to_csv(TABLAS / "planA_plazos_por_familia.csv", index=False, encoding="utf-8-sig")

print("\n" + "="*95 + "\n(4) CONTROLES DE VALORES EXTREMOS\n" + "="*95)
ex = q(f"""SELECT
   count(*) FILTER (WHERE alerta_fecha_inicio_sospechosa) fecha_inicio_antes_de_1950,
   count(*) FILTER (WHERE ind_monto_sobre_capital > 1000) monto_mas_de_1000_veces_el_capital,
   count(*) FILTER (WHERE ind_duracion_contrato_dias > 3650) contratos_de_mas_de_10_anios,
   count(*) FILTER (WHERE ind_contrato_sobre_adjudicado > 1.1) contrato_supera_lo_adjudicado,
   count(*) FILTER (WHERE ind_contrato_sobre_adjudicado < 0.9) contrato_por_debajo_de_lo_adjudicado
   FROM read_parquet({sp(IND)})""").iloc[0]
print(ex.to_string())
for k in ex.index: cifra(f"extremos_{k}", int(ex[k]), "446.677 procesos de P1", "celda20")

print("\n" + "="*95 + "\n(5) PUERTA DE CALIDAD: NINGUNA VARIABLE PROHIBIDA PUEDE ESTAR AQUÍ\n" + "="*95)
import re as _re
PROH = _re.compile(r"fantasma|estado_actual|riesgoso|alguna_vez|tras_suspension|^monto_tender$|"
                   r"^red_|^cob_|^com_|^idx_|coreness|dominancia|rivales|^n_comp_|^rep_|^grupo_", _re.I)
malas = [c for c in columnas(IND) if PROH.search(c)]
print(f"Columnas prohibidas encontradas en la tabla de indicadores: {len(malas)}")
print("  " + (", ".join(malas) if malas else "ninguna  ->  CORRECTO"))

print("\n" + "="*95 + "\n(6) CONJUNTO DE VARIABLES CONGELADO\n" + "="*95)
VARS = sorted([c for c in columnas(IND) if c.startswith("pct_")])
AUX  = sorted([c for c in columnas(IND) if c.startswith("falta_")])
print(f"Variables del modelo ({len(VARS)}), todas percentiles dentro de familia x periodo:")
for v in VARS: print("  " + v)
print(f"\nIndicadoras de ausencia que las acompañan ({len(AUX)}):")
for v in AUX: print("  " + v)
import hashlib, json as _json
esp = {"archivo": IND.name, "filas": int(q(f"SELECT count(*) n FROM read_parquet({sp(IND)})").n[0]),
       "variables_modelo": VARS, "indicadoras_ausencia": AUX,
       "grupo_comparable": "planA_familia x planA_periodo_normativo",
       "referencia_precio": "CPC5 x unidad de medida x familia x anio, con retroceso a CPC5, CPC3 y familia; minimo 30 procesos",
       "ventana_historial": "365 dias anteriores, sin incluir el dia del proceso",
       "ventana_fraccionamiento": "30 dias anteriores, mismo par entidad-proveedor y mismo CPC5",
       "fecha_corte_datos": "2026-09-05", "generado": time.strftime("%Y-%m-%d %H:%M")}
esp["sha256_del_parquet"] = hashlib.sha256(IND.read_bytes()).hexdigest()
esp["sha256_de_la_especificacion"] = hashlib.sha256(
    _json.dumps({k: v for k, v in esp.items() if k != "sha256_de_la_especificacion"},
                sort_keys=True, ensure_ascii=False).encode()).hexdigest()
ESP = PLAN_A / "documentacion" / "especificacion_indicadores.json"
ESP.write_text(_json.dumps(esp, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nHuella del parquet de indicadores : {esp['sha256_del_parquet'][:32]}...")
print(f"Huella de la especificación       : {esp['sha256_de_la_especificacion'][:32]}...")
print(f"  -> {ESP}")
print("\nEstas dos huellas son la base del preregistro: si alguien cambia una variable o una regla")
print("después de mirar la validación, la huella cambia y se nota. Guárdalas en el documento.")
cifra("variables_modelo_congeladas", len(VARS), "tabla de indicadores v3", "celda20")

In [ ]:
# ===== CELDA 21 · Cierre del pulido: decisiones declaradas y especificación firmada =====
# No decide a ciegas: primero mide, después fija la decisión en el archivo de especificación.
import json as _json, hashlib, time
from pathlib import Path
IND = PLAN_A / "planA_indicadores_v3.parquet"
P1  = PLAN_A / "planA_v2_P1_modelo_sin_catalogo.parquet"
VER_P = PLAN_A / "planA_validacion_sri_verificada.parquet"
ESP = PLAN_A / "documentacion" / "especificacion_indicadores.json"
c_fini = elegir(["proveedor_fecha_inicio_actividades"], columnas(P1))

print("="*95 + "\n(1) LAS 4.761 FECHAS DE INICIO ANTERIORES A 1950: ¿centinelas o empresas viejas?\n" + "="*95)
if c_fini:
    d = q(f"""WITH x AS (SELECT p.proveedor_ruc_final ruc, TRY_CAST(p.{qc(c_fini)} AS DATE) fi
              FROM read_parquet({sp(P1)}) p WHERE TRY_CAST(p.{qc(c_fini)} AS DATE) < DATE '1950-01-01')
      SELECT year(fi) anio_inicio, count(*) procesos, count(DISTINCT ruc) proveedores
      FROM x GROUP BY 1 ORDER BY 2 DESC LIMIT 12""")
    print(d.to_string(index=False))
    t = q(f"""WITH x AS (SELECT p.proveedor_ruc_final ruc FROM read_parquet({sp(P1)}) p
              WHERE TRY_CAST(p.{qc(c_fini)} AS DATE) < DATE '1950-01-01')
      SELECT count(*) procesos, count(DISTINCT ruc) proveedores FROM x""").iloc[0]
    pr, pv = int(t.procesos), int(t.proveedores)
    print(f"\nTotal: {pr:,} procesos de {pv:,} proveedores distintos "
          f"({pr/max(pv,1):.1f} procesos por proveedor)")
    print("\nCómo se lee. Si unos pocos proveedores concentran casi todo y sus fechas se repiten en el")
    print("mismo año, son valores por defecto de la fuente. Si son cientos de proveedores con años")
    print("dispersos, son empresas antiguas de verdad y la antigüedad se conserva tal cual.")
    cifra("fechas_antes_1950_procesos", pr, "446.677 procesos de P1", "celda21")
    cifra("fechas_antes_1950_proveedores", pv, "ídem", "celda21")

print("\n" + "="*95 + "\n(2) MONTO MUY POR ENCIMA DEL CAPITAL: ¿error o señal?\n" + "="*95)
k = q(f"""SELECT count(*) procesos, count(DISTINCT proveedor_ruc) proveedores,
   round(median(ind_monto_sobre_capital),1) razon_mediana, round(max(ind_monto_sobre_capital),0) razon_maxima,
   round(sum(monto_final)/1e6,1) monto_millones
   FROM read_parquet({sp(IND)}) WHERE ind_monto_sobre_capital > 1000""").iloc[0]
print(k.to_string())
print("\nDECISIÓN: no se recorta ni se winsoriza. Un capital suscrito mínimo frente a un contrato")
print("grande es exactamente la señal de capacidad que el indicador debe capturar, y entra al modelo")
print("por su percentil, que es insensible a la magnitud de la cola. Lo que cambia es la redacción:")
print("las cifras descriptivas se reportan con mediana, nunca con promedio.")
for kk in k.index: cifra(f"monto_sobre_capital_alto_{kk}", float(k[kk]), "procesos con razón mayor a 1000", "celda21")

print("\n" + "="*95 + "\n(3) CONJUNTO DE VALIDACIÓN: 14 PRIMARIO, 15 COMO SENSIBILIDAD\n" + "="*95)
v = q(f"""SELECT grupo, count(*) ruc, CAST(sum(n_procesos) AS BIGINT) procesos,
   round(sum(monto_millones),3) mm FROM read_parquet({sp(VER_P)}) GROUP BY 1 ORDER BY 3 DESC""")
print(v.to_string(index=False))
sin_marca = q(f"SELECT ruc, razon_social FROM read_parquet({sp(VER_P)}) WHERE grupo='sin_marca_actual'").to_dict('records')
print(f"\nDECISIÓN, tomada ANTES de mirar ningún resultado de validación:")
print("  - Conjunto PRIMARIO: los 14 contribuyentes con marca vigente en el SRI al 9 de octubre de 2026.")
print("  - Conjunto de SENSIBILIDAD: los 15, incluyendo el que ya no tiene marca.")
print("  - Se reportan los dos. Si difieren, se declara; no se elige el que convenga.")
for r in sin_marca: print(f"  - Excluido del primario: {r['ruc']} ({r['razon_social']}), sin ninguna de las dos marcas hoy.")
print("\nMotivo: la pertenencia al catastro descargado la contradice la consulta vigente del SRI.")
print("Excluirlo es lo conservador; reportar los dos es lo transparente. Son 1 proceso de 170, así")
print("que no cambia ninguna cifra: por eso mismo sale barato ser riguroso.")

print("\n" + "="*95 + "\n(4) ESPECIFICACIÓN FIRMADA, CON LAS DECISIONES DENTRO\n" + "="*95)
esp = _json.loads(ESP.read_text(encoding="utf-8")) if ESP.exists() else {}
esp.update({
  "decisiones_declaradas": {
    "extremos_monto_sobre_capital": "no se recorta ni se winsoriza; entra por percentil",
    "fechas_inicio_anteriores_a_1950": "se conservan con su marca alerta_fecha_inicio_sospechosa; ver diagnóstico de la celda 21",
    "conjunto_validacion_primario": "14 contribuyentes con marca vigente en el SRI al 2026-10-09",
    "conjunto_validacion_sensibilidad": "15 contribuyentes, incluyendo el que ya no tiene marca",
    "historial": "solo días anteriores al proceso; el mismo día no cuenta como previo",
    "estadistico_descriptivo": "mediana, nunca promedio, en todas las cifras con cola larga"},
  "validacion_excluidos_del_primario": [r["ruc"] for r in sin_marca],
  "fecha_de_congelado": time.strftime("%Y-%m-%d %H:%M"),
  "sha256_del_parquet_indicadores": hashlib.sha256(IND.read_bytes()).hexdigest(),
  "sha256_del_parquet_universo": hashlib.sha256((PLAN_A/"planA_universo_v2.parquet").read_bytes()).hexdigest(),
  "sha256_del_parquet_validacion": hashlib.sha256(VER_P.read_bytes()).hexdigest()})
esp.pop("sha256_de_la_especificacion", None)
esp["sha256_de_la_especificacion"] = hashlib.sha256(
    _json.dumps(esp, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
ESP.write_text(_json.dumps(esp, indent=2, ensure_ascii=False), encoding="utf-8")
print("Huellas congeladas:")
for k2 in ("sha256_del_parquet_universo","sha256_del_parquet_indicadores","sha256_del_parquet_validacion",
           "sha256_de_la_especificacion"):
    print(f"  {k2:34s} {esp[k2][:40]}")
print(f"\n  -> {ESP}")
print("\nA partir de aquí, cualquier cambio en los datos, en las variables o en las reglas altera una")
print("huella. Eso es lo que hace creíble el resultado de validación cuando llegue.")

print("\n" + "="*95 + "\n(5) ESTADO DE LA ENTREGA\n" + "="*95)
arch = sorted(PLAN_A.glob("*.parquet"))
tot = sum(f.stat().st_size for f in arch)/1e6
print(f"Archivos de datos: {len(arch)} | {tot:,.0f} MB")
doc = PLAN_A / "documentacion"
print(f"Documentación: {', '.join(sorted(p.name for p in doc.glob('*')))}")
print(f"\nSiguiente paso: volver a correr la CELDA 15 para que el README recoja los indicadores,")
print("y después abrir un cuaderno NUEVO para los modelos, que parte de estos archivos congelados.")

In [ ]:
# ===== CELDA 15 · Documentación de la base para los compañeros =====
# Requiere las celdas 1, 3, 6, 13 y 14. Genera los textos DESDE los datos, no a mano.
import hashlib, time, re
U2 = PLAN_A / "planA_universo_v2.parquet"
P1_2 = PLAN_A / "planA_v2_P1_modelo_sin_catalogo.parquet"
DOC = PLAN_A / "documentacion"; DOC.mkdir(exist_ok=True)

REGLAS = [
    ("llave_o_texto",    r"^ocid$|^planA_|_ruc$|_ruc_final$|_id$|nombre$|titulo$|^razon|descripcion|^tipo_contratacion$|^metodo_contratacion$|^categoria_principal$"),
    ("prohibido_fuga",   r"fantasma|^f_marca$|^f_reactivacion$|estado_actual|riesgoso|^desviacion_precio_log_z$|alguna_vez|tras_suspension"),
    ("prohibido_canal_lateral", r"^monto_tender$|^tiene_periodo_competitivo$|^es_catalogo$"),
    ("fuera_de_alcance_redes",  r"^red_|^cob_|^com_|^idx_|coreness|dominancia|rivales|^n_comp_|^rep_|^grupo_|^c_conc_rival$|^c_dominancia$|coincidencias|^n_pares_"),
    ("descriptivo_estado_actual", r"^entidad_superintendencia|^proveedor_superintendencia|^entidad_fecha|_sri$"),
]
desc = q(f"DESCRIBE SELECT * FROM read_parquet({sp(P1_2)})")
filas = []
for c, tp in zip(desc.column_name, desc.column_type):
    uso = "modelo_candidato"
    for etq, pat in REGLAS:
        if re.search(pat, c, re.I): uso = etq; break
    r = q(f"""SELECT round(100.0*count(*) FILTER (WHERE {qc(c)} IS NULL)/count(*),2) pct_nulo,
          count(DISTINCT {qc(c)}) distintos FROM read_parquet({sp(P1_2)})""").iloc[0]
    if int(r.distintos) <= 1 and uso == "modelo_candidato": uso = "sin_varianza"
    filas.append((c, tp, uso, float(r.pct_nulo), int(r.distintos)))
dic = pd.DataFrame(filas, columns=["columna","tipo","uso_permitido","pct_nulo_en_P1","valores_distintos"])
dic.sort_values(["uso_permitido","columna"]).to_csv(DOC / "diccionario_columnas.csv", index=False, encoding="utf-8-sig")
print("Columnas por uso permitido:\n"); print(dic.uso_permitido.value_counts().rename_axis("uso").reset_index(name="n").to_string(index=False))

seg = q(f"""SELECT planA_segmento segmento, count(*) procesos,
   round(100.0*count(*)/sum(count(*)) OVER (),2) pct, round(sum(COALESCE({qc(c_monto)},0))/1e6,1) mm
   FROM read_parquet({sp(U2)}) GROUP BY 1 ORDER BY 1""")
org = q(f"""SELECT proveedor_ruc_origen origen, count(*) n, round(100.0*count(*)/sum(count(*)) OVER (),2) pct
   FROM read_parquet({sp(U2)}) GROUP BY 1 ORDER BY 2 DESC""")
fam = q(f"""SELECT planA_familia familia, planA_periodo_normativo periodo, count(*) n
   FROM read_parquet({sp(P1_2)}) GROUP BY 1,2 ORDER BY 1,2""")
VER_P = PLAN_A / "planA_validacion_sri_verificada.parquet"
val = q(f"""SELECT grupo, count(*) ruc, CAST(sum(n_procesos) AS BIGINT) procesos, round(sum(monto_millones),3) mm
   FROM read_parquet({sp(VER_P)}) GROUP BY 1 ORDER BY 3 DESC""") if VER_P.exists() else None
nP1 = int(q(f"SELECT count(*) n FROM read_parquet({sp(P1_2)})").n[0])
nTot = int(seg.procesos.sum())

seg2 = seg.rename(columns={"segmento":"segmento","procesos":"procesos","pct":"% del total","mm":"monto (millones USD)"})
org2 = org.rename(columns={"origen":"origen del RUC","n":"procesos","pct":"% del total"})
fam2 = fam.rename(columns={"familia":"familia","periodo":"período normativo","n":"procesos"})
val2 = val.rename(columns={"grupo":"grupo","ruc":"contribuyentes","procesos":"procesos","mm":"monto (millones)"}) if val is not None else None
uso2 = dic.uso_permitido.value_counts().rename_axis("uso permitido").reset_index(name="columnas")

def tabla(df, cols=None):
    d = df if cols is None else df[cols]
    enc = "| " + " | ".join(d.columns) + " |\n|" + "|".join([" --- "]*len(d.columns)) + "|\n"
    return enc + "".join("| " + " | ".join(f"{v:,}" if isinstance(v,(int,)) else str(v) for v in r) + " |\n"
                         for r in d.itertuples(index=False))

R = f"""# Base de datos del plan A · documentación para el equipo

Generado automáticamente el {time.strftime('%d-%m-%Y')} desde los propios archivos.
Fuente: portal de datos abiertos del SERCOP, estándar OCDS, años 2015 a 2026.
**Fecha de corte de la descarga: 5 de septiembre de 2026.**

## 1. Advertencia que hay que leer antes de usar cualquier cifra

Los datos son **una fotografía del 5 de septiembre de 2026**. Un proceso que seguía vivo ese día
aparece incompleto: sin adjudicación ni contrato aunque después los haya tenido. Verificado con el
proceso RE-CEP-MSP-2026-003 del Ministerio de Salud, adjudicado el 8 de septiembre y firmado el 15,
los dos posteriores al corte. Esto afecta sobre todo a 2026.

## 2. Los archivos

| Archivo | Grano | Llave primaria | Filas |
| --- | --- | --- | --- |
| `planA_universo_v2.parquet` | un proceso de contratación | `ocid` | {nTot:,} |
| `planA_v2_P1_modelo_sin_catalogo.parquet` | un proceso | `ocid` | {nP1:,} |
| `planA_v2_P0_catalogo.parquet` | un proceso | `ocid` | {int(seg.loc[seg.segmento=='S2_catalogo','procesos'].iloc[0]):,} |
| `planA_v2_excluidos_solo_planeacion.parquet` | un proceso | `ocid` | {int(seg.loc[seg.segmento=='S0_solo_planeacion','procesos'].iloc[0]):,} |
| `planA_v2_excluidos_sin_monto.parquet` | un proceso | `ocid` | {int(seg.loc[seg.segmento=='S1_sin_monto','procesos'].iloc[0]):,} |
| `planA_v2_sin_proveedor_identificable.parquet` | un proceso | `ocid` | {int(seg.loc[seg.segmento=='S4_sin_catalogo_sin_proveedor_o_monto','procesos'].iloc[0]):,} |
| `planA_dim_proveedor.parquet` | un proveedor | RUC | una fila por RUC |
| `planA_dim_entidad.parquet` | una entidad compradora | RUC | una fila por RUC |
| `planA_validacion_sri_verificada.parquet` | un contribuyente marcado | RUC | 15 |

**El RUC no es llave primaria de la tabla de procesos.** Un mismo proveedor aparece en cientos de
procesos. Las tablas de proveedor y de entidad se unen por RUC cuando hacen falta; así no se repiten
treinta y siete columnas del SRI dos millones de veces.

## 3. Los segmentos

{tabla(seg2)}
Definiciones, aplicadas en este orden:

- **S0 solo planeación**: el registro trae únicamente la etapa `planning`. Verificado: el 100 % de ellos.
- **S1 sin monto**: no tiene monto en ninguna de las tres columnas de monto.
- **S2 catálogo**: compra por catálogo electrónico (convenio marco). Se describe pero no se modela.
- **S3 universo de modelado (P1)**: fuera del catálogo, con RUC de proveedor y monto positivo.
- **S4 sin proveedor identificable**: fuera del catálogo, sin RUC de proveedor aun tras la recuperación.

## 4. El RUC del proveedor: de dónde sale cada uno

{tabla(org2)}
En parte de los registros el campo de RUC venía vacío pero el identificador del proveedor contenía el
número, con el formato `EC-RUC-<ruc>-<id>` o `ID-<ruc>-<id>`. Ese RUC se recupera **solo si** pasa
cuatro controles: tiene trece dígitos, termina en 001, no coincide con el RUC de la entidad compradora
y existe en el catastro del SRI. La columna **`proveedor_ruc_origen`** dice de dónde salió cada uno, de
modo que la decisión es auditable y reversible. La columna **`planA_segmento_v1`** conserva la
clasificación anterior para poder comparar.

**Nunca uses `proveedor_ruc` a secas: usa `proveedor_ruc_final`.**

## 5. Grupos comparables del universo de modelado

Los modelos se ajustan dentro de cada familia de procedimiento y período normativo. El corte de período
es el **7 de octubre de 2025**, fecha de la reforma a la LOSNCP, y se asigna con la fecha de inicio de
la licitación.

{tabla(fam2)}

## 6. El conjunto de validación externa

{tabla(val2) if val2 is not None else '(falta correr la celda 13)'}
Los 15 contribuyentes se verificaron uno por uno en `consultas.sri.gob.ec` el 9 de octubre de 2026.
**No son «empresas fantasma»**: nueve están marcados por transacciones inexistentes, cinco como
empresas fantasma y uno ya no tiene ninguna marca. **Siete de los quince son personas naturales.**

Este conjunto **solo sirve para validar y describir. No es variable del modelo ni objetivo a
predecir**, y por eso vive en un archivo aparte. La tasa base es de un proceso marcado por cada 4.095
del universo de modelado, así que cualquier medida de precisión sería engañosa: se reportan percentil y
lift, y se declara que la potencia es baja.

## 7. Qué columnas NO pueden ser variables del modelo

{tabla(uso2)}
El detalle columna por columna está en `documentacion/diccionario_columnas.csv`. Los motivos:

- **prohibido_fuga**: describen el estado de hoy o vienen de la misma fuente que la etiqueta de
  validación. Las variables del tipo «fue suspendido alguna vez» informan hechos **posteriores** al
  proceso en el 85 % de los casos.
- **prohibido_canal_lateral**: identifican el tipo de procedimiento por la puerta de atrás, que es lo
  que ya usamos para agrupar.
- **fuera_de_alcance_redes**: variables de co-oferta y comunidades, que corresponden al otro tema del
  catálogo del programa y quedan fuera de esta tesis.
- **descriptivo_estado_actual**: sirven para describir, no para predecir.

## 8. Problemas de calidad conocidos de la fuente

- Los datos abiertos **no publican** modificaciones de contrato, ejecución, ni el estado de desierto,
  cancelado o suspendido. El portal transaccional sí los muestra en la ficha del proceso: es una
  brecha entre los dos sistemas del propio SERCOP.
- El identificador de la entidad compradora no coincide con el del bloque de participantes.
- Hay fechas imposibles en el catastro del SRI (104 anteriores a 1900, mínimo en el año 199) y un valor
  centinela 9999-01-01.
- La variable de año de la base es el **año de publicación del registro**, no el año en que se lanzó el
  proceso; coinciden solo en el 78,93 % de los casos. Para análisis económico se usa la fecha de inicio
  de la licitación.
- El número de oferentes no es comparable entre familias: en menor cuantía de obras refleja
  manifestaciones de interés previas al sorteo, no ofertas económicas.

## 9. Reglas de trabajo del equipo

1. Ninguna tasa ni total se escribe sin declarar sobre cuántos registros se calculó.
2. El monto canónico es `monto_final`. En el universo de modelado es siempre el monto adjudicado.
3. Toda cifra que vaya a la tesis se anota en el registro de cifras con su universo y la celda que la
   produjo.
4. Las decisiones que no resuelve el dato se verifican a mano en el portal o en el SRI, y se documenta
   qué se vio y cuándo.
"""
(DOC / "README_base_plan_A.md").write_text(R, encoding="utf-8")
print(f"\n  -> {DOC/'README_base_plan_A.md'}\n  -> {DOC/'diccionario_columnas.csv'}")

man = []
for f in sorted(PLAN_A.glob("*.parquet")):
    n = q(f"SELECT count(*) n FROM read_parquet({sp(f)})").n[0]
    h = hashlib.sha256(f.read_bytes()).hexdigest()[:16] if f.stat().st_size < 3e9 else "(omitido)"
    man.append((f.name, int(n), len(columnas(f)), round(f.stat().st_size/1e6,1),
                time.strftime("%Y-%m-%d %H:%M", time.localtime(f.stat().st_mtime)), h))
M = pd.DataFrame(man, columns=["archivo","filas","columnas","MB","creado","sha256_16"])
M.to_csv(DOC / "manifiesto.csv", index=False, encoding="utf-8-sig")
print("\nMANIFIESTO:\n"); print(M.to_string(index=False))
print(f"\nTamaño total de la carpeta: {M.MB.sum():,.0f} MB")
print("\nPara enviar a los compañeros: comprime la carpeta plan_A completa, o solo documentacion/ y")
print("planA_v2_P1_modelo_sin_catalogo.parquet si solo necesitan el universo de modelado.")

In [ ]:
# ===== CELDA 22 · Cierre: limpiar duplicados, ver los 57 proveedores antiguos y completar el README =====
import shutil, hashlib, time, json as _json
from pathlib import Path
IND = PLAN_A / "planA_indicadores_v3.parquet"
P1  = PLAN_A / "planA_v2_P1_modelo_sin_catalogo.parquet"
DOC = PLAN_A / "documentacion"
c_fini = elegir(["proveedor_fecha_inicio_actividades"], columnas(P1))

print("="*95 + "\n(1) LOS 57 PROVEEDORES CON FECHA DE INICIO ANTERIOR A 1950\n" + "="*95)
d = q(f"""SELECT p.proveedor_ruc_final ruc, any_value(d.sri_RAZON_SOCIAL) razon_social,
   CAST(min(TRY_CAST(p.{qc(c_fini)} AS DATE)) AS VARCHAR) fecha_inicio,
   count(*) procesos, round(sum(COALESCE(p.{qc(c_monto)},0))/1e6,2) monto_millones
   FROM read_parquet({sp(P1)}) p
   LEFT JOIN read_parquet({sp(PLAN_A/'planA_dim_proveedor.parquet')}) d ON d.ruc = p.proveedor_ruc_final
   WHERE TRY_CAST(p.{qc(c_fini)} AS DATE) < DATE '1950-01-01'
   GROUP BY 1 ORDER BY 4 DESC""")
print(d.to_string(index=False))
d.to_csv(TABLAS / "planA_proveedores_fecha_antigua.csv", index=False, encoding="utf-8-sig")
print(f"\nSon {len(d)} proveedores. Míralos: si son instituciones conocidas (bancos, universidades,")
print("empresas públicas) la fecha es real y la antigüedad se conserva. Los anteriores a 1900 ya")
print("quedan con antigüedad nula por el filtro del indicador, así que no contaminan el modelo.")

print("\n" + "="*95 + "\n(2) BORRAR LAS VERSIONES VIEJAS DE INDICADORES\n" + "="*95)
HIST = PLAN_A / "_version_1_historico"; HIST.mkdir(exist_ok=True)
for n in ("planA_indicadores_v1.parquet", "planA_indicadores_v2.parquet"):
    f = PLAN_A / n
    if f.exists(): shutil.move(str(f), str(HIST / n)); print(f"  {n} -> {HIST.name}/")
print("  (no se borran: quedan en el histórico por si hay que reconstruir la trazabilidad)")

print("\n" + "="*95 + "\n(3) COMPLETAR EL README CON LA TABLA DE INDICADORES\n" + "="*95)
R = DOC / "README_base_plan_A.md"
VARS = sorted([c for c in columnas(IND) if c.startswith("pct_")])
AUX  = sorted([c for c in columnas(IND) if c.startswith("falta_")])
nI = int(q(f"SELECT count(*) n FROM read_parquet({sp(IND)})").n[0])
esp = _json.loads((DOC / "especificacion_indicadores.json").read_text(encoding="utf-8"))
cob = q(f"""SELECT round(100.0*count(ind_desv_precio_unit_z)/count(*),1) precio,
   round(100.0*count(ind_cuota_prov_en_entidad_12m)/count(*),1) cuota,
   round(100.0*count(ind_plazo_inicio_adj_dias)/count(*),1) plazo_adj,
   round(100.0*count(ind_plazo_adj_firma_dias)/count(*),1) plazo_firma,
   round(100.0*count(ind_monto_sobre_capital)/count(*),1) capital
   FROM read_parquet({sp(IND)})""").iloc[0]
txt = f"""

## 10. La tabla de indicadores

`planA_indicadores_v3.parquet` — una fila por `ocid`, {nI:,} filas. **No está dentro del universo:
es una tabla aparte que se une por `ocid`.** Si se corrige un indicador se regenera este archivo y
nada más.

### Las {len(VARS)} variables del modelo

Todas son **percentiles dentro del grupo comparable** (familia de procedimiento × período
normativo), nunca valores en crudo:

{chr(10).join('- `' + v + '`' for v in VARS)}

Las acompañan {len(AUX)} indicadoras de ausencia (`falta_*`), una por cada indicador que puede
faltar. **No se imputa ningún valor:** un indicador que no existe para ese procedimiento se marca
como ausente y el modelo lo trata como tal.

### Las tres reglas que cumplen todos

1. **Punto en el tiempo.** El historial cuenta solo **días anteriores** al proceso; el mismo día no
   cuenta como previo. Verificado: de los 62.613 proveedores debutantes, **ninguno** tiene historial
   previo.
2. **Relativo al grupo comparable.** El precio se compara dentro de CPC de 5 dígitos × unidad de
   medida × familia × año, retrocediendo a CPC5, CPC3 y familia cuando el grupo no llega a 30
   procesos. El 72,5 % se compara al nivel más fino y el 90,6 % a nivel de CPC.
3. **Nada prohibido entra.** Una puerta de calidad verifica que ninguna variable de fuga, de redes o
   de canal lateral esté en esta tabla. Resultado: cero.

### Cobertura de los principales

| Indicador | Procesos con dato |
| --- | --- |
| Desviación del precio unitario | {cob.precio} % |
| Cuota del proveedor en la entidad, 12 meses | {cob.cuota} % |
| Plazo de inicio a adjudicación | {cob.plazo_adj} % |
| Plazo de adjudicación a firma | {cob.plazo_firma} % |
| Monto sobre capital suscrito | {cob.capital} % |

La menor cuantía no tiene fecha de adjudicación en la fuente, de modo que ese plazo no existe para
sus 68.459 procesos.

## 11. El preregistro

`documentacion/especificacion_indicadores.json` fija, **antes de mirar ningún resultado de
validación**, la lista de variables, las reglas de ventana, el nivel de referencia del precio, el
tratamiento de los extremos y la composición del conjunto de validación. Lleva cuatro huellas
SHA-256: del universo, de los indicadores, del conjunto de validación y de la propia especificación.

Congelado el {esp.get('fecha_de_congelado','(ver el archivo)')}.

**Conjunto de validación: 14 contribuyentes en el análisis primario** (los que tienen marca vigente
del SRI al 9 de octubre de 2026) **y 15 como análisis de sensibilidad**. Se reportan los dos.

**Si alguien cambia una variable o una regla después de ver la validación, la huella cambia y se
nota.** Esa es la garantía que hace creíble el resultado.
"""
cur = R.read_text(encoding="utf-8")
cur = cur.split("\n## 10. La tabla de indicadores")[0].rstrip()
R.write_text(cur + txt, encoding="utf-8")
print(f"  README actualizado: {R}")
print(f"  ahora tiene {len(R.read_text(encoding='utf-8').split(chr(10)))} líneas")

print("\n" + "="*95 + "\n(4) MANIFIESTO FINAL DE LA ENTREGA\n" + "="*95)
man = []
for f in sorted(PLAN_A.glob("*.parquet")):
    n = q(f"SELECT count(*) n FROM read_parquet({sp(f)})").n[0]
    man.append((f.name, int(n), len(columnas(f)), round(f.stat().st_size/1e6,1),
                hashlib.sha256(f.read_bytes()).hexdigest()[:16]))
M = pd.DataFrame(man, columns=["archivo","filas","columnas","MB","sha256_16"])
M.to_csv(DOC / "manifiesto.csv", index=False, encoding="utf-8-sig")
print(M.to_string(index=False))
print(f"\n{len(M)} archivos | {M.MB.sum():,.0f} MB")
print(f"Documentación: {', '.join(sorted(p.name for p in DOC.glob('*')))}")
print("\nLa carpeta plan_A queda lista para comprimir y enviar.")

In [ ]:
# =================== CELDA 23 · PERFIL DESCRIPTIVO DE LAS BASES FINALES ===================
# Solo lectura: no reescribe ningún parquet.
# Mide por cada base: grano, unicidad de la llave, nulos y vacíos columna por columna,
# columnas inservibles, distribuciones, cobertura temporal e integridad entre bases.
# Detalle -> plan_A\documentacion\ ; resumen -> pantalla.

import os, hashlib, json
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC = 'plan_A/documentacion'
os.makedirs(DOC, exist_ok=True)
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 80)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")

CIFRAS = []
def cifra(n, v, nota=''):
    CIFRAS.append({'cifra': n, 'valor': v, 'nota': nota}); return v
def t(x):
    print('\n' + '=' * 95); print(x); print('=' * 95)

# --- catálogo de archivos esperados: nombre, grano declarado, caja ---
ESPERADOS = [
    ('planA_universo_v2',                    'release OCDS',              'B auditoria'),
    ('planA_v2_P1_modelo_sin_catalogo',      'proceso modelable',         'A modelado'),
    ('planA_indicadores_v3',                 'proceso modelable',         'A modelado'),
    ('planA_dim_proveedor',                  'proveedor (RUC)',           'A modelado'),
    ('planA_dim_entidad',                    'entidad (RUC)',             'A modelado'),
    ('planA_v2_P0_catalogo',                 'proceso de catalogo',       'B descriptivo'),
    ('planA_v2_excluidos_solo_planeacion',   'proceso excluido',          'C evidencia'),
    ('planA_v2_excluidos_sin_monto',         'proceso excluido',          'C evidencia'),
    ('planA_v2_sin_proveedor_identificable', 'proceso excluido',          'C evidencia'),
    ('planA_validacion_sri_verificada',      'proveedor marcado por SRI', 'C validacion'),
]
RUTA = {n: f'plan_A/{n}.parquet' for n, _, _ in ESPERADOS}
faltan = [n for n in RUTA if not os.path.exists(RUTA[n])]
if faltan: print('OJO, no encuentro:', faltan)
ARCH = [(n, g, c) for n, g, c in ESPERADOS if n not in faltan]

# --- utilidades de esquema ---
def esquema(n):
    d = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RUTA[n]}')").df()
    return list(zip(d.column_name.tolist(), d.column_type.tolist()))
def anidado(ty):                      # listas/structs: no se les puede contar distintos
    return any(s in ty.upper() for s in ('[', 'STRUCT', 'MAP', 'UNION'))
def es_num(ty):
    return any(k in ty.upper() for k in ('TINYINT','SMALLINT','INTEGER','BIGINT','HUGEINT',
                                         'DOUBLE','FLOAT','REAL','DECIMAL'))
def es_fecha(ty):
    return any(k in ty.upper() for k in ('DATE','TIMESTAMP'))
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

# =========================== (1) INVENTARIO Y UNICIDAD DE LA LLAVE ===========================
t('(1) INVENTARIO: tamano, grano y unicidad de la llave')
CAND = ['ocid', 'ruc', 'proveedor_ruc_final', 'proveedor_ruc', 'entidad_ruc', 'ruc_entidad']
inv, LLAVE, COLS = [], {}, {}
for n, grano, caja in ARCH:
    esq = esquema(n); cols = [c for c, _ in esq]; COLS[n] = esq
    llave = next((c for c in CAND if c in cols), None); LLAVE[n] = llave
    filas = con.sql(f"SELECT count(*) FROM read_parquet('{RUTA[n]}')").fetchone()[0]
    if llave:
        nd, nul = con.sql(f"""SELECT count(DISTINCT "{llave}"), count(*) - count("{llave}")
                              FROM read_parquet('{RUTA[n]}')""").fetchone()
    else:
        nd, nul = None, None
    inv.append({'archivo': n, 'caja': caja, 'grano': grano, 'filas': filas,
                'columnas': len(cols), 'MB': round(os.path.getsize(RUTA[n]) / 1e6, 1),
                'llave': llave or '-', 'llaves_distintas': nd,
                'llave_nula': nul, 'llave_unica': (nd == filas and nul == 0) if llave else None,
                'sha256_16': sha16(RUTA[n])})
INV = pd.DataFrame(inv)
print(INV.to_string(index=False))
INV.to_csv(f'{DOC}/perfil_inventario.csv', index=False, encoding='utf-8-sig')
cifra('archivos_entregados', len(INV))
cifra('MB_entregados', round(INV.MB.sum(), 1))
cifra('bases_con_llave_unica', int(INV.llave_unica.fillna(False).sum()),
      f'de {int(INV.llave.ne("-").sum())} con llave detectada')

# =========================== (2) NULOS, VACIOS Y COLUMNAS INSERVIBLES ===========================
t('(2) NULOS Y VACIOS COLUMNA POR COLUMNA  (detalle completo en los CSV)')
def perfil(n):
    """Una fila por columna: % con dato, % vacio de texto, valores distintos, veredicto."""
    esq, filas = COLS[n], int(INV.loc[INV.archivo == n, 'filas'].iloc[0])
    out = []
    for i in range(0, len(esq), 35):                      # por trozos para no armar un SQL gigante
        trozo = esq[i:i + 35]; sel = []
        for c, ty in trozo:
            sel.append(f'count("{c}") AS "nn_{i}_{c}"')
            if ty.upper().startswith('VARCHAR'):
                sel.append(f"""count(*) FILTER (WHERE trim("{c}") = '') AS "vc_{i}_{c}" """)
            if not anidado(ty):
                sel.append(f'approx_count_distinct("{c}") AS "nd_{i}_{c}"')
        r = con.sql(f"SELECT {', '.join(sel)} FROM read_parquet('{RUTA[n]}')").df().iloc[0]
        for c, ty in trozo:
            nn = int(r[f'nn_{i}_{c}'])
            vc = int(r[f'vc_{i}_{c}']) if f'vc_{i}_{c}' in r.index else 0
            nd = int(r[f'nd_{i}_{c}']) if f'nd_{i}_{c}' in r.index else -1
            util = nn - vc
            if util == 0:                   ver = 'INSERVIBLE: sin un solo dato'
            elif nd == 1:                   ver = 'CONSTANTE: un unico valor'
            elif util / filas < 0.05:       ver = 'casi vacia (<5% con dato)'
            elif util / filas < 0.50:       ver = 'mitad vacia'
            elif util == filas:             ver = 'completa'
            else:                           ver = 'con faltantes'
            out.append({'columna': c, 'tipo': ty, 'con_dato': nn, 'vacio_texto': vc,
                        'pct_con_dato': round(100 * util / filas, 2),
                        'distintos_aprox': None if nd < 0 else nd, 'veredicto': ver})
    return pd.DataFrame(out)

RES = {}
for n, _, _ in ARCH:
    P = perfil(n); RES[n] = P
    P.to_csv(f'{DOC}/perfil_nulos_{n}.csv', index=False, encoding='utf-8-sig')
    v = P.veredicto
    print(f'\n{n}  ({len(P)} columnas)')
    print(f'   completas (100% con dato) ....... {int((v == "completa").sum()):>4}')
    print(f'   con faltantes ................... {int((v == "con faltantes").sum()):>4}')
    print(f'   mitad vacia (50-95% nulo) ....... {int((v == "mitad vacia").sum()):>4}')
    print(f'   casi vacia (>95% nulo) .......... {int(v.str.startswith("casi").sum()):>4}')
    print(f'   constantes (un solo valor) ...... {int(v.str.startswith("CONSTANTE").sum()):>4}')
    print(f'   inservibles (ni un dato) ........ {int(v.str.startswith("INSERVIBLE").sum()):>4}')
    cifra(f'{n}__columnas_completas', int((v == 'completa').sum()))
    cifra(f'{n}__columnas_inservibles', int(v.str.startswith('INSERVIBLE').sum()))
    cifra(f'{n}__columnas_constantes', int(v.str.startswith('CONSTANTE').sum()))

t('(2b) LAS 25 COLUMNAS MAS VACIAS DE LA BASE DE MODELADO (P1)')
P = RES['planA_v2_P1_modelo_sin_catalogo']
print(P.sort_values('pct_con_dato').head(25)[
      ['columna', 'tipo', 'pct_con_dato', 'distintos_aprox', 'veredicto']].to_string(index=False))

t('(2c) NULOS EN LA TABLA DE INDICADORES (la que entra al modelo)')
Q = RES['planA_indicadores_v3'].sort_values('pct_con_dato')
print(Q[['columna', 'tipo', 'pct_con_dato', 'distintos_aprox', 'veredicto']].to_string(index=False))

t('(2d) NULOS EN LAS DIMENSIONES')
for n in ['planA_dim_proveedor', 'planA_dim_entidad']:
    D = RES[n].sort_values('pct_con_dato')
    print(f'\n--- {n} ---')
    print(D[['columna', 'tipo', 'pct_con_dato', 'distintos_aprox']].head(18).to_string(index=False))

# =========================== (3) DISTRIBUCIONES NUMERICAS ===========================
t('(3) DISTRIBUCION DE LOS INDICADORES  (mediana y cuantiles, no promedios)')
con.sql(f"CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('{RUTA['planA_indicadores_v3']}')")
num_i = [c for c, ty in COLS['planA_indicadores_v3'] if es_num(ty)]
if num_i:
    piezas = [f"""SELECT '{c}' AS variable, count("{c}") AS con_dato,
                   round(100.0*count("{c}")/count(*),1) AS pct,
                   min("{c}") AS minimo, quantile_cont("{c}",0.25) AS p25,
                   median("{c}") AS mediana, quantile_cont("{c}",0.75) AS p75,
                   quantile_cont("{c}",0.95) AS p95, quantile_cont("{c}",0.99) AS p99,
                   max("{c}") AS maximo FROM ind""" for c in num_i]
    D = con.sql('\nUNION ALL\n'.join(piezas)).df()
    for c in ['minimo', 'p25', 'mediana', 'p75', 'p95', 'p99', 'maximo']:
        D[c] = pd.to_numeric(D[c], errors='coerce').round(4)
    print(D.to_string(index=False))
    D.to_csv(f'{DOC}/perfil_distribucion_indicadores.csv', index=False, encoding='utf-8-sig')

t('(3b) MONTO: distribucion por familia de procedimiento en la base de modelado')
con.sql(f"CREATE OR REPLACE VIEW p1 AS SELECT * FROM read_parquet('{RUTA['planA_v2_P1_modelo_sin_catalogo']}')")
fam = 'familia_procedimiento' if 'familia_procedimiento' in [c for c, _ in COLS['planA_v2_P1_modelo_sin_catalogo']] else None
if fam:
    M = con.sql(f"""SELECT {fam} AS familia, count(*) AS procesos,
                      round(100.0*count(monto_final)/count(*),1) AS pct_con_monto,
                      round(median(monto_final),0)                AS mediana_usd,
                      round(quantile_cont(monto_final,0.75),0)    AS p75_usd,
                      round(quantile_cont(monto_final,0.95),0)    AS p95_usd,
                      round(max(monto_final),0)                   AS maximo_usd,
                      round(sum(monto_final)/1e6,1)               AS suma_millones
                    FROM p1 GROUP BY 1 ORDER BY procesos DESC""").df()
    print(M.to_string(index=False))
    M.to_csv(f'{DOC}/perfil_monto_por_familia.csv', index=False, encoding='utf-8-sig')

t('(3c) ASIMETRIA DEL MONTO: por que se trabaja en logaritmo')
print(con.sql("""SELECT count(*) AS procesos,
                   round(avg(monto_final),0)                  AS promedio_usd,
                   round(median(monto_final),0)               AS mediana_usd,
                   round(avg(monto_final)/median(monto_final),1) AS veces_promedio_sobre_mediana,
                   round(quantile_cont(monto_final,0.99),0)   AS p99_usd,
                   round(max(monto_final)/1e6,1)              AS maximo_millones,
                   count(*) FILTER (WHERE monto_final <= 0)   AS monto_cero_o_negativo
                 FROM p1 WHERE monto_final IS NOT NULL""").df().to_string(index=False))

# =========================== (4) COBERTURA TEMPORAL ===========================
t('(4) COBERTURA TEMPORAL de la base de modelado')
fechas = [c for c, ty in COLS['planA_v2_P1_modelo_sin_catalogo'] if es_fecha(ty)][:18]
if fechas:
    piezas = [f"""SELECT '{c}' AS columna, round(100.0*count("{c}")/count(*),1) AS pct_con_dato,
                   min("{c}")::VARCHAR AS minimo, max("{c}")::VARCHAR AS maximo,
                   count(*) FILTER (WHERE "{c}" > DATE '2026-09-05') AS posteriores_al_corte
                  FROM p1""" for c in fechas]
    F = con.sql('\nUNION ALL\n'.join(piezas)).df()
    print(F.to_string(index=False))
    F.to_csv(f'{DOC}/perfil_fechas.csv', index=False, encoding='utf-8-sig')

t('(4b) FILAS Y MONTO POR ANIO en la base de modelado')
A = con.sql("""SELECT anio_proceso AS anio, count(*) AS procesos,
                 count(DISTINCT proveedor_ruc_final) AS proveedores,
                 round(sum(monto_final)/1e6,1) AS millones,
                 round(median(monto_final),0)   AS mediana_usd
               FROM p1 GROUP BY 1 ORDER BY 1""").df()
print(A.to_string(index=False))
A.to_csv(f'{DOC}/perfil_por_anio.csv', index=False, encoding='utf-8-sig')

# =========================== (5) INTEGRIDAD ENTRE BASES ===========================
t('(5) INTEGRIDAD REFERENCIAL Y PARTICION DEL UNIVERSO')
for n in ['planA_universo_v2', 'planA_v2_P0_catalogo', 'planA_v2_excluidos_solo_planeacion',
          'planA_v2_excluidos_sin_monto', 'planA_v2_sin_proveedor_identificable',
          'planA_dim_proveedor', 'planA_dim_entidad']:
    con.sql(f"CREATE OR REPLACE VIEW {n.replace('planA_','').replace('planA','')[:28]} AS "
            f"SELECT * FROM read_parquet('{RUTA[n]}')")
con.sql(f"CREATE OR REPLACE VIEW uni AS SELECT * FROM read_parquet('{RUTA['planA_universo_v2']}')")
con.sql(f"CREATE OR REPLACE VIEW p0  AS SELECT * FROM read_parquet('{RUTA['planA_v2_P0_catalogo']}')")
con.sql(f"CREATE OR REPLACE VIEW ex1 AS SELECT * FROM read_parquet('{RUTA['planA_v2_excluidos_solo_planeacion']}')")
con.sql(f"CREATE OR REPLACE VIEW ex2 AS SELECT * FROM read_parquet('{RUTA['planA_v2_excluidos_sin_monto']}')")
con.sql(f"CREATE OR REPLACE VIEW ex3 AS SELECT * FROM read_parquet('{RUTA['planA_v2_sin_proveedor_identificable']}')")
con.sql(f"CREATE OR REPLACE VIEW dpr AS SELECT * FROM read_parquet('{RUTA['planA_dim_proveedor']}')")
con.sql(f"CREATE OR REPLACE VIEW den AS SELECT * FROM read_parquet('{RUTA['planA_dim_entidad']}')")

part = con.sql("""WITH u AS (
    SELECT ocid, 'P1' AS parte FROM p1  UNION ALL
    SELECT ocid, 'P0'          FROM p0  UNION ALL
    SELECT ocid, 'solo_planeacion' FROM ex1 UNION ALL
    SELECT ocid, 'sin_monto'       FROM ex2 UNION ALL
    SELECT ocid, 'sin_proveedor'   FROM ex3)
  SELECT count(*) AS filas_sumadas, count(DISTINCT ocid) AS ocid_distintos FROM u""").fetchone()
n_uni = con.sql("SELECT count(*) FROM uni").fetchone()[0]
print(f'universo_v2 ................... {n_uni:,}')
print(f'suma de las 5 partes .......... {part[0]:,}   ({"cuadra" if part[0]==n_uni else "NO CUADRA"})')
print(f'ocid distintos en las partes .. {part[1]:,}   ({"sin solapamiento" if part[1]==part[0] else "HAY SOLAPAMIENTO"})')
fuera = con.sql("""WITH u AS (SELECT ocid FROM p1 UNION ALL SELECT ocid FROM p0 UNION ALL
                              SELECT ocid FROM ex1 UNION ALL SELECT ocid FROM ex2 UNION ALL
                              SELECT ocid FROM ex3)
                   SELECT count(*) FROM uni ANTI JOIN u USING (ocid)""").fetchone()[0]
print(f'procesos del universo en ninguna parte ... {fuera:,}')
cifra('particion_cuadra', part[0] == n_uni and part[1] == part[0] and fuera == 0,
      'P1+P0+3 exclusiones = universo_v2, sin solapamiento')

print('\n-- indicadores vs base de modelado --')
print(con.sql("""SELECT (SELECT count(*) FROM p1)  AS filas_p1,
                        (SELECT count(*) FROM ind) AS filas_indicadores,
                        (SELECT count(*) FROM ind ANTI JOIN p1 USING (ocid)) AS indicador_sin_proceso,
                        (SELECT count(*) FROM p1 ANTI JOIN ind USING (ocid)) AS proceso_sin_indicador,
                        (SELECT count(*) FROM p1 LEFT JOIN ind USING (ocid)) AS filas_tras_unir
              """).df().to_string(index=False))

print('\n-- dimensiones --')
kpr, ken = LLAVE['planA_dim_proveedor'], LLAVE['planA_dim_entidad']
col_ent = next((c for c, _ in COLS['planA_v2_P1_modelo_sin_catalogo']
                if 'ruc' in c.lower() and ('entidad' in c.lower() or 'compradora' in c.lower()
                                           or 'buyer' in c.lower())), None)
print(con.sql(f"""SELECT (SELECT count(DISTINCT proveedor_ruc_final) FROM p1) AS ruc_proveedor_en_p1,
                         (SELECT count(*) FROM dpr)                           AS filas_dim_proveedor,
                         (SELECT count(*) FROM (SELECT DISTINCT proveedor_ruc_final AS k FROM p1
                             WHERE proveedor_ruc_final IS NOT NULL)
                           ANTI JOIN dpr ON k = dpr."{kpr}")                   AS proveedor_sin_dimension
               """).df().to_string(index=False))
if col_ent:
    print(f'columna de RUC de entidad detectada en P1: {col_ent}')
    print(con.sql(f"""SELECT (SELECT count(DISTINCT "{col_ent}") FROM p1) AS ruc_entidad_en_p1,
                             (SELECT count(*) FROM den)                   AS filas_dim_entidad,
                             (SELECT count(*) FROM (SELECT DISTINCT "{col_ent}" AS k FROM p1
                                 WHERE "{col_ent}" IS NOT NULL)
                               ANTI JOIN den ON k = den."{ken}")           AS entidad_sin_dimension
                   """).df().to_string(index=False))
else:
    print('no detecte columna de RUC de entidad en P1; dime como se llama y la reviso')

print('\n-- conjunto de validacion (los 15 del SRI): donde aparecen --')
con.sql(f"CREATE OR REPLACE VIEW val AS SELECT * FROM read_parquet('{RUTA['planA_validacion_sri_verificada']}')")
kval = next((c for c, _ in COLS['planA_validacion_sri_verificada'] if 'ruc' in c.lower()), 'ruc')
print(con.sql(f"""SELECT (SELECT count(*) FROM val)                                   AS ruc_verificados,
                         (SELECT count(*) FROM val SEMI JOIN dpr ON val."{kval}" = dpr."{kpr}") AS estan_en_dim_proveedor,
                         (SELECT count(DISTINCT p1.ocid) FROM p1 SEMI JOIN val
                             ON p1.proveedor_ruc_final = val."{kval}")                AS procesos_de_esos_ruc
               """).df().to_string(index=False))

print('\n-- control de fuga: columnas prohibidas en la tabla de indicadores --')
prohibidas = [c for c, _ in COLS['planA_indicadores_v3']
              if c.startswith(('red_', 'cob_', 'com_', 'idx_'))
              or 'monto_tender' in c or 'alguna_vez' in c or 'fantasma' in c.lower()]
print('prohibidas encontradas:', prohibidas if prohibidas else 'ninguna (correcto)')
cifra('columnas_prohibidas_en_indicadores', len(prohibidas))

print('\n-- cuadre de plata --')
print(con.sql("""SELECT round((SELECT sum(monto_final) FROM p1)/1e6,1) AS p1_millones,
                        round((SELECT sum(monto_final) FROM p0)/1e6,1) AS p0_millones,
                        round(((SELECT sum(monto_final) FROM p1)+(SELECT sum(monto_final) FROM p0))/1e6,1) AS suma_millones,
                        round((SELECT sum(monto_final) FROM uni)/1e6,1) AS universo_millones
              """).df().to_string(index=False))

# =========================== (6) CENTINELAS DE ANTIGUEDAD (para la celda 24) ===========================
t('(6) CENTINELAS DE FECHA EN LA DIMENSION DE PROVEEDOR')
fcol = next((c for c, ty in COLS['planA_dim_proveedor'] if es_fecha(ty) and 'inicio' in c.lower()), None)
acol = [c for c, _ in COLS['planA_indicadores_v3'] if 'antigued' in c.lower()]
print('columna de fecha de inicio en la dimension:', fcol)
print('columnas de antiguedad en los indicadores :', acol if acol else 'ninguna')
if fcol:
    print(con.sql(f"""SELECT count(*) FILTER (WHERE "{fcol}" = DATE '1900-01-01') AS piso_1900_01_01,
                             count(*) FILTER (WHERE "{fcol}" <  DATE '1900-01-01') AS anteriores_a_1900,
                             count(*) FILTER (WHERE "{fcol}" <  DATE '1950-01-01') AS anteriores_a_1950,
                             count(*) FILTER (WHERE "{fcol}" IS NULL)              AS sin_fecha,
                             count(*)                                              AS proveedores
                      FROM dpr""").df().to_string(index=False))
print('\ncolumnas de la tabla de indicadores (las necesito tal cual para la celda 24):')
print([c for c, _ in COLS['planA_indicadores_v3']])

# =========================== (7) SALIDAS ===========================
t('(7) ARCHIVOS ESCRITOS')
pd.DataFrame(CIFRAS).to_csv(f'{DOC}/cifras_celda23.csv', index=False, encoding='utf-8-sig')
lineas = ['# Inventario de entregables — base Plan A', '',
          f'Generado desde los datos. Total: {len(INV)} archivos, {round(INV.MB.sum(),1)} MB.', '',
          INV[['archivo','caja','grano','filas','columnas','MB','llave','llave_unica','sha256_16']]
             .to_markdown(index=False), '',
          '## Columnas por archivo', '']
for n, _, _ in ARCH:
    v = RES[n].veredicto
    lineas.append(f'- **{n}**: {len(RES[n])} columnas · {int((v=="completa").sum())} completas · '
                  f'{int(v.str.startswith("CONSTANTE").sum())} constantes · '
                  f'{int(v.str.startswith("INSERVIBLE").sum())} sin un solo dato')
open(f'{DOC}/INVENTARIO_ENTREGABLES.md', 'w', encoding='utf-8').write('\n'.join(lineas))
for f in sorted(os.listdir(DOC)):
    print(f'  {f}')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 23-bis · CONTINUACION (corrige el tipo VARCHAR[]) ============
# Continua la celda 23 desde la parte (2). Reutiliza lo que ya quedo en memoria
# (con, RUTA, ARCH, COLS, LLAVE, INV, CIFRAS, cifra, t, DOC). Solo lectura.

def anidado(ty): return any(s in ty.upper() for s in ('[', 'STRUCT', 'MAP', 'UNION'))
def es_texto(ty): return ty.upper() == 'VARCHAR'          # <-- la correccion: igualdad exacta
def es_num(ty):  return any(k in ty.upper() for k in ('TINYINT','SMALLINT','INTEGER','BIGINT',
                            'HUGEINT','DOUBLE','FLOAT','REAL','DECIMAL'))
def es_fecha(ty):return any(k in ty.upper() for k in ('DATE','TIMESTAMP'))

# =========================== (2) NULOS, VACIOS Y COLUMNAS INSERVIBLES ===========================
t('(2) NULOS Y VACIOS COLUMNA POR COLUMNA  (detalle completo en los CSV)')
def perfil(n):
    """Una fila por columna: % con dato, % vacio de texto, valores distintos, veredicto."""
    esq, filas = COLS[n], int(INV.loc[INV.archivo == n, 'filas'].iloc[0])
    out = []
    for i in range(0, len(esq), 35):
        trozo = esq[i:i + 35]; sel = []
        for c, ty in trozo:
            sel.append(f'count("{c}") AS "nn_{i}_{c}"')
            if es_texto(ty):
                sel.append(f"""count(*) FILTER (WHERE trim("{c}") = '') AS "vc_{i}_{c}" """)
            if not anidado(ty):
                sel.append(f'approx_count_distinct("{c}") AS "nd_{i}_{c}"')
        r = con.sql(f"SELECT {', '.join(sel)} FROM read_parquet('{RUTA[n]}')").df().iloc[0]
        for c, ty in trozo:
            nn = int(r[f'nn_{i}_{c}'])
            vc = int(r[f'vc_{i}_{c}']) if f'vc_{i}_{c}' in r.index else 0
            nd = int(r[f'nd_{i}_{c}']) if f'nd_{i}_{c}' in r.index else -1
            util = nn - vc
            if   util == 0:              ver = 'INSERVIBLE: sin un solo dato'
            elif nd == 1:                ver = 'CONSTANTE: un unico valor'
            elif util / filas < 0.05:    ver = 'casi vacia (<5% con dato)'
            elif util / filas < 0.50:    ver = 'mitad vacia'
            elif util == filas:          ver = 'completa'
            else:                        ver = 'con faltantes'
            out.append({'columna': c, 'tipo': ty, 'con_dato': nn, 'vacio_texto': vc,
                        'pct_con_dato': round(100 * util / filas, 2),
                        'distintos_aprox': None if nd < 0 else nd, 'veredicto': ver})
    return pd.DataFrame(out)

RES = {}
for n, _, _ in ARCH:
    P = perfil(n); RES[n] = P
    P.to_csv(f'{DOC}/perfil_nulos_{n}.csv', index=False, encoding='utf-8-sig')
    v = P.veredicto
    print(f'\n{n}  ({len(P)} columnas)')
    print(f'   completas (100% con dato) ....... {int((v == "completa").sum()):>4}')
    print(f'   con faltantes ................... {int((v == "con faltantes").sum()):>4}')
    print(f'   mitad vacia (50-95% nulo) ....... {int((v == "mitad vacia").sum()):>4}')
    print(f'   casi vacia (>95% nulo) .......... {int(v.str.startswith("casi").sum()):>4}')
    print(f'   constantes (un solo valor) ...... {int(v.str.startswith("CONSTANTE").sum()):>4}')
    print(f'   inservibles (ni un dato) ........ {int(v.str.startswith("INSERVIBLE").sum()):>4}')
    cifra(f'{n}__columnas_completas',    int((v == 'completa').sum()))
    cifra(f'{n}__columnas_inservibles',  int(v.str.startswith('INSERVIBLE').sum()))
    cifra(f'{n}__columnas_constantes',   int(v.str.startswith('CONSTANTE').sum()))

t('(2b) LAS 25 COLUMNAS MAS VACIAS DE LA BASE DE MODELADO (P1)')
P = RES['planA_v2_P1_modelo_sin_catalogo']
print(P.sort_values('pct_con_dato').head(25)[
      ['columna','tipo','pct_con_dato','distintos_aprox','veredicto']].to_string(index=False))

t('(2c) COLUMNAS INSERVIBLES Y CONSTANTES DE P1 (candidatas a no enviar)')
mal = P[P.veredicto.str.startswith(('INSERVIBLE','CONSTANTE'))]
print(f'{len(mal)} columnas de 203')
print(mal[['columna','tipo','pct_con_dato','distintos_aprox','veredicto']].to_string(index=False))

t('(2d) NULOS EN LA TABLA DE INDICADORES (la que entra al modelo)')
Q = RES['planA_indicadores_v3'].sort_values('pct_con_dato')
print(Q[['columna','tipo','pct_con_dato','distintos_aprox','veredicto']].to_string(index=False))

t('(2e) NULOS EN LAS DIMENSIONES')
for n in ['planA_dim_proveedor', 'planA_dim_entidad']:
    D = RES[n].sort_values('pct_con_dato')
    print(f'\n--- {n} ---')
    print(D[['columna','tipo','pct_con_dato','distintos_aprox']].head(20).to_string(index=False))

# =========================== (3) DISTRIBUCIONES ===========================
t('(3) DISTRIBUCION DE LOS INDICADORES  (mediana y cuantiles, no promedios)')
con.sql(f"CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('{RUTA['planA_indicadores_v3']}')")
con.sql(f"CREATE OR REPLACE VIEW p1  AS SELECT * FROM read_parquet('{RUTA['planA_v2_P1_modelo_sin_catalogo']}')")
num_i = [c for c, ty in COLS['planA_indicadores_v3'] if es_num(ty)]
if num_i:
    piezas = [f"""SELECT '{c}' AS variable, count("{c}") AS con_dato,
                   round(100.0*count("{c}")/count(*),1) AS pct,
                   min("{c}") AS minimo, quantile_cont("{c}",0.25) AS p25,
                   median("{c}") AS mediana, quantile_cont("{c}",0.75) AS p75,
                   quantile_cont("{c}",0.95) AS p95, quantile_cont("{c}",0.99) AS p99,
                   max("{c}") AS maximo FROM ind""" for c in num_i]
    D = con.sql('\nUNION ALL\n'.join(piezas)).df()
    for c in ['minimo','p25','mediana','p75','p95','p99','maximo']:
        D[c] = pd.to_numeric(D[c], errors='coerce').round(4)
    print(D.to_string(index=False))
    D.to_csv(f'{DOC}/perfil_distribucion_indicadores.csv', index=False, encoding='utf-8-sig')

t('(3b) MONTO POR FAMILIA DE PROCEDIMIENTO en la base de modelado')
cols_p1 = [c for c, _ in COLS['planA_v2_P1_modelo_sin_catalogo']]
fam = 'familia_procedimiento' if 'familia_procedimiento' in cols_p1 else None
if fam:
    M = con.sql(f"""SELECT {fam} AS familia, count(*) AS procesos,
                      round(100.0*count(monto_final)/count(*),1) AS pct_con_monto,
                      round(median(monto_final),0)             AS mediana_usd,
                      round(quantile_cont(monto_final,0.75),0) AS p75_usd,
                      round(quantile_cont(monto_final,0.95),0) AS p95_usd,
                      round(max(monto_final),0)                AS maximo_usd,
                      round(sum(monto_final)/1e6,1)            AS suma_millones
                    FROM p1 GROUP BY 1 ORDER BY procesos DESC""").df()
    print(M.to_string(index=False))
    M.to_csv(f'{DOC}/perfil_monto_por_familia.csv', index=False, encoding='utf-8-sig')

t('(3c) ASIMETRIA DEL MONTO: por que se modela en logaritmo')
print(con.sql("""SELECT count(*) AS procesos,
                   round(avg(monto_final),0)    AS promedio_usd,
                   round(median(monto_final),0) AS mediana_usd,
                   round(avg(monto_final)/nullif(median(monto_final),0),1) AS promedio_sobre_mediana,
                   round(quantile_cont(monto_final,0.99),0) AS p99_usd,
                   round(max(monto_final)/1e6,1)            AS maximo_millones,
                   count(*) FILTER (WHERE monto_final <= 0)  AS monto_cero_o_negativo
                 FROM p1 WHERE monto_final IS NOT NULL""").df().to_string(index=False))

# =========================== (4) COBERTURA TEMPORAL ===========================
t('(4) COBERTURA TEMPORAL de la base de modelado')
fechas = [c for c, ty in COLS['planA_v2_P1_modelo_sin_catalogo'] if es_fecha(ty)][:18]
if fechas:
    piezas = [f"""SELECT '{c}' AS columna, round(100.0*count("{c}")/count(*),1) AS pct_con_dato,
                   min("{c}")::VARCHAR AS minimo, max("{c}")::VARCHAR AS maximo,
                   count(*) FILTER (WHERE "{c}" > DATE '2026-09-05') AS posteriores_al_corte
                  FROM p1""" for c in fechas]
    F = con.sql('\nUNION ALL\n'.join(piezas)).df()
    print(F.to_string(index=False))
    F.to_csv(f'{DOC}/perfil_fechas.csv', index=False, encoding='utf-8-sig')

t('(4b) FILAS, PROVEEDORES Y MONTO POR ANIO')
A = con.sql("""SELECT anio_proceso AS anio, count(*) AS procesos,
                 count(DISTINCT proveedor_ruc_final) AS proveedores,
                 round(sum(monto_final)/1e6,1) AS millones,
                 round(median(monto_final),0)   AS mediana_usd
               FROM p1 GROUP BY 1 ORDER BY 1""").df()
print(A.to_string(index=False))
A.to_csv(f'{DOC}/perfil_por_anio.csv', index=False, encoding='utf-8-sig')

# =========================== (5) INTEGRIDAD ENTRE BASES ===========================
t('(5) INTEGRIDAD REFERENCIAL Y PARTICION DEL UNIVERSO')
for alias, n in [('uni','planA_universo_v2'), ('p0','planA_v2_P0_catalogo'),
                 ('ex1','planA_v2_excluidos_solo_planeacion'), ('ex2','planA_v2_excluidos_sin_monto'),
                 ('ex3','planA_v2_sin_proveedor_identificable'), ('dpr','planA_dim_proveedor'),
                 ('den','planA_dim_entidad'), ('val','planA_validacion_sri_verificada')]:
    con.sql(f"CREATE OR REPLACE VIEW {alias} AS SELECT * FROM read_parquet('{RUTA[n]}')")

part = con.sql("""WITH u AS (SELECT ocid FROM p1  UNION ALL SELECT ocid FROM p0 UNION ALL
                             SELECT ocid FROM ex1 UNION ALL SELECT ocid FROM ex2 UNION ALL
                             SELECT ocid FROM ex3)
                  SELECT count(*), count(DISTINCT ocid) FROM u""").fetchone()
n_uni = con.sql("SELECT count(*) FROM uni").fetchone()[0]
fuera = con.sql("""WITH u AS (SELECT ocid FROM p1 UNION ALL SELECT ocid FROM p0 UNION ALL
                              SELECT ocid FROM ex1 UNION ALL SELECT ocid FROM ex2 UNION ALL
                              SELECT ocid FROM ex3)
                   SELECT count(*) FROM uni ANTI JOIN u USING (ocid)""").fetchone()[0]
print(f'universo_v2 ............................. {n_uni:,}')
print(f'suma de las 5 partes .................... {part[0]:,}   ({"cuadra" if part[0]==n_uni else "NO CUADRA"})')
print(f'ocid distintos en las partes ............ {part[1]:,}   ({"sin solapamiento" if part[1]==part[0] else "HAY SOLAPAMIENTO"})')
print(f'procesos del universo en ninguna parte .. {fuera:,}')
cifra('particion_cuadra', bool(part[0]==n_uni and part[1]==part[0] and fuera==0),
      'P1+P0+3 exclusiones = universo_v2, sin solapamiento')

print('\n-- indicadores vs base de modelado --')
print(con.sql("""SELECT (SELECT count(*) FROM p1)  AS filas_p1,
                        (SELECT count(*) FROM ind) AS filas_indicadores,
                        (SELECT count(*) FROM ind ANTI JOIN p1 USING (ocid)) AS indicador_sin_proceso,
                        (SELECT count(*) FROM p1 ANTI JOIN ind USING (ocid)) AS proceso_sin_indicador,
                        (SELECT count(*) FROM p1 LEFT JOIN ind USING (ocid)) AS filas_tras_unir
              """).df().to_string(index=False))

print('\n-- dimension proveedor --')
kpr = LLAVE['planA_dim_proveedor']; ken = LLAVE['planA_dim_entidad']
print(con.sql(f"""SELECT (SELECT count(DISTINCT proveedor_ruc_final) FROM p1) AS ruc_en_p1,
                         (SELECT count(*) FROM dpr) AS filas_dim,
                         (SELECT count(*) FROM (SELECT DISTINCT proveedor_ruc_final AS k FROM p1
                              WHERE proveedor_ruc_final IS NOT NULL) s
                            ANTI JOIN dpr ON s.k = dpr."{kpr}") AS proveedor_sin_dimension,
                         (SELECT count(*) FROM p1 WHERE proveedor_ruc_final IS NULL) AS proceso_sin_ruc
              """).df().to_string(index=False))

col_ent = next((c for c in cols_p1 if 'ruc' in c.lower() and
                any(k in c.lower() for k in ('entidad','compradora','buyer'))), None)
print(f'\n-- dimension entidad (columna detectada en P1: {col_ent}) --')
if col_ent:
    print(con.sql(f"""SELECT (SELECT count(DISTINCT "{col_ent}") FROM p1) AS ruc_en_p1,
                             (SELECT count(*) FROM den) AS filas_dim,
                             (SELECT count(*) FROM (SELECT DISTINCT "{col_ent}" AS k FROM p1
                                  WHERE "{col_ent}" IS NOT NULL) s
                                ANTI JOIN den ON s.k = den."{ken}") AS entidad_sin_dimension
                  """).df().to_string(index=False))
else:
    print('no la detecte; aqui estan las columnas con "ruc" en el nombre:')
    print([c for c in cols_p1 if 'ruc' in c.lower()])

print('\n-- conjunto de validacion (los 15 del SRI) --')
kval = next((c for c, _ in COLS['planA_validacion_sri_verificada'] if 'ruc' in c.lower()), 'ruc')
print(con.sql(f"""SELECT (SELECT count(*) FROM val) AS ruc_verificados,
                         (SELECT count(*) FROM val SEMI JOIN dpr ON val."{kval}" = dpr."{kpr}") AS en_dim_proveedor,
                         (SELECT count(*) FROM p1 SEMI JOIN val ON p1.proveedor_ruc_final = val."{kval}") AS procesos_de_esos_ruc
              """).df().to_string(index=False))

print('\n-- control de fuga: columnas prohibidas en la tabla de indicadores --')
prohib = [c for c, _ in COLS['planA_indicadores_v3']
          if c.startswith(('red_','cob_','com_','idx_')) or 'monto_tender' in c
          or 'alguna_vez' in c or 'fantasma' in c.lower()]
print('prohibidas encontradas:', prohib if prohib else 'ninguna (correcto)')
cifra('columnas_prohibidas_en_indicadores', len(prohib))

print('\n-- cuadre de plata --')
print(con.sql("""SELECT round((SELECT sum(monto_final) FROM p1)/1e6,1) AS p1_millones,
                        round((SELECT sum(monto_final) FROM p0)/1e6,1) AS p0_millones,
                        round(((SELECT sum(monto_final) FROM p1)+(SELECT sum(monto_final) FROM p0))/1e6,1) AS suma_millones,
                        round((SELECT sum(monto_final) FROM uni)/1e6,1) AS universo_millones
              """).df().to_string(index=False))

# =========================== (6) CENTINELAS DE ANTIGUEDAD ===========================
t('(6) CENTINELAS DE FECHA EN LA DIMENSION DE PROVEEDOR')
fcol = next((c for c, ty in COLS['planA_dim_proveedor'] if es_fecha(ty) and 'inicio' in c.lower()), None)
acol = [c for c, _ in COLS['planA_indicadores_v3'] if 'antigued' in c.lower()]
print('fecha de inicio en la dimension :', fcol)
print('columnas de antiguedad          :', acol if acol else 'ninguna')
if fcol:
    print(con.sql(f"""SELECT count(*) FILTER (WHERE "{fcol}" = DATE '1900-01-01') AS piso_1900,
                             count(*) FILTER (WHERE "{fcol}" <  DATE '1900-01-01') AS antes_de_1900,
                             count(*) FILTER (WHERE "{fcol}" <  DATE '1950-01-01') AS antes_de_1950,
                             count(*) FILTER (WHERE "{fcol}" IS NULL)              AS sin_fecha,
                             count(*) AS proveedores FROM dpr""").df().to_string(index=False))
print('\ncolumnas de planA_indicadores_v3 (las necesito tal cual para la celda 24):')
print([c for c, _ in COLS['planA_indicadores_v3']])

# =========================== (7) SALIDAS ===========================
t('(7) ARCHIVOS ESCRITOS')
pd.DataFrame(CIFRAS).to_csv(f'{DOC}/cifras_celda23.csv', index=False, encoding='utf-8-sig')
L = ['# Inventario de entregables - base Plan A', '',
     f'Generado desde los datos. {len(INV)} archivos, {round(INV.MB.sum(),1)} MB.', '', '```',
     INV[['archivo','caja','grano','filas','columnas','MB','llave','llave_unica','sha256_16']].to_string(index=False),
     '```', '', '## Columnas por archivo', '']
for n, _, _ in ARCH:
    v = RES[n].veredicto
    L.append(f'- **{n}**: {len(RES[n])} columnas | {int((v=="completa").sum())} completas | '
             f'{int(v.str.startswith("CONSTANTE").sum())} constantes | '
             f'{int(v.str.startswith("INSERVIBLE").sum())} sin un solo dato')
open(f'{DOC}/INVENTARIO_ENTREGABLES.md','w',encoding='utf-8').write('\n'.join(L))
for f in sorted(os.listdir(DOC)): print(f'  {f}')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 24 · PARCHE -> planA_indicadores_v3_1.parquet ============
# 1) cuadra la plata con las 5 partes   2) diagnostica valores imposibles y saca muestras
# 3) anula centinelas e imposibles      4) reconstruye las 9 pct_ SOLO sobre no nulos
# 5) controles de aceptacion            6) escribe v3.1, re-firma y registra la enmienda
import os, json, hashlib
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST = 'plan_A/documentacion', 'plan_A/_version_1_historico'
os.makedirs(HIST, exist_ok=True)
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 80)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
CIF = []
def cifra(n, v, nota=''): CIF.append({'cifra': n, 'valor': v, 'nota': nota}); return v
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

V = {'ind':'planA_indicadores_v3', 'p1':'planA_v2_P1_modelo_sin_catalogo', 'p0':'planA_v2_P0_catalogo',
     'ex1':'planA_v2_excluidos_solo_planeacion', 'ex2':'planA_v2_excluidos_sin_monto',
     'ex3':'planA_v2_sin_proveedor_identificable', 'uni':'planA_universo_v2', 'dpr':'planA_dim_proveedor'}
for a, n in V.items():
    con.sql(f"CREATE OR REPLACE VIEW {a} AS SELECT * FROM read_parquet('plan_A/{n}.parquet')")
COLS_IND = [c for c in con.sql("DESCRIBE SELECT * FROM ind").df().column_name]
COLS_P1  = [c for c in con.sql("DESCRIBE SELECT * FROM p1").df().column_name]

# =================== (1) CUADRE DE PLATA CON LAS 5 PARTES ===================
t('(1) CUADRE DE PLATA: las 5 partes contra el universo')
C = con.sql("""
  SELECT 'P1 modelo (sin catalogo)' AS parte, count(*) AS procesos,
         round(coalesce(sum(monto_final),0)/1e6,1) AS millones FROM p1
  UNION ALL SELECT 'P0 catalogo electronico', count(*), round(coalesce(sum(monto_final),0)/1e6,1) FROM p0
  UNION ALL SELECT 'excluido: solo planeacion', count(*), round(coalesce(sum(monto_final),0)/1e6,1) FROM ex1
  UNION ALL SELECT 'excluido: sin monto', count(*), round(coalesce(sum(monto_final),0)/1e6,1) FROM ex2
  UNION ALL SELECT 'excluido: sin proveedor identificable', count(*), round(coalesce(sum(monto_final),0)/1e6,1) FROM ex3
  ORDER BY millones DESC""").df()
tot = con.sql("SELECT count(*), round(coalesce(sum(monto_final),0)/1e6,1) FROM uni").fetchone()
C.loc[len(C)] = ['SUMA DE LAS PARTES', C.procesos.sum(), round(C.millones.sum(), 1)]
C.loc[len(C)] = ['universo_v2 (referencia)', tot[0], tot[1]]
C['pct_plata'] = (100 * C.millones / tot[1]).round(2)
print(C.to_string(index=False))
dif = round(C.millones.iloc[-2] - tot[1], 1)
print(f'\ndiferencia suma - universo: {dif} MM  ({"CUADRA" if abs(dif) < 0.2 else "NO CUADRA"})')
cifra('plata_cuadra_5_partes', abs(dif) < 0.2, f'diferencia {dif} MM')
cifra('plata_excluida_sin_proveedor_MM',
      float(C.loc[C.parte.str.contains('sin proveedor'), 'millones'].iloc[0]),
      'monto que sale del modelo por no poder identificar al proveedor')

# =================== (2) VALORES IMPOSIBLES Y MUESTRAS PARA REVISAR ===================
t('(2) VALORES IMPOSIBLES EN LA TABLA DE INDICADORES')
print(con.sql("""SELECT
   count(*) FILTER (WHERE ind_duracion_contrato_dias > 7300)      AS duracion_mayor_20_anios,
   count(*) FILTER (WHERE ind_duracion_contrato_dias > 3650)      AS duracion_mayor_10_anios,
   count(*) FILTER (WHERE ind_contrato_sobre_adjudicado > 100)    AS contrato_100_veces_lo_adjudicado,
   count(*) FILTER (WHERE ind_contrato_sobre_adjudicado BETWEEN 2 AND 100) AS contrato_2_a_100_veces,
   count(*) FILTER (WHERE monto_final < 1)                        AS monto_menor_a_1_dolar,
   count(*) FILTER (WHERE monto_final < 100)                      AS monto_menor_a_100
 FROM ind""").df().to_string(index=False))

def pick(cols, *pats):
    for p in pats:
        for c in cols:
            if p in c.lower(): return c
    return None
nom_ent = pick(COLS_P1, 'entidad_razon', 'razon_social_entidad', 'entidad_nombre', 'sri_razon_social')
print(f'\n(2a) LOS 10 MONTOS MAS BAJOS  -> revisalos en el portal (columna de entidad: {nom_ent})')
print(con.sql(f"""SELECT i.ocid, i.familia, i.fecha_proceso, i.monto_final,
                         {'p1."'+nom_ent+'" AS entidad' if nom_ent else "'-' AS entidad"}
                  FROM ind i JOIN p1 USING (ocid) ORDER BY i.monto_final LIMIT 10""").df().to_string(index=False))
print('\n(2b) LAS 10 DURACIONES DE CONTRATO IMPOSIBLES  -> revisalas en el portal')
print(con.sql("""SELECT ocid, familia, fecha_proceso, monto_final, ind_duracion_contrato_dias,
                        round(ind_duracion_contrato_dias/365.25,1) AS anios
                 FROM ind ORDER BY ind_duracion_contrato_dias DESC LIMIT 10""").df().to_string(index=False))
print('\n(2c) LOS 10 RATIOS contrato/adjudicado MAS ALTOS  -> revisalos en el portal')
print(con.sql("""SELECT ocid, familia, fecha_proceso, monto_final, ind_contrato_sobre_adjudicado
                 FROM ind ORDER BY ind_contrato_sobre_adjudicado DESC LIMIT 10""").df().to_string(index=False))

# =================== (3) PARCHE: CENTINELAS E IMPOSIBLES ===================
t('(3) PARCHE')
fini = pick([c for c in con.sql("DESCRIBE SELECT * FROM dpr").df().column_name],
            'sri_fecha_inicio_actividades_fecha', 'fecha_inicio_actividades')
print('columna de fecha en la dimension:', fini)
con.sql(f"""CREATE OR REPLACE TABLE ruc_centinela AS
            SELECT ruc FROM dpr WHERE "{fini}" <= DATE '1900-01-01'""")
n_cent = con.sql("SELECT count(*) FROM ruc_centinela").fetchone()[0]
afect = con.sql("""SELECT count(*) FROM ind SEMI JOIN ruc_centinela ON ind.proveedor_ruc = ruc_centinela.ruc
                   WHERE ind_antiguedad_dias IS NOT NULL""").fetchone()[0]
print(f'RUC centinela (fecha <= 1900-01-01): {n_cent}   procesos con antiguedad a anular: {afect}')
cifra('ruc_centinela_fecha', n_cent); cifra('procesos_antiguedad_anulada', afect)

con.sql("""CREATE OR REPLACE TABLE base31 AS
  SELECT * REPLACE (
    CASE WHEN proveedor_ruc IN (SELECT ruc FROM ruc_centinela) THEN NULL
         ELSE ind_antiguedad_dias END                                   AS ind_antiguedad_dias,
    CASE WHEN proveedor_ruc IN (SELECT ruc FROM ruc_centinela) THEN TRUE
         ELSE falta_antiguedad END                                      AS falta_antiguedad,
    CASE WHEN ind_duracion_contrato_dias > 7300 THEN NULL
         ELSE ind_duracion_contrato_dias END                            AS ind_duracion_contrato_dias,
    CASE WHEN ind_duracion_contrato_dias > 7300 THEN TRUE
         ELSE falta_duracion END                                        AS falta_duracion,
    CASE WHEN ind_contrato_sobre_adjudicado > 100 THEN NULL
         ELSE ind_contrato_sobre_adjudicado END                         AS ind_contrato_sobre_adjudicado
  ) FROM ind""")
print(con.sql("""SELECT count(*) AS filas,
                   count(ind_antiguedad_dias)           AS antiguedad_con_dato,
                   count(ind_duracion_contrato_dias)    AS duracion_con_dato,
                   count(ind_contrato_sobre_adjudicado) AS ratio_con_dato,
                   max(ind_duracion_contrato_dias)      AS duracion_maxima,
                   max(ind_contrato_sobre_adjudicado)   AS ratio_maximo
                 FROM base31""").df().to_string(index=False))

# =================== (4) RECONSTRUCCION DE LAS 9 pct_ ===================
t('(4) RECONSTRUCCION DE LOS RANGOS PERCENTILES (solo sobre no nulos)')
# (columna del modelo, indicador fuente, direccion: ASC = valor alto es mas anomalo)
VARS = [('pct_precio_alto',       'ind_desv_precio_unit_z',        'ASC'),
        ('pct_cuota_12m',         'ind_cuota_prov_en_entidad_12m', 'ASC'),
        ('pct_recurrencia_par',   'ind_recurrencia_par_12m',       'ASC'),
        ('pct_monto_capital',     'ind_monto_sobre_capital',       'ASC'),
        ('pct_fraccionamiento',   'ind_fraccionamiento_monto_30d', 'ASC'),
        ('pct_proveedor_nuevo',   'ind_antiguedad_dias',           'DESC'),
        ('pct_poca_concurrencia', 'ind_n_oferentes',               'DESC'),
        ('pct_plazo_corto',       'ind_plazo_ofertas_dias',        'DESC'),
        ('pct_firma_rapida',      'ind_plazo_adj_firma_dias',      'DESC')]
# grupo comparable = familia x periodo normativo; si el grupo tiene menos de 30 casos con dato
# baja a familia, y si tampoco llega, a global. Asi ningun rango se calcula sobre poquitos casos.
for i, (pct, src, d) in enumerate(VARS):
    con.sql(f"""CREATE OR REPLACE TABLE r{i} AS
      WITH c AS (SELECT ocid, familia, periodo, "{src}" AS x,
                        count("{src}") OVER (PARTITION BY familia, periodo) AS n_fp,
                        count("{src}") OVER (PARTITION BY familia)          AS n_f
                 FROM base31),
           d AS (SELECT ocid, x,
                        CASE WHEN n_fp >= 30 THEN familia || '|' || periodo
                             WHEN n_f  >= 30 THEN familia
                             ELSE 'GLOBAL' END AS grupo
                 FROM c WHERE x IS NOT NULL)
      SELECT ocid, percent_rank() OVER (PARTITION BY grupo ORDER BY x {d}) AS "{pct}" FROM d""")
    print(f'  {pct:24s} <- {src:32s} {d:4s}  filas con rango: '
          f'{con.sql(f"SELECT count(*) FROM r{i}").fetchone()[0]:,}')

pctset = {v[0]: i for i, v in enumerate(VARS)}
sel = ', '.join((f'r{pctset[c]}."{c}"' if c in pctset else f'base31."{c}"') for c in COLS_IND)
joins = ' '.join(f'LEFT JOIN r{i} USING (ocid)' for i in range(len(VARS)))
con.sql(f"CREATE OR REPLACE TABLE v31 AS SELECT {sel} FROM base31 {joins}")
print(f'\nv3.1 armada: {con.sql("SELECT count(*) FROM v31").fetchone()[0]:,} filas, '
      f'{len(con.sql("DESCRIBE SELECT * FROM v31").df()):,} columnas')

# =================== (5) CONTROLES DE ACEPTACION ===================
t('(5) CONTROLES DE ACEPTACION')
piezas = [f"""SELECT '{pct}' AS variable, '{src}' AS indicador, '{d}' AS direccion,
   count("{src}") AS fuente_con_dato, count("{pct}") AS rango_con_dato,
   CASE WHEN count("{src}") = count("{pct}") THEN 'OK' ELSE 'DESCUADRE' END AS nulos_coinciden,
   round(min("{pct}"),4) AS pct_min, round(max("{pct}"),4) AS pct_max,
   round(corr("{src}", "{pct}"),3) AS correlacion,
   CASE WHEN ('{d}' = 'ASC' AND corr("{src}","{pct}") > 0)
          OR ('{d}' = 'DESC' AND corr("{src}","{pct}") < 0) THEN 'OK' ELSE 'INVERTIDA' END AS direccion_ok
   FROM v31""" for pct, src, d in VARS]
K = con.sql('\nUNION ALL\n'.join(piezas)).df()
print(K.to_string(index=False))
K.to_csv(f'{DOC}/control_rangos_v3_1.csv', index=False, encoding='utf-8-sig')
cifra('rangos_con_nulos_coherentes', int((K.nulos_coinciden == 'OK').sum()), f'de {len(K)}')
cifra('rangos_con_direccion_correcta', int((K.direccion_ok == 'OK').sum()), f'de {len(K)}')

print('\n-- cuanto cambio frente a v3 --')
cmp_ = [f"""SELECT '{pct}' AS variable,
   count(*) FILTER (WHERE abs(coalesce(a."{pct}",-1) - coalesce(b."{pct}",-1)) > 0.01) AS filas_que_cambian,
   count(*) FILTER (WHERE b."{pct}" IS NULL AND a."{pct}" IS NOT NULL) AS tenian_rango_sin_dato
   FROM ind a JOIN v31 b USING (ocid)""" for pct, _, _ in VARS]
print(con.sql('\nUNION ALL\n'.join(cmp_)).df().to_string(index=False))

print('\n-- llave, filas y punto en el tiempo --')
print(con.sql("""SELECT count(*) AS filas, count(DISTINCT ocid) AS ocid_distintos,
   count(*) FILTER (WHERE ind_proveedor_debutante AND ind_procesos_proveedor_12m > 0) AS debutante_con_historia,
   count(*) FILTER (WHERE ind_primera_vez_con_la_entidad AND ind_recurrencia_par_12m > 0) AS primera_vez_con_historia
 FROM v31""").df().to_string(index=False))
prohib = [c for c in COLS_IND if c.startswith(('red_','cob_','com_','idx_'))
          or 'monto_tender' in c or 'alguna_vez' in c or 'fantasma' in c.lower()]
print('columnas prohibidas:', prohib if prohib else 'ninguna (correcto)')

# =================== (6) ESCRITURA, FIRMA Y ENMIENDA ===================
t('(6) ESCRITURA Y FIRMA')
out = 'plan_A/planA_indicadores_v3_1.parquet'
con.sql(f"COPY v31 TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
h = sha16(out)
print(f'  {out}  |  {round(os.path.getsize(out)/1e6,1)} MB  |  sha256_16 = {h}')
if os.path.exists('plan_A/planA_indicadores_v3.parquet'):
    os.replace('plan_A/planA_indicadores_v3.parquet', f'{HIST}/planA_indicadores_v3.parquet')
    print('  v3 movida a _version_1_historico/ (no se borra: sostiene la traza)')

esp = f'{DOC}/especificacion_indicadores.json'
E = json.load(open(esp, encoding='utf-8')) if os.path.exists(esp) else {}
E.setdefault('enmiendas', []).append({
    'version': 'v3.1', 'fecha': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'motivo': 'correccion anterior a cualquier modelado; ningun resultado de modelo existia aun',
    'cambios': [
        'los 9 rangos percentiles se recalculan solo sobre filas con dato; antes los nulos entraban '
        'empatados al ordenamiento y recibian un puntaje de anomalia sin haber sido medidos',
        'grupo comparable: familia x periodo normativo, con descenso a familia y luego a global '
        'cuando el grupo tiene menos de 30 casos con dato',
        f'antiguedad anulada en los {n_cent} RUC con fecha de inicio <= 1900-01-01 (valores centinela '
        'del registro; la fecha del SRI es la de fundacion institucional, no la de inscripcion)',
        'ind_duracion_contrato_dias anulada sobre 7300 dias (20 anios) e '
        'ind_contrato_sobre_adjudicado anulada sobre 100 veces: errores de la fuente, no hallazgos',
        'la ausencia de dato la carga el flag falta_* correspondiente, no el rango'],
    'sha256_16_indicadores_v3_1': h})
json.dump(E, open(esp, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print(f'  enmienda registrada en {esp}')

M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        d = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()
        M.append({'archivo': f, 'filas': con.sql(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0],
                  'columnas': len(d), 'MB': round(os.path.getsize(p)/1e6,1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print('\n' + MAN.to_string(index=False))
print(f'\n{len(MAN)} archivos | {round(MAN.MB.sum(),1)} MB')
pd.DataFrame(CIF).to_csv(f'{DOC}/cifras_celda24.csv', index=False, encoding='utf-8-sig')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 25 · FORENSE: monto_final, duracion y ratio ============
# Solo lectura. Contesta tres preguntas con datos, no con hipotesis:
#  (1) cuantos procesos tienen "dias de contrato = monto" en TODA la base
#  (2) que guardamos nosotros en el proceso de las 4 adjudicaciones
#  (3) de que columna sale el ratio de 8.000.000
import os
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 250); pd.set_option('display.max_colwidth', 45)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
qq = lambda cs: ', '.join('"%s"' % c for c in cs)

con.sql("""CREATE OR REPLACE VIEW v3 AS SELECT * FROM
           read_parquet('plan_A/_version_1_historico/planA_indicadores_v3.parquet')""")
con.sql("""CREATE OR REPLACE VIEW p1 AS SELECT * FROM
           read_parquet('plan_A/planA_v2_P1_modelo_sin_catalogo.parquet')""")
hay_j = os.path.exists('03_resultados/c15_complemento_json.parquet')
if hay_j:
    con.sql("""CREATE OR REPLACE VIEW j AS SELECT * FROM
               read_parquet('03_resultados/c15_complemento_json.parquet')""")
    COLS_J = list(con.sql("DESCRIBE SELECT * FROM j").df().column_name)
    print('columnas del complemento JSON:', COLS_J)
else:
    print('OJO: no encuentro 03_resultados/c15_complemento_json.parquet')
COLS_P1 = list(con.sql("DESCRIBE SELECT * FROM p1").df().column_name)

# ---------- (1) LA IDENTIDAD dias = monto ----------
t('(1) DIAS DE CONTRATO = MONTO ADJUDICADO: cuantos casos hay en toda la base')
print(con.sql("""SELECT
   count(*) FILTER (WHERE ind_duracion_contrato_dias IS NOT NULL)                   AS con_duracion,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(monto_final))           AS dias_igual_monto,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(monto_final*100))       AS dias_igual_monto_centavos,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(monto_final)
                       OR  ind_duracion_contrato_dias = round(monto_final*100))      AS total_sospechoso,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(monto_final) + 7)       AS control_falso_positivo
 FROM v3""").df().to_string(index=False))
print('\nla ultima columna es el control: compara contra monto+7, que no tiene por que coincidir.')
print('si "total_sospechoso" es muy superior al control, la identidad no es casualidad.\n')

print('-- los sospechosos, por tramo de duracion --')
print(con.sql("""SELECT CASE WHEN ind_duracion_contrato_dias <= 365 THEN 'a) hasta 1 anio'
                             WHEN ind_duracion_contrato_dias <= 1095 THEN 'b) 1 a 3 anios'
                             WHEN ind_duracion_contrato_dias <= 3650 THEN 'c) 3 a 10 anios'
                             ELSE 'd) mas de 10 anios' END AS tramo,
                        count(*) AS casos, round(sum(monto_final)/1e6,2) AS millones,
                        round(median(monto_final),0) AS mediana_usd
                 FROM v3
                 WHERE ind_duracion_contrato_dias = round(monto_final)
                    OR ind_duracion_contrato_dias = round(monto_final*100)
                 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

print('\n-- por familia y por anio --')
print(con.sql("""SELECT familia, count(*) AS casos FROM v3
                 WHERE ind_duracion_contrato_dias = round(monto_final)
                    OR ind_duracion_contrato_dias = round(monto_final*100)
                 GROUP BY 1 ORDER BY casos DESC""").df().to_string(index=False))
print(con.sql("""SELECT anio, count(*) AS casos FROM v3
                 WHERE ind_duracion_contrato_dias = round(monto_final)
                    OR ind_duracion_contrato_dias = round(monto_final*100)
                 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

print('\n-- 15 casos al azar para que los revises en el portal --')
print(con.sql("""SELECT ocid, familia, fecha_proceso, monto_final, ind_duracion_contrato_dias
                 FROM v3
                 WHERE (ind_duracion_contrato_dias = round(monto_final)
                     OR ind_duracion_contrato_dias = round(monto_final*100))
                   AND ind_duracion_contrato_dias <= 1095
                 USING SAMPLE 15 ROWS""").df().to_string(index=False))

# ---------- (2) EL PROCESO DE LAS 4 ADJUDICACIONES ----------
t('(2) ocds-5wno2w-MCBS-006-S.S.-2016-72693: el portal suma 12.640,40; que tenemos nosotros')
OC = 'ocds-5wno2w-MCBS-006-S.S.-2016-72693'
F = con.sql(f"SELECT * FROM p1 WHERE ocid = '{OC}'").df().T
F.columns = ['valor']; F = F[F.valor.notna()]
print(f'columnas con dato en P1 para ese proceso: {len(F)} de {len(COLS_P1)}')
pat = ('monto','valor','precio','cantidad','contrato','item','oferta','fuente','adjud','n_')
print(F[[any(p in str(i).lower() for p in pat) for i in F.index]].to_string())
numf = pd.to_numeric(F.valor, errors='coerce')
cerca = F[(numf - 12640.40).abs() < 1.0]
print('\ncolumnas con un valor cercano a 12.640,40 (el total real del portal):')
print(cerca.to_string() if len(cerca) else '  NINGUNA -> el total de las 4 adjudicaciones no esta en la base')
if hay_j:
    G = con.sql(f"SELECT * FROM j WHERE ocid = '{OC}'").df().T
    G.columns = ['valor']; print('\nlo que trajo el complemento JSON:'); print(G.to_string())

# ---------- (3) DE DONDE SALE EL RATIO ----------
t('(3) EL RATIO contrato/adjudicado: de que columna sale')
MALOS = ['ocds-5wno2w-RE-SENAE-006-2021-2656', 'ocds-5wno2w-RE-CEP-UNAEP-01-2022-486900',
         'ocds-5wno2w-RE-PU-SENAE-001-2022-2656', 'ocds-5wno2w-CS-CD-GADMS-002-2018-3450',
         'ocds-5wno2w-RE-GADPO-FP-37-2022-29112']
lista_sql = ', '.join(f"'{o}'" for o in MALOS)
print(con.sql(f"""SELECT ocid, monto_final, ind_contrato_sobre_adjudicado AS ratio,
                         cantidad, precio_unit, unidad, ind_duracion_contrato_dias
                  FROM v3 WHERE ocid IN ({lista_sql})""").df().to_string(index=False))
if hay_j:
    print('\n-- el complemento JSON para esos mismos procesos --')
    print(con.sql(f"SELECT * FROM j WHERE ocid IN ({lista_sql})").df().to_string(index=False))
    print('\n-- hipotesis probadas sobre TODA la base --')
    print(con.sql("""SELECT
       count(*) FILTER (WHERE v3.ind_contrato_sobre_adjudicado = j.j_item_cantidad)        AS ratio_igual_a_cantidad,
       count(*) FILTER (WHERE v3.ind_contrato_sobre_adjudicado = round(j.j_item_cantidad)) AS ratio_igual_a_cantidad_red,
       count(*) FILTER (WHERE v3.ind_contrato_sobre_adjudicado > 2)                        AS ratio_mayor_a_2,
       count(*) FILTER (WHERE v3.ind_contrato_sobre_adjudicado > 2
                          AND v3.ind_contrato_sobre_adjudicado = j.j_item_cantidad)        AS de_esos_igual_a_cantidad
     FROM v3 JOIN j USING (ocid)""").df().to_string(index=False))
cols_m = [c for c in COLS_P1 if any(p in c.lower() for p in ('monto','valor','precio','contrato'))]
print('\ncolumnas de P1 que pueden ser el numerador:', cols_m)
if cols_m:
    print(con.sql(f"SELECT ocid, {qq(cols_m)} FROM p1 WHERE ocid IN ({lista_sql})").df().T.to_string())
print('\nPega aqui toda la salida.')

In [ ]:
# ============ Plan A · Paso 3 · Adjudicaciones y contratos por proceso ============
# Cuenta cuantas adjudicaciones tiene cada proceso y suma sus montos, para comprobar
# si monto_final se quedo con una sola adjudicacion de varias.
# Guarda 03_resultados/c25_awards_<anio>.parquet (reanudable) y luego c25_awards.parquet
import os, glob, re, time
import pandas as pd, duckdb
try:
    import ijson
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ijson']); import ijson

os.chdir(r'C:\SERCOP_FINAL')
R = '03_resultados'
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")

def prefijo_de(ruta):                      # algunos archivos vienen envueltos en una lista
    with open(ruta, 'rb') as fh:
        while True:
            c = fh.read(1)
            if not c: return 'releases.item'
            if not c.isspace():
                return 'item.releases.item' if c == b'[' else 'releases.item'

def lista(v):                              # a veces el array viene como dict con llaves "0","1"
    if isinstance(v, list): return [x for x in v if isinstance(x, dict)]
    if isinstance(v, dict) and v and all(str(k).isdigit() for k in v):
        return [x for x in v.values() if isinstance(x, dict)]
    return []

def g(d, *ruta):
    for k in ruta:
        if not isinstance(d, dict): return None
        d = d.get(k)
    return d

def num(v):
    try: return float(v)
    except (TypeError, ValueError): return None

def fila(rel):
    aws, cts = lista(rel.get('awards')), lista(rel.get('contracts'))
    vals, vals_act, est, provs = [], [], set(), set()
    for a in aws:
        v = num(g(a, 'value', 'amount'))
        s = str(a.get('status') or '').strip().lower() or 'sin_estado'
        est.add(s)
        if v is not None:
            vals.append(v)
            if s in ('sin_estado', 'active', 'pending'): vals_act.append(v)
        for sp in lista(a.get('suppliers')):
            if sp.get('id'): provs.add(str(sp['id']))
    cv = [x for x in (num(g(c, 'value', 'amount')) for c in cts) if x is not None]
    cest = {str(c.get('status') or 'sin_estado').strip().lower() for c in cts}
    return {'ocid': rel.get('ocid'),
            'n_awards': len(aws), 'n_awards_con_monto': len(vals),
            'suma_awards':         sum(vals) if vals else None,
            'suma_awards_activas': sum(vals_act) if vals_act else None,
            'max_award': max(vals) if vals else None,
            'min_award': min(vals) if vals else None,
            'estados_award': '|'.join(sorted(est)) if est else None,
            'n_prov_distintos_award': len(provs),
            'n_contracts': len(cts),
            'suma_contratos': sum(cv) if cv else None,
            'max_contrato': max(cv) if cv else None,
            'estados_contrato': '|'.join(sorted(cest)) if cest else None}

for ruta in sorted(glob.glob('00_json_crudo/releases_*.json')):
    if ruta.endswith('releases_2024.json') and os.path.exists(ruta.replace('.json', '_reparado.json')):
        continue                                   # se usa el reparado, no el original
    m = re.search(r'releases_(\d{4})', ruta); etq = m.group(1) if m else 'x'
    dest = f'{R}/c25_awards_{etq}.parquet'
    if os.path.exists(dest):
        print(f'  {os.path.basename(ruta)}: ya estaba, se salta'); continue
    t0, filas = time.time(), []
    with open(ruta, 'rb') as fh:
        for rel in ijson.items(fh, prefijo_de(ruta)):
            if isinstance(rel, dict): filas.append(fila(rel))
    pd.DataFrame(filas).to_parquet(dest, index=False)
    print(f'  {os.path.basename(ruta)}: {len(filas):,} procesos ({time.time()-t0:.0f} s)')
    del filas

A = pd.concat([pd.read_parquet(p) for p in sorted(glob.glob(f'{R}/c25_awards_*.parquet'))],
              ignore_index=True)
dup = A.ocid.duplicated().sum()
if dup: A = A.drop_duplicates('ocid', keep='first')
A.to_parquet(f'{R}/c25_awards.parquet', index=False)
print(f'\n{len(A):,} procesos | ocid repetidos: {dup} | -> {R}/c25_awards.parquet')

# ---------------- La comparacion que importa ----------------
con.sql(f"CREATE OR REPLACE VIEW a AS SELECT * FROM read_parquet('{R}/c25_awards.parquet')")
con.sql("""CREATE OR REPLACE VIEW p1 AS SELECT * FROM
           read_parquet('plan_A/planA_v2_P1_modelo_sin_catalogo.parquet')""")
print('\n================ CUANTAS ADJUDICACIONES TIENE CADA PROCESO ================')
print(con.sql("""SELECT CASE WHEN n_awards = 0 THEN '0'
                             WHEN n_awards = 1 THEN '1'
                             WHEN n_awards BETWEEN 2 AND 4 THEN '2 a 4'
                             WHEN n_awards BETWEEN 5 AND 10 THEN '5 a 10'
                             ELSE 'mas de 10' END AS adjudicaciones,
                        count(*) AS procesos_en_P1,
                        round(sum(p1.monto_final)/1e6,1) AS millones_segun_monto_final,
                        round(sum(a.suma_awards)/1e6,1)  AS millones_sumando_adjudicaciones
                 FROM p1 JOIN a USING (ocid) GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

print('\n================ QUE ES monto_final CUANDO HAY VARIAS ADJUDICACIONES ================')
print(con.sql("""SELECT count(*) AS procesos_con_varias,
   count(*) FILTER (WHERE abs(p1.monto_final - a.suma_awards) < 0.01) AS monto_final_es_la_suma,
   count(*) FILTER (WHERE abs(p1.monto_final - a.max_award)   < 0.01) AS monto_final_es_el_mayor,
   count(*) FILTER (WHERE abs(p1.monto_final - a.min_award)   < 0.01) AS monto_final_es_el_menor,
   count(*) FILTER (WHERE abs(p1.monto_final - a.suma_awards) >= 0.01
                     AND abs(p1.monto_final - a.max_award)  >= 0.01
                     AND abs(p1.monto_final - a.min_award)  >= 0.01) AS ninguno_de_los_tres,
   round(sum(a.suma_awards - p1.monto_final)/1e6, 1) AS millones_que_faltarian
 FROM p1 JOIN a USING (ocid) WHERE a.n_awards > 1""").df().to_string(index=False))

print('\n================ ESTADOS DE LAS ADJUDICACIONES (hay que saber si se suman todas) ================')
print(con.sql("""SELECT estados_award, count(*) AS procesos FROM p1 JOIN a USING (ocid)
                 GROUP BY 1 ORDER BY procesos DESC LIMIT 15""").df().to_string(index=False))

print('\n================ 15 EJEMPLOS CON VARIAS ADJUDICACIONES ================')
print(con.sql("""SELECT p1.ocid, p1.familia_procedimiento AS familia, a.n_awards,
                        a.n_prov_distintos_award AS proveedores, p1.monto_final,
                        a.suma_awards, a.min_award, a.max_award, a.estados_award
                 FROM p1 JOIN a USING (ocid) WHERE a.n_awards > 1
                 ORDER BY a.suma_awards - p1.monto_final DESC LIMIT 15""").df().to_string(index=False))
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 26 · CUANTO DANO HIZO monto_final (y el ratio) ============
# Solo lectura. Mide el tamano exacto del problema antes de reconstruir nada.
import os
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)

con.sql("CREATE OR REPLACE VIEW a   AS SELECT * FROM read_parquet('03_resultados/c25_awards.parquet')")
con.sql("CREATE OR REPLACE VIEW jj  AS SELECT * FROM read_parquet('03_resultados/c15_complemento_json.parquet')")
con.sql("CREATE OR REPLACE VIEW p1  AS SELECT * FROM read_parquet('plan_A/planA_v2_P1_modelo_sin_catalogo.parquet')")
con.sql("CREATE OR REPLACE VIEW p0  AS SELECT * FROM read_parquet('plan_A/planA_v2_P0_catalogo.parquet')")
con.sql("CREATE OR REPLACE VIEW uni AS SELECT * FROM read_parquet('plan_A/planA_universo_v2.parquet')")
con.sql("""CREATE OR REPLACE VIEW v3 AS SELECT * FROM
           read_parquet('plan_A/_version_1_historico/planA_indicadores_v3.parquet')""")

# ---------- (1) ESTADOS DE LAS ADJUDICACIONES ----------
t('(1) ESTADOS DE LAS ADJUDICACIONES: hay que saber si se suman todas')
print(con.sql("""SELECT a.estados_award, count(*) AS procesos,
                        round(sum(a.suma_awards)/1e6,1) AS millones
                 FROM a GROUP BY 1 ORDER BY procesos DESC LIMIT 20""").df().to_string(index=False))

# ---------- (2) CUANTAS ADJUDICACIONES TIENE CADA PROCESO ----------
t('(2) DISTRIBUCION DE ADJUDICACIONES POR PROCESO')
tramo = """CASE WHEN a.n_awards = 0 THEN 'a) 0'
                WHEN a.n_awards = 1 THEN 'b) 1'
                WHEN a.n_awards BETWEEN 2 AND 4  THEN 'c) 2 a 4'
                WHEN a.n_awards BETWEEN 5 AND 10 THEN 'd) 5 a 10'
                ELSE 'e) mas de 10' END"""
print('-- en la base de modelado (P1) --')
print(con.sql(f"""SELECT {tramo} AS adjudicaciones, count(*) AS procesos,
                    round(sum(p1.monto_final)/1e6,1)        AS mm_monto_final_actual,
                    round(sum(a.suma_awards)/1e6,1)         AS mm_sumando_adjudicaciones,
                    round(sum(a.suma_awards_activas)/1e6,1) AS mm_solo_activas
                  FROM p1 JOIN a ON a.ocid = p1.ocid
                  GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('\n-- en el universo completo --')
print(con.sql(f"""SELECT {tramo} AS adjudicaciones, count(*) AS procesos,
                    round(sum(uni.monto_final)/1e6,1) AS mm_monto_final_actual,
                    round(sum(a.suma_awards)/1e6,1)   AS mm_sumando_adjudicaciones
                  FROM uni JOIN a ON a.ocid = uni.ocid
                  GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

# ---------- (3) COHERENCIA DEL CONTEO ----------
t('(3) EL n_awards QUE YA TENIA P1 COINCIDE CON EL DEL JSON?')
print(con.sql("""SELECT count(*) AS procesos,
   count(*) FILTER (WHERE p1.n_awards = a.n_awards)  AS coinciden,
   count(*) FILTER (WHERE p1.n_awards <> a.n_awards) AS no_coinciden,
   count(*) FILTER (WHERE p1.n_awards IS NULL)       AS p1_sin_dato
 FROM p1 JOIN a ON a.ocid = p1.ocid""").df().to_string(index=False))

# ---------- (4) QUE ES monto_final CUANDO HAY VARIAS ----------
t('(4) QUE GUARDO monto_final EN LOS PROCESOS CON VARIAS ADJUDICACIONES')
print(con.sql("""SELECT count(*) AS procesos_con_varias,
   count(*) FILTER (WHERE abs(p1.monto_final - a.suma_awards) < 0.01) AS es_la_suma,
   count(*) FILTER (WHERE abs(p1.monto_final - a.max_award)   < 0.01) AS es_el_mayor,
   count(*) FILTER (WHERE abs(p1.monto_final - a.min_award)   < 0.01) AS es_el_menor,
   count(*) FILTER (WHERE abs(p1.monto_final - a.suma_awards) >= 0.01
                     AND abs(p1.monto_final - a.max_award)  >= 0.01
                     AND abs(p1.monto_final - a.min_award)  >= 0.01) AS ninguno_de_los_tres,
   round(sum(a.suma_awards - p1.monto_final)/1e6, 1) AS millones_que_faltan,
   round(median(a.suma_awards / nullif(p1.monto_final,0)), 2) AS veces_mediana
 FROM p1 JOIN a ON a.ocid = p1.ocid WHERE a.n_awards > 1""").df().to_string(index=False))

print('\n-- y cuando hay una sola adjudicacion, cuadra? --')
print(con.sql("""SELECT count(*) AS procesos_con_una,
   count(*) FILTER (WHERE abs(p1.monto_final - a.suma_awards) < 0.01) AS cuadra,
   count(*) FILTER (WHERE abs(p1.monto_final - a.suma_awards) >= 0.01) AS no_cuadra
 FROM p1 JOIN a ON a.ocid = p1.ocid WHERE a.n_awards = 1""").df().to_string(index=False))

# ---------- (5) IMPACTO ----------
t('(5) IMPACTO DE RECONSTRUIR monto_final = SUMA DE ADJUDICACIONES')
print(con.sql("""SELECT
   round(sum(p1.monto_final)/1e6,1)                        AS mm_actual,
   round(sum(coalesce(a.suma_awards, p1.monto_final))/1e6,1) AS mm_reconstruido,
   round((sum(coalesce(a.suma_awards, p1.monto_final)) - sum(p1.monto_final))/1e6,1) AS mm_diferencia,
   round(100.0*(sum(coalesce(a.suma_awards, p1.monto_final)) - sum(p1.monto_final))
         / sum(p1.monto_final), 2)                         AS pct_diferencia,
   count(*) FILTER (WHERE abs(coalesce(a.suma_awards,p1.monto_final) - p1.monto_final) >= 0.01) AS filas_que_cambian,
   count(DISTINCT p1.proveedor_ruc_final) FILTER (WHERE abs(coalesce(a.suma_awards,p1.monto_final)
         - p1.monto_final) >= 0.01)                        AS proveedores_afectados,
   count(DISTINCT p1.entidad_ruc) FILTER (WHERE abs(coalesce(a.suma_awards,p1.monto_final)
         - p1.monto_final) >= 0.01)                        AS entidades_afectadas
 FROM p1 JOIN a ON a.ocid = p1.ocid""").df().to_string(index=False))

print('\n-- por familia: donde se concentra el dano --')
print(con.sql("""SELECT p1.familia_procedimiento AS familia, count(*) AS procesos,
   count(*) FILTER (WHERE a.n_awards > 1) AS con_varias_adjudicaciones,
   round(100.0*count(*) FILTER (WHERE a.n_awards > 1)/count(*),2) AS pct,
   round(sum(coalesce(a.suma_awards,p1.monto_final) - p1.monto_final)/1e6,1) AS mm_que_faltan
 FROM p1 JOIN a ON a.ocid = p1.ocid GROUP BY 1 ORDER BY mm_que_faltan DESC""").df().to_string(index=False))

print('\n-- por anio --')
print(con.sql("""SELECT p1.anio_proceso AS anio, count(*) FILTER (WHERE a.n_awards > 1) AS con_varias,
   round(sum(coalesce(a.suma_awards,p1.monto_final) - p1.monto_final)/1e6,1) AS mm_que_faltan
 FROM p1 JOIN a ON a.ocid = p1.ocid GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

print('\n-- los 12 casos donde falta mas plata --')
print(con.sql("""SELECT p1.ocid, p1.familia_procedimiento AS familia, a.n_awards,
   a.n_prov_distintos_award AS proveedores, p1.monto_final, a.suma_awards, a.min_award, a.max_award
 FROM p1 JOIN a ON a.ocid = p1.ocid WHERE a.n_awards > 1
 ORDER BY a.suma_awards - p1.monto_final DESC LIMIT 12""").df().to_string(index=False))

# ---------- (6) CUANTOS INDICADORES QUEDAN TOCADOS ----------
t('(6) QUE TAN LEJOS LLEGA EL DANO EN LOS INDICADORES')
print(con.sql("""WITH m AS (SELECT p1.ocid, p1.monto_final AS viejo,
                    coalesce(a.suma_awards, p1.monto_final) AS nuevo
                  FROM p1 JOIN a ON a.ocid = p1.ocid)
  SELECT count(*) FILTER (WHERE abs(m.nuevo - m.viejo) >= 0.01) AS procesos_con_monto_distinto,
         count(*) FILTER (WHERE abs(m.nuevo - m.viejo) >= 0.01
                            AND abs(v3.ind_desv_precio_unit_z) > 3) AS de_esos_marcados_como_precio_atipico,
         count(*) FILTER (WHERE abs(v3.ind_desv_precio_unit_z) > 3) AS total_marcados_precio_atipico,
         count(*) FILTER (WHERE m.nuevo > 0 AND m.viejo > 0
                            AND abs(ln(m.nuevo) - ln(m.viejo)) > 1) AS cambio_de_mas_de_1_en_log
  FROM m JOIN v3 ON v3.ocid = m.ocid""").df().to_string(index=False))

# ---------- (7) EL RATIO: CONFIRMAR LA FORMULA ----------
t('(7) j_contrato_monto = cantidad x monto ? (la formula del error de la fuente)')
print(con.sql("""SELECT count(*) AS con_los_tres_datos,
   count(*) FILTER (WHERE abs(jj.j_contrato_monto
            - jj.j_item_cantidad * p1.monto_final) < 1.0)              AS cumple_la_identidad,
   round(100.0*count(*) FILTER (WHERE abs(jj.j_contrato_monto
            - jj.j_item_cantidad * p1.monto_final) < 1.0)/count(*),1)  AS pct,
   count(*) FILTER (WHERE abs(jj.j_contrato_monto - p1.monto_final) < 1.0) AS contrato_igual_al_monto,
   count(*) FILTER (WHERE jj.j_item_cantidad = 1)                      AS cantidad_igual_a_1
 FROM p1 JOIN jj ON jj.ocid = p1.ocid
 WHERE jj.j_contrato_monto IS NOT NULL AND jj.j_item_cantidad IS NOT NULL
   AND p1.monto_final IS NOT NULL""").df().to_string(index=False))
print('\n-- lo mismo pero solo donde la cantidad es distinta de 1 (ahi se nota de verdad) --')
print(con.sql("""SELECT count(*) AS casos,
   count(*) FILTER (WHERE abs(jj.j_contrato_monto
            - jj.j_item_cantidad * p1.monto_final) < 1.0)              AS cumple_la_identidad,
   round(100.0*count(*) FILTER (WHERE abs(jj.j_contrato_monto
            - jj.j_item_cantidad * p1.monto_final) < 1.0)/count(*),1)  AS pct,
   count(*) FILTER (WHERE abs(jj.j_contrato_monto - p1.monto_final) < 1.0) AS contrato_igual_al_monto
 FROM p1 JOIN jj ON jj.ocid = p1.ocid
 WHERE jj.j_contrato_monto IS NOT NULL AND jj.j_item_cantidad IS NOT NULL
   AND jj.j_item_cantidad <> 1 AND p1.monto_final IS NOT NULL""").df().to_string(index=False))

# ---------- (8) LAS COLUMNAS DE RED QUE SE COLARON EN P1 ----------
t('(8) COLUMNAS DE RED DENTRO DE P1 (hay que sacarlas del entregable)')
cols = list(con.sql("DESCRIBE SELECT * FROM p1").df().column_name)
pro = [c for c in cols if c.startswith(('red_','cob_','com_','idx_','rep_','grupo_'))]
print(f'{len(pro)} columnas:'); print(pro)
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 27 · P1 v3: monto corregido, sin columnas de red ============
# A) imprime la especificacion de indicadores para no adivinar definiciones en la celda 28
# B) reconstruye monto_final = suma de adjudicaciones y saca las 56 columnas de red
# C) controles  D) escribe, firma y actualiza el manifiesto
import os, json, hashlib
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST = 'plan_A/documentacion', 'plan_A/_version_1_historico'
os.makedirs(HIST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

con.sql("CREATE OR REPLACE VIEW a  AS SELECT * FROM read_parquet('03_resultados/c25_awards.parquet')")
con.sql("CREATE OR REPLACE VIEW p1 AS SELECT * FROM read_parquet('plan_A/planA_v2_P1_modelo_sin_catalogo.parquet')")
con.sql("CREATE OR REPLACE VIEW dpr AS SELECT * FROM read_parquet('plan_A/planA_dim_proveedor.parquet')")

# ---------- (A) LO QUE DECLARAMOS EN LA ESPECIFICACION ----------
t('(A) ESPECIFICACION DE INDICADORES (la necesito literal para la celda 28)')
esp = f'{DOC}/especificacion_indicadores.json'
if os.path.exists(esp):
    print(json.dumps(json.load(open(esp, encoding='utf-8')), ensure_ascii=False, indent=1)[:12000])
else:
    print('no existe', esp)
rd = f'{DOC}/README_base_plan_A.md'
if os.path.exists(rd):
    print('\n--- lineas del README que definen indicadores ---')
    for l in open(rd, encoding='utf-8').read().splitlines():
        if 'ind_' in l or 'pct_' in l or 'falta_' in l: print(l)
print('\n--- columnas de capital en la dimension de proveedor ---')
dc = list(con.sql("DESCRIBE SELECT * FROM dpr").df().column_name)
print([c for c in dc if any(k in c.lower() for k in ('capital','cap_','patrimonio','activo'))])

# ---------- (B) RECONSTRUCCION ----------
t('(B) P1 v3: monto_final = suma de adjudicaciones, fuera las columnas de red')
cols = list(con.sql("DESCRIBE SELECT * FROM p1").df().column_name)
PREF  = ('red_', 'cob_', 'com_', 'idx_', 'rep_', 'grupo_')      # item #48, no es nuestro alcance
RANCIAS = ['log_monto', 'desviacion_precio_log_z', 'desviacion_precio_log_z_sinfuga',
           'monto_acumulado_30d_no_catalogo', 'monto_total_historico_par',
           'monto_total_historico_entidad', 'concentracion_entidad_proveedor',
           'ratio_monto_capital', 'ent_monto_total']            # derivadas del monto viejo
fuera = [c for c in cols if c.startswith(PREF)] + [c for c in RANCIAS if c in cols]
quedan = [c for c in cols if c not in set(fuera)]
print(f'columnas que salen: {len(fuera)}  ({len([c for c in cols if c.startswith(PREF)])} de red, '
      f'{len([c for c in RANCIAS if c in cols])} derivadas del monto viejo)')
print('derivadas del monto que se eliminan por quedar rancias:', [c for c in RANCIAS if c in cols])
print(f'columnas que quedan: {len(quedan)}')

sel = ', '.join(('coalesce(a.suma_awards, p1.monto_final) AS monto_final' if c == 'monto_final'
                 else f'p1."{c}"') for c in quedan)
sel += (', p1.monto_final AS monto_adjudicacion_unica_v1'
        ', (a.n_awards > 1) AS es_multilote'
        ', a.suma_awards AS monto_suma_adjudicaciones'
        ', a.min_award   AS monto_adj_min'
        ', a.max_award   AS monto_adj_max'
        ', a.n_prov_distintos_award')
con.sql(f"CREATE OR REPLACE TABLE p1v3 AS SELECT {sel} FROM p1 JOIN a ON a.ocid = p1.ocid")

# ---------- (C) CONTROLES ----------
t('(C) CONTROLES DE ACEPTACION')
print(con.sql("""SELECT count(*) AS filas, count(DISTINCT ocid) AS ocid_distintos,
   count(*) FILTER (WHERE monto_final IS NULL)                          AS monto_nulo,
   count(*) FILTER (WHERE monto_final <= 0)                             AS monto_cero_o_negativo,
   count(*) FILTER (WHERE abs(monto_final - monto_suma_adjudicaciones) >= 0.01) AS monto_no_es_la_suma,
   count(*) FILTER (WHERE es_multilote)                                 AS multilote,
   round(sum(monto_final)/1e6,1)                 AS millones,
   round(sum(monto_adjudicacion_unica_v1)/1e6,1) AS millones_antes,
   round(median(monto_final),0)                  AS mediana_usd
 FROM p1v3""").df().to_string(index=False))
cv3 = list(con.sql("DESCRIBE SELECT * FROM p1v3").df().column_name)
pro = [c for c in cv3 if c.startswith(PREF)] + [c for c in RANCIAS if c in cv3]
print('columnas prohibidas o rancias que quedaron:', pro if pro else 'ninguna (correcto)')
print('\n-- el proceso de las 4 adjudicaciones, ahora --')
print(con.sql("""SELECT ocid, n_awards, es_multilote, monto_adjudicacion_unica_v1,
                        monto_final, monto_tender
                 FROM p1v3 WHERE ocid = 'ocds-5wno2w-MCBS-006-S.S.-2016-72693'""").df().to_string(index=False))
print('\n-- multilote por familia --')
print(con.sql("""SELECT familia_procedimiento AS familia, count(*) AS procesos,
   count(*) FILTER (WHERE es_multilote) AS multilote,
   round(100.0*count(*) FILTER (WHERE es_multilote)/count(*),2) AS pct,
   round(median(CASE WHEN es_multilote THEN n_awards END),0) AS mediana_adjudicaciones,
   max(n_awards) AS max_adjudicaciones,
   round(median(CASE WHEN es_multilote THEN monto_final END),0) AS mediana_monto_multilote,
   round(median(CASE WHEN NOT es_multilote THEN monto_final END),0) AS mediana_monto_simple
 FROM p1v3 GROUP BY 1 ORDER BY multilote DESC""").df().to_string(index=False))
print('\n-- el monto del pliego frente al adjudicado, ahora que el adjudicado esta completo --')
print(con.sql("""SELECT es_multilote, count(*) AS procesos,
   count(monto_tender) AS con_pliego,
   round(median(monto_final / nullif(monto_tender,0)),4) AS mediana_adjudicado_sobre_pliego,
   count(*) FILTER (WHERE monto_final > monto_tender * 1.01) AS adjudicado_mayor_que_pliego
 FROM p1v3 GROUP BY 1""").df().to_string(index=False))

# ---------- (D) ESCRITURA ----------
t('(D) ESCRITURA Y FIRMA')
out = 'plan_A/planA_P1_v3_modelo.parquet'
con.sql(f"COPY p1v3 TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
print(f'  {out} | {round(os.path.getsize(out)/1e6,1)} MB | sha256_16 = {sha16(out)}')
vieja = 'plan_A/planA_v2_P1_modelo_sin_catalogo.parquet'
if os.path.exists(vieja):
    os.replace(vieja, f'{HIST}/planA_v2_P1_modelo_sin_catalogo.parquet')
    print('  P1 v2 movida a _version_1_historico/')
M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        M.append({'archivo': f,
                  'filas': con.sql(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0],
                  'columnas': len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()),
                  'MB': round(os.path.getsize(p)/1e6, 1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print('\n' + MAN.to_string(index=False))
print(f'\n{len(MAN)} archivos | {round(MAN.MB.sum(),1)} MB')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 28 · INDICADORES v4 CON EL MONTO CORREGIDO ============
# Metodo: para cada indicador que depende del monto, primero lo REPRODUCE con el monto viejo
# y lo compara con el valor guardado. Solo si coincide (>99%) lo recalcula con el monto nuevo.
# Si no coincide, lo deja como esta y lo reporta: no se reemplaza lo que no se entiende.
import os, json, hashlib
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST = 'plan_A/documentacion', 'plan_A/_version_1_historico'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
CIF = []
def cifra(n, v, nota=''): CIF.append({'cifra': n, 'valor': v, 'nota': nota}); return v
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

con.sql("CREATE OR REPLACE VIEW v31 AS SELECT * FROM read_parquet('plan_A/planA_indicadores_v3_1.parquet')")
con.sql("CREATE OR REPLACE VIEW p3  AS SELECT * FROM read_parquet('plan_A/planA_P1_v3_modelo.parquet')")
con.sql(f"""CREATE OR REPLACE VIEW p2 AS SELECT * FROM
            read_parquet('{HIST}/planA_v2_P1_modelo_sin_catalogo.parquet')""")

# ---------- (0) MESA DE TRABAJO ----------
cp2 = list(con.sql("DESCRIBE SELECT * FROM p2").df().column_name)
cap_cols = [c for c in ('ratio_monto_capital', 'k_ratio_capital') if c in cp2]
print('columnas candidatas de capital en el P1 historico:', cap_cols)
extra = ', '.join(f'p2."{c}"' for c in cap_cols)
con.sql(f"""CREATE OR REPLACE TABLE w AS
  SELECT v31.*, p3.monto_final AS monto_nuevo,
         p3.monto_adjudicacion_unica_v1 AS monto_viejo,
         p3.es_multilote, p3.n_awards {(', ' + extra) if extra else ''}
  FROM v31 JOIN p3 ON p3.ocid = v31.ocid JOIN p2 ON p2.ocid = v31.ocid""")
print(con.sql("""SELECT count(*) AS filas, count(DISTINCT ocid) AS ocid,
                   round(sum(monto_viejo)/1e6,1) AS mm_viejo,
                   round(sum(monto_nuevo)/1e6,1) AS mm_nuevo,
                   count(*) FILTER (WHERE abs(monto_nuevo-monto_viejo)>=0.01) AS filas_que_cambian
                 FROM w""").df().to_string(index=False))

# ---------- (1) DE DONDE SALE ind_monto_sobre_capital ----------
t('(1) REPRODUCIR ind_monto_sobre_capital')
cap_ok = None
for c in cap_cols:
    r = con.sql(f"""SELECT count(*) FILTER (WHERE ind_monto_sobre_capital IS NOT NULL) AS con_dato,
      count(*) FILTER (WHERE abs(ind_monto_sobre_capital - "{c}") < 1e-6) AS coincide
      FROM w""").fetchone()
    pct = 100 * r[1] / r[0] if r[0] else 0
    print(f'  ind_monto_sobre_capital == {c}: {r[1]:,} de {r[0]:,} ({pct:.2f}%)')
    if pct > 99 and cap_ok is None: cap_ok = c
print('  se usara:', cap_ok or 'NINGUNA -> no se recalcula, se deja como esta')

# ---------- (2) REPRODUCIR ind_fraccionamiento_monto_30d ----------
t('(2) REPRODUCIR ind_fraccionamiento_monto_30d  (30 dias, mismo par y mismo CPC5)')
con.sql("""CREATE OR REPLACE TABLE fr AS
  SELECT ocid,
    coalesce(sum(monto_viejo) OVER v, 0) AS repro_viejo,
    coalesce(sum(monto_nuevo) OVER v, 0) AS repro_nuevo
  FROM w
  WINDOW v AS (PARTITION BY entidad_ruc, proveedor_ruc, cpc5 ORDER BY fecha_proceso
               RANGE BETWEEN INTERVAL 30 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING)""")
r = con.sql("""SELECT count(*) AS filas,
  count(*) FILTER (WHERE abs(w.ind_fraccionamiento_monto_30d - fr.repro_viejo) < 0.01) AS coincide,
  round(100.0*count(*) FILTER (WHERE abs(w.ind_fraccionamiento_monto_30d - fr.repro_viejo) < 0.01)
        /count(*),2) AS pct
  FROM w JOIN fr USING (ocid)""").df()
print(r.to_string(index=False)); frac_ok = float(r.pct.iloc[0]) > 99
print('  reproducido:', frac_ok)

# ---------- (3) QUE ES ind_cuota_prov_en_entidad_12m: monto o conteo? ----------
t('(3) REPRODUCIR ind_cuota_prov_en_entidad_12m: cuota de monto o cuota de procesos?')
con.sql("""CREATE OR REPLACE TABLE cu AS
  SELECT ocid,
    sum(monto_viejo) OVER par / nullif(sum(monto_viejo) OVER ent, 0) AS monto_viejo_cuota,
    sum(monto_nuevo) OVER par / nullif(sum(monto_nuevo) OVER ent, 0) AS monto_nuevo_cuota,
    count(*) OVER par * 1.0 / nullif(count(*) OVER ent, 0)           AS conteo_cuota
  FROM w
  WINDOW par AS (PARTITION BY entidad_ruc, proveedor_ruc ORDER BY fecha_proceso
                 RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING),
         ent AS (PARTITION BY entidad_ruc ORDER BY fecha_proceso
                 RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING)""")
r = con.sql("""SELECT count(*) FILTER (WHERE w.ind_cuota_prov_en_entidad_12m IS NOT NULL) AS con_dato,
  count(*) FILTER (WHERE abs(w.ind_cuota_prov_en_entidad_12m - cu.monto_viejo_cuota) < 1e-6) AS como_monto,
  count(*) FILTER (WHERE abs(w.ind_cuota_prov_en_entidad_12m - cu.conteo_cuota)      < 1e-6) AS como_conteo
  FROM w JOIN cu USING (ocid)""").df()
print(r.to_string(index=False))
cd = int(r.con_dato.iloc[0]) or 1
cuota_es_monto = 100*int(r.como_monto.iloc[0])/cd > 99
cuota_es_conteo = 100*int(r.como_conteo.iloc[0])/cd > 99
print(f'  es cuota de monto: {cuota_es_monto}   es cuota de procesos: {cuota_es_conteo}')
print('  -> ' + ('se recalcula con el monto nuevo' if cuota_es_monto else
                 'no depende del monto, se deja intacta' if cuota_es_conteo else
                 'NO SE REPRODUCE: se deja intacta y se reporta'))

# ---------- (4) EL RATIO DEL CONTRATO ----------
t('(4) ind_contrato_sobre_adjudicado: recalcular y anular el error de la fuente')
con.sql("CREATE OR REPLACE VIEW jj AS SELECT * FROM read_parquet('03_resultados/c15_complemento_json.parquet')")
con.sql("""CREATE OR REPLACE TABLE rt AS
  SELECT w.ocid,
    CASE WHEN jj.j_contrato_monto IS NULL THEN NULL
         WHEN jj.j_item_cantidad IS NOT NULL AND jj.j_item_cantidad <> 1
          AND abs(jj.j_contrato_monto - jj.j_item_cantidad * w.monto_nuevo) < 1.0 THEN NULL
         WHEN jj.j_item_cantidad IS NOT NULL AND jj.j_item_cantidad <> 1
          AND abs(jj.j_contrato_monto - jj.j_item_cantidad * w.monto_viejo) < 1.0 THEN NULL
         ELSE jj.j_contrato_monto / nullif(w.monto_nuevo, 0) END AS ratio_nuevo,
    (jj.j_item_cantidad IS NOT NULL AND jj.j_item_cantidad <> 1
     AND (abs(jj.j_contrato_monto - jj.j_item_cantidad * w.monto_nuevo) < 1.0
       OR abs(jj.j_contrato_monto - jj.j_item_cantidad * w.monto_viejo) < 1.0)) AS contrato_mal_publicado
  FROM w LEFT JOIN jj ON jj.ocid = w.ocid""")
print(con.sql("""SELECT count(*) FILTER (WHERE contrato_mal_publicado) AS anulados_por_error_de_fuente,
  count(ratio_nuevo) AS con_ratio, round(max(ratio_nuevo),1) AS ratio_maximo,
  round(median(ratio_nuevo),4) AS ratio_mediana FROM rt""").df().to_string(index=False))

# ---------- (5) ARMAR v4 ----------
t('(5) ARMAR LA TABLA v4')
rep = {'monto_final': 'w.monto_nuevo'}
if cap_ok:  rep['ind_monto_sobre_capital'] = f'w."{cap_ok}" * (w.monto_nuevo / nullif(w.monto_viejo,0))'
if frac_ok: rep['ind_fraccionamiento_monto_30d'] = 'fr.repro_nuevo'
if cuota_es_monto: rep['ind_cuota_prov_en_entidad_12m'] = 'cu.monto_nuevo_cuota'
rep['ind_contrato_sobre_adjudicado'] = 'rt.ratio_nuevo'
print('columnas que se reemplazan:'); [print(f'   {k} <- {v}') for k, v in rep.items()]

COLS31 = list(con.sql("DESCRIBE SELECT * FROM v31").df().column_name)
VARS = [('pct_precio_alto','ind_desv_precio_unit_z','ASC'),
        ('pct_cuota_12m','ind_cuota_prov_en_entidad_12m','ASC'),
        ('pct_recurrencia_par','ind_recurrencia_par_12m','ASC'),
        ('pct_monto_capital','ind_monto_sobre_capital','ASC'),
        ('pct_fraccionamiento','ind_fraccionamiento_monto_30d','ASC'),
        ('pct_proveedor_nuevo','ind_antiguedad_dias','DESC'),
        ('pct_poca_concurrencia','ind_n_oferentes','DESC'),
        ('pct_plazo_corto','ind_plazo_ofertas_dias','DESC'),
        ('pct_firma_rapida','ind_plazo_adj_firma_dias','DESC')]
pctnames = {v[0] for v in VARS}
sel = ', '.join(rep.get(c, f'w."{c}"') + f' AS "{c}"' for c in COLS31 if c not in pctnames)
con.sql(f"""CREATE OR REPLACE TABLE base4 AS
  SELECT {sel}, w.es_multilote, w.n_awards, rt.contrato_mal_publicado
  FROM w LEFT JOIN fr USING (ocid) LEFT JOIN cu USING (ocid) LEFT JOIN rt USING (ocid)""")

# los 9 rangos, otra vez con la regla declarada (solo no nulos, grupo con descenso)
for i, (pct, src, d) in enumerate(VARS):
    con.sql(f"""CREATE OR REPLACE TABLE q{i} AS
      WITH c AS (SELECT ocid, familia, periodo, "{src}" AS x,
                        count("{src}") OVER (PARTITION BY familia, periodo) AS n_fp,
                        count("{src}") OVER (PARTITION BY familia)          AS n_f
                 FROM base4),
           d AS (SELECT ocid, x, CASE WHEN n_fp >= 30 THEN familia || '|' || periodo
                                      WHEN n_f  >= 30 THEN familia ELSE 'GLOBAL' END AS grupo
                 FROM c WHERE x IS NOT NULL)
      SELECT ocid, percent_rank() OVER (PARTITION BY grupo ORDER BY x {d}) AS "{pct}" FROM d""")
idx = {v[0]: i for i, v in enumerate(VARS)}
sel4 = ', '.join((f'q{idx[c]}."{c}"' if c in pctnames else f'base4."{c}"') for c in COLS31)
joins = ' '.join(f'LEFT JOIN q{i} USING (ocid)' for i in range(len(VARS)))
con.sql(f"""CREATE OR REPLACE TABLE v4 AS
  SELECT {sel4}, base4.es_multilote, base4.n_awards, base4.contrato_mal_publicado
  FROM base4 {joins}""")
print(f'\nv4: {con.sql("SELECT count(*) FROM v4").fetchone()[0]:,} filas, '
      f'{len(con.sql("DESCRIBE SELECT * FROM v4").df())} columnas')

# ---------- (6) CONTROLES ----------
t('(6) CONTROLES DE ACEPTACION')
piezas = [f"""SELECT '{pct}' AS variable, count("{src}") AS fuente, count("{pct}") AS rango,
   CASE WHEN count("{src}")=count("{pct}") THEN 'OK' ELSE 'DESCUADRE' END AS nulos_ok,
   round(corr("{src}","{pct}"),3) AS corr,
   CASE WHEN ('{d}'='ASC' AND corr("{src}","{pct}")>0) OR ('{d}'='DESC' AND corr("{src}","{pct}")<0)
        THEN 'OK' ELSE 'INVERTIDA' END AS direccion_ok FROM v4""" for pct, src, d in VARS]
K = con.sql('\nUNION ALL\n'.join(piezas)).df(); print(K.to_string(index=False))
print('\n-- cuanto cambio frente a v3.1 --')
cmp_ = [f"""SELECT '{pct}' AS variable,
   count(*) FILTER (WHERE abs(coalesce(a."{pct}",-1)-coalesce(b."{pct}",-1))>0.01) AS filas_que_cambian
   FROM v31 a JOIN v4 b USING (ocid)""" for pct, _, _ in VARS]
print(con.sql('\nUNION ALL\n'.join(cmp_)).df().to_string(index=False))
print('\n-- llave, monto y punto en el tiempo --')
print(con.sql("""SELECT count(*) AS filas, count(DISTINCT ocid) AS ocid,
   round(sum(monto_final)/1e6,1) AS millones,
   count(*) FILTER (WHERE ind_proveedor_debutante AND ind_procesos_proveedor_12m>0) AS debutante_con_historia,
   count(*) FILTER (WHERE ind_primera_vez_con_la_entidad AND ind_recurrencia_par_12m>0) AS primera_con_historia,
   count(*) FILTER (WHERE es_multilote) AS multilote
 FROM v4""").df().to_string(index=False))

# ---------- (7) ESCRIBIR Y FIRMAR ----------
t('(7) ESCRITURA Y FIRMA')
out = 'plan_A/planA_indicadores_v4.parquet'
con.sql(f"COPY v4 TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
h = sha16(out); print(f'  {out} | {round(os.path.getsize(out)/1e6,1)} MB | sha256_16 = {h}')
if os.path.exists('plan_A/planA_indicadores_v3_1.parquet'):
    os.replace('plan_A/planA_indicadores_v3_1.parquet', f'{HIST}/planA_indicadores_v3_1.parquet')
esp = f'{DOC}/especificacion_indicadores.json'
E = json.load(open(esp, encoding='utf-8'))
E['enmiendas'].append({
    'version': 'v4', 'fecha': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'motivo': 'monto_final estaba tomando una sola adjudicacion en procesos multilote; '
              'corregido antes de cualquier modelado',
    'cambios': [
        'monto_final = suma de las adjudicaciones del proceso (no hay adjudicaciones canceladas '
        'en la fuente: los unicos estados son active y vacio). P1 pasa de 53550.2 a 54012.3 MM',
        f'ind_monto_sobre_capital recalculado{"" if cap_ok else " NO: no se pudo reproducir"}',
        f'ind_fraccionamiento_monto_30d recalculado{"" if frac_ok else " NO: no se pudo reproducir"}',
        'ind_cuota_prov_en_entidad_12m: ' + ('recalculada como cuota de monto' if cuota_es_monto
            else 'es cuota de procesos, no depende del monto' if cuota_es_conteo else 'no reproducida, intacta'),
        'ind_contrato_sobre_adjudicado recalculado y anulado donde la fuente publica el valor del '
        'contrato como cantidad x monto; se marca con contrato_mal_publicado',
        'se agregan es_multilote y n_awards como variables de estratificacion, no de anomalia',
        'ind_desv_precio_unit_z NO cambia: se calcula sobre el precio unitario del item, '
        'independiente del monto'],
    'sha256_16_indicadores_v4': h})
json.dump(E, open(esp, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        M.append({'archivo': f, 'filas': con.sql(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0],
                  'columnas': len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()),
                  'MB': round(os.path.getsize(p)/1e6,1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print('\n' + MAN.to_string(index=False)); print(f'\n{len(MAN)} archivos | {round(MAN.MB.sum(),1)} MB')
pd.DataFrame(CIF).to_csv(f'{DOC}/cifras_celda28.csv', index=False, encoding='utf-8-sig')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 29 · AUDITORIA DE DURACION, TOPE DEL RATIO Y LA CUOTA ============
# (1) resuelve que es ind_cuota_prov_en_entidad_12m, probando solo donde las definiciones difieren
# (2) audita la duracion del contrato con tres redes distintas
# (3) repone el tope del ratio que yo deshice en v4
# (4) escribe v4.1 con nulos + banderas + una columna imputada SOLO para descriptivo
import os, json, hashlib
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST = 'plan_A/documentacion', 'plan_A/_version_1_historico'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
CIF = []
def cifra(n, v, nota=''): CIF.append({'cifra': n, 'valor': v, 'nota': nota}); return v
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

con.sql("CREATE OR REPLACE VIEW v4  AS SELECT * FROM read_parquet('plan_A/planA_indicadores_v4.parquet')")
con.sql(f"CREATE OR REPLACE VIEW v3 AS SELECT * FROM read_parquet('{HIST}/planA_indicadores_v3.parquet')")
con.sql("CREATE OR REPLACE VIEW jj  AS SELECT * FROM read_parquet('03_resultados/c15_complemento_json.parquet')")
con.sql("CREATE OR REPLACE VIEW p3  AS SELECT * FROM read_parquet('plan_A/planA_P1_v3_modelo.parquet')")

# ---------- (1) QUE ES LA CUOTA: probar solo donde las definiciones difieren ----------
t('(1) ind_cuota_prov_en_entidad_12m: probar SOLO donde monto y conteo dan distinto')
con.sql("""CREATE OR REPLACE TABLE cu AS
  SELECT v4.ocid,
    sum(p3.monto_adjudicacion_unica_v1) OVER par
      / nullif(sum(p3.monto_adjudicacion_unica_v1) OVER ent, 0) AS monto_viejo_cuota,
    sum(v4.monto_final) OVER par / nullif(sum(v4.monto_final) OVER ent, 0) AS monto_nuevo_cuota,
    count(*) OVER par * 1.0 / nullif(count(*) OVER ent, 0)      AS conteo_cuota
  FROM v4 JOIN p3 ON p3.ocid = v4.ocid
  WINDOW par AS (PARTITION BY v4.entidad_ruc, v4.proveedor_ruc ORDER BY v4.fecha_proceso
                 RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING),
         ent AS (PARTITION BY v4.entidad_ruc ORDER BY v4.fecha_proceso
                 RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING)""")
print(con.sql("""SELECT count(*) AS filas_donde_difieren,
   count(*) FILTER (WHERE abs(v4.ind_cuota_prov_en_entidad_12m - cu.monto_viejo_cuota) < 1e-6) AS es_cuota_de_monto,
   count(*) FILTER (WHERE abs(v4.ind_cuota_prov_en_entidad_12m - cu.conteo_cuota)      < 1e-6) AS es_cuota_de_procesos,
   count(*) FILTER (WHERE abs(v4.ind_cuota_prov_en_entidad_12m - cu.monto_viejo_cuota) >= 1e-6
                      AND abs(v4.ind_cuota_prov_en_entidad_12m - cu.conteo_cuota)      >= 1e-6) AS ninguna
 FROM v4 JOIN cu USING (ocid)
 WHERE v4.ind_cuota_prov_en_entidad_12m IS NOT NULL
   AND cu.monto_viejo_cuota IS NOT NULL AND cu.conteo_cuota IS NOT NULL
   AND abs(cu.monto_viejo_cuota - cu.conteo_cuota) > 1e-6""").df().to_string(index=False))
print('\n-- y la otra cuota, para no dejarla en el aire --')
print(con.sql("""SELECT count(*) AS filas_donde_difieren,
   count(*) FILTER (WHERE abs(v4.ind_cuota_procesos_12m - cu.monto_viejo_cuota) < 1e-6) AS es_cuota_de_monto,
   count(*) FILTER (WHERE abs(v4.ind_cuota_procesos_12m - cu.conteo_cuota)      < 1e-6) AS es_cuota_de_procesos
 FROM v4 JOIN cu USING (ocid)
 WHERE v4.ind_cuota_procesos_12m IS NOT NULL
   AND abs(cu.monto_viejo_cuota - cu.conteo_cuota) > 1e-6""").df().to_string(index=False))

# ---------- (2) AUDITORIA DE LA DURACION, TRES REDES ----------
t('(2a) AÑO DE FIN DEL CONTRATO: la red mas limpia')
print(con.sql("""SELECT CASE WHEN year(j_contrato_fin) > 2100 THEN 'e) despues de 2100 (imposible)'
                             WHEN year(j_contrato_fin) > 2060 THEN 'd) 2061 a 2100 (imposible)'
                             WHEN year(j_contrato_fin) > 2040 THEN 'c) 2041 a 2060 (dudoso)'
                             WHEN year(j_contrato_fin) > 2030 THEN 'b) 2031 a 2040 (posible)'
                             ELSE 'a) hasta 2030 (normal)' END AS tramo,
                        count(*) AS procesos
                 FROM jj WHERE j_contrato_fin IS NOT NULL GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
print('\n-- los que terminan despues de 2040, uno por uno --')
print(con.sql("""SELECT jj.ocid, v3.familia, v3.monto_final, jj.j_contrato_inicio::DATE AS inicio,
                        jj.j_contrato_fin::DATE AS fin, v3.ind_duracion_contrato_dias AS dias_v3
                 FROM jj JOIN v3 ON v3.ocid = jj.ocid
                 WHERE year(jj.j_contrato_fin) > 2040
                 ORDER BY jj.j_contrato_fin DESC LIMIT 40""").df().to_string(index=False))

t('(2b) IDENTIDADES: la duracion es en realidad otro campo?')
print(con.sql("""SELECT count(*) FILTER (WHERE ind_duracion_contrato_dias IS NOT NULL) AS con_duracion,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(monto_final))      AS igual_al_monto,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(monto_final*100))  AS igual_al_monto_centavos,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(precio_unit))      AS igual_al_precio_unit,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(cantidad))         AS igual_a_la_cantidad,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(monto_final) + 7)  AS control_monto_mas_7,
   count(*) FILTER (WHERE ind_duracion_contrato_dias = round(cantidad) + 7)     AS control_cantidad_mas_7
 FROM v3""").df().to_string(index=False))

t('(2c) PLAUSIBILIDAD POR FAMILIA: hasta donde llega una duracion creible')
print(con.sql("""SELECT familia, count(ind_duracion_contrato_dias) AS con_dato,
   round(median(ind_duracion_contrato_dias),0) AS mediana,
   round(quantile_cont(ind_duracion_contrato_dias,0.95),0)   AS p95,
   round(quantile_cont(ind_duracion_contrato_dias,0.999),0)  AS p999,
   max(ind_duracion_contrato_dias) AS maximo,
   count(*) FILTER (WHERE ind_duracion_contrato_dias > 3650) AS mas_de_10_anios,
   count(*) FILTER (WHERE ind_duracion_contrato_dias <= 0)   AS cero_o_negativa
 FROM v3 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

# ---------- (3) CONSTRUIR LAS BANDERAS Y LA IMPUTACION ----------
t('(3) REGLAS APLICADAS')
con.sql("""CREATE OR REPLACE TABLE dur AS
  SELECT v3.ocid, v3.ind_duracion_contrato_dias AS dias_original, jj.j_contrato_fin,
    (v3.ind_duracion_contrato_dias IS NOT NULL AND (
        v3.ind_duracion_contrato_dias <= 0
     OR v3.ind_duracion_contrato_dias > 7300
     OR v3.ind_duracion_contrato_dias = round(v3.monto_final)
     OR v3.ind_duracion_contrato_dias = round(v3.monto_final*100)
     OR year(jj.j_contrato_fin) > 2060)) AS duracion_invalida
  FROM v3 LEFT JOIN jj ON jj.ocid = v3.ocid""")
print(con.sql("""SELECT count(*) FILTER (WHERE duracion_invalida) AS invalidas,
   count(*) FILTER (WHERE duracion_invalida AND dias_original <= 0)       AS por_cero_o_negativa,
   count(*) FILTER (WHERE duracion_invalida AND dias_original > 7300)     AS por_mas_de_20_anios,
   count(*) FILTER (WHERE duracion_invalida AND year(j_contrato_fin) > 2060) AS por_anio_imposible,
   count(dias_original) AS con_duracion
 FROM dur""").df().to_string(index=False))

# mediana del grupo comparable: familia x CPC5 si hay 30 casos validos, si no familia
con.sql("""CREATE OR REPLACE TABLE med_fc AS
  SELECT v4.familia, v4.cpc5, median(v4.ind_duracion_contrato_dias) AS m, count(*) AS n
  FROM v4 JOIN dur USING (ocid)
  WHERE v4.ind_duracion_contrato_dias IS NOT NULL AND NOT dur.duracion_invalida
  GROUP BY 1,2 HAVING count(*) >= 30""")
con.sql("""CREATE OR REPLACE TABLE med_f AS
  SELECT v4.familia, median(v4.ind_duracion_contrato_dias) AS m
  FROM v4 JOIN dur USING (ocid)
  WHERE v4.ind_duracion_contrato_dias IS NOT NULL AND NOT dur.duracion_invalida
  GROUP BY 1""")
med_glob = con.sql("""SELECT median(v4.ind_duracion_contrato_dias) FROM v4 JOIN dur USING (ocid)
                      WHERE NOT dur.duracion_invalida""").fetchone()[0]
print(f'\ngrupos familia x CPC5 con 30 o mas casos validos: {con.sql("SELECT count(*) FROM med_fc").fetchone()[0]:,}')
print(f'mediana global de respaldo: {med_glob} dias')

# ---------- (4) v4.1 ----------
t('(4) ARMAR v4.1')
C4 = list(con.sql("DESCRIBE SELECT * FROM v4").df().column_name)
sel = ', '.join(
    ('CASE WHEN dur.duracion_invalida THEN NULL ELSE v4."ind_duracion_contrato_dias" END '
     'AS ind_duracion_contrato_dias' if c == 'ind_duracion_contrato_dias' else
     'CASE WHEN dur.duracion_invalida THEN TRUE ELSE v4."falta_duracion" END AS falta_duracion'
     if c == 'falta_duracion' else
     'CASE WHEN v4."ind_contrato_sobre_adjudicado" > 10 '
     '       OR v4."ind_contrato_sobre_adjudicado" < 0.1 THEN NULL '
     '  ELSE v4."ind_contrato_sobre_adjudicado" END AS ind_contrato_sobre_adjudicado'
     if c == 'ind_contrato_sobre_adjudicado' else f'v4."{c}"')
    for c in C4)
con.sql(f"""CREATE OR REPLACE TABLE v41 AS
  SELECT {sel},
    dur.duracion_invalida,
    (v4."ind_contrato_sobre_adjudicado" > 10 OR v4."ind_contrato_sobre_adjudicado" < 0.1)
      AS ratio_contrato_implausible,
    -- SOLO PARA DESCRIPTIVO: nunca entra al modelo
    CASE WHEN dur.duracion_invalida OR v4."ind_duracion_contrato_dias" IS NULL
         THEN coalesce(mfc.m, mf.m, {med_glob})
         ELSE v4."ind_duracion_contrato_dias" END AS duracion_dias_imputada,
    (dur.duracion_invalida OR v4."ind_duracion_contrato_dias" IS NULL) AS duracion_imputada
  FROM v4 JOIN dur USING (ocid)
       LEFT JOIN med_fc mfc ON mfc.familia = v4.familia AND mfc.cpc5 = v4.cpc5
       LEFT JOIN med_f  mf  ON mf.familia  = v4.familia""")

t('(5) CONTROLES')
print(con.sql("""SELECT count(*) AS filas, count(DISTINCT ocid) AS ocid,
   round(sum(monto_final)/1e6,1) AS millones,
   count(ind_duracion_contrato_dias) AS duracion_con_dato,
   max(ind_duracion_contrato_dias)   AS duracion_maxima,
   count(*) FILTER (WHERE duracion_invalida) AS duracion_invalida,
   count(*) FILTER (WHERE duracion_imputada) AS duracion_imputada,
   count(ind_contrato_sobre_adjudicado) AS ratio_con_dato,
   round(max(ind_contrato_sobre_adjudicado),2) AS ratio_maximo,
   count(*) FILTER (WHERE ratio_contrato_implausible) AS ratio_anulado,
   count(*) FILTER (WHERE duracion_invalida AND NOT falta_duracion) AS bandera_mal_puesta
 FROM v41""").df().to_string(index=False))
print('\n-- la duracion, antes y despues --')
print(con.sql("""SELECT 'original (v3)' AS version, count(ind_duracion_contrato_dias) AS con_dato,
   round(median(ind_duracion_contrato_dias),0) AS mediana, max(ind_duracion_contrato_dias) AS maximo FROM v3
 UNION ALL SELECT 'limpia (v4.1)', count(ind_duracion_contrato_dias),
   round(median(ind_duracion_contrato_dias),0), max(ind_duracion_contrato_dias) FROM v41
 UNION ALL SELECT 'imputada (descriptivo)', count(duracion_dias_imputada),
   round(median(duracion_dias_imputada),0), max(duracion_dias_imputada) FROM v41""").df().to_string(index=False))

t('(6) ESCRITURA Y FIRMA')
out = 'plan_A/planA_indicadores_v4_1.parquet'
con.sql(f"COPY v41 TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
h = sha16(out); print(f'  {out} | {round(os.path.getsize(out)/1e6,1)} MB | sha256_16 = {h}')
if os.path.exists('plan_A/planA_indicadores_v4.parquet'):
    os.replace('plan_A/planA_indicadores_v4.parquet', f'{HIST}/planA_indicadores_v4.parquet')
esp = f'{DOC}/especificacion_indicadores.json'
E = json.load(open(esp, encoding='utf-8'))
E['enmiendas'].append({
    'version': 'v4.1', 'fecha': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'motivo': 'limpieza de la duracion del contrato y reposicion del tope del ratio',
    'cambios': [
        'ind_duracion_contrato_dias se anula cuando es <= 0, mayor a 7300 dias, igual al monto '
        '(o al monto en centavos), o cuando el contrato termina despues de 2060; la ausencia la '
        'carga falta_duracion y la marca duracion_invalida',
        'duracion_dias_imputada: mediana de familia x CPC5 (minimo 30 casos validos), con '
        'respaldo en familia y mediana global. ES SOLO PARA ESTADISTICA DESCRIPTIVA, nunca entra '
        'al modelo, y se identifica con la bandera duracion_imputada',
        'ind_contrato_sobre_adjudicado: se repone el tope de plausibilidad (se anula sobre 10 y '
        'bajo 0.1), que se habia perdido al recalcular el ratio en v4; se marca con '
        'ratio_contrato_implausible'],
    'sha256_16_indicadores_v4_1': h})
json.dump(E, open(esp, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        M.append({'archivo': f, 'filas': con.sql(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0],
                  'columnas': len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()),
                  'MB': round(os.path.getsize(p)/1e6,1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print('\n' + MAN.to_string(index=False)); print(f'\n{len(MAN)} archivos | {round(MAN.MB.sum(),1)} MB')
pd.DataFrame(CIF).to_csv(f'{DOC}/cifras_celda29.csv', index=False, encoding='utf-8-sig')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 30 · CUOTA DE MONTO RECALCULADA -> v4.2 (cierre) ============
# (1) averigua la regla de ind_fraccionamiento_marca, por si tambien depende del monto
# (2) recalcula ind_cuota_prov_en_entidad_12m con el monto corregido (definicion ya verificada)
# (3) recalcula los 9 rangos  (4) controles  (5) escribe v4.2, firma y cierra la entrega
import os, json, hashlib
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST = 'plan_A/documentacion', 'plan_A/_version_1_historico'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
CIF = []
def cifra(n, v, nota=''): CIF.append({'cifra': n, 'valor': v, 'nota': nota}); return v
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

con.sql("CREATE OR REPLACE VIEW v41 AS SELECT * FROM read_parquet('plan_A/planA_indicadores_v4_1.parquet')")
con.sql(f"CREATE OR REPLACE VIEW v3 AS SELECT * FROM read_parquet('{HIST}/planA_indicadores_v3.parquet')")

# ---------- (1) LA REGLA DE ind_fraccionamiento_marca ----------
t('(1) DE QUE DEPENDE ind_fraccionamiento_marca (se prueba contra los valores ORIGINALES)')
print(con.sql("""SELECT count(*) AS filas,
   count(*) FILTER (WHERE ind_fraccionamiento_marca = (ind_fraccionamiento_n_30d > 0))     AS regla_n_mayor_0,
   count(*) FILTER (WHERE ind_fraccionamiento_marca = (ind_fraccionamiento_n_30d >= 2))    AS regla_n_mayor_igual_2,
   count(*) FILTER (WHERE ind_fraccionamiento_marca = (ind_fraccionamiento_monto_30d > 0)) AS regla_monto_mayor_0
 FROM v3""").df().to_string(index=False))
print('\n-- tabla cruzada: numero de procesos previos en 30 dias contra la marca --')
print(con.sql("""SELECT ind_fraccionamiento_n_30d AS procesos_previos_30d,
                        ind_fraccionamiento_marca AS marca, count(*) AS filas
                 FROM v3 GROUP BY 1,2 ORDER BY 1,2""").df().to_string(index=False))

# ---------- (2) CUOTA DE MONTO CON EL MONTO CORREGIDO ----------
t('(2) RECALCULO DE ind_cuota_prov_en_entidad_12m  (cuota de monto, definicion verificada al 100%)')
con.sql("""CREATE OR REPLACE TABLE cu2 AS
  SELECT ocid,
    sum(monto_final) OVER par / nullif(sum(monto_final) OVER ent, 0) AS cuota_nueva
  FROM v41
  WINDOW par AS (PARTITION BY entidad_ruc, proveedor_ruc ORDER BY fecha_proceso
                 RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING),
         ent AS (PARTITION BY entidad_ruc ORDER BY fecha_proceso
                 RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING)""")
print(con.sql("""SELECT count(v41.ind_cuota_prov_en_entidad_12m) AS cuota_vieja_con_dato,
   count(cu2.cuota_nueva) AS cuota_nueva_con_dato,
   count(*) FILTER (WHERE abs(coalesce(v41.ind_cuota_prov_en_entidad_12m,-1)
                            - coalesce(cu2.cuota_nueva,-1)) > 1e-9) AS filas_que_cambian,
   round(max(abs(v41.ind_cuota_prov_en_entidad_12m - cu2.cuota_nueva)),4) AS cambio_maximo,
   round(min(cu2.cuota_nueva),4) AS minimo, round(max(cu2.cuota_nueva),4) AS maximo
 FROM v41 LEFT JOIN cu2 USING (ocid)""").df().to_string(index=False))

# ---------- (3) v4.2 CON LOS 9 RANGOS RECALCULADOS ----------
t('(3) ARMAR v4.2')
C = list(con.sql("DESCRIBE SELECT * FROM v41").df().column_name)
VARS = [('pct_precio_alto','ind_desv_precio_unit_z','ASC'),
        ('pct_cuota_12m','ind_cuota_prov_en_entidad_12m','ASC'),
        ('pct_recurrencia_par','ind_recurrencia_par_12m','ASC'),
        ('pct_monto_capital','ind_monto_sobre_capital','ASC'),
        ('pct_fraccionamiento','ind_fraccionamiento_monto_30d','ASC'),
        ('pct_proveedor_nuevo','ind_antiguedad_dias','DESC'),
        ('pct_poca_concurrencia','ind_n_oferentes','DESC'),
        ('pct_plazo_corto','ind_plazo_ofertas_dias','DESC'),
        ('pct_firma_rapida','ind_plazo_adj_firma_dias','DESC')]
pctn = {v[0] for v in VARS}
sel = ', '.join(('cu2.cuota_nueva AS ind_cuota_prov_en_entidad_12m'
                 if c == 'ind_cuota_prov_en_entidad_12m' else f'v41."{c}"')
                for c in C if c not in pctn)
con.sql(f"CREATE OR REPLACE TABLE b42 AS SELECT {sel} FROM v41 LEFT JOIN cu2 USING (ocid)")
for i, (pct, src, d) in enumerate(VARS):
    con.sql(f"""CREATE OR REPLACE TABLE z{i} AS
      WITH c AS (SELECT ocid, familia, periodo, "{src}" AS x,
                        count("{src}") OVER (PARTITION BY familia, periodo) AS n_fp,
                        count("{src}") OVER (PARTITION BY familia)          AS n_f
                 FROM b42),
           d AS (SELECT ocid, x, CASE WHEN n_fp >= 30 THEN familia || '|' || periodo
                                      WHEN n_f  >= 30 THEN familia ELSE 'GLOBAL' END AS grupo
                 FROM c WHERE x IS NOT NULL)
      SELECT ocid, percent_rank() OVER (PARTITION BY grupo ORDER BY x {d}) AS "{pct}" FROM d""")
idx = {v[0]: i for i, v in enumerate(VARS)}
sel2 = ', '.join((f'z{idx[c]}."{c}"' if c in pctn else f'b42."{c}"') for c in C)
joins = ' '.join(f'LEFT JOIN z{i} USING (ocid)' for i in range(len(VARS)))
con.sql(f"CREATE OR REPLACE TABLE v42 AS SELECT {sel2} FROM b42 {joins}")
print(f'v4.2: {con.sql("SELECT count(*) FROM v42").fetchone()[0]:,} filas, '
      f'{len(con.sql("DESCRIBE SELECT * FROM v42").df())} columnas')

# ---------- (4) CONTROLES ----------
t('(4) CONTROLES DE ACEPTACION')
piezas = [f"""SELECT '{pct}' AS variable, count("{src}") AS fuente, count("{pct}") AS rango,
   CASE WHEN count("{src}")=count("{pct}") THEN 'OK' ELSE 'DESCUADRE' END AS nulos_ok,
   round(min("{pct}"),4) AS pmin, round(max("{pct}"),4) AS pmax, round(corr("{src}","{pct}"),3) AS corr,
   CASE WHEN ('{d}'='ASC' AND corr("{src}","{pct}")>0) OR ('{d}'='DESC' AND corr("{src}","{pct}")<0)
        THEN 'OK' ELSE 'INVERTIDA' END AS direccion_ok FROM v42""" for pct, src, d in VARS]
K = con.sql('\nUNION ALL\n'.join(piezas)).df(); print(K.to_string(index=False))
cifra('rangos_nulos_ok', int((K.nulos_ok == 'OK').sum()), f'de {len(K)}')
cifra('rangos_direccion_ok', int((K.direccion_ok == 'OK').sum()), f'de {len(K)}')
print('\n-- cambios frente a v4.1 --')
cmp_ = [f"""SELECT '{pct}' AS variable,
   count(*) FILTER (WHERE abs(coalesce(a."{pct}",-1)-coalesce(b."{pct}",-1))>0.01) AS filas_que_cambian
   FROM v41 a JOIN v42 b USING (ocid)""" for pct, _, _ in VARS]
print(con.sql('\nUNION ALL\n'.join(cmp_)).df().to_string(index=False))
print('\n-- integridad final --')
print(con.sql("""SELECT count(*) AS filas, count(DISTINCT ocid) AS ocid,
   round(sum(monto_final)/1e6,1) AS millones,
   count(*) FILTER (WHERE ind_proveedor_debutante AND ind_procesos_proveedor_12m>0) AS debutante_con_historia,
   count(*) FILTER (WHERE ind_primera_vez_con_la_entidad AND ind_recurrencia_par_12m>0) AS primera_con_historia,
   count(*) FILTER (WHERE ind_cuota_prov_en_entidad_12m > 1.0000001) AS cuota_mayor_que_uno,
   count(*) FILTER (WHERE ind_cuota_prov_en_entidad_12m < 0) AS cuota_negativa,
   max(ind_duracion_contrato_dias) AS duracion_maxima,
   round(max(ind_contrato_sobre_adjudicado),2) AS ratio_maximo,
   count(*) FILTER (WHERE es_multilote) AS multilote
 FROM v42""").df().to_string(index=False))
prohib = [c for c in C if c.startswith(('red_','cob_','com_','idx_','rep_','grupo_'))
          or 'monto_tender' in c or 'alguna_vez' in c or 'fantasma' in c.lower()]
print('columnas prohibidas:', prohib if prohib else 'ninguna (correcto)')

# ---------- (5) ESCRITURA, FIRMA Y CIERRE ----------
t('(5) ESCRITURA, FIRMA Y CIERRE DE LA ENTREGA')
out = 'plan_A/planA_indicadores_v4_2.parquet'
con.sql(f"COPY v42 TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
h = sha16(out); print(f'  {out} | {round(os.path.getsize(out)/1e6,1)} MB | sha256_16 = {h}')
if os.path.exists('plan_A/planA_indicadores_v4_1.parquet'):
    os.replace('plan_A/planA_indicadores_v4_1.parquet', f'{HIST}/planA_indicadores_v4_1.parquet')
esp = f'{DOC}/especificacion_indicadores.json'
E = json.load(open(esp, encoding='utf-8'))
E['enmiendas'].append({
    'version': 'v4.2', 'fecha': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'motivo': 'ultima dependencia del monto corregido',
    'cambios': [
        'ind_cuota_prov_en_entidad_12m es cuota de MONTO (identificada al 100% en las 145.956 '
        'filas donde la definicion de monto y la de conteo difieren); se recalcula con el monto '
        'corregido. ind_cuota_procesos_12m es cuota de PROCESOS, igualmente verificada al 100%, '
        'y no depende del monto',
        'los 9 rangos percentiles se recalculan en consecuencia'],
    'sha256_16_indicadores_v4_2': h})
E['definiciones_verificadas'] = {
    'ind_monto_sobre_capital': 'monto adjudicado / capital social; reproducida al 100%',
    'ind_fraccionamiento_monto_30d': 'suma del monto del mismo par entidad-proveedor y mismo CPC5 '
                                     'en los 30 dias anteriores, sin el dia del proceso; reproducida al 100%',
    'ind_cuota_prov_en_entidad_12m': 'monto del par / monto de la entidad, 365 dias anteriores; '
                                     'reproducida al 100% en las filas discriminantes',
    'ind_cuota_procesos_12m': 'procesos del par / procesos de la entidad, 365 dias anteriores; '
                              'reproducida al 100% en las filas discriminantes',
    'ind_desv_precio_unit_z': 'z robusto del ln del precio unitario del item en su grupo de '
                              'referencia de precio; NO depende de monto_final',
    'monto_final': 'suma de las adjudicaciones del proceso'}
json.dump(E, open(esp, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('  especificacion actualizada con las definiciones verificadas')
M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        M.append({'archivo': f, 'filas': con.sql(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0],
                  'columnas': len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()),
                  'MB': round(os.path.getsize(p)/1e6,1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print('\n' + MAN.to_string(index=False)); print(f'\n{len(MAN)} archivos | {round(MAN.MB.sum(),1)} MB')
pd.DataFrame(CIF).to_csv(f'{DOC}/cifras_celda30.csv', index=False, encoding='utf-8-sig')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 31 · ARREGLO DE LA COBERTURA DE LA CUOTA -> v4.3 ============
# El numerador de la cuota necesita coalesce(...,0): sin historial previo la cuota es CERO,
# no ausente. Esta vez la reproduccion se prueba sobre TODAS las filas, no solo las que difieren.
import os, json, hashlib
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST = 'plan_A/documentacion', 'plan_A/_version_1_historico'
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
CIF = []
def cifra(n, v, nota=''): CIF.append({'cifra': n, 'valor': v, 'nota': nota}); return v
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

con.sql("CREATE OR REPLACE VIEW v42 AS SELECT * FROM read_parquet('plan_A/planA_indicadores_v4_2.parquet')")
con.sql(f"CREATE OR REPLACE VIEW v3 AS SELECT * FROM read_parquet('{HIST}/planA_indicadores_v3.parquet')")
con.sql("CREATE OR REPLACE VIEW p3 AS SELECT * FROM read_parquet('plan_A/planA_P1_v3_modelo.parquet')")

# ---------- (1) REPRODUCCION COMPLETA, SOBRE TODAS LAS FILAS ----------
t('(1) REPRODUCCION COMPLETA DE LAS DOS CUOTAS (monto viejo, todas las filas)')
con.sql("""CREATE OR REPLACE TABLE rep AS
  SELECT v42.ocid,
    coalesce(sum(p3.monto_adjudicacion_unica_v1) OVER par, 0)
      / nullif(sum(p3.monto_adjudicacion_unica_v1) OVER ent, 0) AS cuota_monto_viejo,
    coalesce(sum(v42.monto_final) OVER par, 0)
      / nullif(sum(v42.monto_final) OVER ent, 0)                AS cuota_monto_nuevo,
    coalesce(count(*) OVER par, 0) * 1.0 / nullif(count(*) OVER ent, 0) AS cuota_procesos
  FROM v42 JOIN p3 ON p3.ocid = v42.ocid
  WINDOW par AS (PARTITION BY v42.entidad_ruc, v42.proveedor_ruc ORDER BY v42.fecha_proceso
                 RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING),
         ent AS (PARTITION BY v42.entidad_ruc ORDER BY v42.fecha_proceso
                 RANGE BETWEEN INTERVAL 365 DAY PRECEDING AND INTERVAL 1 DAY PRECEDING)""")
print(con.sql("""SELECT count(*) AS filas,
   count(v3.ind_cuota_prov_en_entidad_12m) AS original_con_dato,
   count(rep.cuota_monto_viejo)            AS reproducido_con_dato,
   count(*) FILTER (WHERE abs(v3.ind_cuota_prov_en_entidad_12m - rep.cuota_monto_viejo) < 1e-9
                 OR (v3.ind_cuota_prov_en_entidad_12m IS NULL AND rep.cuota_monto_viejo IS NULL)) AS coincide,
   round(100.0*count(*) FILTER (WHERE abs(v3.ind_cuota_prov_en_entidad_12m - rep.cuota_monto_viejo) < 1e-9
                 OR (v3.ind_cuota_prov_en_entidad_12m IS NULL AND rep.cuota_monto_viejo IS NULL))
         /count(*),3) AS pct
 FROM v3 JOIN rep USING (ocid)""").df().to_string(index=False))
print('\n-- la cuota de procesos, igual de completa --')
print(con.sql("""SELECT count(v3.ind_cuota_procesos_12m) AS original_con_dato,
   count(rep.cuota_procesos) AS reproducido_con_dato,
   round(100.0*count(*) FILTER (WHERE abs(v3.ind_cuota_procesos_12m - rep.cuota_procesos) < 1e-9
                 OR (v3.ind_cuota_procesos_12m IS NULL AND rep.cuota_procesos IS NULL))
         /count(*),3) AS pct
 FROM v3 JOIN rep USING (ocid)""").df().to_string(index=False))
r = con.sql("""SELECT round(100.0*count(*) FILTER (WHERE abs(v3.ind_cuota_prov_en_entidad_12m
       - rep.cuota_monto_viejo) < 1e-9 OR (v3.ind_cuota_prov_en_entidad_12m IS NULL
       AND rep.cuota_monto_viejo IS NULL))/count(*),3) FROM v3 JOIN rep USING (ocid)""").fetchone()[0]
ok = float(r) > 99.9
print(f'\nreproduccion completa: {r}%  -> {"se aplica" if ok else "NO SE APLICA, revisar"}')

print('\n-- y la bandera falta_cuota_12m, a que corresponde --')
print(con.sql("""SELECT count(*) FILTER (WHERE falta_cuota_12m = (ind_cuota_prov_en_entidad_12m IS NULL)) AS regla_cuota_nula,
   count(*) FILTER (WHERE falta_cuota_12m) AS marcadas, count(*) AS filas FROM v3""").df().to_string(index=False))

# ---------- (2) v4.3 ----------
t('(2) ARMAR v4.3')
C = list(con.sql("DESCRIBE SELECT * FROM v42").df().column_name)
VARS = [('pct_precio_alto','ind_desv_precio_unit_z','ASC'),
        ('pct_cuota_12m','ind_cuota_prov_en_entidad_12m','ASC'),
        ('pct_recurrencia_par','ind_recurrencia_par_12m','ASC'),
        ('pct_monto_capital','ind_monto_sobre_capital','ASC'),
        ('pct_fraccionamiento','ind_fraccionamiento_monto_30d','ASC'),
        ('pct_proveedor_nuevo','ind_antiguedad_dias','DESC'),
        ('pct_poca_concurrencia','ind_n_oferentes','DESC'),
        ('pct_plazo_corto','ind_plazo_ofertas_dias','DESC'),
        ('pct_firma_rapida','ind_plazo_adj_firma_dias','DESC')]
pctn = {v[0] for v in VARS}
def expr(c):
    if c == 'ind_cuota_prov_en_entidad_12m': return 'rep.cuota_monto_nuevo AS ind_cuota_prov_en_entidad_12m'
    if c == 'falta_cuota_12m': return '(rep.cuota_monto_nuevo IS NULL) AS falta_cuota_12m'
    return f'v42."{c}"'
sel = ', '.join(expr(c) for c in C if c not in pctn)
con.sql(f"CREATE OR REPLACE TABLE b43 AS SELECT {sel} FROM v42 JOIN rep USING (ocid)")
for i, (pct, src, d) in enumerate(VARS):
    con.sql(f"""CREATE OR REPLACE TABLE y{i} AS
      WITH c AS (SELECT ocid, familia, periodo, "{src}" AS x,
                        count("{src}") OVER (PARTITION BY familia, periodo) AS n_fp,
                        count("{src}") OVER (PARTITION BY familia)          AS n_f
                 FROM b43),
           d AS (SELECT ocid, x, CASE WHEN n_fp >= 30 THEN familia || '|' || periodo
                                      WHEN n_f  >= 30 THEN familia ELSE 'GLOBAL' END AS grupo
                 FROM c WHERE x IS NOT NULL)
      SELECT ocid, percent_rank() OVER (PARTITION BY grupo ORDER BY x {d}) AS "{pct}" FROM d""")
idx = {v[0]: i for i, v in enumerate(VARS)}
sel2 = ', '.join((f'y{idx[c]}."{c}"' if c in pctn else f'b43."{c}"') for c in C)
joins = ' '.join(f'LEFT JOIN y{i} USING (ocid)' for i in range(len(VARS)))
con.sql(f"CREATE OR REPLACE TABLE v43 AS SELECT {sel2} FROM b43 {joins}")

# ---------- (3) CONTROLES ----------
t('(3) CONTROLES DE ACEPTACION')
piezas = [f"""SELECT '{pct}' AS variable, count("{src}") AS fuente, count("{pct}") AS rango,
   CASE WHEN count("{src}")=count("{pct}") THEN 'OK' ELSE 'DESCUADRE' END AS nulos_ok,
   round(min("{pct}"),4) AS pmin, round(max("{pct}"),4) AS pmax, round(corr("{src}","{pct}"),3) AS corr,
   CASE WHEN ('{d}'='ASC' AND corr("{src}","{pct}")>0) OR ('{d}'='DESC' AND corr("{src}","{pct}")<0)
        THEN 'OK' ELSE 'INVERTIDA' END AS direccion_ok FROM v43""" for pct, src, d in VARS]
K = con.sql('\nUNION ALL\n'.join(piezas)).df(); print(K.to_string(index=False))
print('\n-- cobertura de la cuota: tiene que volver a 438.465 --')
print(con.sql("""SELECT count(ind_cuota_prov_en_entidad_12m) AS cuota_con_dato,
   count(ind_cuota_procesos_12m) AS cuota_procesos_con_dato,
   count(*) FILTER (WHERE falta_cuota_12m) AS marcadas_como_faltante,
   count(*) FILTER (WHERE falta_cuota_12m <> (ind_cuota_prov_en_entidad_12m IS NULL)) AS bandera_incoherente,
   count(*) FILTER (WHERE ind_cuota_prov_en_entidad_12m = 0) AS cuota_exactamente_cero,
   round(median(ind_cuota_prov_en_entidad_12m),4) AS mediana,
   round(max(ind_cuota_prov_en_entidad_12m),4) AS maximo FROM v43""").df().to_string(index=False))
print('\n-- cambios frente a v4.2 y frente al original v3 --')
print(con.sql("""SELECT
   count(*) FILTER (WHERE abs(coalesce(b.ind_cuota_prov_en_entidad_12m,-1)
                            - coalesce(a.ind_cuota_prov_en_entidad_12m,-1)) > 1e-9) AS cambian_vs_v42,
   count(*) FILTER (WHERE abs(coalesce(b.ind_cuota_prov_en_entidad_12m,-1)
                            - coalesce(c.ind_cuota_prov_en_entidad_12m,-1)) > 1e-9) AS cambian_vs_v3_original
 FROM v42 a JOIN v43 b USING (ocid) JOIN v3 c ON c.ocid = b.ocid""").df().to_string(index=False))
print('\n-- integridad final --')
print(con.sql("""SELECT count(*) AS filas, count(DISTINCT ocid) AS ocid,
   round(sum(monto_final)/1e6,1) AS millones,
   count(*) FILTER (WHERE ind_proveedor_debutante AND ind_procesos_proveedor_12m>0) AS debutante_con_historia,
   count(*) FILTER (WHERE ind_primera_vez_con_la_entidad AND ind_recurrencia_par_12m>0) AS primera_con_historia,
   count(*) FILTER (WHERE ind_cuota_prov_en_entidad_12m > 1.0000001
                       OR ind_cuota_prov_en_entidad_12m < 0) AS cuota_fuera_de_rango,
   max(ind_duracion_contrato_dias) AS duracion_maxima,
   round(max(ind_contrato_sobre_adjudicado),2) AS ratio_maximo,
   count(*) FILTER (WHERE es_multilote) AS multilote FROM v43""").df().to_string(index=False))

# ---------- (4) ESCRITURA Y FIRMA ----------
t('(4) ESCRITURA Y FIRMA')
out = 'plan_A/planA_indicadores_v4_3.parquet'
con.sql(f"COPY v43 TO '{out}' (FORMAT PARQUET, COMPRESSION ZSTD)")
h = sha16(out); print(f'  {out} | {round(os.path.getsize(out)/1e6,1)} MB | sha256_16 = {h}')
if os.path.exists('plan_A/planA_indicadores_v4_2.parquet'):
    os.replace('plan_A/planA_indicadores_v4_2.parquet', f'{HIST}/planA_indicadores_v4_2.parquet')
esp = f'{DOC}/especificacion_indicadores.json'
E = json.load(open(esp, encoding='utf-8'))
E['enmiendas'].append({
    'version': 'v4.3', 'fecha': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'motivo': 'corregir una regresion introducida en v4.2 al recalcular la cuota',
    'cambios': [
        'el numerador de ind_cuota_prov_en_entidad_12m lleva coalesce a cero: sin historial previo '
        'del par la cuota es CERO, no ausente. En v4.2 quedaba nula y la cobertura cayo de 438.465 '
        'a 146.881 filas',
        'falta_cuota_12m se recalcula como (cuota IS NULL), coherente con la regla original',
        'los 9 rangos percentiles se recalculan en consecuencia',
        'leccion de metodo: la prueba de reproduccion de v4.2 se restringio a las filas donde dos '
        'definiciones candidatas difieren, que son precisamente las filas con historial; validaba '
        'la forma de la formula y era ciega al tratamiento de la ausencia de historial. Desde v4.3 '
        'toda reproduccion se prueba sobre el total de filas, contando tambien las coincidencias '
        'de nulo contra nulo'],
    'sha256_16_indicadores_v4_3': h})
json.dump(E, open(esp, 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        M.append({'archivo': f, 'filas': con.sql(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0],
                  'columnas': len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()),
                  'MB': round(os.path.getsize(p)/1e6,1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print('\n' + MAN.to_string(index=False)); print(f'\n{len(MAN)} archivos | {round(MAN.MB.sum(),1)} MB')
pd.DataFrame(CIF).to_csv(f'{DOC}/cifras_celda31.csv', index=False, encoding='utf-8-sig')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 32 · EL RESTO DE LA BASE AL MISMO ESTANDAR -> familia v3 ============
# Aplica a universo, catalogo y los tres archivos de exclusion exactamente lo mismo que a P1:
# monto_final = suma de adjudicaciones, fuera las 56 columnas de red y las 9 derivadas rancias,
# mas es_multilote. Luego cuadra la particion y la plata, y arma el paquete de envio.
import os, shutil, hashlib
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST = 'plan_A/documentacion', 'plan_A/_version_1_historico'
os.makedirs(HIST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

con.sql("CREATE OR REPLACE VIEW a AS SELECT * FROM read_parquet('03_resultados/c25_awards.parquet')")
PREF = ('red_', 'cob_', 'com_', 'idx_', 'rep_', 'grupo_')
RANCIAS = ['log_monto', 'desviacion_precio_log_z', 'desviacion_precio_log_z_sinfuga',
           'monto_acumulado_30d_no_catalogo', 'monto_total_historico_par',
           'monto_total_historico_entidad', 'concentracion_entidad_proveedor',
           'ratio_monto_capital', 'ent_monto_total']
TRABAJOS = [('planA_universo_v2',                    'planA_universo_v3'),
            ('planA_v2_P0_catalogo',                 'planA_P0_catalogo_v3'),
            ('planA_v2_excluidos_solo_planeacion',   'planA_excluidos_solo_planeacion_v3'),
            ('planA_v2_excluidos_sin_monto',         'planA_excluidos_sin_monto_v3'),
            ('planA_v2_sin_proveedor_identificable', 'planA_sin_proveedor_identificable_v3')]

t('(1) RECONSTRUCCION')
for src, dst in TRABAJOS:
    pin, pout = f'plan_A/{src}.parquet', f'plan_A/{dst}.parquet'
    if not os.path.exists(pin):
        print(f'  {src}: no esta (ya reconstruido?), se salta'); continue
    con.sql(f"CREATE OR REPLACE VIEW s AS SELECT * FROM read_parquet('{pin}')")
    cols = list(con.sql("DESCRIBE SELECT * FROM s").df().column_name)
    fuera = [c for c in cols if c.startswith(PREF)] + [c for c in RANCIAS if c in cols]
    quedan = [c for c in cols if c not in set(fuera)]
    # el n_awards que ya traia el archivo debe coincidir con el del JSON
    desc = con.sql("""SELECT count(*) FILTER (WHERE s.n_awards <> a.n_awards) AS no_coinciden
                      FROM s JOIN a ON a.ocid = s.ocid""").fetchone()[0] if 'n_awards' in cols else None
    sel = ', '.join(('coalesce(a.suma_awards, s.monto_final) AS monto_final' if c == 'monto_final'
                     else f's."{c}"') for c in quedan)
    sel += (', s.monto_final AS monto_adjudicacion_unica_v1'
            ', coalesce(a.n_awards > 1, false) AS es_multilote')
    con.sql(f"""COPY (SELECT {sel} FROM s LEFT JOIN a ON a.ocid = s.ocid)
                TO '{pout}' (FORMAT PARQUET, COMPRESSION ZSTD)""")
    n = con.sql(f"SELECT count(*) FROM read_parquet('{pout}')").fetchone()[0]
    nc = len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{pout}')").df())
    print(f'  {src} -> {dst}: {n:,} filas, {len(cols)} -> {nc} columnas, '
          f'{round(os.path.getsize(pout)/1e6,1)} MB'
          + (f' | n_awards que no coinciden con el JSON: {desc}' if desc is not None else ''))
    os.replace(pin, f'{HIST}/{src}.parquet')

t('(2) CUADRE DE LA PARTICION Y DE LA PLATA, CON TODO CORREGIDO')
V = {'p1': 'planA_P1_v3_modelo', 'p0': 'planA_P0_catalogo_v3', 'uni': 'planA_universo_v3',
     'ex1': 'planA_excluidos_solo_planeacion_v3', 'ex2': 'planA_excluidos_sin_monto_v3',
     'ex3': 'planA_sin_proveedor_identificable_v3'}
for k, f in V.items():
    con.sql(f"CREATE OR REPLACE VIEW {k} AS SELECT * FROM read_parquet('plan_A/{f}.parquet')")
C = con.sql("""SELECT 'P1 modelo' AS estrato, count(*) AS procesos,
     round(coalesce(sum(monto_final),0)/1e6,1) AS millones,
     round(coalesce(sum(monto_adjudicacion_unica_v1),0)/1e6,1) AS millones_antes FROM p1
  UNION ALL SELECT 'P0 catalogo', count(*), round(coalesce(sum(monto_final),0)/1e6,1),
     round(coalesce(sum(monto_adjudicacion_unica_v1),0)/1e6,1) FROM p0
  UNION ALL SELECT 'excl: solo planeacion', count(*), round(coalesce(sum(monto_final),0)/1e6,1),
     round(coalesce(sum(monto_adjudicacion_unica_v1),0)/1e6,1) FROM ex1
  UNION ALL SELECT 'excl: sin monto', count(*), round(coalesce(sum(monto_final),0)/1e6,1),
     round(coalesce(sum(monto_adjudicacion_unica_v1),0)/1e6,1) FROM ex2
  UNION ALL SELECT 'excl: sin proveedor', count(*), round(coalesce(sum(monto_final),0)/1e6,1),
     round(coalesce(sum(monto_adjudicacion_unica_v1),0)/1e6,1) FROM ex3
  ORDER BY millones DESC""").df()
tot = con.sql("SELECT count(*), round(coalesce(sum(monto_final),0)/1e6,1) FROM uni").fetchone()
C.loc[len(C)] = ['SUMA DE LAS PARTES', C.procesos.sum(), round(C.millones.sum(), 1),
                 round(C.millones_antes.sum(), 1)]
C.loc[len(C)] = ['universo_v3', tot[0], tot[1], None]
C['pct_plata'] = (100 * C.millones / tot[1]).round(2)
print(C.to_string(index=False))
part = con.sql("""WITH u AS (SELECT ocid FROM p1 UNION ALL SELECT ocid FROM p0 UNION ALL
                             SELECT ocid FROM ex1 UNION ALL SELECT ocid FROM ex2 UNION ALL
                             SELECT ocid FROM ex3)
                  SELECT count(*), count(DISTINCT ocid) FROM u""").fetchone()
fuera_u = con.sql("""WITH u AS (SELECT ocid FROM p1 UNION ALL SELECT ocid FROM p0 UNION ALL
                                SELECT ocid FROM ex1 UNION ALL SELECT ocid FROM ex2 UNION ALL
                                SELECT ocid FROM ex3)
                     SELECT count(*) FROM uni ANTI JOIN u USING (ocid)""").fetchone()[0]
print(f'\nparticion: {part[0]:,} filas, {part[1]:,} ocid distintos, universo {tot[0]:,} | '
      f'{"CUADRA" if part[0]==tot[0]==part[1] else "NO CUADRA"} | huerfanos: {fuera_u}')
print(f'diferencia suma de partes vs universo: {round(C.millones.iloc[-2]-tot[1],1)} MM')

t('(3) CONTROL FINAL: NINGUNA COLUMNA DE RED EN NINGUN ARCHIVO')
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        cc = list(con.sql(f"DESCRIBE SELECT * FROM read_parquet('plan_A/{f}')").df().column_name)
        mal = [c for c in cc if c.startswith(PREF)] + [c for c in RANCIAS if c in cc]
        print(f'  {f:48s} {len(cc):>4} columnas  {"OK" if not mal else "PROHIBIDAS: " + str(mal)}')

t('(4) MANIFIESTO FINAL')
M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        M.append({'archivo': f, 'filas': con.sql(f"SELECT count(*) FROM read_parquet('{p}')").fetchone()[0],
                  'columnas': len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()),
                  'MB': round(os.path.getsize(p)/1e6,1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print(MAN.to_string(index=False)); print(f'\n{len(MAN)} archivos | {round(MAN.MB.sum(),1)} MB')

t('(5) PAQUETE DE ENVIO PARA LOS COMPAÑEROS')
ENV = 'plan_A/_envio_companeros'
if os.path.exists(ENV): shutil.rmtree(ENV)
os.makedirs(ENV)
NUCLEO = ['planA_P1_v3_modelo.parquet', 'planA_indicadores_v4_3.parquet',
          'planA_dim_proveedor.parquet', 'planA_dim_entidad.parquet',
          'planA_validacion_sri_verificada.parquet']
for f in NUCLEO:
    if os.path.exists(f'plan_A/{f}'): shutil.copy2(f'plan_A/{f}', f'{ENV}/{f}')
    else: print(f'  FALTA {f}')
shutil.copytree(DOC, f'{ENV}/documentacion')
tam = sum(os.path.getsize(os.path.join(r, x)) for r, _, fs in os.walk(ENV) for x in fs)
print(f'  {ENV}  |  {round(tam/1e6,1)} MB')
for r, _, fs in os.walk(ENV):
    for x in sorted(fs): print('   ', os.path.relpath(os.path.join(r, x), ENV))
print('\nEsa carpeta es la que se comprime y se envia. Pega aqui toda la salida.')

In [ ]:
# ============ CELDA 33 · PERFIL FINAL, DOCUMENTACION REGENERADA Y PAQUETE ============
# Perfila los 10 archivos definitivos, regenera README y diccionario DESDE LOS DATOS,
# archiva la documentacion vieja y rearma el paquete de envio. Tarda unos 5 a 8 minutos.
import os, shutil, hashlib, json
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, VIEJO = 'plan_A/documentacion', 'plan_A/documentacion/_perfiles_de_versiones_anteriores'
os.makedirs(VIEJO, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80); pd.set_option('display.max_rows', 300)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]
def anidado(ty): return any(s in ty.upper() for s in ('[', 'STRUCT', 'MAP', 'UNION'))
def es_texto(ty): return ty.upper() == 'VARCHAR'
def es_num(ty): return any(k in ty.upper() for k in ('TINYINT','SMALLINT','INTEGER','BIGINT',
                           'HUGEINT','DOUBLE','FLOAT','REAL','DECIMAL'))
def es_fecha(ty): return any(k in ty.upper() for k in ('DATE','TIMESTAMP'))

ARCH = [('planA_P1_v3_modelo',                   'proceso modelable',         'A nucleo'),
        ('planA_indicadores_v4_3',               'proceso modelable',         'A nucleo'),
        ('planA_dim_proveedor',                  'proveedor (RUC)',           'A nucleo'),
        ('planA_dim_entidad',                    'entidad (RUC)',             'A nucleo'),
        ('planA_universo_v3',                    'release OCDS',              'B auditoria'),
        ('planA_P0_catalogo_v3',                 'proceso de catalogo',       'B descriptivo'),
        ('planA_excluidos_solo_planeacion_v3',   'proceso excluido',          'C evidencia'),
        ('planA_excluidos_sin_monto_v3',         'proceso excluido',          'C evidencia'),
        ('planA_sin_proveedor_identificable_v3', 'proceso excluido',          'C evidencia'),
        ('planA_validacion_sri_verificada',      'proveedor marcado por SRI', 'C validacion')]
RUTA = {n: f'plan_A/{n}.parquet' for n, _, _ in ARCH}
ARCH = [x for x in ARCH if os.path.exists(RUTA[x[0]])]

# ---------- (0) ARCHIVAR LA DOCUMENTACION VIEJA ----------
t('(0) ARCHIVAR LOS PERFILES DE VERSIONES ANTERIORES')
vivos = {f'perfil_nulos_{n}.csv' for n, _, _ in ARCH}
for f in sorted(os.listdir(DOC)):
    if not os.path.isfile(f'{DOC}/{f}'): continue
    viejo = (f.startswith('perfil_nulos_') and f not in vivos) or f.startswith('cifras_celda2') \
            or f in ('perfil_inventario.csv', 'control_rangos_v3_1.csv',
                     'perfil_distribucion_indicadores.csv', 'perfil_monto_por_familia.csv',
                     'perfil_fechas.csv', 'perfil_por_anio.csv')
    if viejo:
        shutil.move(f'{DOC}/{f}', f'{VIEJO}/{f}'); print(f'  archivado: {f}')

# ---------- (1) INVENTARIO Y LLAVE ----------
t('(1) INVENTARIO: grano, llave y huella')
CAND = ['ocid', 'ruc']
inv, COLS, LLAVE = [], {}, {}
for n, grano, caja in ARCH:
    esq = list(zip(*[con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RUTA[n]}')").df()[c]
                     for c in ('column_name', 'column_type')]))
    COLS[n] = esq; cols = [c for c, _ in esq]
    k = next((c for c in CAND if c in cols), None); LLAVE[n] = k
    filas = con.sql(f"SELECT count(*) FROM read_parquet('{RUTA[n]}')").fetchone()[0]
    nd, nul = con.sql(f"""SELECT count(DISTINCT "{k}"), count(*) - count("{k}")
                          FROM read_parquet('{RUTA[n]}')""").fetchone() if k else (None, None)
    inv.append({'archivo': n, 'caja': caja, 'grano': grano, 'filas': filas, 'columnas': len(cols),
                'MB': round(os.path.getsize(RUTA[n])/1e6, 1), 'llave': k or '-',
                'llave_unica': (nd == filas and nul == 0) if k else None, 'sha256_16': sha16(RUTA[n])})
INV = pd.DataFrame(inv); print(INV.to_string(index=False))
INV.to_csv(f'{DOC}/perfil_inventario.csv', index=False, encoding='utf-8-sig')

# ---------- (2) NULOS COLUMNA POR COLUMNA ----------
t('(2) COMPLETITUD POR ARCHIVO')
def perfil(n):
    esq, filas = COLS[n], int(INV.loc[INV.archivo == n, 'filas'].iloc[0])
    out = []
    for i in range(0, len(esq), 35):
        tr = esq[i:i+35]; sel = []
        for c, ty in tr:
            sel.append(f'count("{c}") AS "nn_{i}_{c}"')
            if es_texto(ty): sel.append(f"""count(*) FILTER (WHERE trim("{c}")='') AS "vc_{i}_{c}" """)
            if not anidado(ty): sel.append(f'approx_count_distinct("{c}") AS "nd_{i}_{c}"')
        r = con.sql(f"SELECT {', '.join(sel)} FROM read_parquet('{RUTA[n]}')").df().iloc[0]
        for c, ty in tr:
            nn = int(r[f'nn_{i}_{c}'])
            vc = int(r[f'vc_{i}_{c}']) if f'vc_{i}_{c}' in r.index else 0
            nd = int(r[f'nd_{i}_{c}']) if f'nd_{i}_{c}' in r.index else -1
            u = nn - vc
            ver = ('INSERVIBLE' if u == 0 else 'CONSTANTE' if nd == 1 else
                   'casi vacia' if u/filas < 0.05 else 'mitad vacia' if u/filas < 0.5 else
                   'completa' if u == filas else 'con faltantes')
            out.append({'archivo': n, 'columna': c, 'tipo': ty, 'con_dato': nn, 'vacio_texto': vc,
                        'pct_con_dato': round(100*u/filas, 2),
                        'distintos_aprox': None if nd < 0 else nd, 'veredicto': ver})
    return pd.DataFrame(out)

DIC, RES = [], {}
for n, _, _ in ARCH:
    P = perfil(n); RES[n] = P; DIC.append(P)
    P.drop(columns=['archivo']).to_csv(f'{DOC}/perfil_nulos_{n}.csv', index=False, encoding='utf-8-sig')
    v = P.veredicto
    print(f'{n:40s} {len(P):>4} col | completas {int((v=="completa").sum()):>3} | '
          f'faltantes {int((v=="con faltantes").sum()):>3} | mitad {int((v=="mitad vacia").sum()):>3} | '
          f'casi vacias {int((v=="casi vacia").sum()):>3} | constantes {int((v=="CONSTANTE").sum()):>3} | '
          f'inservibles {int((v=="INSERVIBLE").sum()):>3}')
DICC = pd.concat(DIC, ignore_index=True)
DICC.to_csv(f'{DOC}/diccionario_columnas.csv', index=False, encoding='utf-8-sig')
print(f'\ndiccionario_columnas.csv regenerado: {len(DICC):,} filas (una por columna de cada archivo)')

t('(2b) LA TABLA DE INDICADORES, COLUMNA POR COLUMNA')
print(RES['planA_indicadores_v4_3'].sort_values('pct_con_dato')[
      ['columna','tipo','pct_con_dato','distintos_aprox','veredicto']].to_string(index=False))

# ---------- (3) DISTRIBUCIONES FINALES ----------
t('(3) DISTRIBUCION DE LOS INDICADORES (base definitiva)')
con.sql(f"CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('{RUTA['planA_indicadores_v4_3']}')")
con.sql(f"CREATE OR REPLACE VIEW p1  AS SELECT * FROM read_parquet('{RUTA['planA_P1_v3_modelo']}')")
num = [c for c, ty in COLS['planA_indicadores_v4_3'] if es_num(ty)]
pz = [f"""SELECT '{c}' AS variable, count("{c}") AS con_dato,
      round(100.0*count("{c}")/count(*),1) AS pct, min("{c}") AS minimo,
      quantile_cont("{c}",0.25) AS p25, median("{c}") AS mediana, quantile_cont("{c}",0.75) AS p75,
      quantile_cont("{c}",0.95) AS p95, quantile_cont("{c}",0.99) AS p99, max("{c}") AS maximo
      FROM ind""" for c in num]
D = con.sql('\nUNION ALL\n'.join(pz)).df()
for c in ['minimo','p25','mediana','p75','p95','p99','maximo']:
    D[c] = pd.to_numeric(D[c], errors='coerce').round(4)
print(D.to_string(index=False))
D.to_csv(f'{DOC}/perfil_distribucion_indicadores.csv', index=False, encoding='utf-8-sig')

t('(3b) MONTO POR FAMILIA Y POR AÑO (base definitiva)')
MF = con.sql("""SELECT familia_procedimiento AS familia, count(*) AS procesos,
     count(*) FILTER (WHERE es_multilote) AS multilote,
     round(median(monto_final),0) AS mediana_usd, round(quantile_cont(monto_final,0.75),0) AS p75_usd,
     round(quantile_cont(monto_final,0.95),0) AS p95_usd, round(max(monto_final),0) AS maximo_usd,
     round(sum(monto_final)/1e6,1) AS suma_millones
   FROM p1 GROUP BY 1 ORDER BY procesos DESC""").df()
print(MF.to_string(index=False)); MF.to_csv(f'{DOC}/perfil_monto_por_familia.csv', index=False, encoding='utf-8-sig')
AN = con.sql("""SELECT anio_proceso AS anio, count(*) AS procesos,
     count(DISTINCT proveedor_ruc_final) AS proveedores, count(DISTINCT entidad_ruc) AS entidades,
     round(sum(monto_final)/1e6,1) AS millones, round(median(monto_final),0) AS mediana_usd
   FROM p1 GROUP BY 1 ORDER BY 1""").df()
print('\n' + AN.to_string(index=False)); AN.to_csv(f'{DOC}/perfil_por_anio.csv', index=False, encoding='utf-8-sig')

# ---------- (4) README DESDE LOS DATOS ----------
t('(4) REGENERAR README_base_plan_A.md')
nuc = INV[INV.caja.str.startswith('A')]
tot_mm = con.sql(f"""SELECT round(sum(monto_final)/1e6,1) FROM
                     read_parquet('{RUTA['planA_universo_v3']}')""").fetchone()[0]
p1_mm = float(MF.suma_millones.sum())
L = []
A = L.append
A('# Base Plan A - Deteccion de anomalias en compras publicas del Ecuador (SERCOP, OCDS)')
A('')
A(f'Generado desde los datos el {pd.Timestamp.today().strftime("%Y-%m-%d %H:%M")}. '
  'No editar a mano: lo regenera la celda 33.')
A('')
A(f'- Corte de descarga de los datos: 2026-09-05')
A(f'- Universo: {int(INV.loc[INV.archivo=="planA_universo_v3","filas"].iloc[0]):,} releases OCDS, 2015-2026')
A(f'- Valor contratado del universo: {tot_mm} millones USD')
A(f'- Base de modelado: {int(nuc.loc[nuc.archivo=="planA_P1_v3_modelo","filas"].iloc[0]):,} '
  f'procesos, {p1_mm} millones USD')
A('')
A('## Archivos')
A('')
A('```')
A(INV[['archivo','caja','grano','filas','columnas','MB','llave','llave_unica','sha256_16']].to_string(index=False))
A('```')
A('')
A('## Como se unen')
A('')
A('- `planA_P1_v3_modelo` <-- ocid --> `planA_indicadores_v4_3`  (1 a 1, verificado)')
A('- `planA_P1_v3_modelo.proveedor_ruc_final` --> `planA_dim_proveedor.ruc`')
A('- `planA_P1_v3_modelo.entidad_ruc` --> `planA_dim_entidad.ruc`')
A('')
A('## Reglas de uso')
A('')
A('1. `monto_final` es la suma de las adjudicaciones del proceso. Nunca usar `monto_tender`.')
A('2. `planA_validacion_sri_verificada` no entra como variable en ningun modelo. Es validacion')
A('   externa, vive en archivo aparte para que no pueda filtrarse a una tabla de features.')
A('3. Prohibido usar columnas de estado actual del SRI o de la Superintendencia para explicar')
A('   procesos pasados: eso es anacronismo y rompe el punto en el tiempo.')
A('4. Las 56 columnas de red (`red_`, `cob_`, `com_`, `rep_`, `grupo_`, `idx_`) fueron eliminadas:')
A('   corresponden al item #48 del catalogo, que es de otro grupo.')
A('5. Los nueve `pct_` son rangos percentiles calculados SOLO sobre filas con dato. La ausencia la')
A('   carga la indicadora `falta_*` correspondiente. `pct_poca_concurrencia` satura en 0.8315 por')
A('   los empates de los procesos con un solo oferente.')
A('6. `duracion_dias_imputada` es solo para estadistica descriptiva y lleva su bandera')
A('   `duracion_imputada`. Para las tablas de la tesis usar `ind_duracion_contrato_dias`')
A('   declarando su cobertura.')
A('7. `es_multilote` y `n_awards` son variables de estratificacion, no de anomalia.')
A('8. Toda cifra con cola larga se reporta con mediana, nunca con promedio.')
A('')
A('## Definiciones verificadas por reproduccion')
A('')
DEFS = json.load(open(f'{DOC}/especificacion_indicadores.json', encoding='utf-8'))
for k, v in DEFS.get('definiciones_verificadas', {}).items():
    A(f'- `{k}`: {v}')
A('')
A('- Grupo comparable: familia x periodo normativo, con descenso a familia y luego a global')
A('  cuando el grupo tiene menos de 30 casos con dato.')
A('- Ventana de historial: 365 dias anteriores, sin incluir el dia del proceso.')
A('- Ventana de fraccionamiento: 30 dias anteriores, mismo par entidad-proveedor y mismo CPC5.')
A('')
A('## Completitud por archivo')
A('')
for n, _, _ in ARCH:
    v = RES[n].veredicto
    A(f'- **{n}**: {len(RES[n])} columnas | {int((v=="completa").sum())} completas | '
      f'{int((v=="con faltantes").sum())} con faltantes | {int((v=="CONSTANTE").sum())} constantes | '
      f'{int((v=="INSERVIBLE").sum())} sin un solo dato')
A('')
A('El detalle columna por columna esta en `diccionario_columnas.csv` y en los')
A('`perfil_nulos_<archivo>.csv`. Las enmiendas de version, con fecha, motivo y huella, estan en')
A('`especificacion_indicadores.json`. Las versiones anteriores de cada archivo se conservan en')
A('`_version_1_historico\\`: no se borra nada.')
open(f'{DOC}/README_base_plan_A.md', 'w', encoding='utf-8').write('\n'.join(L))
print(f'  README regenerado: {len(L)} lineas')

# ---------- (5) REARMAR EL PAQUETE ----------
t('(5) PAQUETE DE ENVIO')
ENV = 'plan_A/_envio_companeros'
if os.path.exists(ENV): shutil.rmtree(ENV)
os.makedirs(ENV)
for f in ['planA_P1_v3_modelo.parquet', 'planA_indicadores_v4_3.parquet',
          'planA_dim_proveedor.parquet', 'planA_dim_entidad.parquet',
          'planA_validacion_sri_verificada.parquet']:
    shutil.copy2(f'plan_A/{f}', f'{ENV}/{f}')
shutil.copytree(DOC, f'{ENV}/documentacion',
                ignore=shutil.ignore_patterns('_perfiles_de_versiones_anteriores'))
tam = sum(os.path.getsize(os.path.join(r, x)) for r, _, fs in os.walk(ENV) for x in fs)
print(f'  {ENV}  |  {round(tam/1e6,1)} MB')
for r, _, fs in os.walk(ENV):
    for x in sorted(fs): print('   ', os.path.relpath(os.path.join(r, x), ENV))
print('\nListo para comprimir. Pega aqui toda la salida.')

In [ ]:
# ============ CELDA 34 · ESTADISTICAS PARA LA TESIS (1 de 2) ============
# (0) regenera el INVENTARIO que quedo viejo
# (1) concentracion de mercado: Herfindahl por entidad, top proveedores, curva de concentracion
# (2) estructura de competencia: oferentes por modalidad y por anio, oferente unico
# (3) ley de Benford del primer digito, con chi-cuadrado y MAD de Nigrini
import os, math
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC = 'plan_A/documentacion'; EST = f'{DOC}/estadisticas'
os.makedirs(EST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 120)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def guardar(df, nombre):
    df.to_csv(f'{EST}/{nombre}.csv', index=False, encoding='utf-8-sig'); return df

con.sql("CREATE OR REPLACE VIEW p1  AS SELECT * FROM read_parquet('plan_A/planA_P1_v3_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores_v4_3.parquet')")
con.sql("CREATE OR REPLACE VIEW dpr AS SELECT * FROM read_parquet('plan_A/planA_dim_proveedor.parquet')")
con.sql("CREATE OR REPLACE VIEW den AS SELECT * FROM read_parquet('plan_A/planA_dim_entidad.parquet')")
cdpr = list(con.sql("DESCRIBE SELECT * FROM dpr").df().column_name)
cden = list(con.sql("DESCRIBE SELECT * FROM den").df().column_name)
cp1  = list(con.sql("DESCRIBE SELECT * FROM p1").df().column_name)
nom_pr = next((c for c in cdpr if 'razon' in c.lower()), 'ruc')
nom_en = next((c for c in cden if 'razon' in c.lower()), 'ruc')
print('nombre del proveedor:', nom_pr, '| nombre de la entidad:', nom_en)
print('columnas de provincia disponibles (para la celda 35):')
print('  en P1 :', [c for c in cp1  if 'provincia' in c.lower()])
print('  en dpr:', [c for c in cdpr if 'provincia' in c.lower()])
print('  en den:', [c for c in cden if 'provincia' in c.lower()])

# ---------- (0) INVENTARIO REGENERADO ----------
t('(0) INVENTARIO_ENTREGABLES.md REGENERADO')
INV = pd.read_csv(f'{DOC}/perfil_inventario.csv')
L = ['# Inventario de entregables - base Plan A', '',
     f'Generado desde los datos el {pd.Timestamp.today().strftime("%Y-%m-%d %H:%M")}.',
     f'{len(INV)} archivos, {round(INV.MB.sum(),1)} MB.', '',
     '```', INV.to_string(index=False), '```', '',
     '## Que enviar', '',
     '- **Caja A (nucleo de modelado)**: planA_P1_v3_modelo, planA_indicadores_v4_3,',
     '  planA_dim_proveedor, planA_dim_entidad. Es lo que se necesita para trabajar.',
     '- **Caja B (auditoria)**: planA_universo_v3 y planA_P0_catalogo_v3. No hacen falta para',
     '  modelar; sirven para rastrear cualquier cifra hasta el universo completo.',
     '- **Caja C (evidencia)**: los tres archivos de exclusion y el conjunto de validacion del SRI.',
     '  Documentan que nada se descarto en silencio.', '',
     'Las reglas de uso, las definiciones verificadas y la completitud de cada archivo estan en',
     '`README_base_plan_A.md`. El detalle columna por columna, en `diccionario_columnas.csv`.']
open(f'{DOC}/INVENTARIO_ENTREGABLES.md', 'w', encoding='utf-8').write('\n'.join(L))
print(f'  regenerado: {len(L)} lineas')

# ---------- (1) CONCENTRACION DE MERCADO ----------
t('(1a) INDICE DE HERFINDAHL POR ENTIDAD  (concentracion de proveedores en cada entidad)')
con.sql("""CREATE OR REPLACE TABLE hhi AS
  WITH s AS (SELECT entidad_ruc, proveedor_ruc_final, sum(monto_final) AS m, count(*) AS n
             FROM p1 GROUP BY 1,2),
       e AS (SELECT entidad_ruc, sum(m) AS total, count(*) AS proveedores, sum(n) AS procesos
             FROM s GROUP BY 1)
  SELECT e.entidad_ruc, e.proveedores, e.procesos, e.total,
         sum(power(s.m / nullif(e.total,0), 2)) AS hhi,
         max(s.m / nullif(e.total,0))           AS cuota_del_mayor
  FROM s JOIN e USING (entidad_ruc) GROUP BY 1,2,3,4""")
print(guardar(con.sql("""SELECT
     CASE WHEN hhi >= 0.9 THEN 'f) 0.90 a 1.00  practicamente un solo proveedor'
          WHEN hhi >= 0.5 THEN 'e) 0.50 a 0.90  muy concentrada'
          WHEN hhi >= 0.25 THEN 'd) 0.25 a 0.50  concentrada'
          WHEN hhi >= 0.15 THEN 'c) 0.15 a 0.25  moderada'
          WHEN hhi >= 0.05 THEN 'b) 0.05 a 0.15  poco concentrada'
          ELSE 'a) menos de 0.05  dispersa' END AS tramo_hhi,
     count(*) AS entidades, sum(procesos) AS procesos,
     round(sum(total)/1e6,1) AS millones,
     round(100.0*sum(total)/(SELECT sum(total) FROM hhi),2) AS pct_plata,
     round(median(proveedores),0) AS mediana_proveedores
   FROM hhi GROUP BY 1 ORDER BY 1""").df(), 'concentracion_hhi_por_entidad').to_string(index=False))
print('\nEl HHI va de 0 a 1: 1 significa que una sola empresa se lleva todo el monto de esa entidad.')
print(con.sql("""SELECT count(*) AS entidades, round(median(hhi),4) AS hhi_mediana,
     round(quantile_cont(hhi,0.25),4) AS p25, round(quantile_cont(hhi,0.75),4) AS p75,
     count(*) FILTER (WHERE hhi > 0.9999) AS con_un_solo_proveedor,
     round(median(cuota_del_mayor),4) AS cuota_mediana_del_mayor,
     round(median(proveedores),0) AS mediana_proveedores_por_entidad
   FROM hhi""").df().to_string(index=False))

t('(1b) LOS 20 PROVEEDORES CON MAS MONTO ADJUDICADO EN TODO EL PERIODO')
print(guardar(con.sql(f"""SELECT p1.proveedor_ruc_final AS ruc, any_value(dpr."{nom_pr}") AS proveedor,
     count(*) AS procesos, count(DISTINCT p1.entidad_ruc) AS entidades,
     round(sum(p1.monto_final)/1e6,1) AS millones,
     round(100.0*sum(p1.monto_final)/(SELECT sum(monto_final) FROM p1),2) AS pct_del_total,
     round(median(p1.monto_final),0) AS mediana_usd
   FROM p1 LEFT JOIN dpr ON dpr.ruc = p1.proveedor_ruc_final
   GROUP BY 1 ORDER BY millones DESC LIMIT 20""").df(),
   'top20_proveedores').to_string(index=False))

t('(1c) CURVA DE CONCENTRACION: cuanta plata se lleva el uno por ciento de proveedores')
con.sql("""CREATE OR REPLACE TABLE curva AS
  WITH s AS (SELECT proveedor_ruc_final AS ruc, sum(monto_final) AS m FROM p1 GROUP BY 1),
       r AS (SELECT ruc, m, row_number() OVER (ORDER BY m DESC) AS puesto,
                    count(*) OVER () AS n, sum(m) OVER () AS total,
                    sum(m) OVER (ORDER BY m DESC ROWS UNBOUNDED PRECEDING) AS acumulado
             FROM s)
  SELECT * FROM r""")
print(guardar(con.sql("""SELECT corte AS top_pct_de_proveedores,
     (SELECT max(puesto) FROM curva WHERE puesto <= n*corte/100.0) AS proveedores,
     (SELECT round(100.0*max(acumulado)/max(total),2) FROM curva WHERE puesto <= n*corte/100.0)
       AS pct_de_la_plata
   FROM (SELECT unnest([0.1, 0.5, 1, 5, 10, 25, 50]) AS corte)
   ORDER BY corte""").df(), 'curva_concentracion_proveedores').to_string(index=False))
print('\nLo mismo por el lado de las entidades:')
print(con.sql("""WITH s AS (SELECT entidad_ruc, sum(monto_final) AS m FROM p1 GROUP BY 1),
       r AS (SELECT m, row_number() OVER (ORDER BY m DESC) AS puesto, count(*) OVER () AS n,
                    sum(m) OVER () AS total,
                    sum(m) OVER (ORDER BY m DESC ROWS UNBOUNDED PRECEDING) AS acum FROM s)
  SELECT 10 AS top_pct_de_entidades,
     (SELECT round(100.0*max(acum)/max(total),2) FROM r WHERE puesto <= n*0.10) AS pct_de_la_plata
  UNION ALL SELECT 1, (SELECT round(100.0*max(acum)/max(total),2) FROM r WHERE puesto <= n*0.01)
  ORDER BY 1""").df().to_string(index=False))

# ---------- (2) ESTRUCTURA DE COMPETENCIA ----------
t('(2a) OFERENTES POR MODALIDAD')
print(guardar(con.sql("""SELECT familia, count(*) AS procesos,
     round(median(ind_n_oferentes),0) AS mediana_oferentes,
     round(quantile_cont(ind_n_oferentes,0.75),0) AS p75,
     round(quantile_cont(ind_n_oferentes,0.95),0) AS p95, max(ind_n_oferentes) AS maximo,
     count(*) FILTER (WHERE ind_oferente_unico) AS con_un_solo_oferente,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico,
     round(100.0*sum(CASE WHEN ind_oferente_unico THEN monto_final ELSE 0 END)
           /sum(monto_final),2) AS pct_plata_con_un_oferente
   FROM ind GROUP BY 1 ORDER BY procesos DESC""").df(),
   'competencia_por_familia').to_string(index=False))

t('(2b) EVOLUCION DEL OFERENTE UNICO POR AÑO')
print(guardar(con.sql("""SELECT anio, count(*) AS procesos,
     round(median(ind_n_oferentes),0) AS mediana_oferentes,
     round(100.0*count(*) FILTER (WHERE ind_oferente_unico)/count(*),2) AS pct_oferente_unico,
     round(100.0*sum(CASE WHEN ind_oferente_unico THEN monto_final ELSE 0 END)
           /sum(monto_final),2) AS pct_plata_con_un_oferente
   FROM ind GROUP BY 1 ORDER BY 1""").df(), 'competencia_por_anio').to_string(index=False))

# ---------- (3) LEY DE BENFORD ----------
t('(3) LEY DE BENFORD DEL PRIMER DIGITO DEL MONTO ADJUDICADO')
B = con.sql("""SELECT CAST(floor(monto_final / pow(10, floor(log10(monto_final)))) AS INTEGER) AS digito,
     count(*) AS observado
   FROM p1 WHERE monto_final > 0 GROUP BY 1 ORDER BY 1""").df()
B = B[B.digito.between(1, 9)].copy()
n = B.observado.sum()
B['esperado_pct'] = [100*math.log10(1 + 1/d) for d in B.digito]
B['observado_pct'] = 100*B.observado/n
B['esperado'] = (B.esperado_pct/100*n).round(0)
B['diferencia_pp'] = (B.observado_pct - B.esperado_pct).round(3)
chi2 = float((((B.observado - B.esperado)**2)/B.esperado).sum())
mad = float((B.observado_pct/100 - B.esperado_pct/100).abs().mean())
print(guardar(B[['digito','observado','esperado','observado_pct','esperado_pct','diferencia_pp']]
              .round(3), 'benford_primer_digito').to_string(index=False))
print(f'\nprocesos analizados: {n:,}')
print(f'chi-cuadrado (8 grados de libertad): {chi2:,.1f}   valor critico al 5%: 15.507')
print(f'MAD de Nigrini: {mad:.5f}')
print('  interpretacion del MAD: menos de 0.006 conformidad estrecha | 0.006 a 0.012 aceptable |')
print('  0.012 a 0.015 marginal | mas de 0.015 no conformidad')
print(f'  -> {"conformidad estrecha" if mad<0.006 else "aceptable" if mad<0.012 else "marginal" if mad<0.015 else "NO CONFORMIDAD"}')
print('\nOJO al interpretar: con 446.677 casos el chi-cuadrado rechaza casi cualquier desvio, por eso')
print('se reporta tambien el MAD, que no depende del tamano de la muestra.')

print('\n-- Benford por modalidad --')
BF = con.sql("""SELECT familia_procedimiento AS familia,
     CAST(floor(monto_final / pow(10, floor(log10(monto_final)))) AS INTEGER) AS digito,
     count(*) AS observado FROM p1 WHERE monto_final > 0 GROUP BY 1,2""").df()
BF = BF[BF.digito.between(1, 9)]
filas = []
for fam, g in BF.groupby('familia'):
    tot = g.observado.sum()
    esp = pd.Series({d: tot*math.log10(1+1/d) for d in range(1, 10)})
    obs = g.set_index('digito').observado.reindex(range(1, 10), fill_value=0)
    filas.append({'familia': fam, 'procesos': int(tot),
                  'chi2': round(float((((obs-esp)**2)/esp).sum()), 1),
                  'MAD': round(float((obs/tot - esp/tot).abs().mean()), 5),
                  'pct_digito_1': round(100*obs[1]/tot, 2),
                  'esperado_digito_1': 30.10})
BT = pd.DataFrame(filas).sort_values('MAD')
BT['veredicto'] = pd.cut(BT.MAD, [0, .006, .012, .015, 9],
                         labels=['conformidad estrecha','aceptable','marginal','no conformidad'])
print(guardar(BT, 'benford_por_familia').to_string(index=False))
print(f'\nCSV guardados en {EST}')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 35 · QUE FECHA ES LA FECHA DEL PROCESO (diagnostico) ============
# No escribe nada. Tres preguntas:
#  (1) que columna de origen es fecha_proceso
#  (2) contra el año del codigo del proceso, que campo acierta mas
#  (3) cuanto se retrasa la publicacion del release respecto del proceso
import os
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 200)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)

con.sql("CREATE OR REPLACE VIEW p1  AS SELECT * FROM read_parquet('plan_A/planA_P1_v3_modelo.parquet')")
con.sql("CREATE OR REPLACE VIEW ind AS SELECT * FROM read_parquet('plan_A/planA_indicadores_v4_3.parquet')")
con.sql("CREATE OR REPLACE VIEW jj  AS SELECT * FROM read_parquet('03_resultados/c15_complemento_json.parquet')")

def fechas_de(vista):
    d = con.sql(f"DESCRIBE SELECT * FROM {vista}").df()
    return [c for c, ty in zip(d.column_name, d.column_type)
            if any(k in ty.upper() for k in ('DATE', 'TIMESTAMP'))]
FP1, FJJ = fechas_de('p1'), fechas_de('jj')
print('columnas de fecha en P1 :', FP1)
print('columnas de fecha en JSON:', FJJ)

# ---------- (1) QUE ES fecha_proceso ----------
t('(1) CONTRA QUE COLUMNA COINCIDE fecha_proceso  (dia exacto)')
pz = []
for c in FP1:
    pz.append(f"""SELECT 'P1.{c}' AS candidata,
      count(*) FILTER (WHERE ind.fecha_proceso = CAST(p1."{c}" AS DATE)) AS coincide,
      count(p1."{c}") AS candidata_con_dato
      FROM ind JOIN p1 USING (ocid)""")
for c in FJJ:
    pz.append(f"""SELECT 'JSON.{c}',
      count(*) FILTER (WHERE ind.fecha_proceso = CAST(jj."{c}" AS DATE)),
      count(jj."{c}")
      FROM ind LEFT JOIN jj USING (ocid)""")
M = con.sql('\nUNION ALL\n'.join(pz)).df()
M['pct_de_446677'] = (100 * M.coincide / 446677).round(2)
print(M.sort_values('coincide', ascending=False).to_string(index=False))

# ---------- (2) EL AÑO DEL CODIGO DEL PROCESO COMO ANCLA ----------
t('(2a) EXTRAER EL AÑO DEL CODIGO DEL PROCESO')
con.sql("""CREATE OR REPLACE TABLE cod AS
  SELECT ocid,
    list_distinct(list_filter(regexp_extract_all(ocid, '20[0-2][0-9]'),
                              x -> CAST(x AS INTEGER) BETWEEN 2014 AND 2027)) AS candidatos
  FROM p1""")
con.sql("""CREATE OR REPLACE TABLE codigo AS
  SELECT ocid, len(candidatos) AS n_candidatos,
         CASE WHEN len(candidatos) = 1 THEN CAST(candidatos[1] AS INTEGER) END AS anio_codigo
  FROM cod""")
print(con.sql("""SELECT n_candidatos, count(*) AS procesos,
                   round(100.0*count(*)/sum(count(*)) OVER (),2) AS pct
                 FROM codigo GROUP BY 1 ORDER BY 1""").df().to_string(index=False))
usable = con.sql("SELECT count(*) FROM codigo WHERE anio_codigo IS NOT NULL").fetchone()[0]
print(f'\nprocesos con un unico año identificable en el codigo: {usable:,} '
      f'({100*usable/446677:.2f}% de la base). Sobre esos se hace la prueba.')
print('\n-- muestra de 10 codigos con su año extraido, para que lo verifiques a ojo --')
print(con.sql("""SELECT ocid, anio_codigo FROM codigo WHERE anio_codigo IS NOT NULL
                 USING SAMPLE 10 ROWS""").df().to_string(index=False))
print('\n-- muestra de los ambiguos (mas de un año en el codigo) --')
print(con.sql("""SELECT c.ocid, c.candidatos FROM cod c JOIN codigo k USING (ocid)
                 WHERE k.n_candidatos > 1 LIMIT 8""").df().to_string(index=False))

t('(2b) LA PRUEBA: que campo de fecha acierta el año del codigo')
cands = [('ind.fecha_proceso', 'ind', 'fecha_proceso'),
         ('P1.anio_proceso (columna, no fecha)', None, None)] \
        + [(f'P1.{c}', 'p1', c) for c in FP1] \
        + [(f'JSON.{c}', 'jj', c) for c in FJJ]
pz = []
for etq, vista, col in cands:
    if vista is None:
        expr = 'p1.anio_proceso'
    else:
        expr = f'year({vista}."{col}")'
    pz.append(f"""SELECT '{etq}' AS campo,
      count({expr}) AS con_dato,
      count(*) FILTER (WHERE {expr} = k.anio_codigo) AS acierta,
      round(100.0*count(*) FILTER (WHERE {expr} = k.anio_codigo)/nullif(count({expr}),0),2) AS pct_acierto,
      count(*) FILTER (WHERE {expr} = k.anio_codigo + 1) AS un_anio_despues,
      count(*) FILTER (WHERE {expr} > k.anio_codigo + 1) AS dos_o_mas_despues,
      count(*) FILTER (WHERE {expr} < k.anio_codigo) AS antes_del_codigo
      FROM codigo k JOIN p1 USING (ocid) JOIN ind USING (ocid) LEFT JOIN jj USING (ocid)
      WHERE k.anio_codigo IS NOT NULL""")
R = con.sql('\nUNION ALL\n'.join(pz)).df()
print(R.sort_values('pct_acierto', ascending=False).to_string(index=False))
print('\nEl campo que corresponde al procedimiento debe acertar el año del codigo casi siempre.')
print('Las columnas "un año despues" y "dos o mas despues" miden retraso de publicacion.')

t('(2c) A DONDE SE MUEVEN LOS PROCESOS: año del codigo contra los dos candidatos')
print(con.sql("""SELECT k.anio_codigo AS anio_del_codigo, count(*) AS procesos,
     count(*) FILTER (WHERE year(ind.fecha_proceso) = k.anio_codigo) AS acierta_fecha_proceso,
     count(*) FILTER (WHERE p1.anio_proceso = k.anio_codigo)        AS acierta_anio_proceso,
     round(100.0*count(*) FILTER (WHERE year(ind.fecha_proceso) = k.anio_codigo)/count(*),1) AS pct_fecha_proceso,
     round(100.0*count(*) FILTER (WHERE p1.anio_proceso = k.anio_codigo)/count(*),1) AS pct_anio_proceso
   FROM codigo k JOIN p1 USING (ocid) JOIN ind USING (ocid)
   WHERE k.anio_codigo IS NOT NULL GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

print('\n-- los procesos de 2021 segun el codigo: en que año los pone cada campo --')
print(con.sql("""SELECT p1.anio_proceso AS lo_pone_anio_proceso_en,
                        count(*) AS procesos,
                        count(*) FILTER (WHERE year(ind.fecha_proceso) = 2021) AS de_esos_fecha_proceso_dice_2021
   FROM codigo k JOIN p1 USING (ocid) JOIN ind USING (ocid)
   WHERE k.anio_codigo = 2021 GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

# ---------- (3) RETRASO DE PUBLICACION ----------
t('(3) RETRASO ENTRE EL PROCESO Y LA PUBLICACION DEL RELEASE')
print(con.sql("""SELECT year(ind.fecha_proceso) AS anio_del_proceso, count(*) AS procesos,
     round(median(date_diff('day', ind.fecha_proceso, CAST(p1.fecha_release AS DATE))),0) AS mediana_dias,
     round(quantile_cont(date_diff('day', ind.fecha_proceso, CAST(p1.fecha_release AS DATE)),0.95),0) AS p95_dias,
     count(*) FILTER (WHERE date_diff('day', ind.fecha_proceso, CAST(p1.fecha_release AS DATE)) > 365) AS publicado_mas_de_1_anio_despues,
     count(*) FILTER (WHERE CAST(p1.fecha_release AS DATE) < ind.fecha_proceso) AS release_anterior_al_proceso
   FROM ind JOIN p1 USING (ocid) GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

t('(4) COHERENCIA INTERNA DE fecha_proceso')
print(con.sql("""SELECT
     count(*) AS procesos,
     count(*) FILTER (WHERE ind.fecha_proceso < CAST(p1.fecha_inicio_tender AS DATE)) AS antes_del_inicio_del_tender,
     count(*) FILTER (WHERE ind.fecha_proceso > CAST(jj.j_fecha_adjudicacion AS DATE)) AS posterior_a_la_adjudicacion,
     count(*) FILTER (WHERE ind.fecha_proceso > DATE '2026-09-05') AS posterior_al_corte,
     min(ind.fecha_proceso) AS minima, max(ind.fecha_proceso) AS maxima
   FROM ind JOIN p1 USING (ocid) LEFT JOIN jj USING (ocid)""").df().to_string(index=False))

t('(5) SI CAMBIA EL AÑO, CUANTO SE MUEVE EL GRUPO DE REFERENCIA DE PRECIOS')
print(con.sql("""SELECT count(*) AS procesos,
     count(*) FILTER (WHERE year(ind.fecha_proceso) <> p1.anio_proceso) AS cambian_de_anio,
     round(100.0*count(*) FILTER (WHERE year(ind.fecha_proceso) <> p1.anio_proceso)/count(*),2) AS pct,
     count(DISTINCT ind.cpc5) AS grupos_cpc5
   FROM ind JOIN p1 USING (ocid)""").df().to_string(index=False))
print('\nEl grupo de referencia de precios es CPC5 x unidad x familia x AÑO. Esta cifra dice cuantos')
print('procesos quedarian en otro grupo de referencia si cambiamos el año, y por tanto si hay que')
print('recalcular ind_desv_precio_unit_z o no.')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 36 · AÑO CANONICO, NOMBRES DEFINITIVOS Y TABLERO 10/10 ============
import os, shutil, hashlib, json
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC, HIST = 'plan_A/documentacion', 'plan_A/_version_1_historico'
os.makedirs(HIST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 60)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]

# ---------- (1) CONFIRMAR QUE LOS INDICADORES YA USAN EL AÑO CORRECTO ----------
t('(1) LA TABLA DE INDICADORES, USA EL AÑO DE fecha_proceso?')
con.sql("CREATE OR REPLACE VIEW i0 AS SELECT * FROM read_parquet('plan_A/planA_indicadores_v4_3.parquet')")
print(con.sql("""SELECT count(*) AS filas,
   count(*) FILTER (WHERE anio = year(fecha_proceso)) AS coincide,
   count(*) FILTER (WHERE anio <> year(fecha_proceso)) AS no_coincide
 FROM i0""").df().to_string(index=False))
ok_anio = con.sql("SELECT count(*) FROM i0 WHERE anio <> year(fecha_proceso)").fetchone()[0] == 0
print('  -> ' + ('las ventanas y la referencia de precios usan el año correcto: NO hay que recalcular'
                 if ok_anio else 'OJO: el año de los indicadores NO es el de fecha_proceso, hay que recalcular'))

# ---------- (2) RECONSTRUIR LOS SEIS ARCHIVOS DE PROCESOS CON EL AÑO CANONICO ----------
t('(2) AÑO CANONICO Y NOMBRES DEFINITIVOS')
REHACER = [('planA_P1_v3_modelo',                   'planA_procesos_modelo'),
           ('planA_universo_v3',                    'planA_universo'),
           ('planA_P0_catalogo_v3',                 'planA_catalogo'),
           ('planA_excluidos_solo_planeacion_v3',   'planA_excluidos_solo_planeacion'),
           ('planA_excluidos_sin_monto_v3',         'planA_excluidos_sin_monto'),
           ('planA_sin_proveedor_identificable_v3', 'planA_excluidos_sin_proveedor')]
RENOMBRAR = [('planA_indicadores_v4_3',          'planA_indicadores'),
             ('planA_dim_proveedor',             'planA_dim_proveedor'),
             ('planA_dim_entidad',               'planA_dim_entidad'),
             ('planA_validacion_sri_verificada', 'planA_validacion_sri')]

for src, dst in REHACER:
    pin, pout = f'plan_A/{src}.parquet', f'plan_A/{dst}.parquet'
    if not os.path.exists(pin):
        print(f'  {src}: no esta, se salta'); continue
    con.sql(f"CREATE OR REPLACE VIEW s AS SELECT * FROM read_parquet('{pin}')")
    cols = list(con.sql("DESCRIBE SELECT * FROM s").df().column_name)
    con.sql("""CREATE OR REPLACE TABLE k AS
      SELECT ocid, CASE WHEN len(a) = 1 THEN CAST(a[1] AS INTEGER) END AS anio_codigo
      FROM (SELECT ocid, list_distinct(list_filter(regexp_extract_all(ocid, '20[0-2][0-9]'),
                       x -> CAST(x AS INTEGER) BETWEEN 2014 AND 2027)) AS a FROM s)""")
    quedan = [c for c in cols if c != 'anio_proceso']
    sel = ', '.join(f's."{c}"' for c in quedan)
    sel += """, coalesce(year(s.fecha_inicio_tender), k.anio_codigo, year(s.fecha_release)) AS anio_proceso
      , year(s.fecha_release) AS anio_publicacion_release
      , CASE WHEN s.fecha_inicio_tender IS NOT NULL THEN 'inicio del tender'
             WHEN k.anio_codigo IS NOT NULL THEN 'anio del codigo del proceso'
             WHEN s.fecha_release IS NOT NULL THEN 'anio del release'
             ELSE 'sin fuente' END AS anio_proceso_origen
      , date_diff('day', CAST(s.fecha_inicio_tender AS DATE), CAST(s.fecha_release AS DATE))
          AS dias_retraso_publicacion"""
    con.sql(f"COPY (SELECT {sel} FROM s LEFT JOIN k USING (ocid)) TO '{pout}' (FORMAT PARQUET, COMPRESSION ZSTD)")
    n = con.sql(f"SELECT count(*) FROM read_parquet('{pout}')").fetchone()[0]
    o = con.sql(f"""SELECT anio_proceso_origen, count(*) AS filas FROM read_parquet('{pout}')
                    GROUP BY 1 ORDER BY filas DESC""").df()
    print(f'  {src} -> {dst}: {n:,} filas, {len(cols)} -> '
          f'{len(con.sql(f"DESCRIBE SELECT * FROM read_parquet(chr(39)||chr(39))").df()) if False else len(quedan)+4} columnas')
    print('     origen del año: ' + ' | '.join(f'{r.anio_proceso_origen}: {r.filas:,}' for r in o.itertuples()))
    os.replace(pin, f'{HIST}/{src}.parquet')

for src, dst in RENOMBRAR:
    pin, pout = f'plan_A/{src}.parquet', f'plan_A/{dst}.parquet'
    if os.path.exists(pin) and src != dst:
        shutil.move(pin, pout); print(f'  {src} -> {dst} (sin cambios de contenido)')

# ---------- (3) LA SERIE ANUAL CORRECTA ----------
t('(3) SERIE ANUAL CON EL AÑO CANONICO (la que reemplaza al Bloque 19)')
con.sql("CREATE OR REPLACE VIEW pm AS SELECT * FROM read_parquet('plan_A/planA_procesos_modelo.parquet')")
A = con.sql("""SELECT anio_proceso AS anio, count(*) AS procesos,
     count(DISTINCT proveedor_ruc_final) AS proveedores, count(DISTINCT entidad_ruc) AS entidades,
     round(sum(monto_final)/1e6,1) AS millones, round(median(monto_final),0) AS mediana_usd,
     round(median(dias_retraso_publicacion),0) AS retraso_mediano_dias
   FROM pm GROUP BY 1 ORDER BY 1""").df()
print(A.to_string(index=False))
A.to_csv(f'{DOC}/estadisticas/serie_anual_canonica.csv', index=False, encoding='utf-8-sig')
print('\n-- comparacion de las dos series, para el bloque metodologico --')
print(con.sql("""SELECT anio_proceso AS anio,
     count(*) AS con_anio_canonico,
     (SELECT count(*) FROM pm b WHERE b.anio_publicacion_release = a.anio_proceso) AS con_anio_de_release
   FROM pm a GROUP BY 1 ORDER BY 1""").df().to_string(index=False))

# ---------- (4) TABLERO 10/10 ----------
t('(4) TABLERO DE CONTROL DE LA BASE')
F = {'pm': 'planA_procesos_modelo', 'ind': 'planA_indicadores', 'dpr': 'planA_dim_proveedor',
     'den': 'planA_dim_entidad', 'uni': 'planA_universo', 'cat': 'planA_catalogo',
     'ex1': 'planA_excluidos_solo_planeacion', 'ex2': 'planA_excluidos_sin_monto',
     'ex3': 'planA_excluidos_sin_proveedor', 'val': 'planA_validacion_sri'}
for a, f in F.items():
    con.sql(f"CREATE OR REPLACE VIEW {a} AS SELECT * FROM read_parquet('plan_A/{f}.parquet')")
PREF = ('red_', 'cob_', 'com_', 'idx_', 'rep_', 'grupo_')
VARS = [('pct_precio_alto','ind_desv_precio_unit_z','ASC'),('pct_cuota_12m','ind_cuota_prov_en_entidad_12m','ASC'),
        ('pct_recurrencia_par','ind_recurrencia_par_12m','ASC'),('pct_monto_capital','ind_monto_sobre_capital','ASC'),
        ('pct_fraccionamiento','ind_fraccionamiento_monto_30d','ASC'),('pct_proveedor_nuevo','ind_antiguedad_dias','DESC'),
        ('pct_poca_concurrencia','ind_n_oferentes','DESC'),('pct_plazo_corto','ind_plazo_ofertas_dias','DESC'),
        ('pct_firma_rapida','ind_plazo_adj_firma_dias','DESC')]
def uno(q): return con.sql(q).fetchone()[0]
pruebas = []
P = pruebas.append

n_uni = uno("SELECT count(*) FROM uni")
suma = uno("""WITH u AS (SELECT ocid FROM pm UNION ALL SELECT ocid FROM cat UNION ALL SELECT ocid FROM ex1
              UNION ALL SELECT ocid FROM ex2 UNION ALL SELECT ocid FROM ex3) SELECT count(*) FROM u""")
dist = uno("""WITH u AS (SELECT ocid FROM pm UNION ALL SELECT ocid FROM cat UNION ALL SELECT ocid FROM ex1
              UNION ALL SELECT ocid FROM ex2 UNION ALL SELECT ocid FROM ex3) SELECT count(DISTINCT ocid) FROM u""")
huerf = uno("""WITH u AS (SELECT ocid FROM pm UNION ALL SELECT ocid FROM cat UNION ALL SELECT ocid FROM ex1
               UNION ALL SELECT ocid FROM ex2 UNION ALL SELECT ocid FROM ex3)
               SELECT count(*) FROM uni ANTI JOIN u USING (ocid)""")
P(('1. Particion exacta del universo', suma == n_uni and dist == suma and huerf == 0,
   f'{suma:,} = {n_uni:,} | distintos {dist:,} | huerfanos {huerf}'))

malas = []
for a, f in F.items():
    cc = list(con.sql(f"DESCRIBE SELECT * FROM {a}").df().column_name)
    llave = 'ocid' if 'ocid' in cc else 'ruc'
    nf, nd, nn = con.sql(f'SELECT count(*), count(DISTINCT "{llave}"), count(*)-count("{llave}") FROM {a}').fetchone()
    if not (nf == nd and nn == 0): malas.append(f)
P(('2. Llave unica sin nulos en los 10 archivos', not malas, 'fallan: ' + str(malas) if malas else 'las 10 correctas'))

a1 = uno("SELECT count(*) FROM ind ANTI JOIN pm USING (ocid)")
a2 = uno("SELECT count(*) FROM pm ANTI JOIN ind USING (ocid)")
a3 = uno("SELECT count(*) FROM pm LEFT JOIN ind USING (ocid)")
a4 = uno("""SELECT count(*) FROM (SELECT DISTINCT proveedor_ruc_final AS k FROM pm
            WHERE proveedor_ruc_final IS NOT NULL) s ANTI JOIN dpr ON s.k = dpr.ruc""")
a5 = uno("""SELECT count(*) FROM (SELECT DISTINCT entidad_ruc AS k FROM pm
            WHERE entidad_ruc IS NOT NULL) s ANTI JOIN den ON s.k = den.ruc""")
P(('3. Integridad referencial', a1 == 0 and a2 == 0 and a3 == uno("SELECT count(*) FROM pm") and a4 == 0 and a5 == 0,
   f'union 1 a 1 ({a3:,}) | proveedores sin dimension {a4} | entidades sin dimension {a5}'))

mm_u = uno("SELECT round(sum(monto_final)/1e6,1) FROM uni")
mm_s = uno("""SELECT round((coalesce((SELECT sum(monto_final) FROM pm),0)
   + coalesce((SELECT sum(monto_final) FROM cat),0) + coalesce((SELECT sum(monto_final) FROM ex1),0)
   + coalesce((SELECT sum(monto_final) FROM ex2),0) + coalesce((SELECT sum(monto_final) FROM ex3),0))/1e6,1)""")
mal_monto = uno("SELECT count(*) FROM pm WHERE monto_final IS NULL OR monto_final <= 0 OR abs(monto_final - monto_suma_adjudicaciones) >= 0.01")
P(('4. Monto canonico cuadrado', abs(mm_u - mm_s) < 0.2 and mal_monto == 0,
   f'universo {mm_u} MM = partes {mm_s} MM | filas con monto incoherente {mal_monto}'))

proh = []
for a, f in F.items():
    cc = list(con.sql(f"DESCRIBE SELECT * FROM {a}").df().column_name)
    if [c for c in cc if c.startswith(PREF)]: proh.append(f)
P(('5. Cero columnas de red en cualquier archivo', not proh, 'con prohibidas: ' + str(proh) if proh else 'los 10 limpios'))

mal_r = []
for pct, src, d in VARS:
    r = con.sql(f"""SELECT count("{src}"), count("{pct}"), min("{pct}"), max("{pct}"), corr("{src}","{pct}")
                    FROM ind""").fetchone()
    bien = r[0] == r[1] and r[2] >= 0 and r[3] <= 1 and ((d == 'ASC' and r[4] > 0) or (d == 'DESC' and r[4] < 0))
    if not bien: mal_r.append(pct)
P(('6. Los 9 rangos: nulos, recorrido y sentido', not mal_r, 'fallan: ' + str(mal_r) if mal_r else '9 de 9 correctos'))

f1 = uno("SELECT count(*) FROM ind WHERE ind_proveedor_debutante AND ind_procesos_proveedor_12m > 0")
f2 = uno("SELECT count(*) FROM ind WHERE ind_primera_vez_con_la_entidad AND ind_recurrencia_par_12m > 0")
P(('7. Punto en el tiempo sin fugas', f1 == 0 and f2 == 0, f'debutantes con historia {f1} | primeras veces con historia {f2}'))

d1 = uno("SELECT count(*) FROM ind WHERE anio <> year(fecha_proceso)")
d2 = uno("SELECT count(*) FROM pm JOIN ind USING (ocid) WHERE pm.anio_proceso <> ind.anio")
P(('8. Año canonico coherente entre las dos tablas', d1 == 0 and d2 == 0,
   f'indicadores con año distinto a su fecha {d1} | desacuerdo entre tablas {d2}'))

g1 = uno("SELECT coalesce(max(ind_duracion_contrato_dias),0) FROM ind")
g2 = uno("SELECT coalesce(max(ind_contrato_sobre_adjudicado),0) FROM ind")
g3 = uno("SELECT count(*) FROM ind WHERE ind_cuota_prov_en_entidad_12m < 0 OR ind_cuota_prov_en_entidad_12m > 1.0000001")
P(('9. Depuracion de valores imposibles', g1 <= 7300 and g2 <= 10.0001 and g3 == 0,
   f'duracion maxima {g1} dias | ratio maximo {g2} | cuota fuera de rango {g3}'))

E = json.load(open(f'{DOC}/especificacion_indicadores.json', encoding='utf-8'))
n_hist = len([x for x in os.listdir(HIST) if x.endswith('.parquet')])
P(('10. Trazabilidad y preregistro', len(E.get('enmiendas', [])) >= 6 and n_hist >= 8,
   f'{len(E.get("enmiendas", []))} enmiendas firmadas | {n_hist} archivos en el historico'))

T = pd.DataFrame([{'control': c, 'resultado': 'PASA' if ok else 'FALLA', 'detalle': d} for c, ok, d in pruebas])
print(T.to_string(index=False))
print('\n' + ('=' * 95))
print(f'  {sum(1 for _, ok, _ in pruebas if ok)} de {len(pruebas)} controles PASAN')
print('=' * 95)
T.to_csv(f'{DOC}/tablero_de_control.csv', index=False, encoding='utf-8-sig')

t('(5) MANIFIESTO DEFINITIVO')
M = []
for f in sorted(os.listdir('plan_A')):
    if f.endswith('.parquet'):
        p = f'plan_A/{f}'
        M.append({'archivo': f, 'filas': uno(f"SELECT count(*) FROM read_parquet('{p}')"),
                  'columnas': len(con.sql(f"DESCRIBE SELECT * FROM read_parquet('{p}')").df()),
                  'MB': round(os.path.getsize(p)/1e6, 1), 'sha256_16': sha16(p)})
MAN = pd.DataFrame(M); MAN.to_csv(f'{DOC}/manifiesto.csv', index=False, encoding='utf-8-sig')
print(MAN.to_string(index=False)); print(f'\n{len(MAN)} archivos | {round(MAN.MB.sum(),1)} MB')
E.setdefault('enmiendas', []).append({
    'version': 'v5', 'fecha': pd.Timestamp.today().strftime('%Y-%m-%d'),
    'motivo': 'fijar el año canonico del proceso y estabilizar los nombres de archivo',
    'cambios': [
        'anio_proceso pasa a ser coalesce(year(fecha_inicio_tender), anio del codigo del proceso, '
        'year(fecha_release)), con la columna anio_proceso_origen que declara de donde sale cada valor',
        'el anterior anio_proceso, que era el año de publicacion del release, se conserva como '
        'anio_publicacion_release: acierta el año del codigo del proceso solo en el 77,49% de los '
        'casos frente al 99,36% de la fecha de inicio del tender',
        'se agrega dias_retraso_publicacion = fecha_release - fecha_inicio_tender',
        'la tabla de indicadores NO cambia: su columna anio ya era el año de fecha_proceso, que es '
        'la fecha de inicio del tender, de modo que las ventanas de 365 dias y la referencia de '
        'precios siempre usaron el año correcto',
        'nombres de archivo definitivos sin sufijo de version; la version vive en el manifiesto y '
        'en esta especificacion']})
json.dump(E, open(f'{DOC}/especificacion_indicadores.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('\nenmienda v5 registrada. Pega aqui toda la salida.')

In [ ]:
# ============ CELDA 37 · DOCUMENTACION DEFINITIVA, PAQUETE Y TABLERO ============
# Perfila los 10 archivos con sus nombres definitivos, regenera README y diccionario desde los
# datos, arma el paquete de envio y vuelve a correr el tablero completo. Unos 6 a 10 minutos.
import os, shutil, hashlib, json
import pandas as pd, duckdb

os.chdir(r'C:\SERCOP_FINAL')
DOC = 'plan_A/documentacion'; EST = f'{DOC}/estadisticas'
VIEJO = f'{DOC}/_perfiles_de_versiones_anteriores'
os.makedirs(VIEJO, exist_ok=True); os.makedirs(EST, exist_ok=True)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 80)
con = duckdb.connect(); con.sql("SET preserve_insertion_order=false")
def t(x): print('\n' + '=' * 95); print(x); print('=' * 95)
def sha16(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 20), b''): h.update(b)
    return h.hexdigest()[:16]
def anidado(ty): return any(s in ty.upper() for s in ('[', 'STRUCT', 'MAP', 'UNION'))
def es_texto(ty): return ty.upper() == 'VARCHAR'

ARCH = [('planA_procesos_modelo',           'proceso modelable',         'A nucleo'),
        ('planA_indicadores',               'proceso modelable',         'A nucleo'),
        ('planA_dim_proveedor',             'proveedor (RUC)',           'A nucleo'),
        ('planA_dim_entidad',               'entidad (RUC)',             'A nucleo'),
        ('planA_universo',                  'release OCDS',              'B auditoria'),
        ('planA_catalogo',                  'proceso de catalogo',       'B descriptivo'),
        ('planA_excluidos_solo_planeacion', 'proceso excluido',          'C evidencia'),
        ('planA_excluidos_sin_monto',       'proceso excluido',          'C evidencia'),
        ('planA_excluidos_sin_proveedor',   'proceso excluido',          'C evidencia'),
        ('planA_validacion_sri',            'proveedor marcado por SRI', 'C validacion')]
RUTA = {n: f'plan_A/{n}.parquet' for n, _, _ in ARCH}
ARCH = [x for x in ARCH if os.path.exists(RUTA[x[0]])]

# ---------- (0) ARCHIVAR PERFILES DE NOMBRES VIEJOS ----------
t('(0) ARCHIVAR PERFILES DE LOS NOMBRES ANTERIORES')
vivos = {f'perfil_nulos_{n}.csv' for n, _, _ in ARCH}
for f in sorted(os.listdir(DOC)):
    if os.path.isfile(f'{DOC}/{f}') and f.startswith('perfil_nulos_') and f not in vivos:
        shutil.move(f'{DOC}/{f}', f'{VIEJO}/{f}'); print(f'  archivado: {f}')

# ---------- (1) INVENTARIO ----------
t('(1) INVENTARIO DEFINITIVO')
inv, COLS = [], {}
for n, grano, caja in ARCH:
    d = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{RUTA[n]}')").df()
    esq = list(zip(d.column_name.tolist(), d.column_type.tolist())); COLS[n] = esq
    cols = [c for c, _ in esq]
    k = 'ocid' if 'ocid' in cols else 'ruc'
    filas, nd, nn = con.sql(f'SELECT count(*), count(DISTINCT "{k}"), count(*)-count("{k}") FROM read_parquet(\'{RUTA[n]}\')').fetchone()
    inv.append({'archivo': n, 'caja': caja, 'grano': grano, 'filas': filas, 'columnas': len(cols),
                'MB': round(os.path.getsize(RUTA[n])/1e6, 1), 'llave': k,
                'llave_unica': bool(nd == filas and nn == 0), 'sha256_16': sha16(RUTA[n])})
INV = pd.DataFrame(inv); print(INV.to_string(index=False))
INV.to_csv(f'{DOC}/perfil_inventario.csv', index=False, encoding='utf-8-sig')

# ---------- (2) COMPLETITUD Y DICCIONARIO ----------
t('(2) COMPLETITUD POR ARCHIVO')
def perfil(n):
    esq, filas = COLS[n], int(INV.loc[INV.archivo == n, 'filas'].iloc[0])
    out = []
    for i in range(0, len(esq), 35):
        tr = esq[i:i+35]; sel = []
        for c, ty in tr:
            sel.append(f'count("{c}") AS "nn_{i}_{c}"')
            if es_texto(ty): sel.append(f"""count(*) FILTER (WHERE trim("{c}")='') AS "vc_{i}_{c}" """)
            if not anidado(ty): sel.append(f'approx_count_distinct("{c}") AS "nd_{i}_{c}"')
        r = con.sql(f"SELECT {', '.join(sel)} FROM read_parquet('{RUTA[n]}')").df().iloc[0]
        for c, ty in tr:
            nn = int(r[f'nn_{i}_{c}'])
            vc = int(r[f'vc_{i}_{c}']) if f'vc_{i}_{c}' in r.index else 0
            nd = int(r[f'nd_{i}_{c}']) if f'nd_{i}_{c}' in r.index else -1
            u = nn - vc
            ver = ('INSERVIBLE' if u == 0 else 'CONSTANTE' if nd == 1 else
                   'casi vacia' if u/filas < 0.05 else 'mitad vacia' if u/filas < 0.5 else
                   'completa' if u == filas else 'con faltantes')
            out.append({'archivo': n, 'columna': c, 'tipo': ty, 'con_dato': nn, 'vacio_texto': vc,
                        'pct_con_dato': round(100*u/filas, 2),
                        'distintos_aprox': None if nd < 0 else nd, 'veredicto': ver})
    return pd.DataFrame(out)

DIC, RES = [], {}
for n, _, _ in ARCH:
    P = perfil(n); RES[n] = P; DIC.append(P)
    P.drop(columns=['archivo']).to_csv(f'{DOC}/perfil_nulos_{n}.csv', index=False, encoding='utf-8-sig')
    v = P.veredicto
    print(f'{n:34s} {len(P):>4} col | completas {int((v=="completa").sum()):>3} | '
          f'faltantes {int((v=="con faltantes").sum()):>3} | mitad {int((v=="mitad vacia").sum()):>3} | '
          f'casi vacias {int((v=="casi vacia").sum()):>3} | constantes {int((v=="CONSTANTE").sum()):>3} | '
          f'inservibles {int((v=="INSERVIBLE").sum()):>3}')
DICC = pd.concat(DIC, ignore_index=True)
DICC.to_csv(f'{DOC}/diccionario_columnas.csv', index=False, encoding='utf-8-sig')
print(f'\ndiccionario_columnas.csv: {len(DICC):,} filas')

print('\n-- las columnas nuevas del año canonico --')
print(con.sql("""SELECT anio_proceso_origen, count(*) AS procesos,
     round(median(dias_retraso_publicacion),0) AS retraso_mediano
   FROM read_parquet('plan_A/planA_procesos_modelo.parquet')
   GROUP BY 1 ORDER BY procesos DESC""").df().to_string(index=False))

# ---------- (3) README E INVENTARIO ----------
t('(3) README E INVENTARIO REGENERADOS')
DEFS = json.load(open(f'{DOC}/especificacion_indicadores.json', encoding='utf-8'))
tot_mm = con.sql("SELECT round(sum(monto_final)/1e6,1) FROM read_parquet('plan_A/planA_universo.parquet')").fetchone()[0]
p1_mm = con.sql("SELECT round(sum(monto_final)/1e6,1) FROM read_parquet('plan_A/planA_procesos_modelo.parquet')").fetchone()[0]
L = []; A = L.append
A('# Base Plan A - Deteccion de anomalias en compras publicas del Ecuador (SERCOP, OCDS)')
A('')
A(f'Generado desde los datos el {pd.Timestamp.today().strftime("%Y-%m-%d %H:%M")}. No editar a mano.')
A('')
A('- Corte de descarga de los datos: 2026-09-05')
A(f'- Universo: {int(INV.loc[INV.archivo=="planA_universo","filas"].iloc[0]):,} releases OCDS, 2015-2026')
A(f'- Valor contratado del universo: {tot_mm} millones USD')
A(f'- Base de modelado: {int(INV.loc[INV.archivo=="planA_procesos_modelo","filas"].iloc[0]):,} procesos, {p1_mm} millones USD')
A('')
A('## Archivos'); A(''); A('```')
A(INV[['archivo','caja','grano','filas','columnas','MB','llave','llave_unica','sha256_16']].to_string(index=False))
A('```'); A('')
A('## Como se unen'); A('')
A('- `planA_procesos_modelo` <-- ocid --> `planA_indicadores`  (1 a 1, verificado)')
A('- `planA_procesos_modelo.proveedor_ruc_final` --> `planA_dim_proveedor.ruc`')
A('- `planA_procesos_modelo.entidad_ruc` --> `planA_dim_entidad.ruc`')
A('')
A('## Reglas de uso'); A('')
A('1. `monto_final` es la suma de las adjudicaciones del proceso. Nunca usar `monto_tender`.')
A('2. `anio_proceso` es el año de inicio del periodo de ofertas, con retroceso al año del codigo')
A('   del proceso y luego al del release; `anio_proceso_origen` declara de donde sale cada valor.')
A('   `anio_publicacion_release` NO es el año del proceso: acierta el año del codigo solo en el')
A('   77,49% de los casos frente al 99,36% del inicio del tender.')
A('3. LAS SERIES TEMPORALES SE CORTAN EN 2024. Por el retraso de publicacion (mediana 106 a 147')
A('   dias desde 2021, p95 sobre 450) los años 2025 y 2026 estan incompletos y sus totales no son')
A('   comparables. El modelo si puede usar todas las filas.')
A('4. `planA_validacion_sri` no entra como variable en ningun modelo: es validacion externa y vive')
A('   aparte para que no pueda filtrarse a una tabla de features.')
A('5. Prohibido usar columnas de estado administrativo actual (SRI, Superintendencia) para explicar')
A('   procesos pasados: eso es anacronismo y rompe el punto en el tiempo.')
A('6. Las 56 columnas de red (`red_`, `cob_`, `com_`, `rep_`, `grupo_`, `idx_`) fueron eliminadas:')
A('   corresponden al item #48 del catalogo, que es de otro grupo.')
A('7. Los nueve `pct_` son rangos percentiles calculados SOLO sobre filas con dato; la ausencia la')
A('   carga la indicadora `falta_*`. `pct_poca_concurrencia` satura en 0.8315 por los empates de')
A('   los procesos con un solo oferente.')
A('8. `duracion_dias_imputada` es solo para estadistica descriptiva (19,0% de valores sinteticos) y')
A('   lleva su bandera `duracion_imputada`. Para la tesis usar `ind_duracion_contrato_dias`')
A('   declarando su cobertura.')
A('9. `es_multilote` y `n_awards` son variables de estratificacion, no de anomalia.')
A('10. Toda cifra con cola larga se reporta con mediana, nunca con promedio.')
A('')
A('## Definiciones verificadas por reproduccion'); A('')
for k, v in DEFS.get('definiciones_verificadas', {}).items():
    A(f'- `{k}`: {v}')
A('')
A('- Grupo comparable: familia x periodo normativo, con descenso a familia y luego a global cuando')
A('  el grupo tiene menos de 30 casos con dato.')
A('- Referencia de precio: CPC5 x unidad x familia x año, con retroceso a CPC5, CPC3 y familia,')
A('  minimo 30 procesos.')
A('- Ventana de historial: 365 dias anteriores, sin incluir el dia del proceso.')
A('- Ventana de fraccionamiento: 30 dias anteriores, mismo par entidad-proveedor y mismo CPC5.')
A('')
A('## Completitud por archivo'); A('')
for n, _, _ in ARCH:
    v = RES[n].veredicto
    A(f'- **{n}**: {len(RES[n])} columnas | {int((v=="completa").sum())} completas | '
      f'{int((v=="con faltantes").sum())} con faltantes | {int((v=="CONSTANTE").sum())} constantes | '
      f'{int((v=="INSERVIBLE").sum())} sin un solo dato')
A('')
A('El detalle columna por columna esta en `diccionario_columnas.csv` y en los')
A('`perfil_nulos_<archivo>.csv`. Las enmiendas de version, con fecha, motivo y huella, en')
A('`especificacion_indicadores.json`. El tablero de control, en `tablero_de_control.csv`. Las')
A('versiones anteriores de cada archivo se conservan en `_version_1_historico\\`: no se borra nada.')
open(f'{DOC}/README_base_plan_A.md', 'w', encoding='utf-8').write('\n'.join(L))
print(f'  README: {len(L)} lineas')
I2 = ['# Inventario de entregables - base Plan A', '',
      f'Generado desde los datos el {pd.Timestamp.today().strftime("%Y-%m-%d %H:%M")}.',
      f'{len(INV)} archivos, {round(INV.MB.sum(),1)} MB.', '', '```', INV.to_string(index=False), '```', '',
      '## Que enviar', '',
      '- **Caja A (nucleo de modelado)**: procesos_modelo, indicadores, dim_proveedor, dim_entidad.',
      '- **Caja B (auditoria)**: universo y catalogo. No hacen falta para modelar.',
      '- **Caja C (evidencia)**: los tres archivos de exclusion y el conjunto de validacion del SRI.',
      '', 'Las reglas de uso y las definiciones verificadas estan en `README_base_plan_A.md`.']
open(f'{DOC}/INVENTARIO_ENTREGABLES.md', 'w', encoding='utf-8').write('\n'.join(I2))
print(f'  INVENTARIO: {len(I2)} lineas')

# ---------- (4) PAQUETE ----------
t('(4) PAQUETE DE ENVIO')
ENV = 'plan_A/_envio_companeros'
if os.path.exists(ENV): shutil.rmtree(ENV)
os.makedirs(ENV)
for f in ['planA_procesos_modelo.parquet', 'planA_indicadores.parquet', 'planA_dim_proveedor.parquet',
          'planA_dim_entidad.parquet', 'planA_validacion_sri.parquet']:
    shutil.copy2(f'plan_A/{f}', f'{ENV}/{f}')
shutil.copytree(DOC, f'{ENV}/documentacion',
                ignore=shutil.ignore_patterns('_perfiles_de_versiones_anteriores'))
tam = sum(os.path.getsize(os.path.join(r, x)) for r, _, fs in os.walk(ENV) for x in fs)
print(f'  {ENV}  |  {round(tam/1e6,1)} MB')
for r, _, fs in os.walk(ENV):
    for x in sorted(fs): print('   ', os.path.relpath(os.path.join(r, x), ENV))

# ---------- (5) TABLERO FINAL ----------
t('(5) TABLERO DE CONTROL — CORRIDA FINAL')
F = {'pm':'planA_procesos_modelo','ind':'planA_indicadores','dpr':'planA_dim_proveedor',
     'den':'planA_dim_entidad','uni':'planA_universo','cat':'planA_catalogo',
     'ex1':'planA_excluidos_solo_planeacion','ex2':'planA_excluidos_sin_monto',
     'ex3':'planA_excluidos_sin_proveedor','val':'planA_validacion_sri'}
for a, f in F.items():
    con.sql(f"CREATE OR REPLACE VIEW {a} AS SELECT * FROM read_parquet('plan_A/{f}.parquet')")
PREF = ('red_','cob_','com_','idx_','rep_','grupo_')
VARS = [('pct_precio_alto','ind_desv_precio_unit_z','ASC'),('pct_cuota_12m','ind_cuota_prov_en_entidad_12m','ASC'),
        ('pct_recurrencia_par','ind_recurrencia_par_12m','ASC'),('pct_monto_capital','ind_monto_sobre_capital','ASC'),
        ('pct_fraccionamiento','ind_fraccionamiento_monto_30d','ASC'),('pct_proveedor_nuevo','ind_antiguedad_dias','DESC'),
        ('pct_poca_concurrencia','ind_n_oferentes','DESC'),('pct_plazo_corto','ind_plazo_ofertas_dias','DESC'),
        ('pct_firma_rapida','ind_plazo_adj_firma_dias','DESC')]
U = "WITH u AS (SELECT ocid FROM pm UNION ALL SELECT ocid FROM cat UNION ALL SELECT ocid FROM ex1 UNION ALL SELECT ocid FROM ex2 UNION ALL SELECT ocid FROM ex3)"
def uno(q): return con.sql(q).fetchone()[0]
pr = []
n_uni = uno("SELECT count(*) FROM uni")
s1 = uno(f"{U} SELECT count(*) FROM u"); s2 = uno(f"{U} SELECT count(DISTINCT ocid) FROM u")
s3 = uno(f"{U} SELECT count(*) FROM uni ANTI JOIN u USING (ocid)")
pr.append(('1. Particion exacta del universo', s1 == n_uni and s2 == s1 and s3 == 0, f'{s1:,} = {n_uni:,} | huerfanos {s3}'))
mal = [f for a, f in F.items() if not INV.loc[INV.archivo == f, 'llave_unica'].iloc[0]]
pr.append(('2. Llave unica sin nulos en los 10', not mal, 'fallan: ' + str(mal) if mal else 'las 10 correctas'))
a1 = uno("SELECT count(*) FROM ind ANTI JOIN pm USING (ocid)"); a2 = uno("SELECT count(*) FROM pm ANTI JOIN ind USING (ocid)")
a3 = uno("SELECT count(*) FROM pm LEFT JOIN ind USING (ocid)"); n_pm = uno("SELECT count(*) FROM pm")
a4 = uno("SELECT count(*) FROM (SELECT DISTINCT proveedor_ruc_final AS k FROM pm WHERE proveedor_ruc_final IS NOT NULL) s ANTI JOIN dpr ON s.k = dpr.ruc")
a5 = uno("SELECT count(*) FROM (SELECT DISTINCT entidad_ruc AS k FROM pm WHERE entidad_ruc IS NOT NULL) s ANTI JOIN den ON s.k = den.ruc")
pr.append(('3. Integridad referencial', a1 == 0 and a2 == 0 and a3 == n_pm and a4 == 0 and a5 == 0, f'union 1 a 1 ({a3:,}) | sin dimension {a4}/{a5}'))
mm_u = uno("SELECT round(sum(monto_final)/1e6,1) FROM uni")
mm_s = uno("SELECT round((coalesce((SELECT sum(monto_final) FROM pm),0)+coalesce((SELECT sum(monto_final) FROM cat),0)+coalesce((SELECT sum(monto_final) FROM ex1),0)+coalesce((SELECT sum(monto_final) FROM ex2),0)+coalesce((SELECT sum(monto_final) FROM ex3),0))/1e6,1)")
mz = uno("SELECT count(*) FROM pm WHERE monto_final IS NULL OR monto_final <= 0 OR abs(monto_final - monto_suma_adjudicaciones) >= 0.01")
pr.append(('4. Monto canonico cuadrado', abs(mm_u - mm_s) < 0.2 and mz == 0, f'{mm_u} MM = {mm_s} MM | incoherentes {mz}'))
proh = [f for n, _, _ in ARCH for f in [n] if [c for c, _ in COLS[n] if c.startswith(PREF)]]
pr.append(('5. Cero columnas de red', not proh, 'con prohibidas: ' + str(proh) if proh else 'los 10 limpios'))
malr = []
for pct, src, d in VARS:
    r = con.sql(f'SELECT count("{src}"), count("{pct}"), min("{pct}"), max("{pct}"), corr("{src}","{pct}") FROM ind').fetchone()
    if not (r[0] == r[1] and r[2] >= 0 and r[3] <= 1 and ((d == 'ASC' and r[4] > 0) or (d == 'DESC' and r[4] < 0))): malr.append(pct)
pr.append(('6. Los 9 rangos: nulos, recorrido, sentido', not malr, 'fallan: ' + str(malr) if malr else '9 de 9'))
f1 = uno("SELECT count(*) FROM ind WHERE ind_proveedor_debutante AND ind_procesos_proveedor_12m > 0")
f2 = uno("SELECT count(*) FROM ind WHERE ind_primera_vez_con_la_entidad AND ind_recurrencia_par_12m > 0")
pr.append(('7. Punto en el tiempo sin fugas', f1 == 0 and f2 == 0, f'debutantes {f1} | primeras veces {f2}'))
d1 = uno("SELECT count(*) FROM ind WHERE anio <> year(fecha_proceso)")
d2 = uno("SELECT count(*) FROM pm JOIN ind USING (ocid) WHERE pm.anio_proceso <> ind.anio")
pr.append(('8. Año canonico coherente', d1 == 0 and d2 == 0, f'{d1} | {d2}'))
g1 = uno("SELECT coalesce(max(ind_duracion_contrato_dias),0) FROM ind")
g2 = uno("SELECT coalesce(max(ind_contrato_sobre_adjudicado),0) FROM ind")
g3 = uno("SELECT count(*) FROM ind WHERE ind_cuota_prov_en_entidad_12m < 0 OR ind_cuota_prov_en_entidad_12m > 1.0000001")
pr.append(('9. Depuracion de valores imposibles', g1 <= 7300 and g2 <= 10.0001 and g3 == 0, f'duracion {g1} | ratio {g2} | cuota {g3}'))
n_enm = len(DEFS.get('enmiendas', []))
n_hist = len([x for x in os.listdir('plan_A/_version_1_historico') if x.endswith('.parquet')])
n_doc = len([x for x in os.listdir(DOC) if os.path.isfile(f'{DOC}/{x}')])
pr.append(('10. Trazabilidad y preregistro', n_enm >= 6 and n_hist >= 8 and n_doc >= 10,
           f'{n_enm} enmiendas | {n_hist} en historico | {n_doc} archivos de documentacion'))
T = pd.DataFrame([{'control': c, 'resultado': 'PASA' if ok else 'FALLA', 'detalle': d} for c, ok, d in pr])
print(T.to_string(index=False))
ok_todo = all(ok for _, ok, _ in pr)
print('\n' + '=' * 95)
print(f'  {sum(1 for _, ok, _ in pr if ok)} de {len(pr)} controles PASAN'
      + ('   ***  BASE CERRADA  ***' if ok_todo else '   <-- revisar los que fallan'))
print('=' * 95)
T.to_csv(f'{DOC}/tablero_de_control.csv', index=False, encoding='utf-8-sig')
print('\nPega aqui toda la salida.')

In [ ]:
# ============ CELDA 38 · ultimo detalle: la serie anual vieja dentro del paquete ============
import os, shutil
import pandas as pd, duckdb
os.chdir(r'C:\SERCOP_FINAL')
DOC = 'plan_A/documentacion'
con = duckdb.connect()
A = con.sql("""SELECT anio_proceso AS anio, count(*) AS procesos,
     count(DISTINCT proveedor_ruc_final) AS proveedores, count(DISTINCT entidad_ruc) AS entidades,
     round(sum(monto_final)/1e6,1) AS millones, round(median(monto_final),0) AS mediana_usd,
     round(median(dias_retraso_publicacion),0) AS retraso_mediano_dias,
     CASE WHEN anio_proceso >= 2025 THEN 'INCOMPLETO por retraso de publicacion' ELSE 'completo' END AS estado
   FROM read_parquet('plan_A/planA_procesos_modelo.parquet')
   GROUP BY 1 ORDER BY 1""").df()
A.to_csv(f'{DOC}/perfil_por_anio.csv', index=False, encoding='utf-8-sig')
print(A.to_string(index=False))
ENV = 'plan_A/_envio_companeros/documentacion'
shutil.copy2(f'{DOC}/perfil_por_anio.csv', f'{ENV}/perfil_por_anio.csv')
print('\nperfil_por_anio.csv regenerado con el año canonico y copiado al paquete.')
print('La carpeta plan_A/_envio_companeros ya se puede comprimir y enviar.')